# olaf — self-contained OneLake security runtime

**OneLake Access Framework — source of truth** (single-sheet config + generate lock-file).
Customer-agnostic: ALL project-specific logic lives in the `onelake_security_config` table and the
runtime parameters — this notebook never changes per project. No `%run`, no separate lib, no test
code here: one self-contained notebook a user/pipeline imports and invokes (all modes via the `mode`
parameter).

The notebook reads top-to-bottom as documented sections: a `parameters` cell first, then the **pure
functions** (constants & parsing → path helpers → control-table schema → catalog & entry resolution →
generate & validation → desired state & DAR payload → diff & merge) — these run anywhere; the CI
extract-and-exec suite execs them directly with plain Python. Then the **module helpers**
(context/target resolution) and the classes in dependency order: `FabricClient` (REST) → `Log`
(`onelake_security_log`) → `Deployment` (one public method per runtime mode; shared
steps are private methods) → `Audit`. Finally the **runtime entrypoint** (`run_mode` /
`run_and_exit`) and the **▶️ Run** dispatch cell.

Cells execute in order, so every name is defined before use — do not reorder the code cells.

Naming rule for maintainers: names must say what they do — internal design codes (C1, B2, …)
appear only inside docstrings/messages as traceability references to docs/architecture.md.

## Contents

Fabric's **Outline** panel (command bar → **Contents**) auto-builds a clickable sidebar from the
headers below and jumps to a section on click — the guaranteed navigation for this single-file
notebook. This list is the in-body mirror.

- **Parameters** — runtime inputs a pipeline injects
- **Constants** — module constants + control-table column definitions (config / mapping / log / member)
- **Exceptions** — the `OLAFError` hierarchy (drives the audit `error_category`)
- **Utility classes** (pure, one per cell)
    - `Parse` — string → structured value parsing; name/glob matching
    - `ScopePath` — dotted catalog name ⇄ `/Tables/…` DAR path
    - `Hash` — config-staleness + mapping lock-file fingerprints
    - `TableSchema` — control-table DDL builder + column-type lookup
    - `RLS` — row-level predicate analysis (rules B2 / C7 / C9 / C10)
    - `CLS` — column-level visibility resolution (rule C11)
    - `Member` — member display name → Entra objectId resolution
    - `Catalog` — Spark-catalog snapshot + include/exclude resolution
    - `Target` — attach / tenant / runtime resolution
    - `DAR` — desired-state build + diff / merge of data-access roles
    - `Generate` — short config → validated role × scope grants (rules A / B / C)
- **Classes** (stateful)
    - `FabricClient` — Fabric REST for data-access roles
    - `Log` — writer/reader for `onelake_security_log`
    - `Deployment` — the pipeline façade (one public method per mode)
    - `Audit` — read-only generation-tracing / audit queries
- **Dispatch** — `run_mode` / `run_and_exit` (the dispatch engine)
- **OLAF** — the interactive facade (every method first-level: `OLAF.generate()` · `OLAF.plan()` · `OLAF.show()` · `OLAF.setup()` · …)
- **▶️ Run** — the mode-dispatch cell (pipeline entrypoint)


## Parameters

Runtime inputs — a Fabric pipeline (or `notebookutils.notebook.run`) overrides these; the cell is
tagged `parameters` so papermill/Fabric injects a values cell right after it.

In [ ]:
# PARAMETERS — a Fabric pipeline overrides these (cell is tagged "parameters").
# Legend: [required] set before run · [auto] resolved at runtime when left "" · [default] safe as-is.

# fmt: off
# -- what to run ---------------------------------------------------------------
mode                               = ""     # [required·choice] setup | generate | validate | plan | apply | rollback | show | trace
#   default "" = no dispatch: %run loads this notebook as a library (the OLAF facade + classes) without
#   running anything; a Fabric pipeline / notebook.run sets `mode` to dispatch. See the Run guard below.
rebuild                            = False  # [default·bool] generate: rebuild the mapping even if the config is unchanged · setup: DROP + recreate any control table whose column types drifted (DATA IS LOST)
#   a Fabric pipeline passes this as a STRING (Base parameters default to the String type): "true"/"false", "1"/"0", "yes"/"no" — case-insensitive, surrounding blanks ignored, "" = left blank = False; a null (None) is also an unset parameter = False.
#   an Int-typed parameter is accepted too, but ONLY 0 / 1 — every other int is ambiguous.
#   anything else is REJECTED with a blocked envelope naming the value — never coerced.
keep_unmanaged                     = False  # [default·bool] apply: False (default) submits the config-derived payload; absent prior-live roles are omission candidates, not deletion proof; True = incremental upsert that carries them
if_match                           = True   # [default·bool] apply/rollback: send the bulk PUT conditionally (If-Match with the ETag from this run's own live read). False = unconditional — the escape hatch if the live service's ETag semantics misbehave (a 412 on every apply)
#   a Fabric pipeline passes this as a STRING (Base parameters default to the String type): "true"/"false", "1"/"0", "yes"/"no" — case-insensitive, surrounding blanks ignored, "" = left blank = False; a null (None) is also an unset parameter = False.
#   an Int-typed parameter is accepted too, but ONLY 0 / 1 — every other int is ambiguous.
#   anything else is REJECTED with a blocked envelope naming the value — never coerced, because bool("false") is True and would silently select the config-payload path.
control_data_isolation_attestation = ""     # [required for sensitive writes] per-run external access-review evidence reference; not a secret and not proof of workspace isolation

# -- tenant --------------------------------------------------------------------
tenant_id = ""  # [auto] "" = auto-resolve from the runtime context · Entra tenant GUID stamped into member payloads
#   the deploy target is ALWAYS the notebook's attached lakehouse — there is no workspace/lakehouse to CHOOSE (mode=setup asserts which one that is, below)

# -- setup only ----------------------------------------------------------------
lakehouse_name = ""  # [required for setup] the lakehouse you intend to create the control tables in — an ASSERTION, never a target
#   setup writes through two-part `olaf.…` names, which always resolve against the ATTACHED lakehouse, so this can never SELECT a different one — it only lets setup REFUSE when the notebook is attached to a lakehouse you did not mean, instead of leaving four control tables + an audit row in the wrong workspace.
#   matched case-insensitively; a lakehouse attached from ANOTHER workspace is refused on the ids too, because display names are not unique across workspaces. Off-Fabric (no notebookutils) only the ATTACHMENT checks are skipped — there is no attachment to be wrong about; the requirement to NAME a lakehouse always applies.

# -- control tables (defaults match the olaf schema convention) ----------------
config_table        = "olaf.onelake_security_config"    # [default] short config (authored by humans)
mapping_table       = "olaf.onelake_security_mapping"   # [default] lock-file (written by mode=generate)
log_table           = "olaf.onelake_security_log"       # [default] audit log (written by every mutating mode: setup/generate/plan/apply/rollback)
member_table        = "olaf.onelake_security_member"    # [default] name->objectId resolution cache (mode=generate resolves member names from it ONLY — it must contain every config member; created by setup)
mapping_history_dir = "Files/security/mapping-history"  # [default] folder for the versioned mapping-history CSV export (one file per generation: {mapping_basename}_{ts}_v{mapping_version}_{mapping_hash}.csv)
role_backup_dir     = "Files/security/role-backups"     # [default] folder for the pre-apply role backup (one JSON per apply, never deduped — the break-glass restore point; see docs/runbook.md 3c and 3f)
verbosity           = "info"                            # [default·choice] how much to PRINT — silent | quiet | info | detail | verbose (cumulative; the returned DataFrame is unaffected). blocked/error always prints, even at silent

# -- run labels ----------------------------------------------------------------
env      = ""  # [optional] label written to every log row — "" = none · e.g. dev | qa | prod
batch_id = ""  # [auto] "" = new uuid · pipelines pass their run id here to link plan → apply
#   workspace_name and the attached lakehouse's name are auto-resolved from the runtime context (stamped into mapping/log) — the `lakehouse_name` parameter above is setup's assertion, never a target

# -- show only -----------------------------------------------------------------
by      = "table"  # [required for show·choice] table | role | member — the pivot axis
subject = ""       # [required for show] by=role: role name · by=table: schema.table or /Tables/ path · by=member: objectId / name · globs allowed (e.g. Sales.Order*)

# -- rollback only -------------------------------------------------------------
rollback_to_version = ""  # [rollback] "" = previous config version · N = that exact Delta version (error if absent)
rollback_reason     = ""  # [required for rollback] why — stamped into the audit log note
# fmt: on

## Constants — no Spark, no side effects; the CI extract-and-exec suite runs these directly.

Module constants (member-column maps, rule/limit thresholds, regexes) plus the smallest pure helpers: the config `config_hash` staleness fingerprint and the RLS predicate/column-reference extractors. Everything below is importable and runs anywhere.

In [ ]:
__version__ = "1.1.0"

import re
import json
import hashlib
import fnmatch
import difflib
import uuid
import datetime
import time
import traceback
from dataclasses import dataclass

LIST_SEP = ";"
# A member value is a PATTERN only if it declares itself one. Entra permits `*` and `?` in a
# displayName, so sniffing for those characters cannot tell a glob from a real name -- and guessing
# wrong grants a DIFFERENT principal with no error at all (the finding that added this marker):
# `Sales? Reporting` silently expanded against `Salesx Reporting`. An explicit marker removes the
# ambiguity instead of detecting it. Scope globs need no such marker: a Delta table cannot be named
# `*`, so there is nothing to confuse them with.
GLOB_PREFIX = "glob:"

# The string spellings accepted for a boolean Base parameter, lowercased. Fabric Base parameters
# default to the String type, so a pipeline hands a boolean parameter over as a STRING — and every
# non-empty string is truthy, so bool("false") is True. Values must be PARSED against this map
# (Parse.bool_param), never coerced.
BOOL_PARAM_SPELLINGS = {
    "true": True,
    "false": False,
    "1": True,
    "0": False,
    "yes": True,
    "no": False,
}

# Member types: one include/exclude column pair per Entra principal type. Config values are DISPLAY
# NAMES (group displayName / user UPN-or-mail / SP displayName) — generate resolves them to objectIds.
# (Group / User / ServicePrincipal — the objectType the DAR payload expects.)
MEMBER_KINDS = [
    ("include_group_names", "exclude_group_names", "Group"),
    ("include_user_names", "exclude_user_names", "User"),
    ("include_sp_names", "exclude_sp_names", "ServicePrincipal"),
    ("include_mi_names", "exclude_mi_names", "ManagedIdentity"),
]

# the eight member columns, in canonical order — C1 compares each one across a role's rows
MEMBER_ALL_COLUMNS = [c for inc, exc, _ in MEMBER_KINDS for c in (inc, exc)]

# mapping-table member columns: per type, the resolved effective NAME-list column and its resolved
# objectId-list column (';' lists, name<->id positionally aligned), plus the DAR objectType. Names are
# the human-facing effective set; the ids are what the DAR payload / OneLake security actually consume. Both
# round-trip through onelake_security_mapping into the plan/apply payload.
MAPPING_MEMBER_COLUMNS = [
    ("member_group_names", "member_group_ids", "Group"),
    ("member_user_names", "member_user_ids", "User"),
    ("member_sp_names", "member_sp_ids", "ServicePrincipal"),
    ("member_mi_names", "member_mi_ids", "ManagedIdentity"),
]

_MTYPE_TO_NAME_COL = {mtype: name_col for name_col, _id_col, mtype in MAPPING_MEMBER_COLUMNS}

# valid onelake_security_member.member_type values — generate rejects any other value
MEMBER_TYPES = {mtype for _inc, _exc, mtype in MEMBER_KINDS}

# The DAR Action enum as a case-normalization map: lowercased input -> canonical token.
# A MAP rather than a set so validation and normalization are one lookup — rule B4 rejects
# anything absent from it, and rule B3 (ReadWrite + RLS/CLS forbidden) then compares the
# CANONICAL token, which closes the case-sensitivity bypass a bare `perm == "ReadWrite"`
# comparison had: 'readwrite' + RLS used to sail past the rule and die at the platform.
PERMISSIONS = {"read": "Read", "readwrite": "ReadWrite"}

# The two modes that run the shared desired-vs-live pipeline (_desired_state: read the
# mapping, diff against the live DAR, and — for apply — write it back). The CONTRACT this
# set gates is "needs a tenant id and a live-DAR client for the diff path", NOT "writes
# state": plan writes only log rows, while generate/setup/rollback write control tables
# without being in it. Renamed from MUTATING_MODES during pre-release development — that
# name claimed the latter.
DESIRED_STATE_MODES = {"plan", "apply"}
# The grant table `show` returns, in its canonical order, and which column each axis LEADS
# with. A pivot that buries its own key mid-row makes the reader hunt for the thing they
# searched by — and, across three axes returning the same eleven columns, makes it hard to
# tell at a glance WHICH axis produced the frame in front of you. Every axis returns all
# eleven; only the order differs. first_applied/first_granted_by and last_applied/
# last_granted_by sit adjacent because each actor belongs to ONE end: the reader must never
# have to guess which timestamp a lone `granted_by` was answering for. by=member leads with the objectId — the key the live DAR
# actually holds and the one column that is never null — display name immediately beside it.
GRANT_COLUMNS = [
    "role_name",
    "scope_path",
    "member",
    "member_name",
    "permission",
    "first_applied",
    "first_granted_by",
    "last_applied",
    "last_granted_by",
    "config_version",
    "provenance",
]
GRANT_LEAD_COLUMNS = {
    "table": ["scope_path"],
    "role": ["role_name"],
    "member": ["member", "member_name"],
}

SHOW_AXES = (
    "table",
    "role",
    "member",
)  # show mode pivot axes (by=...); order = error-message order
ROLE_NAME_RE = re.compile(r"^[A-Za-z][A-Za-z0-9]*$")
GUID_RE = re.compile(
    r"^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$"
)
# What an `env` label may be. Shape is a boundary here, not a formatting preference: `env` is an
# operator-supplied string that reaches Spark SQL as a WHERE literal on the log reads
# (Log.find_plan_record, Log.grant_provenance), so a value carrying a quote would close that
# literal and hand the remainder to the engine. Letters, digits, '_' and '-', 1-64 characters:
# every environment label anyone actually writes ("dev", "prod", "uat-2"), and nothing that can
# terminate a string literal or a comment. Validated at the two boundaries that can SET it,
# never sanitised at the point of use — see Parse.env_param.
ENV_RE = re.compile(r"^[A-Za-z0-9_-]{1,64}$")
CONTROL_EVIDENCE_RE = re.compile(r"^[A-Za-z0-9._:/#-]{1,128}$")


# Keywords stripped when pulling column references out of an rls_condition (column-existence check).
RLS_KEYWORDS = {
    "AND",
    "OR",
    "NOT",
    "IN",
    "IS",
    "NULL",
    "LIKE",
    "BETWEEN",
    "TRUE",
    "FALSE",
    "SELECT",
    "FROM",
    "WHERE",
    "AS",
    "EXISTS",
    "ANY",
    "ALL",
    "SOME",
    "CASE",
    "WHEN",
    "THEN",
    "ELSE",
    "END",
    "CAST",
}

# Platform ceilings (OneLake security): fail at generate, not at apply. Warn at ~80%.
MAX_PATHS_PER_ROLE = 500
MAX_MEMBERS_PER_ROLE = 500
MAX_ROLES_PER_ITEM = 250
WARN_FRACTION = 0.8

# role_name over 124 chars hard-fails the SQL analytics-endpoint security sync, with no workaround
# (rule C6). RLS predicate limits: a 1000-char max (rule C7) and the supported operator/keyword
# subset (rule C9). Refs: troubleshoot-onelake-security-for-sql-analytics-endpoints +
# row-level-security #syntax-rules / #supported-operators.
MAX_ROLE_NAME_CHARS = 124
MAX_RLS_CONDITION_CHARS = 1000

# The only operators/keywords OneLake RLS honors in a predicate (plus identifiers, string/number
# literals, parentheses, commas). Anything else — LIKE, BETWEEN, arithmetic, a function call — is
# unsupported syntax that fails at query time (rule C9).
RLS_SUPPORTED_OPERATORS = {
    "=",
    "<>",
    ">",
    ">=",
    "<",
    "<=",
    "IN",
    "NOT",
    "AND",
    "OR",
    "IS",
    "BLANK",
    "NULL",
    "TRUE",
    "FALSE",
}

# Known SQL keywords NOT in the supported subset — derived from RLS_KEYWORDS (the same set the
# column lexer already treats as non-columns) minus the supported alphabetic keywords. C9 flags a
# bareword only when it is one of these (confidently a keyword, not a column). BLANK is supported and
# absent from RLS_KEYWORDS, so subtracting it is harmless.
RLS_UNSUPPORTED_KEYWORDS = RLS_KEYWORDS - {op for op in RLS_SUPPORTED_OPERATORS if op.isalpha()}

# Follow-set that decides OPERAND position: a bareword whose NEXT significant token matches this is
# the LEFT OPERAND of a comparison — a column literally named e.g. `Between` / `End` — not an
# operator. SHARED BY CONSTRUCTION between RLS.referenced_columns_all and RLS.unsupported_tokens:
# the column lexer readmits such a bareword as a column exactly when C9 declines to flag it as an
# operator. Widening one copy alone would restore the T1 validation gap (C9 stops flagging while
# the column lexer still drops the column, so the column-existence check and rule C11 never run), so the
# two directions MUST move together. Only the follow-set is shared — the RLS_SUPPORTED_OPERATORS
# early-`continue` asymmetry between the two functions is deliberate and stays.
RLS_OPERAND_FOLLOW_RE = re.compile(r"<=|>=|<>|!=|[=<>]|(?:IN|IS)\b", re.IGNORECASE)

# The mirror of the follow-set: a comparison operator at the END of the text BEFORE a token, which
# puts that token in VALUE position. RLS_OPERAND_FOLLOW_RE answers "is this bareword a left
# operand"; this answers "is this bareword a right operand". Rule C14 needs the second question,
# and only for TRUE/FALSE, so this deliberately stays a separate small constant rather than
# widening the shared follow-set — nothing about C9 or the column lexer changes.
# Value position: after a comparison operator, after IS / IS NOT, or anywhere inside an IN list
# (the list is all values by definition, so every slot in it counts, not just the first).
RLS_VALUE_PRECEDE_RE = re.compile(
    r"(?:<=|>=|<>|!=|[=<>]|\bIS(?:\s+NOT)?|\bIN\s*\((?:[^()]*,)?\s*)$", re.IGNORECASE
)

# AND/OR connective count above which a predicate is warned as over-complex (rule C10). A tunable
# heuristic, NOT a platform-published hard limit — highly complex roles can fail the security sync,
# so this nudges an author to split the role.
RLS_COMPLEXITY_WARN_CONNECTIVES = 20

In [ ]:
# ---------- control-table definitions (mode=setup creates these; names are parameters) ----------
CONFIG_AUTHOR_COLUMNS = [
    "role_name",
    "lakehouse_name",
    "include_tables",
    "exclude_tables",
    "include_folders",
    "exclude_folders",
    "permission",
    "rls_condition",
    "include_columns",
    "exclude_columns",
    "include_group_names",
    "exclude_group_names",
    "include_user_names",
    "exclude_user_names",
    "include_sp_names",
    "exclude_sp_names",
    "include_mi_names",
    "exclude_mi_names",
    "active",
    "notes",
]

# One row per (role x scope): the mapping/lock-file grain. Members are typed name+id column pairs
# (identical on every row of a role): the *_names carry the effective display-name set, the *_ids the
# preloaded objectIds plan/apply build the payload from — so the right objectType round-trips.
MAPPING_COLUMNS = [
    "role_name",
    "workspace_name",
    "lakehouse_name",
    "workspace_id",
    "lakehouse_id",
    "tenant_id",
    "scope_path",
    "scope_type",
    "permission",
    "rls_condition",
    "visible_columns",
    "member_group_names",
    "member_group_ids",
    "member_user_names",
    "member_user_ids",
    "member_sp_names",
    "member_sp_ids",
    "member_mi_names",
    "member_mi_ids",
]

# workspace_name/lakehouse_name make the lock-file self-describing about its target — carried from
# the config rows; the stamped workspace_id/lakehouse_id are what the target-identity guard in
# Deployment._desired_state re-validates against the runtime target at plan/apply time.
MAPPING_PROVENANCE_COLUMNS = ["generated_at", "config_hash", "config_version", "framework_version"]
LOG_COLUMNS = [
    "batch_id",
    "run_id",
    "run_at",
    "env",
    "mode",
    "workspace_name",
    "lakehouse_name",
    "role_name",
    "scope_path",
    "scope_type",
    "member_name",
    "member_id",
    "member_type",
    "action",
    "status",
    "error_category",
    "message",
    "run_by",
    "run_duration",
    "config_hash",
    "config_version",
    "workspace_id",
    "lakehouse_id",
    "tenant_id",
    "mapping_hash",
    "mapping_version",
    "framework_version",
]

# name->objectId resolution table. mode=generate resolves member names from it ONLY (No-Graph gate);
# logical PK (member_type, lower(member_name)); every column STRING. PRELOADED ENTIRELY from the
# member sheet of onelake_security.xlsx (fill the template first) — every member a config row
# DECLARES (all eight member columns, include and exclude alike) must be
# present with its objectId, else generate blocks. Reloaded whenever the file changes (single
# source, no per-row timestamp — hence no source/resolved_at columns). setup creates it.
# Recovery input for apply: every apply snapshots the LIVE data access roles here BEFORE it
# submits its request. The artifact captures prior request input; it does not prove omission
# deletion or guarantee exact later platform recovery.
#
# This was a CONSTANT, on the argument that mapping_history_dir earns a parameter because operators
# CONSUME its CSV while a role backup is "an artifact nobody reads on the happy path". That argument
# no longer holds: RUNBOOK 3c has operators RESTORE from this directory and 3f has them PRUNE it —
# two operator procedures pointing at a path they could neither move nor even see in their own
# parameters pane. Promoted to the parameter role_backup_dir (the old comment said it was one line
# away). The default is unchanged, so nothing moves for anyone who does not set it.
# The control-table names a run falls back to when the caller names none. ONE map, because
# these four literals were repeated at every entry point and a rename that missed one would
# silently split a deployment across two sets of tables.
DEFAULT_CONTROL_TABLES = {
    "config_table": "olaf.onelake_security_config",
    "mapping_table": "olaf.onelake_security_mapping",
    "log_table": "olaf.onelake_security_log",
    "member_table": "olaf.onelake_security_member",
}

# How much a run prints. Five levels, each ADDING to the one before, so a level is "everything up
# to here": silent (nothing) → quiet (the verdict line) → info (+ the result keys and the returned
# frame's columns) → detail (+ in-flight progress) → verbose (+ the last_result JSON echo).
# `info` is the default because the returned DataFrame already carries the detail — the printout is
# there to say what happened, not to restate the frame.
# A blocked/error verdict prints at EVERY level, silent included: a failed apply that printed
# nothing is the one silence nobody asked for.
VERBOSITY_LEVELS = ("silent", "quiet", "info", "detail", "verbose")

# Destructive utilities that exist on the OLAF facade and are NOT modes. run_mode refuses them by
# name, so no pipeline can reach them by setting `mode` — the guard is the absence of a mode, and
# the named refusal is what makes that absence readable instead of looking like a typo.
INTERACTIVE_ONLY = frozenset({"reset", "cleanup"})


def interactive_only_refusal(mode):
    """The one sentence both refusal sites emit, so they cannot drift apart.

    Two callers say this: run_mode raises it as SystemExit, and the entrypoint folds it into an
    envelope message. They were separate byte-identical literals differing only in where the source
    line wrapped, and NEITHER was pinned by a test -- so a caller arriving through the two paths
    could have started getting two different explanations of the same refusal without anything
    noticing. That matters here more than for a typical duplicated string: this is the guard keeping
    reset() and cleanup() -- the two destructive, irreversible operations -- off the `mode` string a
    scheduled pipeline can set, and the message is how an operator learns the omission is deliberate
    rather than a bug to work around.
    """
    return (
        f"{mode} is interactive-only, by design — call OLAF.{mode}() from a notebook. "
        "It is destructive and irreversible, so it is deliberately not a mode a pipeline can select."
    )


class Say:
    """The output gate. Every print in this notebook goes through Say.out(level, ...), where
    `level` is the LOWEST verbosity at which that line appears — so reading a call site tells you
    when it shows, and "silent" reads as "always"."""

    # run_mode pins the run's resolved value for its duration; outside a run (the interactive
    # utilities, which never enter run_mode) the sticky facade setting is the answer.
    override = None

    @staticmethod
    def level():
        if Say.override is not None:
            return Say.override
        return OLAF._base_params.get("verbosity", PARAM_DEFAULTS["verbosity"])

    @staticmethod
    def at(level):
        """Is `level` reached by the current setting?"""
        return VERBOSITY_LEVELS.index(Say.level()) >= VERBOSITY_LEVELS.index(level)

    @staticmethod
    def out(level, *args, **kw):
        if Say.at(level):
            print(*args, **kw)  # the ONE real print in the notebook — everything else routes here


# Every parameter run_mode resolves, with the value it falls back to when the caller set none.
# THE single source: run_mode reads its own defaults from here, and OLAF.params fills the same
# gaps to answer "what will this run actually use". A second copy of these values is the drift
# this codebase keeps paying for, so there is exactly one — and test_param_defaults_is_complete
# reads run_mode back to prove no parameter was added without landing here.
PARAM_DEFAULTS = {
    "env": "",
    "tenant_id": "",
    "lakehouse_name": "",
    **DEFAULT_CONTROL_TABLES,
    "mapping_history_dir": "Files/security/mapping-history",
    "role_backup_dir": "Files/security/role-backups",
    "verbosity": "info",
    "batch_id": "",
    "by": "table",
    "subject": "",
    "rollback_to_version": "",
    "rollback_reason": "",
    # Concurrency escape hatch: if_match=false sends the bulk PUT unconditionally (no
    # If-Match header). STICKY-able via configure() on purpose — it exists for the case
    # where the live service's ETag semantics misbehave and every conditional apply 412s;
    # a per-call-only switch would have to be repeated on every apply while that lasts.
    "if_match": True,
    "control_data_isolation_attestation": "",
    # Per-call, and configure() refuses both. Listed anyway because a call that passes neither
    # gets exactly these — leaving them out would make OLAF.params a partial answer to the one
    # question it exists to answer. show_params() labels them so nobody reads them as sticky.
    "keep_unmanaged": False,
    "rebuild": False,
}

# The params table's columns. `source` is not decoration: it is the difference between "someone
# chose dev" and "nobody chose anything", which renders identically without it.
PARAMS_COLUMNS = ["parameter", "value", "source"]
MEMBER_CACHE_COLUMNS = [
    "member_type",
    "member_name",
    "member_id",
]

In [ ]:
# ══ Exceptions ═══════════════════════════════════════════════════════════════
class OLAFError(Exception):
    """Base for every framework-raised error. `category` feeds the audit error_category field."""

    category = "unexpected"

    @staticmethod
    def classify(exc):
        """Any exception → the audit error_category vocabulary (http|validation|guard|unexpected)."""
        if isinstance(exc, OLAFError):
            return exc.category
        if isinstance(exc, SystemExit):
            return "guard"  # load-bearing pipeline-fail signal — unchanged
        if isinstance(exc, ValueError):
            return "validation"  # safety net for any stray built-in
        module = type(exc).__module__ or ""
        if "requests" in module or "HTTP" in type(exc).__name__ or hasattr(exc, "response"):
            return "http"
        return "unexpected"


class ValidationError(OLAFError):  # config / rule A·B·C violation
    category = "validation"


class ZeroMatchError(ValidationError):  # a scope entry resolved to nothing
    """An include/exclude entry matched no table or folder.

    Distinct from its siblings because it is the one failure that is a statement about the
    OPERATOR'S CONFIG: a TargetResolutionError says the environment could not be read at all,
    and blaming that on the row's scope would point the reader at the wrong thing. Callers that
    want to explain the consequence of a dead scope catch this, and nothing else.
    """


class TargetResolutionError(
    ValidationError
):  # lakehouse target guard (name kept — minimises churn)
    """The attached lakehouse can't be resolved (not found / ambiguous)."""


class TargetNotFound(TargetResolutionError):
    pass


class TargetAmbiguous(TargetResolutionError):
    pass


class DARHTTPError(OLAFError):  # Fabric DAR REST call failed
    category = "http"


class DARConflictError(DARHTTPError):  # 412 — live roles changed since this run's read
    """The bulk PUT's If-Match precondition failed: live OneLake security changed between
    this run's list_roles read and its PUT. On a FIRST-attempt 412 the service refused the
    write before anything landed (zero blast radius, like a dryRun rejection). A 412 on a
    RETRIED attempt (`ambiguous` True) proves no such thing: the earlier attempt may itself
    have COMMITTED — rotating the very ETag the re-send then failed against — before its
    response was lost to the transient status that triggered the retry, so what this run
    wrote is undecidable without a live re-read (see Deployment._push_or_record_failure).
    Never retried further: re-sending the same stale ETag cannot succeed. The remedy is a
    fresh plan."""

    def __init__(self, message, ambiguous=False):
        super().__init__(message)
        # True iff the 412 arrived on attempt >1 of the SAME conditional PUT — only the
        # client can know that, so the fact rides on the exception (see FabricClient.put_roles).
        self.ambiguous = ambiguous


class UsageError(OLAFError):  # called wrong, e.g. out_of_band() with no client
    category = "validation"


class ControlDataGuardError(OLAFError):
    """A sensitive operation could not establish its narrow DAR/attestation boundary."""

    category = "guard"


class PostWriteAuditError(OLAFError):
    """A sensitive write returned success but its durable completion row was not confirmed."""

    category = "audit"

    def __init__(self, operation, push_status, backup_path, batch_id, run_id, cause):
        self.changed = True
        self.possible_exposure = True
        self.operation = operation
        self.push_status = push_status
        self.backup_path = backup_path
        self.batch_id = batch_id
        self.run_id = run_id
        self.cause = cause
        write_result = (
            "the Delta RESTORE completed" if operation == "rollback" else "the DAR PUT returned 2xx"
        )
        super().__init__(
            f"{operation}: {write_result}; audit completion was not confirmed "
            f"({type(cause).__name__}: {cause}). Inspect the prepared row, backup "
            f"{backup_path}, and a bounded live DAR reread before recovery."
        )

    def as_data(self):
        return {
            "operation": self.operation,
            "push_status": self.push_status,
            "backup_path": self.backup_path,
            "batch_id": self.batch_id,
            "run_id": self.run_id,
            "possible_exposure": self.possible_exposure,
        }


class PostWriteBoundaryError(ControlDataGuardError):
    """A confirmed sensitive write was followed by an unsafe or unreadable boundary."""

    def __init__(self, operation, backup_path, cause, changed=True):
        self.changed = changed
        self.possible_exposure = True
        self.operation = operation
        self.backup_path = backup_path
        self.cause = cause
        super().__init__(
            f"{operation}: post-write control-data boundary was not confirmed "
            f"({type(cause).__name__}: {cause}); the incident sentinel and recovery "
            f"artifact are retained"
        )

    def as_data(self):
        return {
            "operation": self.operation,
            "backup_path": self.backup_path,
            "possible_exposure": self.possible_exposure,
        }

In [ ]:
# ══ Parse ════════════════════════════════════════════════════════════════════
class Parse:
    """String → structured value parsing; parameter typing; name/glob matching."""

    @staticmethod
    def bool_param(name, value):
        """Strictly parse a boolean notebook parameter -> (parsed, error). A real bool passes
        through (the OLAF facade and direct run_mode calls hand over real booleans); an int 0/1
        passes through as its bool (Fabric also offers Int as a Base-parameter type, and 0/1 are
        the only unambiguous int spellings — every OTHER int is rejected rather than guessed at);
        a string is matched case-insensitively against BOOL_PARAM_SPELLINGS. Absence has three
        spellings — a missing key (callers do params.get(name, False)), a blank string, and None
        (Fabric can genuinely deliver null for a Base parameter left unset) — and all three mean
        the same thing: no value was given -> False, not an error. ANYTHING else is rejected —
        never coerced with bool(), under which every non-empty string is truthy, so a pipeline
        passing the STRING "false" (Fabric Base parameters default to the String type) would get
        True and apply would select the config-payload path, making prior-live roles omission
        candidates rather than carrying them in an incremental payload. The reason is
        RETURNED rather than raised so the caller can finish building the envelope's param echo
        before refusing (run_mode raises its SystemExit guard alongside the other guards, so the
        refusal comes back as a blocked envelope)."""
        # bool FIRST: isinstance(True, int) is True, so a real bool would otherwise fall into the
        # int branch below.
        if isinstance(value, bool):
            return value, None
        if isinstance(value, int) and value in (0, 1):
            return bool(value), None
        if value is None:
            return False, None  # unset, same as a blank string — not a guess, not an error
        if isinstance(value, str):
            text = value.strip().lower()
            if not text:
                return False, None
            if text in BOOL_PARAM_SPELLINGS:
                return BOOL_PARAM_SPELLINGS[text], None
        return False, (
            f"{name} must be a boolean, the int 0 or 1, or one of {sorted(BOOL_PARAM_SPELLINGS)} "
            f"(case-insensitive), got {value!r}"
        )

    @staticmethod
    def env_param(value):
        """Validate the optional `env` label -> (value, error), the same (parsed, reason) shape
        bool_param uses, so a caller can finish building its envelope before refusing.

        BLANK IS VALID and is the default: env exists to tell one environment's log rows from
        another's when several workspaces share a control-table shape, so a deployment that
        needs no such split leaves it unset rather than inventing a label. A missing pipeline
        base parameter (None) reads as blank for the same reason — there is nothing to refuse,
        only nothing to stamp. A value that IS given still has to be a label.

        ONE definition for the two boundaries that can SET env — run_mode's parameter parse and
        OLAF.configure() — so the pipeline path and the interactive facade cannot come to disagree
        about what an environment may be called. Each boundary translates the refusal into its own
        currency (run_mode a SystemExit alongside its other pre-target guards, configure a
        UsageError alongside its other refusals); what may NOT differ is the rule itself.

        REFUSED, never cleaned up. Stripping the offending characters would run the operator's
        deploy under a DIFFERENT environment label than the one they passed, and stamp that label
        on every audit row — a quiet wrong answer where a loud refusal costs one re-run. Same
        house rule as bool_param and the verbosity guard."""
        text = "" if value is None else str(value)
        if not text or ENV_RE.match(text):
            return text, None
        return text, (
            f"env must be blank (no label) or 1-64 characters of letters, digits, '_' or '-' "
            f"(matching "
            f"{ENV_RE.pattern}), got {value!r} — env is stamped on every audit row and read back "
            f"as a SQL literal, so a value that could close that literal is refused, not repaired"
        )

    @staticmethod
    def list(cell):
        """Split a ';' list cell: trim, drop empties, dedupe case-insensitively, tolerate trailing ';'."""
        if cell is None:
            return []
        out, seen = [], set()
        for item in str(cell).split(LIST_SEP):
            item = item.strip()
            if item and item.lower() not in seen:
                seen.add(item.lower())
                out.append(item)
        return out

    @staticmethod
    def table_entry(entry):
        """Validate + split one include/exclude_tables entry into (schema, table_pattern).
        Accepts name form 'schema.table' and path form '/Tables/schema/table'.
        Schema part is LITERAL only (wildcards are table-part only, rule A2);
        a '/Files/...' entry is redirected to the folder column (rule A3)."""
        e = str(entry).strip()
        if e.startswith("/Files") or e.lower().lstrip("/").startswith("files/"):
            raise ValidationError(
                f"'{entry}' looks like a folder path — did you mean include_folders? (rule A3)"
            )
        if e.startswith("/Tables"):
            parts = e.strip("/").split("/")
            if len(parts) != 3 or parts[0] != "Tables" or not parts[1] or not parts[2]:
                raise ValidationError(
                    f"table path must be /Tables/schema/table: '{entry}' (rule A3)"
                )
            schema, table = parts[1], parts[2]
        # ONE raise for the two ways an entry can name neither form: a leading slash that is not
        # /Tables (already handled) and not /Files (rejected above), or no dot at all. These were
        # two separate raises carrying byte-identical text -- the FOURTH duplicate of this shape
        # this codebase has merged. The first three were the RLS literal strip (four copies), the
        # pagination loop (two) and the member-gate checks (two, merged in the commit immediately
        # before this one). Both conditions genuinely mean the same thing to a reader, so they get
        # the same sentence from the same place.
        elif e.startswith("/") or "." not in e:
            raise ValidationError(
                f"table entry must be schema.table or /Tables/schema/table: '{entry}' (rule A3)"
            )
        else:
            schema, table = e.split(".", 1)
            # DELIBERATELY a different, shorter sentence -- do NOT fold it into the one above.
            # It is the ORIGINAL message from the name-form-only contract (fa1e05d): when the
            # path form arrived, that commit moved the no-dot case onto the longer sentence and
            # gave this brand-new branch the short one, with the long text three lines above --
            # a choice, not residue. (Why is not recorded anywhere; the reading that it would be
            # noise to offer the path form to someone who has already typed a dot is inference.)
            # Pinned by test_table_entry_dotted_but_empty_side_keeps_the_shorter_message, so a
            # later tidy-up reading the difference as an oversight goes red.
            if not schema or not table:
                raise ValidationError(f"table entry must be schema.table: '{entry}' (rule A3)")
        if "*" in schema or "?" in schema:
            raise ValidationError(
                f"schema part must be literal — wildcards are table-part only: '{entry}' (rule A2)"
            )
        return schema, table

    @staticmethod
    def subject_match(subject, *candidates):
        """show-* matcher: glob when the subject has wildcards, else case-insensitive EQUALITY.

        Equality, not substring. The subjects here are identifiers whose names nest —
        `sg-sales` is a prefix of `sg-sales-managers`, `sales.order` of
        `sales.orders` — so a substring match silently answered about principals and tables
        the caller had not named. On a mode whose whole job is "who can reach what", extra
        rows are worse than none: they read as access that was asked about and confirmed.

        Partial matching is still available and now has to be ASKED for, with the wildcard
        the error messages already advertise: `sg-sales*` matches both."""
        s = str(subject).lower()
        cands = [str(c).lower() for c in candidates]
        if "*" in s or "?" in s:
            return any(fnmatch.fnmatch(c, s) for c in cands)
        return any(s == c for c in cands)

    @staticmethod
    def trim_row(row):
        """Strip every STRING value in a Row.asDict() dict at the config/member-cache read
        seam, so all downstream logic sees trimmed values (role_name, table/folder patterns,
        member names/ids, notes, the rls_condition field, ...). Only the field's OUTER
        whitespace is stripped — inner content (e.g. a quoted rls_condition literal like
        'A B') is untouched because the whole field string is stripped, never its substrings.
        Non-string values (the boolean 'active' column, or a NULL column read back as None)
        pass through unchanged."""
        return {k: v.strip() if isinstance(v, str) else v for k, v in row.items()}

In [ ]:
# ══ ScopePath ════════════════════════════════════════════════════════════════
class ScopePath:
    """Scope path ⇄ name conversions: dotted catalog name ↔ `/Tables/...` DAR path,
    and folder-entry normalization to a canonical `/Files/...` path."""

    @staticmethod
    def table(table):
        """'schema.table' -> '/Tables/schema/table' (the path form the DAR API uses)."""
        schema, name = table.split(".", 1)
        return f"/Tables/{schema}/{name}"

    @staticmethod
    def to_table(path):
        """'/Tables/schema/table' -> 'schema.table' (inverse of ScopePath.table)."""
        parts = path.strip("/").split("/")
        return f"{parts[1]}.{parts[2]}" if len(parts) == 3 else path

    @staticmethod
    def folder(folder):
        """Normalize one folder entry to a canonical '/Files/...' path.
        'Files/raw' -> '/Files/raw'. A '/Tables/...' entry is redirected (wrong column);
        anything not under Files/ is rejected."""
        f = str(folder).strip()
        if f.startswith("/Tables") or f.lower().lstrip("/").startswith("tables/"):
            raise ValidationError(
                f"'{folder}' looks like a table path — did you mean include_tables? (rule A3)"
            )
        stripped = f.strip("/")
        if not (stripped.lower() == "files" or stripped.lower().startswith("files/")):
            raise ValidationError(f"folder entry must be under /Files: '{folder}' (rule A3)")
        return "/" + stripped

In [ ]:
# ══ Control-data boundary ════════════════════════════════════════════════════
@dataclass(frozen=True)
class ControlBoundarySnapshot:
    """Immutable point-in-time DAR evidence; never a workspace-isolation claim."""

    etag: str
    roles_digest: str
    reserved_digest: str
    workspace_id: str
    item_id: str
    observed_at: str
    roles_json: tuple[str, ...]

    @property
    def roles(self):
        return [json.loads(text) for text in self.roles_json]


class ControlBoundaryLease:
    """One operation's sentinel ownership and immutable approved DAR snapshot."""

    def __init__(self, boundary, operation, snapshot, owns_sentinel=True):
        self.boundary = boundary
        self.operation = operation
        self.snapshot = snapshot
        self.current_snapshot = snapshot
        self.owns_sentinel = owns_sentinel
        self.post_snapshot = None
        # Set the moment prewrite() authorizes a write. A refusal BEFORE that leaves a state
        # nobody has to wonder about, so it need not hold the incident marker.
        self.authorized_write = False

    def require_current(self):
        """Require a fresh sentinel-backed immutable DAR snapshot before a write."""
        self.boundary._read_sentinel()
        fresh = self.boundary.snapshot()
        self.boundary.require_same(self.current_snapshot, fresh)
        return fresh

    def prewrite(self):
        """Revalidate the sentinel and DAR immediately before a sensitive write."""
        self.authorized_write = True
        return self.require_current()

    def prewrite_audit(self):
        """The same revalidation for an AUDIT append, which does not mark the state uncertain.

        A log row is a complete write or no write; it cannot land halfway and leave a state
        nobody can classify, which is the thing the incident marker is for. Counting it as one
        made the forensic 'rejected' row a validation refusal writes strand every operation
        after it -- the refusal's own audit trail causing the incident it recorded."""
        return self.require_current()

    def accept_dar_write(self):
        """Advance only after this lease's own confirmed DAR mutation."""
        self.boundary._read_sentinel()
        self.current_snapshot = self.boundary.snapshot()
        return self.current_snapshot

    def postcheck(self, allow_dar_change=False):
        fresh = self.boundary.snapshot()
        if not allow_dar_change:
            self.boundary.require_same(self.snapshot, fresh)
        elif self.current_snapshot is not self.snapshot:
            self.boundary.require_same(self.current_snapshot, fresh)
        self.post_snapshot = fresh
        return fresh

    def clear(self):
        self.boundary.clear_owned(self)


class ControlBoundary:
    """Fail closed around sensitive same-lakehouse control data.

    A clean result is a DAR snapshot classification plus a separate operator
    attestation. It is not proof of workspace or item isolation.
    """

    SENTINEL_REL_PATH = "Files/security/.olaf-sensitive-write.sentinel"
    SENTINEL_FULL_PATH = "/lakehouse/default/Files/security/.olaf-sensitive-write.sentinel"
    SENTINEL_CONTENT = "OLAF_CONTROL_DATA_WRITE_IN_PROGRESS_V1\n"

    def __init__(self, client, tables, mapping_history_dir, role_backup_dir, attestation=""):
        if client is None:
            raise ControlDataGuardError(
                "control-data gate needs a live Fabric DAR client for the attached target"
            )
        self.client = client
        self.attestation = str(attestation or "")
        reserved = ["/Files/security"]
        for key in ("config_table", "mapping_table", "log_table", "member_table"):
            reserved.append(self._table_path(tables.get(key), key))
        self._require_security_descendant(mapping_history_dir, "mapping_history_dir")
        self._require_security_descendant(role_backup_dir, "role_backup_dir")
        self.reserved = tuple(sorted({self.normalize_path(path) for path in reserved}))
        self.reserved_digest = self._digest(list(self.reserved))

    @staticmethod
    def _digest(value):
        canonical = json.dumps(value, sort_keys=True, separators=(",", ":"), default=str)
        return hashlib.sha256(canonical.encode()).hexdigest()

    @staticmethod
    def normalize_path(path):
        text = str(path or "").strip()
        if not text or "\\" in text or "\x00" in text:
            raise ControlDataGuardError("DAR path is missing or malformed")
        parts = [part for part in text.split("/") if part]
        return "/" if not parts else "/" + "/".join(parts).lower()

    @classmethod
    def _table_path(cls, table, label):
        text = str(table or "").strip()
        parts = text.split(".")
        if (
            len(parts) != 2
            or any(not part for part in parts)
            or any("/" in part or "\\" in part or "\x00" in part for part in parts)
        ):
            raise ControlDataGuardError(
                f"{label} must be an unambiguous two-part schema.table name"
            )
        return f"/Tables/{parts[0]}/{parts[1]}"

    @staticmethod
    def _require_security_descendant(value, label):
        import posixpath

        text = str(value or "").strip()
        normalized = posixpath.normpath(text.lstrip("/"))
        if (
            not text
            or "\\" in text
            or "\x00" in text
            or not normalized.lower().startswith("files/security/")
        ):
            raise ControlDataGuardError(f"{label} must be a descendant of Files/security")

    def overlaps_reserved(self, path):
        candidate = self.normalize_path(path)
        return any(
            candidate == reserved
            or candidate.startswith(reserved + "/")
            or reserved.startswith(candidate.rstrip("/") + "/")
            for reserved in self.reserved
        )

    def require_desired_safe(self, grants):
        for grant in grants:
            if not isinstance(grant, dict) or not grant.get("scope_path"):
                raise ControlDataGuardError("desired grant has an unknown scope path")
            if self.overlaps_reserved(grant["scope_path"]):
                raise ControlDataGuardError(
                    f"desired role {grant.get('role_name') or '(unnamed)'} overlaps "
                    f"reserved control-data scope {grant['scope_path']}"
                )

    def _require_live_safe(self, roles):
        if not isinstance(roles, list):
            raise ControlDataGuardError(
                "DAR snapshot is partial or malformed: expected a role list"
            )
        for role in roles:
            if not isinstance(role, dict) or not isinstance(role.get("decisionRules"), list):
                raise ControlDataGuardError("DAR role or decisionRules shape is unknown")
            for rule in role["decisionRules"]:
                if not isinstance(rule, dict) or rule.get("effect") != "Permit":
                    raise ControlDataGuardError("DAR decision-rule effect is unknown")
                permissions = rule.get("permission")
                if not isinstance(permissions, list):
                    raise ControlDataGuardError("DAR permission shape is unknown")
                attributes = {}
                for entry in permissions:
                    if not isinstance(entry, dict):
                        raise ControlDataGuardError("DAR permission entry is unknown")
                    name = entry.get("attributeName")
                    values = entry.get("attributeValueIncludedIn")
                    if name in attributes or not isinstance(values, list):
                        raise ControlDataGuardError(
                            "DAR permission attribute is duplicated or unknown"
                        )
                    attributes[name] = values
                paths = attributes.get("Path")
                actions = attributes.get("Action")
                if (
                    not isinstance(paths, list)
                    or not paths
                    or any(not isinstance(path, str) or not path.strip() for path in paths)
                ):
                    raise ControlDataGuardError("DAR permission is missing a Path attribute")
                overlapping = any(self.overlaps_reserved(path) for path in paths)
                if not isinstance(actions, list) or not actions:
                    if overlapping:
                        raise ControlDataGuardError(
                            "overlapping DAR permission is missing an Action attribute"
                        )
                    continue
                if any(not isinstance(action, str) or not action.strip() for action in actions):
                    raise ControlDataGuardError("DAR Action permission shape is unknown")
                normalized_actions = {action.lower() for action in actions}
                if not normalized_actions <= {"read", "readwrite"}:
                    raise ControlDataGuardError("DAR Action permission shape is unknown")
                if not overlapping:
                    continue
                members = role.get("members")
                if not isinstance(members, dict):
                    raise ControlDataGuardError("overlapping DAR role has unknown members")
                if "fabricItemMembers" in members:
                    raise ControlDataGuardError(
                        "overlapping dynamic fabricItemMembers rule is unsafe even when empty"
                    )
                if set(members) - {"microsoftEntraMembers"}:
                    raise ControlDataGuardError(
                        "overlapping DAR role has an unknown member container"
                    )
                entra = members.get("microsoftEntraMembers")
                if not isinstance(entra, list):
                    raise ControlDataGuardError("overlapping DAR role has unknown Entra members")
                if entra:
                    raise ControlDataGuardError(
                        f"live role {role.get('name') or '(unnamed)'} grants read access "
                        "to reserved control data"
                    )

    def snapshot_from(self, roles, etag):
        """Classify one already-bounded role-list response without issuing another read."""
        workspace_id = str(getattr(self.client, "workspace_id", "") or "").strip()
        item_id = str(getattr(self.client, "item_id", "") or "").strip()
        if not workspace_id or not item_id:
            raise ControlDataGuardError(
                "bounded DAR snapshot has no resolved workspace or item target"
            )
        if not isinstance(etag, str) or not etag.strip():
            raise ControlDataGuardError("bounded DAR snapshot has no collection ETag")
        self._require_live_safe(roles)
        roles_json = tuple(
            sorted(json.dumps(role, sort_keys=True, separators=(",", ":")) for role in roles)
        )
        return ControlBoundarySnapshot(
            etag=etag,
            roles_digest=self._digest(list(roles_json)),
            reserved_digest=self.reserved_digest,
            workspace_id=workspace_id,
            item_id=item_id,
            observed_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),
            roles_json=roles_json,
        )

    def snapshot(self):
        try:
            roles = self.client.list_roles_quick()
        except Exception as exc:
            raise ControlDataGuardError(
                f"bounded DAR snapshot could not be read: {type(exc).__name__}"
            ) from exc
        return self.snapshot_from(roles, getattr(self.client, "roles_etag", None))

    @staticmethod
    def require_same(approved, fresh):
        before = (
            approved.etag,
            approved.roles_digest,
            approved.reserved_digest,
            approved.workspace_id,
            approved.item_id,
        )
        after = (
            fresh.etag,
            fresh.roles_digest,
            fresh.reserved_digest,
            fresh.workspace_id,
            fresh.item_id,
        )
        if before != after:
            raise ControlDataGuardError(
                "DAR state changed after the approved snapshot; refused instead of "
                "refreshing authorization"
            )

    @staticmethod
    def _require_evidence(value, label):
        if not CONTROL_EVIDENCE_RE.fullmatch(str(value or "")):
            raise ControlDataGuardError(
                f"{label} must be a 1-128 character evidence reference using only "
                "letters, digits, dot, underscore, colon, slash, hash, or hyphen"
            )

    def _create_sentinel(self):
        import os

        try:
            os.makedirs(os.path.dirname(self.SENTINEL_FULL_PATH), exist_ok=True)
            with open(self.SENTINEL_FULL_PATH, "x", encoding="utf-8") as handle:
                handle.write(self.SENTINEL_CONTENT)
        except FileExistsError as exc:
            raise ControlDataGuardError(
                "control-data incident sentinel already exists; explicit reviewed "
                "clearance is required"
            ) from exc
        except OSError as exc:
            raise ControlDataGuardError(
                f"control-data sentinel could not be created: {type(exc).__name__}"
            ) from exc
        self._read_sentinel()

    def isolation_state(self):
        """attested when this run supplied an evidence reference, unknown when it did not.

        Never "safe" or "isolated": OLAF cannot verify that same-lakehouse control data is
        actually isolated, so the record says who claimed it, not that it is true. Optional
        since it gates nothing — an absent reference is recorded as unknown rather than
        refused, because refusing taught callers to type a character to get past it."""
        return (
            "attested" if CONTROL_EVIDENCE_RE.fullmatch(str(self.attestation or "")) else "unknown"
        )

    def begin(self, operation, snapshot=None, sentinel_already_owned=False):
        approved = snapshot or self.snapshot()
        if approved.reserved_digest != self.reserved_digest:
            raise ControlDataGuardError("reserved control-data set changed before the operation")
        if sentinel_already_owned:
            self._read_sentinel()
        else:
            self._create_sentinel()
        self.require_same(approved, self.snapshot())
        return ControlBoundaryLease(
            self, str(operation), approved, owns_sentinel=not sentinel_already_owned
        )

    def _read_sentinel(self):
        try:
            with open(self.SENTINEL_FULL_PATH, encoding="utf-8") as handle:
                content = handle.read()
        except OSError as exc:
            raise ControlDataGuardError(
                f"control-data sentinel is absent or unreadable: {type(exc).__name__}"
            ) from exc
        if content != self.SENTINEL_CONTENT:
            raise ControlDataGuardError("control-data sentinel read-back content is unknown")

    def release_unwritten(self, lease):
        """Drop a lease that refused before authorizing any write. True when released.

        clear_owned() deliberately demands a completed postcheck, because clearing after a
        WRITE requires proving the state is safe. Nothing was written here, so there is no
        post-state to prove -- and holding the marker for a config typo taught operators to
        clear incidents without reading them."""
        import os

        if lease.authorized_write or lease.boundary is not self or not lease.owns_sentinel:
            return False
        try:
            os.remove(self.SENTINEL_FULL_PATH)
        except OSError:
            return False  # leave it rather than guess; the next run will say so plainly
        return True

    def clear_owned(self, lease):
        import os

        if lease.boundary is not self or not lease.owns_sentinel or lease.post_snapshot is None:
            raise ControlDataGuardError(
                "only the owning safely post-checked run may clear the sentinel"
            )
        self._read_sentinel()
        try:
            os.remove(self.SENTINEL_FULL_PATH)
        except OSError as exc:
            raise ControlDataGuardError(
                f"control-data sentinel could not be cleared: {type(exc).__name__}"
            ) from exc

    def clear_incident(self, access_review, approved=None):
        import os

        self._require_evidence(access_review, "access review")
        approved = approved or self.snapshot()
        self._read_sentinel()
        self.require_same(approved, self.snapshot())
        try:
            os.remove(self.SENTINEL_FULL_PATH)
        except OSError as exc:
            raise ControlDataGuardError(
                f"control-data sentinel could not be cleared: {type(exc).__name__}"
            ) from exc
        return {"cleared": True, "exposure_remediated": False}

In [ ]:
# ══ Hash ═════════════════════════════════════════════════════════════════════
class Hash:
    """Content-fingerprint hashing — the staleness guard's config hash and the mapping
    lock-file hash."""

    @staticmethod
    def _digest(rows):
        """sha256 of the canonical JSON serialization, truncated to 16 hex chars — the raw
        digest config() and mapping_content() share AFTER each has imposed its own row
        order. Row order is hashed AS GIVEN here: each caller sorts before digesting, so
        the shared body is a refactor seam, never a second fingerprint."""
        return hashlib.sha256(json.dumps(rows, sort_keys=True, default=str).encode()).hexdigest()[
            :16
        ]

    @staticmethod
    def config(rows):
        """Stable content fingerprint of the short config — the staleness guard's comparison key.
        Same config content -> same hash; any edit -> new hash — the provenance every later
        stage compares against. Rows are sorted by their own canonical serialization first:
        every row list fed here comes off a Spark collect with no ORDER BY, and collect
        order is an engine detail, not a contract — unsorted, the same logical config could
        read STALE after a compaction or table rewrite reordered it, so the promise above
        held per row but not per list. Keyed on the serialization rather than a grain key
        because the config's grain spans several scope columns and this needs to know none
        of them. sorted() (a copy), never .sort(): callers hand in cached row lists."""
        ordered = sorted(rows, key=lambda r: json.dumps(r, sort_keys=True, default=str))
        return Hash._digest(ordered)

    @staticmethod
    def mapping_content(rows):
        """Order-independent content fingerprint of the mapping lock-file — its MAPPING_COLUMNS only
        (excludes per-generation provenance like generated_at). Rows are sorted by
        (role_name, scope_path) — the same primary key 1.0.0 sorted on, so every
        duplicate-free mapping (the normal case) keeps its 1.0.0 hash VALUE and a
        1.0.0-stamped plan row still opens the saved-plan gate after an upgrade — with the
        FULL canonical serialization as a tiebreaker, so even two rows sharing
        (role_name, scope_path), as an imperfectly deduped table can hold, cannot make the
        fingerprint read-order-dependent (a coin-flip hash would make the plan gate reject
        intermittently; the tiebreaker engages ONLY on duplicate grain keys). Identical across
        generate (in-memory) and the plan/apply that read the table back. Logged as mapping_hash
        (the config -> mapping -> run provenance chain). Digested via Hash._digest, the body
        config() shares — one digest, two row orders, and the two can never drift apart."""
        projected = [{c: r.get(c) for c in MAPPING_COLUMNS} for r in rows]
        projected.sort(
            key=lambda d: (
                str(d.get("role_name") or ""),
                str(d.get("scope_path") or ""),
                json.dumps(d, sort_keys=True, default=str),
            )
        )
        return Hash._digest(projected)

In [ ]:
# ══ TableSchema ══════════════════════════════════════════════════════════════
class TableSchema:
    """Control-table DDL: the CREATE TABLE statement builder and the column-type lookup
    schema-drift migration shares with create, so they never disagree."""

    # The columns that are NOT strings. Everything absent from this map is STRING: role/scope
    # names, GUIDs, the `;`-joined LIST columns, and the 16-char hex hashes.
    #
    # These are the PHYSICAL Delta types, and docs/data-model.md documents the same map per
    # column. `run_duration` stays STRING deliberately — it holds a rounded float of elapsed
    # seconds and is read as a label, never arithmetic.
    COLUMN_TYPES = {
        "active": "BOOLEAN",  # config
        "generated_at": "TIMESTAMP",  # mapping
        "run_at": "TIMESTAMP",  # log
        "config_version": "BIGINT",  # mapping + log — a Delta commit version
        "mapping_version": "BIGINT",  # log — the mapping table's Delta commit version
    }

    @staticmethod
    def ddl_type(name):
        """The DDL/ALTER column type for a control-table column: COLUMN_TYPES if it is listed,
        STRING otherwise. Single source of truth shared by TableSchema.definitions (CREATE),
        Deployment.setup (ALTER ADD COLUMNS on schema-drift migration, and the type-drift
        warning), and the two write paths that build a DataFrame schema — so the DDL, the
        migration and the writes can never disagree on a column's type."""
        return TableSchema.COLUMN_TYPES.get(name, "STRING")

    @staticmethod
    def frame_schema(names):
        """(StructType, coerce) for a control-table write: the DataFrame schema and a per-column
        coercion, both derived from ddl_type — so a written frame can never disagree with the DDL
        that created the table. An explicit schema is required either way: Spark cannot infer a
        type for a column that is None in every row (member_name on a 'start' row, rls_condition
        on a folder grant).

        Coercion is by DECLARED type, not by what the caller happens to hold: a timestamp column
        accepts a datetime or the ISO-8601 string the runtime builds, and a bigint accepts an int
        or its string spelling — so a value that has been round-tripped through a read still
        writes back as the right type."""
        from pyspark.sql.types import (
            BooleanType,
            LongType,
            StringType,
            StructField,
            StructType,
            TimestampType,
        )

        spark_types = {
            "STRING": StringType,
            "BOOLEAN": BooleanType,
            "BIGINT": LongType,
            "TIMESTAMP": TimestampType,
        }

        def coerce(ddl, value):
            if value is None or value == "":
                return None
            if ddl == "BOOLEAN":
                return value if isinstance(value, bool) else str(value).lower() == "true"
            if ddl == "BIGINT":
                return int(value)
            if ddl == "TIMESTAMP":
                if isinstance(value, datetime.datetime):
                    return value
                return datetime.datetime.fromisoformat(str(value).replace("Z", "+00:00"))
            return str(value)

        types = [TableSchema.ddl_type(c) for c in names]
        schema = StructType([StructField(c, spark_types[t](), True) for c, t in zip(names, types)])
        return schema, (lambda row: [coerce(t, row.get(c)) for c, t in zip(names, types)])

    @staticmethod
    def definitions(config_table, mapping_table, log_table, member_table):
        """Idempotent CREATE TABLE IF NOT EXISTS DDL for the four control tables.
        All names are backticked defensively; 'active' is BOOLEAN, everything else STRING."""

        def columns(names):
            return ", ".join(f"`{n}` {TableSchema.ddl_type(n)}" for n in names)

        return {
            config_table: f"CREATE TABLE IF NOT EXISTS {config_table} ({columns(CONFIG_AUTHOR_COLUMNS)})",
            mapping_table: f"CREATE TABLE IF NOT EXISTS {mapping_table} "
            f"({columns(MAPPING_COLUMNS + MAPPING_PROVENANCE_COLUMNS)})",
            log_table: f"CREATE TABLE IF NOT EXISTS {log_table} ({columns(LOG_COLUMNS)})",
            member_table: f"CREATE TABLE IF NOT EXISTS {member_table} "
            f"({columns(MEMBER_CACHE_COLUMNS)})",
        }

In [ ]:
# ══ RLS ══════════════════════════════════════════════════════════════════════
class RLS:
    """Row-level security (RLS) predicate analysis: condition -> DAR predicate string,
    referenced-column extraction, unsupported-syntax/complexity checks (B2, C7, C9, C10)."""

    @staticmethod
    def to_predicate(table, condition):
        """WHERE-only condition from config -> full predicate string the DAR API expects.
        Verify once on your live API that the schema-qualified FROM is accepted."""
        return f"SELECT * FROM {table} WHERE {condition}"

    @staticmethod
    def _strip_literals(condition):
        """The condition as a string with every single-quoted literal blanked to a space — the one
        definition of "outside a string literal" that rules C9, C11 and C13 must agree on.

        Hoisted from RLS.referenced_columns_all / names_any_identifier / unsupported_tokens /
        connective_count, which each held a byte-identical copy of this expression with nothing
        forcing the four to stay in agreement. A method rather than a bare compiled constant in the
        ROLE_NAME_RE / RLS_OPERAND_FOLLOW_RE block because the shared step is the whole
        `re.sub(..., str(condition))` — pattern AND the str() coercion; a constant would hoist only
        the pattern and leave four copies of the call.

        NOT a guard. Every caller keeps its own `if not condition:` early return. The four return
        four different empty values ([], False, [], 0) and only the caller knows which one, so those
        guards cannot be hoisted UP into this helper as they stand.
        What breaks is DELETING those guards with nothing strip-neutral in their place: `str(None)`
        is "None", which this reads as a bareword, so names_any_identifier(None) flips False -> True
        and referenced_columns_all(None) returns ['None'] — re-measured against the pytest suite,
        one guard removed at a time: referenced_columns_all's is worth 18 tests,
        names_any_identifier's 2, and test_strip_literals_each_guard_returns_its_own_empty_value
        is in both sets, so 19 distinct. unsupported_tokens' is worth 1, and NOT for None:
        str([]) is "[]", so a falsy CONTAINER makes C9 report a spurious unsupported token
        (an earlier claim that this guard was behaviourally inert was wrong — it was only
        TEST-inert, and the case is pinned now). connective_count's guard genuinely is inert: 0.
        Folding a strip-neutral guard in HERE is a different proposal, and it does work: a leading
        `if not condition: return ""` is behaviour-preserving AND it makes all four callers' guards
        redundant rather than merely survivable, because "" through each caller's body already
        yields that caller's own empty value by construction. Measured out-of-tree with the fold in
        and all four guards deleted: green, and 0 differences over 60 comparisons
        (4 functions x 15 inputs, 8 falsy + 7 truthy).
        The guards are kept anyway, and that is a READABILITY judgement rather than a correctness
        requirement — nothing fails if they go, given the fold. Each function states its own
        empty-value contract at its own top instead of leaving it to emerge from "" flowing through
        two or three regex operations; on the path that gates C9, C11 and C13, four locally stated
        contracts are easier to audit than four emergent ones.

        Does NOT handle the SQL '' escape (`'it''s'`): none of the four copies did either, so this
        is a pure hoist. Whether to handle it is a separate behaviour question."""
        return re.sub(r"'[^']*'", " ", str(condition))

    @staticmethod
    def referenced_columns_all(condition):
        """Every column-name occurrence in an rls_condition, preserving each original spelling
        (duplicates kept, NO case-folding/dedup). Extraction matches RLS.referenced_columns: strips
        string literals via RLS._strip_literals (the shared "outside a string literal" definition),
        drops identifiers immediately followed by '(' (function calls, not columns —
        LOWER/UPPER/current_user/IS_MEMBER/DATE/...), keeps identifiers that aren't SQL keywords or bare
        numbers, and keeps only the trailing part of a qualified name (t.col -> col). Rule C11's case
        scan needs this full-occurrence view: the case-insensitive first-seen dedup in
        RLS.referenced_columns would otherwise drop a LATER wrong-case repeat of an already-referenced
        column (the null-safety idiom `Col NOT IN (...) OR col IS NULL`).

        Keyword-named columns (positional guard, RLS_OPERAND_FOLLOW_RE — the SAME compiled follow-set
        RLS.unsupported_tokens uses, shared so the two can never drift apart): a bareword whose NEXT
        significant token is a comparison operator (=, <>, !=, <, >, <=, >=) or IN / IS is the
        LEFT OPERAND of a comparison — a column literally named e.g. `Between` / `End` — so it IS
        emitted even though it spells a SQL keyword. Without this, such a predicate yields no
        references at all and the caller's column-existence + rule-C11 case check never run. That
        would be an OLAF validation gap; this runtime rejects its own mismatched configuration
        before a DAR request and does not infer platform behavior from the check.

        The readmission is gated on RLS_UNSUPPORTED_KEYWORDS, NOT RLS_KEYWORDS. Unlike
        RLS.unsupported_tokens this function has no early `continue` on RLS_SUPPORTED_OPERATORS, so
        gating on RLS_KEYWORDS would readmit NOT / IN / IS / AND / OR as "columns" whenever one is
        followed by a comparison — `region NOT IN ('north') OR region IS NULL` would report `NOT` as
        a missing column on the happy path.

        RESIDUAL (accepted, filed separately): the 8 supported alphabetic keywords still in
        RLS_KEYWORDS — AND FALSE IN IS NOT NULL OR TRUE — are never readmitted, so a column
        literally named `Null` / `Not` / `Or` stays invisible to this lexical check. That residual
        is an OLAF parser limitation, not a statement about service enforcement or access outcome.
        Covering them anyway needs the follow-set to EXCLUDE IN/IS for those keywords, which would
        break `NOT IN` — a second, subtler change to the same regex, deserving its own review.
        ALSO ACCEPTED (error quality, not correctness): `CASE WHEN ... END = 1` now also yields
        `END`, stacking an extra "column 'END' missing" error onto a predicate rule C9 already
        rejects (OneLake security RLS supports no CASE expression)."""
        if not condition:
            return []
        no_str = RLS._strip_literals(condition)
        cols = []
        for m in re.finditer(r"[A-Za-z_][A-Za-z0-9_.]*", no_str):
            rest = no_str[m.end() :].lstrip(" \t")
            if rest[:1] == "(":  # function call -> not a column
                continue
            name = m.group(0).split(".")[-1]  # t.col -> col
            up = name.upper()
            if not name or (
                up in RLS_KEYWORDS
                and not (up in RLS_UNSUPPORTED_KEYWORDS and RLS_OPERAND_FOLLOW_RE.match(rest))
            ):
                continue
            cols.append(name)
        return cols

    @staticmethod
    def referenced_columns(condition):
        """Best-effort DISTINCT column names referenced by an rls_condition (case-insensitive,
        first-seen spelling), for the column-existence check. Thin dedup over
        RLS.referenced_columns_all — behavior unchanged."""
        cols, seen = [], set()
        for name in RLS.referenced_columns_all(condition):
            if name.lower() not in seen:
                seen.add(name.lower())
                cols.append(name)
        return cols

    @staticmethod
    def names_any_identifier(condition):
        """True when the condition contains at least one bareword outside string literals — the
        weakest honest reading of 'this predicate mentions a column' (rule C13).

        Deliberately weaker than RLS.referenced_columns, which drops SQL keywords. Since T1 that
        lexer readmits a keyword-named bareword in operand position, so `Between = 'x'` now DOES
        read as a column — but only for the 16 keywords in RLS_UNSUPPORTED_KEYWORDS, and only in
        that position. The 8 supported alphabetic keywords (AND FALSE IN IS NOT NULL OR TRUE) are
        still read as no column at all, so `Null = 'x'` would still be invisible to C13 if C13
        used that lexer. Weakening to "any bareword at all" needs no keyword judgement whatsoever
        and so raises zero false C13 errors ON ASCII IDENTIFIERS; the price is letting a
        keyword-bearing constant (`1=0 AND 1=1`) through to the platform's own rejection. Catching
        the plain constant, which is the shorthand an author actually writes, is the point.

        SCOPE OF THAT CLAIM — `[A-Za-z_]` is ASCII-only, so "bareword" means "ASCII bareword".
        Any non-ASCII identifier is already rejected by the pipeline before this matters: C9 reads
        the non-ASCII characters as unsupported operator symbols, so an accented-Latin column name
        has its accented character reported as unsupported exactly as a CJK or Cyrillic name has
        its own. What is additionally WRONG rather than merely rejected is a FULLY non-Latin
        identifier — a bare CJK, Cyrillic or Greek column name, bracket-quoted or not, carrying
        no ASCII letter at all: it returns False here, so C13 additionally reports "RLS condition
        references no column" about a predicate that plainly names one. An accented-Latin
        identifier keeps its ASCII letters, so C13 does not misfire on it.

        Those identifiers are DESCRIBED rather than pasted, so this notebook holds no non-English
        text. The exact strings live as escapes in
        ValidationRules.test_c13_non_ascii_identifier_known_limitation.

        Left as a KNOWN LIMITATION, not a bug fix: the estate has no non-Latin column names
        (confirmed), and widening this to `[^\\W\\d_]` would change which configs generate on a
        rule whose whole design point is raising no false errors. Pinned by
        ValidationRules.test_c13_non_ascii_identifier_known_limitation."""
        if not condition:
            return False
        return bool(re.search(r"[A-Za-z_]", RLS._strip_literals(condition)))

    @staticmethod
    def unsupported_tokens(condition):
        """Unsupported-operator tokens in an rls_condition — operators/keywords OUTSIDE the OneLake-supported
        RLS subset (RLS_SUPPORTED_OPERATORS), returned first-seen and deduped. Empty when the predicate
        uses only supported syntax.

        Tokenization mirrors RLS.referenced_columns_all (the column-extraction lexer): string literals
        are stripped first via the shared RLS._strip_literals, and an identifier immediately followed
        by '(' is read as a function call.
        ASSUMPTION — deliberately conservative (a false error on a valid predicate is worse than a
        missed exotic one; OneLake itself is the backstop): flag ONLY a token we are confident is an
        operator/keyword —
          * a bareword that is a KNOWN SQL keyword not in the supported subset (RLS_UNSUPPORTED_KEYWORDS:
            LIKE / BETWEEN / CAST / EXISTS / ... — the same RLS_KEYWORDS the column lexer already treats
            as non-columns) AND in OPERATOR position — a keyword-spelled bareword whose NEXT significant
            token is a comparison operator (=, <>, !=, <, >, <=, >=) or IN / IS is the left operand of a
            comparison, i.e. a COLUMN literally named e.g. `Between` / `End`, so it reads as an
            identifier and is NOT flagged (positional guard, T1 review),
          * a function call (a name directly followed by '(', e.g. LOWER(...) — OneLake security RLS supports no
            functions),
          * a symbolic operator outside the comparison/grouping set (=, <>, >, >=, <, <=, parens,
            commas) — e.g. + * / % ! ~ ^ & |.
        A bareword that is neither a known keyword nor a function call is assumed to be a COLUMN
        identifier and is never flagged. '-' (unary minus / negative literals) and '.' (decimals,
        qualified names) are treated as value characters, not operators."""
        # C9 — unsupported RLS operator: rejects operators/keywords outside the OneLake-supported subset
        if not condition:
            return []
        no_str = RLS._strip_literals(condition)
        bad, seen = [], set()

        def _flag(tok):
            if tok not in seen:
                seen.add(tok)
                bad.append(tok)

        for m in re.finditer(r"[A-Za-z_][A-Za-z0-9_.]*", no_str):
            word = m.group(0).split(".")[-1]  # t.col -> col; keyword tokens are never qualified
            up = word.upper()
            if up in RLS_SUPPORTED_OPERATORS:
                continue
            if (
                no_str[m.end() :].lstrip(" \t")[:1] == "("
            ):  # function call -> unsupported in OneLake security RLS
                _flag(word + "(")
            elif up in RLS_UNSUPPORTED_KEYWORDS:
                # Positional guard (T1 review), RLS_OPERAND_FOLLOW_RE — the SAME compiled follow-set
                # RLS.referenced_columns_all uses, shared so the two can never drift apart: a
                # known-keyword bareword is an unsupported OPERATOR only when it is NOT in operand
                # (column) position. If the NEXT significant token is a
                # comparison/assignment operator (=, <>, !=, <, >, <=, >=) or IN / IS, the keyword is
                # the left operand of a comparison — a COLUMN literally named e.g. `Between` / `End` —
                # so do NOT flag it. Otherwise it is genuine unsupported operator usage (LIKE, BETWEEN
                # as an operator, ...) and IS flagged. Conservative: a false error on a valid predicate
                # is worse than a missed exotic operator, and OneLake itself is the query-time backstop.
                rest = no_str[m.end() :].lstrip(" \t")
                if not RLS_OPERAND_FOLLOW_RE.match(rest):
                    _flag(up)
        # Symbolic operators: blank out identifiers, number literals, and the supported symbolic/grouping
        # tokens; whatever non-space survives is an unsupported operator symbol.
        sym = re.sub(r"[A-Za-z_][A-Za-z0-9_.]*", " ", no_str)
        sym = re.sub(r"\d+(?:\.\d+)?", " ", sym)
        sym = re.sub(r"<=|>=|<>|[-=<>(),]", " ", sym)
        for m in re.finditer(r"\S+", sym):
            _flag(m.group(0))
        return bad

    @staticmethod
    def bare_boolean_values(condition):
        """Bare TRUE / FALSE used as a VALUE — the right of a comparison (rule C14).

        Lab-verified against the live DAR API on 2026-07-27 (dryRun PUT, with a known-good and a
        known-bad control in the same batch so the probe was shown to discriminate): OneLake
        accepts `Is_Current = 'true'` and refuses `Is_Current = true` with InvalidRLSPredicate.

        The refusal has nothing to do with the column being boolean. An unquoted value is parsed as
        a COLUMN NAME, so `= true` reads as "equals the column named true", which does not exist.
        `Lead_Source = Web` fails for exactly the same reason — and is ALREADY caught, because `Web`
        is not a keyword, so it reaches the column-existence check as a missing column. TRUE and
        FALSE never get that far: they are in RLS_SUPPORTED_OPERATORS, so both lexers skip them.
        That skip is what let a broken predicate through validate and generate and plan, all the way
        to apply, which is the one stage that talks to Fabric.

        Only VALUE position is flagged. A standalone `WHERE TRUE` is left alone: TRUE/FALSE are
        documented supported keywords, and the position guard is the whole difference between the
        documented use and the broken one. Same reason this reads BACKWARDS (RLS_VALUE_PRECEDE_RE)
        rather than reusing the shared follow-set, which answers the opposite question.

        An `IN` list counts as well — every slot in it, not just the first, since a list is all
        values by definition. `Is_Current IN (TRUE)` is refused by the platform exactly like
        `= true`, and C9 does not catch it either: `IN`, `TRUE` and `FALSE` are all supported
        keywords.

        `IS TRUE` / `IS NOT FALSE` count as value position too — checked in the Fabric UI, which
        refuses them with the same InvalidRLSPredicate. `IS` is a documented supported operator and
        so is TRUE, so C9 skips both and nothing else would catch the pair. `IS NULL` and
        `IS BLANK` are untouched: NULL and BLANK are not TRUE/FALSE, so they never reach here.

        Numbers need no quoting — `Contact_Key > 0` and `Contact_Key > '0'` are both accepted
        live — so this flags nothing numeric.

        RESIDUAL (accepted): an unquoted non-numeric literal that is not a bareword, e.g.
        `Birthdate = 1980-01-01`, is invisible here AND to C9 — the identifier lexer only matches
        tokens starting with a letter, and C9 deliberately treats '-' as a value character so
        negative literals keep working. OneLake refuses it, so it fails at apply rather than at
        validate. Catching it needs arithmetic detection inside a value, which cannot be done
        without risking a false error on `Amount > -5`, and this file's standing rule is that a
        false error on a valid predicate is worse than a missed exotic one."""
        if not condition:
            return []
        no_str = RLS._strip_literals(condition)
        bad, seen = [], set()
        for m in re.finditer(r"[A-Za-z_][A-Za-z0-9_]*", no_str):
            up = m.group(0).upper()
            if up not in ("TRUE", "FALSE") or up in seen:
                continue
            if RLS_VALUE_PRECEDE_RE.search(no_str[: m.start()].rstrip(" \t")):
                seen.add(up)
                bad.append(m.group(0))
        return bad

    @staticmethod
    def connective_count(condition):
        """Boolean-connective count (AND/OR) in an rls_condition — the complexity proxy. String literals are
        stripped and the match is whole-word + case-insensitive, so a column like 'BRAND' or a value
        inside quotes is never miscounted. A crude complexity proxy — see
        RLS_COMPLEXITY_WARN_CONNECTIVES."""
        if not condition:
            return 0
        no_str = RLS._strip_literals(condition)
        return len(re.findall(r"\b(?:AND|OR)\b", no_str, re.IGNORECASE))

In [ ]:
# ══ CLS ══════════════════════════════════════════════════════════════════════
class CLS:
    """Column-level security (CLS): CLS pair -> visible (allow-list) columns for one table (rule C11)."""

    @staticmethod
    def visible_for_table(table, cls_mode, inc_cols, exc_cols, canon):
        """CLS pair -> the VISIBLE (allow-list) columns for one table, resolved at generate against the
        catalog (so apply needs no catalog). blacklist -> catalog minus excluded; whitelist -> the
        included columns that exist in the catalog. Returns None when the row has no CLS (no column
        constraint -> every column visible). An empty result (a CLS that would hide EVERY column) is a
        caller-level error, never returned as [] — so downstream None means 'no CLS' unambiguously."""
        tcols = canon.get("columns", {}).get(table.lower(), [])
        if cls_mode == "blacklist":
            exc_lower = {c.lower() for c in exc_cols}
            return [c for c in tcols if c.lower() not in exc_lower]
        if cls_mode == "whitelist":
            lut = {c.lower(): c for c in tcols}
            return [lut[c.lower()] for c in inc_cols if c.lower() in lut]
        return None

In [ ]:
# ══ Member ═══════════════════════════════════════════════════════════════════
class Member:
    """Member resolution: config member display names -> Entra objectIds, resolved against the
    No-Graph member-table cache (rules C4/C5)."""

    # What ids_for writes where a name resolves to nothing. Deliberately neither "" nor None:
    # both are legitimate CONTENTS of a stored id list, so either would let a stale mapping
    # compare EQUAL to a resolution that actually failed. It is not GUID-shaped either, so it
    # cannot be mistaken for an objectId if one ever escaped into a payload.
    UNRESOLVED = "<unresolved>"

    @staticmethod
    def _declared_names(rows):
        """Every (member_type, name) an active config row DECLARES across all eight member
        columns — include and exclude alike, and including an include value its own exclude
        cancels. Generate._members hands the member gate only the EFFECTIVE set, so without this
        the gate never sees these names at all. Wildcard values are skipped, and adding member-wildcard
        expansion changed WHY.
        Expansion now owns them, which looks like it makes this dead -- but a cell whose author
        wrote two spellings of one name is deliberately left unexpanded (so _members' within-cell
        case guard still fires), and a pattern in that same cell rides along into here. 'add sg-*
        with its objectId' is not advice this gate should ever give. Keyed on the LOWERED name, valued with the first spelling seen, so
        the error names the operator's own text."""
        declared = {}
        for row in rows:
            for inc_col, exc_col, mtype in MEMBER_KINDS:
                for col in (inc_col, exc_col):
                    for name in Parse.list(row.get(col)):
                        if "*" in name or "?" in name:
                            continue
                        declared.setdefault((mtype, name.lower()), name)
        return declared

    @staticmethod
    def has_wildcard(rows):
        """True if any row carries `*`/`?` in one of the EIGHT MEMBER columns.

        Scoped to MEMBER_ALL_COLUMNS on purpose. Table and folder globs are legal, common, and sit
        in the canonical fixture (`include_tables: "sales.*"`), so a whole-row scan would match
        almost every real config and silently disable the generate idempotency skip fleet-wide --
        with both of its branches still covered, so the 100 % gate would report nothing wrong.

        Read on the RAW rows, before expansion: it decides whether generate may take the skip, so
        it must be answerable without the member table.
        """
        return any(
            v.lower().startswith(GLOB_PREFIX)
            for row in rows
            for col in MEMBER_ALL_COLUMNS
            for v in Parse.list(row.get(col))
        )

    @staticmethod
    def expand_wildcards(rows, spellings):
        """Expand `*`/`?` in the eight member columns from onelake_security_member, and canonicalise
        every resolvable literal to that table's spelling. Returns (NEW rows, errors). Rule C15.

        Expansion draws ONLY from the member table -- the No-Graph gate's single directory -- and
        only from rows of the column's own member_type, so include_sp_names="sg-*" matches no Group
        however many Groups are named sg-something. Entra is still never enumerated; the member
        table is, and it already had to list every principal the config names. `spellings` is
        _load_member_cache's third value, {(member_type, name.lower()): name AS WRITTEN}; its keys
        are the cache's keys by construction, so the cache itself is not needed here.

        RETURNS NEW DICTS AND NEVER MUTATES `rows`. `config_hash` is a property recomputed on every
        access from `short_rows`, which hands back the same mutable list, and generate reads it for
        the idempotency skip and again when stamping the mapping, with validation in between.
        Mutating a row here stamps a hash that plan/apply -- which never run validation and re-read
        the raw config -- derive differently, so every wildcard config would generate cleanly and
        then fail "STALE: short config changed after generate" forever. No differential over
        wildcard-free configs can see that, which is exactly why it is stated here.

        A 0-match pattern is an error on BOTH sides. That is deliberately unlike a dead LITERAL
        exclusion, which is accepted: Catalog.resolve_tables records the reason -- a dead include
        grants less than intended and fails CLOSED, while a dead exclude leaves everything it meant
        to remove still granted and fails OPEN. For a literal the member-table existence
        requirement already IS the 0-match check, so erroring on both sides needs no exception to
        remember. The pattern is also DROPPED from the cell: left in place it flows to
        _check_known, which advises "add it (with its objectId)" -- nonsense for a glob, and the
        symptom that was reported.

        A cell whose author wrote two spellings of one name is left UNTOUCHED. Generate._members'
        within-cell guard splits the RAW cell precisely because Parse.list dedupes
        case-insensitively and would drop a variant silently; rewriting such a cell collapses the
        two spellings and that guard stops firing, turning a hard error into a silent under-grant.
        Canonicalisation must never resolve a disagreement the author wrote.

        A cell needing no change is not rewritten at all, so incidental spelling in the config --
        a trailing ';', say -- survives untouched for every config that uses no wildcards.
        """
        expanded, errors = [], []
        for i, row in enumerate(rows):
            new = dict(row)
            rid = f"row {i + 1} ({row.get('role_name', '?')})"
            for inc_col, exc_col, mtype in MEMBER_KINDS:
                for col in (inc_col, exc_col):
                    written = [v.strip() for v in str(row.get(col) or "").split(LIST_SEP)]
                    written = [v for v in written if v]
                    if not written:
                        continue
                    spelt = {}
                    for v in written:
                        spelt.setdefault(v.lower(), set()).add(v)
                    if any(len(variants) > 1 for variants in spelt.values()):
                        continue  # the author disagreed with themselves -- _members reports it
                    resolved, changed = [], False
                    for v in written:
                        if v.lower().startswith(GLOB_PREFIX):
                            pattern = v[len(GLOB_PREFIX) :].strip().lower()
                            changed = True
                            hits = sorted(
                                name
                                for (mt, low), name in spellings.items()
                                if mt == mtype and fnmatch.fnmatch(low, pattern)
                            )
                            if not hits:
                                errors.append(
                                    f"{rid}: {col} matched 0 members: '{v}' — no {mtype} in "
                                    f"onelake_security_member matches it (rule C15)"
                                )
                            resolved.extend(hits)
                        else:
                            canon = spellings.get((mtype, v.lower()), v)
                            changed = changed or canon != v
                            resolved.append(canon)
                    if not changed:
                        continue
                    seen, final = set(), []
                    for name in resolved:
                        if name.lower() not in seen:
                            seen.add(name.lower())
                            final.append(name)
                    new[col] = LIST_SEP.join(final)
            expanded.append(new)
        return expanded, errors

    @staticmethod
    def _check_known(mtype, name, cache, errors):
        """One member value against the No-Graph gate: reject a GUID-shaped value, else resolve it
        from `cache`. Returns the objectId on a hit and None otherwise, appending the failure to
        `errors` in place.

        Both passes of resolve_ids call this. They used to spell the same two checks out
        separately with byte-identical message strings, which is the shape this repo has been
        bitten by twice (four copies of the RLS literal strip; two copies of the pagination
        loop) -- both found while still byte-identical, i.e. before they had a chance to
        drift, which is the only reason they were cheap to merge.

        Returning None for BOTH failure modes is safe because a GUID-shaped value and a cache
        miss are equally unusable as an objectId -- neither caller needs to tell them apart, and
        neither does: the effective pass branches on the return, the declared pass discards it."""
        if GUID_RE.match(name):
            errors.append(f"member '{name}' looks like an objectId — config takes display names")
            return None
        cached = cache.get((mtype, name.lower()))
        if cached is not None:
            return cached
        # No-Graph member gate: a config member absent from onelake_security_member is a hard
        # error — no Graph fallback. The member table is a required, complete input; populate it
        # (name + objectId) before generate. The error names the missing member.
        errors.append(
            f"member '{name}' ({mtype}) not found in onelake_security_member "
            f"— add it (with its objectId) before generate"
        )
        return None

    @staticmethod
    def ids_for(row, cache):
        """The four member_*_ids values a row's member NAME columns resolve to under `cache`
        ({(member_type, name.lower()): objectId}) -> {id_col: ';'-joined ids, or None for an
        empty column}. Positionally aligned with the matching member_*_names column, since it is
        built from the same Parse.list of it.

        THE one place names become ids, and it has two callers that MUST agree: resolve_ids stamps
        the mapping through it, and generate's idempotency skip RE-DERIVES through it to notice a
        member table edited since. The second one's whole job is comparing its answer against the
        first one's stored output, so spelled twice they could differ in the join, the dedupe or
        the empty-column case and the skip would then report drift on every run — or on none.

        A name absent from `cache` takes Member.UNRESOLVED rather than raising: to the comparing
        caller a member row deleted since generate IS a mismatch, and a KeyError there would turn
        a detected drift into a crashed generate. resolve_ids only reaches this after its own gate
        has proved every name resolvable, so the sentinel never reaches a written mapping."""
        out = {}
        for name_col, id_col, mtype in MAPPING_MEMBER_COLUMNS:
            ids = [
                cache.get((mtype, n.lower()), Member.UNRESOLVED)
                for n in Parse.list(row.get(name_col))
            ]
            out[id_col] = LIST_SEP.join(ids) if ids else None
        return out

    @staticmethod
    def resolve_ids(grants, cache, rows):
        """Resolve the effective member NAMES on the mapping grants to Entra objectIds — the ALL-OR-NOTHING
        resolve+validate step generate runs BEFORE writing the mapping. Collect every unique
        (member_type, name) across the four name columns, reject any value that is already a GUID
        (config takes display names, not objectIds), then resolve each name from `cache` (built from
        onelake_security_member) — a (member_type, name.lower()) present uses that objectId. No-Graph
        member gate: a name ABSENT from the cache is a HARD error naming it (no Graph fallback — the
        member table is a required, complete input; populate it before generate). `rows` (the same
        short-config rows Generate.rows validated) is REQUIRED, not defaulted: the gate must also
        reach every DECLARED member name — everything in an exclude column, and any include value
        its own exclude cancels — which Generate._members' effective-set subtraction never hands
        this function on its own. A required third parameter means a forgotten wiring is a
        TypeError at every call site, not a silently-unwired guard. Returns `errors`: every failure — when non-empty the caller must NOT write the mapping
        (all-or-nothing: no partial or stale ids ever land). On success every grant's four member_*_ids columns
        are populated in place, positionally aligned with the matching member_*_names column.
        After a clean resolution it also
        runs the resolved-id cross-mix check (Member._cross_role_rls_cls): aliases that collapse
        to one objectId can split RLS and CLS across roles undetected by the value-level C5, so any
        such hit is appended to `errors` and — like every generate error — blocks the write."""
        cache = cache or {}
        # Collect every distinct spelling per (member_type, name.lower()). Two config names that differ
        # ONLY by case (e.g. ONELAKE-x dept group vs OneLake-x matrix group) are DIFFERENT principals;
        # collapsing them to one lookup key would silently resolve to the wrong objectId, so a collision
        # is a hard error naming both (mirrors the table "differ only by case" guard).
        spelling_sets = {}  # (mtype, name.lower()) -> {spellings}
        for a in grants:
            for name_col, _id_col, mtype in MAPPING_MEMBER_COLUMNS:
                for name in Parse.list(a.get(name_col)):
                    spelling_sets.setdefault((mtype, name.lower()), set()).add(name)
        errors, id_by_pair = [], {}
        for (mtype, lower), names in sorted(spelling_sets.items()):
            if len(names) > 1:
                errors.append(
                    f"ambiguous {mtype} member names differing only by case: {sorted(names)} — "
                    f"different principals, rename to disambiguate"
                )
        pairs = {k: sorted(v)[0] for k, v in spelling_sets.items()}  # one representative per key
        for (mtype, lower), name in sorted(pairs.items()):
            oid = Member._check_known(mtype, name, cache, errors)
            # `is not None`, never truthiness. The cache's values are opaque here, and an empty
            # string is a HIT, not a miss. _load_member_cache never stores a blank id (it skips
            # any row failing GUID_RE), so in production this only ever guards a caller-supplied
            # cache -- but flipping it to `if oid:` changes what a blank resolves to, and the
            # whole suite stays green when you do. test_check_known_contract pins it.
            if oid is not None:
                id_by_pair[(mtype, lower)] = oid  # cache hit — no Graph call
        # Guard A: the effective-name loop above only sees names that SURVIVED
        # Generate._members' include/exclude subtraction. Every value a config row DECLARES
        # must still be a known principal -- an exclude column's values, and any include value
        # its own exclude cancelled, never reached `pairs` above. It runs the SAME check, via the
        # same helper, so the two can no longer drift apart -- this is the No-Graph gate reaching
        # columns it was always documented to cover, not a new rule. The resolved id is discarded
        # here: a declared-only name is not in any grant, so nothing would consume it.
        for (mtype, lower), name in sorted(Member._declared_names(rows).items()):
            if (mtype, lower) in pairs:
                continue  # the effective pass above already checked this principal
            Member._check_known(mtype, name, cache, errors)
        if errors:
            return errors
        for a in grants:
            a.update(Member.ids_for(a, id_by_pair))
        # Resolved-id second pass, keyed on the RESOLVED objectId: two spellings of one principal that resolve
        # to the same id and split RLS vs CLS across roles escape the value-level C5 — catch it now
        # that ids are populated (still collect-all: these errors join the caller's reject list).
        errors += Member._cross_role_rls_cls(grants)
        return errors

    @staticmethod
    def _cross_role_rls_cls(grants):
        """Cross-role RLS×CLS check — resolved-id pass. The value-level pass (in Generate._validate_across_rows) keys on the RAW config
        spelling, so two DIFFERENT spellings of one principal (a user's UPN and its mail alias) that
        resolve to the SAME objectId — one placed in an RLS-bearing role, the other in a different
        CLS-bearing role — slip past it, yet OneLake still fails that user's queries. Runs AFTER member
        names are resolved to objectIds (the grants carry populated member_*_ids), groups roles per
        objectId (a plain id key — objectIds are GUIDs, so no per-type namespace is needed) and fires
        the SAME rule-C5 error, naming EVERY original config spelling that resolved to the id plus the
        shared id. Dedupe: a cross role-pair whose RLS role and CLS role hold the same spelling was
        already reported by the value-level pass (identical value on both sides), so an id whose every
        cross pair shares a spelling emits nothing here.

        The id key is LOWER-CASED: objectIds are case-insensitive hex, so one principal whose id is
        written in two letter cases must still group as one — keyed raw, it split into two ids each
        reaching a single role and C5 was bypassed. Same invariant as the member cache: ids are
        compared case-insensitively, the EMITTED id is echoed exactly as it was written (the stored
        spelling reaches the DAR payload and the mapping hash and must never be normalized)."""
        role_has_rls, role_has_cls = {}, {}
        for a in grants:
            role = a["role_name"]
            role_has_rls[role] = role_has_rls.get(role, False) or bool(a.get("rls_condition"))
            role_has_cls[role] = role_has_cls.get(role, False) or bool(a.get("visible_columns"))
        # objectId (LOWERED) -> {role: {lowercased spellings}} / -> {lowercased: first-seen original}
        # / -> the first-seen WRITTEN id spelling (what the error echoes)
        id_role_lowers, id_orig, id_spelling = {}, {}, {}
        for a in grants:
            role = a["role_name"]
            for name_col, id_col, _mtype in MAPPING_MEMBER_COLUMNS:
                names = [x.strip() for x in str(a.get(name_col) or "").split(LIST_SEP) if x.strip()]
                ids = [x.strip() for x in str(a.get(id_col) or "").split(LIST_SEP) if x.strip()]
                for name, oid in zip(names, ids):
                    oid_key = oid.lower()
                    id_spelling.setdefault(oid_key, oid)
                    id_role_lowers.setdefault(oid_key, {}).setdefault(role, set()).add(name.lower())
                    id_orig.setdefault(oid_key, {}).setdefault(name.lower(), name)
        errors = []
        # C5 — cross-role RLS×CLS mix: one member in an RLS role and a different CLS role on a table
        for oid_key, by_role in sorted(id_role_lowers.items()):
            rls_roles = sorted(r for r in by_role if role_has_rls.get(r))
            cls_roles = sorted(r for r in by_role if role_has_cls.get(r))
            cross = [(rr, rc) for rr in rls_roles for rc in cls_roles if rr != rc]
            if not cross:
                continue
            if all(by_role[rr] & by_role[rc] for rr, rc in cross):
                continue  # every cross pair shares a spelling — the value-level pass already fired
            spellings = " / ".join(f"'{s}'" for s in sorted(id_orig[oid_key].values()))
            errors.append(
                f"member {spellings} (same objectId {id_spelling[oid_key]}) is in role(s) "
                f"{rls_roles} carrying RLS "
                f"and role(s) {cls_roles} carrying CLS — OneLake does not support mixing RLS and CLS "
                f"across roles for one user (queries fail); combine both policies into a single role "
                f"or remove the member from one side (rule C5)"
            )
        return errors

In [ ]:
# ══ Catalog ══════════════════════════════════════════════════════════════════
class Catalog:
    """Spark-catalog snapshot (canonical table names + columns, injectable folder listing) and
    table/folder entry resolution against it — generate's one source of truth for what exists."""

    @staticmethod
    def active_config_rows(spark, config_table):
        """The active config rows projected to CONFIG_AUTHOR_COLUMNS — OLAF's own columns
        and nothing else, trimmed at the read seam like every config read. A control table
        may carry columns another framework added (a load timestamp, a file hash, a loader
        identity); those are not OLAF's contract, and hashing them would fire the STALE
        guard on a config nobody edited and stop config_hash comparing across
        environments. A column OLAF itself declares later joins CONFIG_AUTHOR_COLUMNS and
        therefore the fingerprint — the projection follows the contract, not a frozen
        list. A MISSING declared column is refused, never silently projected past: fewer
        fields would hash stable-looking over a config that means something different
        from the one the author edited. The ONE reader behind both
        Deployment.short_rows (the hash the mapping is stamped with) and Audit.is_stale
        (the mirror of the generator's STALE guard) — byte-for-byte agreement between
        those two is an invariant, so they must not carry separate copies of this read.

        Column names are matched CASE-INSENSITIVELY, because that is what the engine
        does: Spark and Delta resolve `Role_Name` and `role_name` to one column, and a
        table adopted from elsewhere may spell the contract in any case. setup()'s
        additive migration and health()'s control-table check already fold case, so a
        case-sensitive match here would declare a table healthy that every run then
        refuses as "missing" — a contradiction the operator cannot act on. The projected
        dict always carries the CONTRACT spelling, so the hash never moves with the
        physical table's casing."""
        rows = []
        for r in spark.table(config_table).where("active = true").collect():
            d = r.asDict()
            by_lower = {str(k).lower(): v for k, v in d.items()}
            missing = [c for c in CONFIG_AUTHOR_COLUMNS if c.lower() not in by_lower]
            if missing:
                raise UsageError(
                    f"{config_table} is missing {missing} — part of OLAF's config column "
                    f"contract; projecting past a missing column would hash a config that "
                    f"means something different from the one the author edited"
                )
            rows.append(Parse.trim_row({c: by_lower[c.lower()] for c in CONFIG_AUTHOR_COLUMNS}))
        return rows

    @staticmethod
    def onelake_uri(workspace_id, item_id, logical_path):
        """Logical DAR scope path ('/Files/raw') -> the physical OneLake URI a filesystem call needs.

        The helper constructs an explicit workspace/item URI instead of relying on a relative
        filesystem path. It validates that both target identifiers are present; operators must
        validate filesystem behavior in their own Fabric environment."""
        if not workspace_id or not item_id:
            raise ValidationError(
                f"OneLake path needs the target workspace/item GUIDs (workspace_id={workspace_id!r}, "
                f"item_id={item_id!r})"
            )
        return (
            f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/"
            f"{item_id}/{str(logical_path).strip('/')}"
        )

    @staticmethod
    def fs_folder_lister(workspace_id, item_id):
        """Production folder lister BOUND TO ONE lakehouse: returns a callable mapping a logical
        scope path to its immediate child directory names, so it drops into the same injectable
        `folders` seam a test dict fills. The GUIDs are captured here because the seam itself only
        ever receives a path. The adapter's observed runtime behavior must be validated by the
        operator in the target environment."""

        def _lister(base):
            import notebookutils

            return [
                entry.name.rstrip("/")
                for entry in notebookutils.fs.ls(Catalog.onelake_uri(workspace_id, item_id, base))
                if getattr(entry, "isDir", False)
            ]

        return _lister

    @staticmethod
    def _export_lister(history_fullpath):
        """Patchable seam (mirrors Catalog.fs_folder_lister): existing filenames in the CSV
        for generate's per-generation dedupe. Sorted for determinism; a missing / unlistable folder (a
        first export, or the driver-local path not present) yields an empty listing, so generate writes a
        fresh versioned file rather than failing."""
        import os

        try:
            return sorted(os.listdir(history_fullpath))
        except OSError:
            return []

    @staticmethod
    def _folder_children(folders, base):
        """Resolve immediate child folder names of `base` from an injectable lister.
        `folders` is a callable (production: notebookutils.fs.ls seam) or a plain dict (tests)."""
        if callable(folders):
            return list(folders(base) or [])
        if isinstance(folders, dict):
            return list(folders.get(base, []) or [])
        return []

    class LazyColumnMap:
        """canon['columns'] as a LAZY, memoized view over spark.catalog.listColumns: one
        catalog round-trip per table, made on FIRST access and never repeated. Eagerly
        listing every table's columns made generate/validate/explain scale with LAKEHOUSE
        size — thousands of listColumns calls to validate a config referencing a handful of
        tables — when the lists are only ever read for tables the config names. Offers the
        exact dict surface those consumers use (.get / [] / in), so the plain dicts the test
        fixtures pass keep working unchanged; deliberately NOT iterable — iterating would
        force the very full-lakehouse listing this exists to avoid. Keys are the lowercase
        'schema.table' spellings `tables` holds; an unknown table reads as absent, exactly
        as it did from the eager dict."""

        def __init__(self, spark, tables):
            self._spark = spark
            self._tables = tables  # lowercase 'schema.table' -> canonical spelling
            self._cache = {}

        def _load(self, key):
            if key not in self._cache and key in self._tables:
                # Spark catalog column list (verify listColumns availability on your runtime)
                self._cache[key] = [
                    c.name for c in self._spark.catalog.listColumns(self._tables[key])
                ]
            return self._cache.get(key)

        def get(self, key, default=None):
            got = self._load(key)
            return default if got is None else got

        def __getitem__(self, key):
            got = self._load(key)
            if got is None:
                raise KeyError(key)
            return got

        def __contains__(self, key):
            return self._load(key) is not None

        def __iter__(self):
            # Without this, __getitem__ alone triggers the legacy sequence protocol:
            # list(map) walks integer keys into a bewildering KeyError(0), and bool(map)
            # is always True. Refuse loudly, with the reason.
            raise TypeError(
                "canon['columns'] is deliberately not iterable — iterating would force the "
                "full-lakehouse listing this lazy view exists to avoid; access it per key "
                "(canon['tables'] enumerates the universe)"
            )

    @staticmethod
    def canonical(spark, list_folders=None, workspace_id=None, item_id=None):
        """Canonical catalog snapshot for generate:
        tables  : lowercase 'schema.table' -> canonical 'schema.table'
        columns : lowercase 'schema.table' -> [column names], listed LAZILY per table on
                  first access and memoized (Catalog.LazyColumnMap) — the cost now scales
                  with the tables the CONFIG references, not with the lakehouse
        folders : injectable child-lister (production notebookutils.fs.ls seam, or a test dict/callable)

        workspace_id/item_id build the production folder lister's OneLake URIs and are required
        only when the config actually resolves folders; callers that read `tables`/`columns` alone
        can omit them, and the seam then raises a named error rather than listing the wrong path.
        """
        tables = {}
        for s in spark.sql("SHOW SCHEMAS").collect():
            # Schema-enabled lakehouses return multi-part namespaces (workspace.lakehouse.schema)
            # from SHOW SCHEMAS; take the trailing segment so SHOW TABLES and the config's bare
            # 'schema.table' keys align.
            schema = s[0].split(".")[-1]
            for t in spark.sql(f"SHOW TABLES IN `{schema}`").collect():
                name = f"{schema}.{t.tableName}"
                if name.lower() in tables:
                    raise ValidationError(f"ambiguous table names differing only by case: {name}")
                tables[name.lower()] = name
        return {
            "tables": tables,
            "columns": Catalog.LazyColumnMap(spark, tables),
            "folders": (
                list_folders
                if list_folders is not None
                else Catalog.fs_folder_lister(workspace_id, item_id)
            ),
        }

    @staticmethod
    def config_version(spark, config_table):
        """Delta commit version of the config table, captured at generate in ONE metadata query.
        A monotonic, human-readable label for VERSION AS OF retrieval — NOT the staleness guard
        (config_hash, content-based, is that). Returns None when the table is not Delta or DESCRIBE
        HISTORY is unavailable, so generate degrades gracefully and relies on config_hash."""
        try:
            rows = spark.sql(f"DESCRIBE HISTORY {config_table}").select("version").collect()
            return max((r["version"] for r in rows), default=None)
        except Exception:
            return None

    @staticmethod
    def _did_you_mean(name, candidates):
        """The closest known name as a message suffix, or "" when nothing is close enough.

        A typo is only actionable if the message says what was meant: 'salse.Dim_Customer'
        against a live catalog is two edits from 'sales.Dim_Customer', and saying so turns a
        refusal into a fix.
        """
        pool = {c.lower(): c for c in candidates}
        near = difflib.get_close_matches(name.lower(), list(pool), n=1, cutoff=0.7)
        return f" — did you mean '{pool[near[0]]}'?" if near else ""

    @staticmethod
    def resolve_tables(entry, canon):
        """One include/exclude_tables entry -> list of canonical 'schema.table'.
        Exact name = case-insensitive dict lookup; glob = fnmatch on the table part.

        0 matches ALWAYS raises, on the exclude side too. It used to warn there, on the
        reasoning that pre-emptively excluding a not-yet-created table is normal. But the two
        directions fail in OPPOSITE directions: a dead include grants less than intended and
        fails closed, while a dead exclude leaves everything it was meant to remove still
        granted — it fails OPEN. Treating the dangerous direction more leniently than the safe
        one inverted the framework's own deny-by-default stance."""
        schema, table = Parse.table_entry(entry)
        if "*" in table or "?" in table:
            pat = f"{schema}.{table}".lower()
            hits = sorted(v for k, v in canon["tables"].items() if fnmatch.fnmatch(k, pat))
        else:
            hit = canon["tables"].get(f"{schema}.{table}".lower())
            hits = [hit] if hit else []
        if not hits:
            # An unknown SCHEMA is reported on its own: schemas are created once, up front, so
            # naming one that does not exist can only be a typo — whereas an unknown table in a
            # real schema may simply not be built yet. Both refuse; the wording differs.
            schemas = sorted({t.split(".", 1)[0] for t in canon["tables"].values()})
            if not any(s.lower() == schema.lower() for s in schemas):
                raise ZeroMatchError(
                    f"matched 0 tables: '{entry}' — unknown schema '{schema}'"
                    f"{Catalog._did_you_mean(schema, schemas)} (rule A2)"
                )
            raise ZeroMatchError(
                f"matched 0 tables (typo?): '{entry}'"
                f"{Catalog._did_you_mean(entry, sorted(canon['tables'].values()))} (rule A2)"
            )
        return hits

    @staticmethod
    def resolve_folders(entry, folders):
        """One include/exclude_folders entry -> list of canonical '/Files/...' paths.
        Per-segment glob (never across '/', no '**'); literal segments must exist in the listing.
        0 matches always raises, include side and exclude side alike — see resolve_tables."""
        path = ScopePath.folder(entry)
        segments = path.strip("/").split("/")  # ["Files", "raw", "region_*"]
        current = ["/Files"]
        for seg in segments[1:]:
            nxt = []
            for base in current:
                children = Catalog._folder_children(folders, base)
                if "*" in seg or "?" in seg:
                    nxt += [
                        f"{base}/{c}" for c in children if fnmatch.fnmatch(c.lower(), seg.lower())
                    ]
                else:
                    nxt += [f"{base}/{c}" for c in children if c.lower() == seg.lower()]
            current = nxt
        hits = sorted(set(current))
        if not hits:
            raise ZeroMatchError(
                f"folder matched 0 paths (does it exist under /Files?): '{entry}'"
                # `folders` is a CALLABLE in production (the notebookutils.fs.ls seam) and a
                # dict only under test. Enumerating it as an iterable raised TypeError on
                # Fabric for every config naming a folder that does not exist -- turning the
                # one message meant to help into one that says nothing. Suggest from what the
                # lister can actually enumerate: the children of the segment above.
                f"{Catalog._did_you_mean(entry, Catalog._folder_children(folders, '/Files'))}"
                " (rule A2)"
            )
        return hits

In [ ]:
# ══ Target ═══════════════════════════════════════════════════════════════════
class Target:
    """Attach / tenant / runtime resolution: the notebook's attached workspace + lakehouse,
    the Entra tenant, who is running, this run's id, and case-insensitive item-name lookup."""

    @staticmethod
    def resolve():
        """The attached workspace + default lakehouse from notebookutils.runtime.context. This framework
        always deploys to the lakehouse the notebook is attached to (OneLake security is a per-lakehouse control, and
        the config/mapping/log tables live in that lakehouse) — there is no target to pass or override.
        Verify the exact context keys once on your Fabric runtime."""
        try:
            import notebookutils
        except ImportError as exc:
            raise SystemExit(
                "an attached Fabric lakehouse is required to establish the DAR control-data "
                "boundary; OLAF will not perform a schema-only bootstrap outside Fabric"
            ) from exc

        ctx = notebookutils.runtime.context
        ws = ctx.get("currentWorkspaceId")
        it = ctx.get("defaultLakehouseId")
        if not ws or not it:
            raise SystemExit(
                "no lakehouse attached — this framework deploys to the notebook's attached lakehouse; "
                "pin a default lakehouse (attach one in the portal, or bind one with a "
                "%%configure cell in a wrapper notebook — a pipeline cannot set a default "
                "lakehouse on the activity itself) and re-run"
            )
        # A notebook may pin a lakehouse that lives in ANOTHER workspace, which the runtime reports
        # as defaultLakehouseWorkspaceId. Deploying then splits in two: the control tables are read
        # and written through the attached lakehouse (that other workspace), while the DAR endpoint
        # addresses an item under THIS workspace — a pairing that names no item at all. Refuse here,
        # naming both sides, instead of surfacing it later as an opaque 404 mid-apply. Run the
        # notebook from the workspace that owns the lakehouse.
        lakehouse_ws = ctx.get("defaultLakehouseWorkspaceId")
        if lakehouse_ws and lakehouse_ws != ws:
            raise SystemExit(
                f"attached lakehouse lives in another workspace "
                f"({Target.ws_label(ctx, 'defaultLakehouseWorkspaceId', 'defaultLakehouseWorkspaceName')}) "
                f"than this notebook ({Target.ws_label(ctx, 'currentWorkspaceId', 'currentWorkspaceName')}) "
                "— OneLake security is a per-lakehouse control and this framework deploys only to a "
                "lakehouse attached from its own workspace; move or copy the notebook into that "
                "workspace, or attach a lakehouse from this one, and re-run"
            )
        return ws, it

    @staticmethod
    def ws_label(ctx, id_key, name_key):
        """A workspace as `'Name' (guid)`, or the bare guid when the runtime gives no name.
        An operator inside a Fabric notebook cannot resolve a workspace GUID without leaving the
        notebook, and these labels appear only in messages written for exactly that reader."""
        wid = ctx.get(id_key)
        name = str(ctx.get(name_key, "") or "").strip()
        return f"'{name}' ({wid})" if name else str(wid)

    @staticmethod
    def tenant(tenant_id=None):
        """The Entra tenant GUID stamped into DAR member payloads. Explicit `tenant_id` wins; else
        best-effort auto-resolve from the notebook runtime context (one tenant per Fabric tenant).
        Returns None if neither is available — the caller decides whether the mode needs it.
        Runtime-context keys vary by environment, so OLAF also attempts spark.conf
        'trident.tenant.id'. Supply tenant_id explicitly if neither source is available."""
        if tenant_id:
            return tenant_id
        # The tenant GUID lives in spark.conf 'trident.tenant.id'; the runtime context carries no
        # tenantId key. spark.synapse.workspace.tenantId is the Fabric service tenant (not the customer
        # tenant), so it must not be used here.
        try:
            from pyspark.sql import SparkSession

            _s = SparkSession.getActiveSession()
            if _s is not None:
                _v = _s.conf.get("trident.tenant.id", None)
                if _v:
                    return _v
        except Exception:
            pass
        return None

    @staticmethod
    def run_by(spark=None):
        """Who is running this, for the audit trail: the runtime context's `userName`, else its
        `userId`, else spark `current_user()`, else None. Best-effort and exception-safe — pure
        Python callers (spark=None, no notebookutils) get None.

        The `userId` layer is what makes a non-interactive run attributable. A pipeline running
        under a service principal or a workspace identity has no user, so `userName` is empty and
        the spark fallback returns a generic service account ('trusted-service-user') — the same
        value for every principal, i.e. no attribution at all on exactly the path production uses.
        `userId`, when present, is preserved as an opaque runtime-provided audit identifier. OLAF
        does not resolve it to a display name or make token-identity claims; resolution would require
        a separate identity integration, while this framework is No-Graph by design."""
        try:
            import notebookutils

            ctx = notebookutils.runtime.context
            for key in ("userName", "userId"):
                value = ctx.get(key)
                if value:
                    return value
        except Exception:
            pass
        try:
            if spark is not None:
                rows = spark.sql("SELECT current_user() AS u").collect()
                if rows and rows[0][0]:
                    return rows[0][0]
        except Exception:
            pass
        return None

    @staticmethod
    def run_id():
        """This run's identity: notebook activityId when available, uuid fallback."""
        try:
            import notebookutils

            rid = notebookutils.runtime.context.get("activityId")
            if rid:
                return rid
        except Exception:
            pass
        return str(uuid.uuid4())

    @staticmethod
    def _single_named(items, name, kind, workspace_id):
        """Resolve a displayName to exactly one workspace item, case-INSENSITIVELY, returning its
        canonical (displayName, id). 0 matches -> TargetNotFound; >1 distinct spelling case-folding to
        the same name -> TargetAmbiguous (mirrors the catalog's 'ambiguous ... differing only by case'
        table guard). The canonical displayName is what generate stamps into the mapping."""
        matches = [
            it for it in items if str(it.get("displayName", "")).lower() == str(name).lower()
        ]
        if not matches:
            raise TargetNotFound(
                f"{kind} '{name}' not found in the attached workspace ({workspace_id}) — "
                f"check onelake_security_config.lakehouse_name"
            )
        spellings = sorted({str(it.get("displayName", "")) for it in matches})
        if len(spellings) > 1:
            raise TargetAmbiguous(
                f"ambiguous {kind} names differing only by case: {spellings} — rename to disambiguate"
            )
        return matches[0].get("displayName"), matches[0].get("id")

In [ ]:
# ══ DAR ══════════════════════════════════════════════════════════════════════
class DAR:
    """Data access role (DAR) payloads: aggregate mapping grants into desired-state DAR objects,
    diff against live for plan/apply, and merge for upsert/replace push semantics."""

    @staticmethod
    def to_role(role_name, role_grants, tenant_id):
        """Aggregate one role's mapping grants -> one DAR object.
        decisionRules are grouped by identical policy (permission + rls_condition + visible_columns);
        RLS rows and CLS hidden-columns travel as per-tablePath constraints. Members are the resolved
        objectIds from the four member_*_ids columns (identical on every grant of a role, so the first
        grant suffices) — the objectType round-trips losslessly whether grants are in-memory or read back
        from the table."""
        groups, order = {}, []
        for a in role_grants:
            rls = a.get("rls_condition") or None
            raw_vis = a.get("visible_columns")
            has_cls = raw_vis is not None
            visible = Parse.list(raw_vis)
            # group key: (permission + rls + visible-column set). has_cls keeps "no CLS" (None) from
            # merging with "CLS present", even when the visible set happens to be empty-after-parse.
            pkey = (
                a.get("permission", "Read"),
                rls,
                has_cls,
                tuple(sorted(c.lower() for c in visible)),
            )
            if pkey not in groups:
                groups[pkey] = {"paths": [], "rls": [], "cls": []}
                order.append(pkey)
            g = groups[pkey]
            g["paths"].append(a["scope_path"])
            if a["scope_type"] == "Table" and rls:
                g["rls"].append(a["scope_path"])
            if a["scope_type"] == "Table" and has_cls:
                g["cls"].append((a["scope_path"], visible))

        rules = []
        for pkey in order:
            perm, rls, _has_cls, _vis = pkey
            g = groups[pkey]
            rule = {
                "effect": "Permit",
                "permission": [
                    {"attributeName": "Path", "attributeValueIncludedIn": sorted(g["paths"])},
                    {"attributeName": "Action", "attributeValueIncludedIn": [perm]},
                ],
            }
            constraints = {}
            if g["rls"]:
                constraints["rows"] = [
                    {"tablePath": p, "value": RLS.to_predicate(ScopePath.to_table(p), rls)}
                    for p in sorted(g["rls"])
                ]
            if g["cls"]:
                # visible_columns is the resolved allow-list (computed at generate, catalog-aware), so
                # apply needs no catalog. DAR CLS is an allow-list: columns not listed default to null
                # (hidden) => a column added after generate is hidden-by-default (fail-closed).
                constraints["columns"] = [
                    {
                        "tablePath": p,
                        "columnNames": cols,
                        "columnEffect": "Permit",
                        "columnAction": ["Read"],
                    }
                    for p, cols in sorted(g["cls"])
                ]
            if constraints:
                rule["constraints"] = constraints
            rules.append(rule)

        members, seen = [], set()
        first = role_grants[0]  # grouped by role: a role with no grants has no group
        for _name_col, id_col, mtype in MAPPING_MEMBER_COLUMNS:
            for value in Parse.list(first.get(id_col)):
                if value.lower() not in seen:
                    seen.add(value.lower())
                    # GATED: objectType "ManagedIdentity" is unverified against the live DAR API — a
                    # managed identity's Entra object is a servicePrincipal, so Fabric may expect
                    # "ServicePrincipal". Verify on real Fabric (docs/live-smoke-test.md) before relying on it.
                    members.append({"objectId": value, "objectType": mtype, "tenantId": tenant_id})
                    # objectId is the preloaded GUID from member_*_ids; the apply gate re-checks it
        return {
            "name": role_name,
            "kind": "Policy",
            "decisionRules": rules,
            "members": {"microsoftEntraMembers": members},
        }

    @staticmethod
    def build_desired(grants, tenant_id):
        order, groups = [], {}
        for a in grants:
            r = a["role_name"]
            if r not in groups:
                order.append(r)
            groups.setdefault(r, []).append(a)
        return [DAR.to_role(r, groups[r], tenant_id) for r in order]

    @staticmethod
    def _canon_rules(rules):
        """Canonicalize decisionRules so a server-normalized live role compares equal to the desired
        role that produced it — without this, plan is never idempotent and the pipeline's
        'changes==false -> skip' gate can never fire). The service stamps constraints.rows[].type =
        "Fabric" on store; that field is server-owned and carries no desired-state meaning, so strip it
        from BOTH sides before diffing.

        ORDER-SENSITIVE by OLAF's comparison model: this compares the serialized list positionally.
        OLAF does not claim how the service normalizes order. A role authored outside the framework
        with the same rules in a different order can therefore read as a permanent `update`; this
        is a deliberate visible drift axis rather than a pre-emptive sort that could hide a
        meaningful reordering."""
        rules = json.loads(json.dumps(rules or []))
        for dr in rules:
            for row in dr.get("constraints", {}).get("rows", []):
                row.pop("type", None)
        return json.dumps(rules, sort_keys=True)

    @staticmethod
    def _canon_members(members):
        """Canonicalize members: drop all empty member arrays from BOTH sides before comparing.
        This is an OLAF comparison normalization, not a claim about service-side storage. It allows
        a live representation with an empty array to compare to an OLAF desired representation that
        omits that array.

        BYTE-EXACT on the objectId, and the ONE place in the framework that is: everywhere else an
        Entra objectId is compared case-insensitively (see Deployment._load_member_cache). This is a
        payload-EQUALITY check between what was PUT and what the service echoes back, not an identity
        comparison, so there is no id-aware layer here in which to fold case. Byte-exact matching is
        an OLAF drift-model choice; validate the target platform's behavior before relying on its
        consequences in an operational decision.
        RESIDUAL (narrow, self-healing): drift appears only when desired and live were produced with
        DIFFERENT case — the member table's spelling edited without re-applying, or the role last
        written by the portal or another tool. One apply settles it; it is not permanent."""
        members = json.loads(json.dumps(members or {}))
        # The service drops objectType from microsoftEntraMembers on store (it re-derives it from the
        # objectId), so the PUT payload's objectType must not read as a diff — strip it from both sides.
        for m in members.get("microsoftEntraMembers", []):
            m.pop("objectType", None)
        return json.dumps({k: v for k, v in members.items() if v}, sort_keys=True)

    @staticmethod
    def diff(desired, live):
        """-> {role_name: create | update | no_change | omit}. `omit` means a role is
        live but absent from config: a request-construction candidate, not a platform deletion
        outcome. Apply records it as an omission candidate and requires post-state review."""
        desired_by_name = {r["name"]: r for r in desired}
        live_by_name = {r["name"]: r for r in live}

        plan = {}
        for name in desired_by_name:
            if name not in live_by_name:
                plan[name] = "create"
            elif DAR._canon_rules(desired_by_name[name]["decisionRules"]) != DAR._canon_rules(
                live_by_name[name].get("decisionRules")
            ) or DAR._canon_members(desired_by_name[name].get("members")) != DAR._canon_members(
                live_by_name[name].get("members")
            ):
                plan[name] = "update"
            else:
                plan[name] = "no_change"
        for name in live_by_name:
            if name not in desired_by_name:
                plan[name] = "omit"
        return plan

    @staticmethod
    def merge_upsert(live, desired):
        """apply semantics (like kubectl apply): create/update managed roles, keep everything else.
        NEVER deletes."""
        out = {r["name"]: r for r in live}
        out.update({r["name"]: r for r in desired})
        return list(out.values())

    @staticmethod
    def merge_replace(desired):
        """Return OLAF's config-derived full payload. Prior-live roles absent from it are
        omission candidates for operator review; the Preview endpoint does not document a full-set
        replacement or deletion-by-omission guarantee."""
        return desired

    @staticmethod
    def paths_and_members(role):
        """DAR role -> (paths, member objectIds) — the two axes the show pivot filters on."""
        paths = [
            p
            for dr in role.get("decisionRules", [])
            for att in dr.get("permission", [])
            if att.get("attributeName") == "Path"
            for p in att.get("attributeValueIncludedIn", [])
        ]
        members = [
            m.get("objectId") for m in role.get("members", {}).get("microsoftEntraMembers", [])
        ]
        return paths, members

    @staticmethod
    def path_permissions(role):
        """DAR role -> {scope_path: permission}. Each decisionRule's Action permission (Read/ReadWrite)
        applies to every Path in that same rule; on the rare duplicate path a later rule wins. Lets show
        surface the effective permission per (role, scope_path) grant."""
        out = {}
        for dr in role.get("decisionRules", []):
            actions = [
                p
                for att in dr.get("permission", [])
                if att.get("attributeName") == "Action"
                for p in att.get("attributeValueIncludedIn", [])
            ]
            perm = actions[0] if actions else None
            for att in dr.get("permission", []):
                if att.get("attributeName") == "Path":
                    for p in att.get("attributeValueIncludedIn", []):
                        out[p] = perm
        return out

    @staticmethod
    def strip_predicate(path, value):
        """The bare WHERE-clause condition inside a stored RLS constraint value -- the inverse of
        RLS.to_predicate's "SELECT * FROM <table> WHERE <condition>" wrapping, returning the value
        unchanged when it does not carry that prefix. ONE formula, shared by row_predicate (the
        per-path reader effective_access/who_can_access use) and Audit._live_policies (the
        whole-role pass report()/drift() use), so the two can never disagree about what a
        predicate says."""
        text = value or ""
        prefix = f"SELECT * FROM {ScopePath.to_table(path)} WHERE "
        return text[len(prefix) :] if text.startswith(prefix) else text

    @staticmethod
    def row_predicate(role, path):
        """DAR role -> the rls_condition it puts on one table path (the raw WHERE-clause
        condition, reversing the "SELECT * FROM <table> WHERE <condition>" wrapping
        RLS.to_predicate applies), or None when the role carries no RLS constraint for that
        path (open -- unfiltered rows). Returns the FIRST matching constraint: a role naming one
        path twice is a shape to_role cannot emit, and Audit._live_policies -- not this reader --
        is where that ambiguity is detected."""
        for dr in role.get("decisionRules", []):
            for row in dr.get("constraints", {}).get("rows", []):
                if row.get("tablePath") == path:
                    return DAR.strip_predicate(path, row.get("value"))
        return None

    @staticmethod
    def column_allowlist(role, path):
        """DAR role -> the CLS visible-column allow-list it puts on one table path, or None
        when the role carries no CLS constraint for that path (open -- every column visible)."""
        for dr in role.get("decisionRules", []):
            for col in dr.get("constraints", {}).get("columns", []):
                if col.get("tablePath") == path:
                    return list(col.get("columnNames") or [])
        return None

## Generate & validation.

`Generate.rows` turns short-config rows into role × scope mapping grants, applying every rule: include−exclude subtraction (exclude wins), pairing, A1/B/C1 checks, CLS whitelist/blacklist resolution, RLS+CLS column-existence, member resolution into the three typed columns, and the cross-row pass (`Generate._validate_across_rows`: C3, carve-out warning, C4, platform ceilings). `Generate.to_log_grants` explodes grants to the log grain. This is the validation heart — it blocks on any error.

In [ ]:
# ══ Generate ═════════════════════════════════════════════════════════════════
# Generate: short config rows -> validated role x scope mapping grants, with the cross-row /
# role rule checks (C1/C3/C4/C5/C8/...). Pure validation + flattening; no Spark, no side effects.
_DUP_KEY_COLS = (
    "role_name",
    "include_tables",
    "exclude_tables",
    "include_folders",
    "exclude_folders",
    "permission",
    "rls_condition",
    "include_columns",
    "exclude_columns",
    "include_group_names",
    "exclude_group_names",
    "include_user_names",
    "exclude_user_names",
    "include_sp_names",
    "exclude_sp_names",
    "include_mi_names",
    "exclude_mi_names",
)


class Generate:
    """Short config rows → validated role×scope mapping grants + cross-row/role rule checks."""

    @staticmethod
    def _dup_key(row):
        return json.dumps({k: str(row.get(k)) for k in _DUP_KEY_COLS}, sort_keys=True)

    @staticmethod
    def _scope_paths(prow):
        """All scope paths a processed row grants: table paths (/Tables/schema/table) + folder paths.
        Shared by the role-wide cross-checks below, which reason over a role's full set of paths."""
        return {ScopePath.table(t) for t, _rls, _hidden in prow["tables"]} | set(prow["folders"])

    @staticmethod
    def _scope_pair(inc_raw, exc_raw, resolver, rid, kind, errors):
        """Shared include/exclude subtract for tables or folders.
        Returns (effective_set, excluded_hits, subtracted_count). Records pairing/0-match/empty
        errors — BOTH sides now, so a scope that resolves to nothing never passes as a warning."""
        if exc_raw and not inc_raw:
            errors.append(
                f"{rid}: exclude_{kind} without include_{kind} has nothing to subtract from "
                f"(there is no ALL keyword) (pairing rule)"
            )
            return set(), set(), 0
        inc_hits, had_err = set(), False
        for e in inc_raw:
            try:
                inc_hits.update(resolver(e))
            except ValidationError as ex:
                errors.append(f"{rid}: {ex}")
                had_err = True
        exc_hits = set()
        for e in exc_raw:
            try:
                exc_hits.update(resolver(e))
            except ZeroMatchError as ex:
                # ONLY a dead scope earns the extra clause. Every other ValidationError reaching
                # here — an unresolvable target above all — is not a statement about this row's
                # exclusion, and has to read byte-identically to the include side so a reader
                # sees one environment problem rather than two config ones.
                errors.append(
                    f"{rid}: {ex} — the exclusion removed nothing, so everything it was meant "
                    f"to take out of this row's {kind} is still granted"
                )
                had_err = True
            except ValidationError as ex:
                errors.append(f"{rid}: {ex}")
                had_err = True
        effective = inc_hits - exc_hits
        if inc_raw and not had_err and inc_hits and not effective:
            errors.append(f"{rid}: row grants no {kind} after exclusion — empty effective set")
        return effective, exc_hits, len(inc_hits & exc_hits)

    @staticmethod
    def _members(row, rid, errors):
        """Resolve the config member columns for one row into the typed member-NAME columns.
        Returns (members_by_col {member_group_names/-user_names/-sp_names/-mi_names: [names]}, member_signature).
        Enforces: at least one include, exclude<-include pairing, non-empty after subtract, and --
        as a fail-closed backstop for member-wildcard expansion -- that no unexpanded wildcard reaches a grant.
        Values are display names here; generate resolves them to objectIds after all rows are built."""
        by_col = {name_col: [] for name_col, _id_col, _mtype in MAPPING_MEMBER_COLUMNS}
        include_total = 0
        for inc_col, exc_col, mtype in MEMBER_KINDS:
            # within-cell case-collision: Parse.list dedupes case-insensitively and would silently drop a
            # case-variant spelling, but two names differing only by case are DIFFERENT principals — flag it
            # (mirrors Member.resolve_ids' cross-grant guard) rather than silently under-granting.
            for _col in (inc_col, exc_col):
                _spell = {}
                for _it in (x.strip() for x in str(row.get(_col) or "").split(LIST_SEP)):
                    if _it:
                        _spell.setdefault(_it.lower(), set()).add(_it)
                for _lo, _variants in _spell.items():
                    if len(_variants) > 1:
                        errors.append(
                            f"{rid}: {_col} names differing only by case: {sorted(_variants)} "
                            f"— different principals, rename to disambiguate"
                        )
            inc, exc = Parse.list(row.get(inc_col)), Parse.list(row.get(exc_col))
            globbed = {v for v in inc + exc if "*" in v or "?" in v}
            for v in sorted(globbed):
                errors.append(
                    f"{rid}: wildcards not allowed in member values: '{v}' "
                    f"— prefix it with '{GLOB_PREFIX}' to expand it from onelake_security_member, "
                    f"or rename the principal (a member name cannot contain '*' or '?')"
                )
            # A REJECTED value is not a member, so it must not reach by_col. Left in, it rides into
            # the grant and the No-Graph gate then adds "add 'sg-*' (with its objectId)" on top of
            # the rejection that already named the real problem, with advice that is besides
            # nonsense for a glob. Reachable only when expand_wildcards deliberately left the cell
            # alone (an author case-disagreement); every other path expands or drops the pattern.
            inc = [v for v in inc if v not in globbed]
            exc = [v for v in exc if v not in globbed]
            if exc and not inc:
                errors.append(
                    f"{rid}: {exc_col} without {inc_col} has nothing to subtract from (pairing rule)"
                )
            include_total += len(inc)
            exc_lower = {e.lower() for e in exc}
            by_col[_MTYPE_TO_NAME_COL[mtype]] = [m for m in inc if m.lower() not in exc_lower]
        if include_total == 0:
            errors.append(
                f"{rid}: every row must declare at least one member "
                f"(include_group_names/include_user_names/include_sp_names/include_mi_names) (rule C1)"
            )
        elif not any(by_col.values()):
            errors.append(
                f"{rid}: member list is empty after exclusion (rule C1) — "
                f"unify the lists or split the roles"
            )
        signature = tuple(
            (col, tuple(sorted(v.lower() for v in Parse.list(row.get(col)))))
            for col in MEMBER_ALL_COLUMNS
        )
        return by_col, signature

    @staticmethod
    def rows(rows, canon):
        """Short-config rows -> (mapping_grants, errors, warnings, summary).

        Grant grain = one row per (role x scope): role_name, scope_path, scope_type, permission,
        rls_condition, visible_columns + the four typed member columns (member_group_names, member_user_names,
        member_sp_names — ';' lists, identical across a role's rows) so the DAR objectType
        round-trips through the mapping table. Rule codes (A1/A2/B*/C1/C3/...) refer to
        docs/architecture.md. Within a row: include first, exclude second, exclude always wins."""
        errors, warnings = [], []
        seen_rows = set()
        processed, role_order, role_members = [], [], {}

        for i, row in enumerate(rows):
            rid = f"row {i + 1} ({row.get('role_name', '?')})"
            key = Generate._dup_key(row)
            if key in seen_rows:
                warnings.append(f"{rid}: exact duplicate row — skipped")
                continue
            seen_rows.add(key)

            name = str(row.get("role_name") or "")
            if not ROLE_NAME_RE.match(name) or len(name) > 124:
                errors.append(
                    f"{rid}: role_name must be alphanumeric, start with a letter, max 124 chars (rule B1)"
                )
            if not str(row.get("lakehouse_name") or "").strip():
                errors.append(f"{rid}: lakehouse_name is required — name the target lakehouse")

            inc_t, inc_f = (
                Parse.list(row.get("include_tables")),
                Parse.list(row.get("include_folders")),
            )
            eff_tables, exc_tables, sub_t = Generate._scope_pair(
                inc_t,
                Parse.list(row.get("exclude_tables")),
                lambda e: Catalog.resolve_tables(e, canon),
                rid,
                "tables",
                errors,
            )
            eff_folders, exc_folders, sub_f = Generate._scope_pair(
                inc_f,
                Parse.list(row.get("exclude_folders")),
                lambda e: Catalog.resolve_folders(e, canon["folders"]),
                rid,
                "folders",
                errors,
            )

            if not inc_t and not inc_f:
                errors.append(
                    f"{rid}: a row must grant at least one table or folder — "
                    f"an exclude alone is not a grant (rule A1)"
                )

            # subtree carve warning (E10): an excluded folder still exposed by a granted ancestor
            for x in exc_folders:
                for p in eff_folders:
                    if x != p and x.startswith(p.rstrip("/") + "/"):
                        warnings.append(
                            f"{rid}: excluded folder '{x}' sits under granted '{p}' — OneLake "
                            f"inherits subtree-wide, so exclude cannot carve out a subtree"
                        )

            raw_perm = str(row.get("permission") or "Read").strip()
            perm = PERMISSIONS.get(raw_perm.lower())
            if perm is None:
                # B4 — the permission column is the DAR Action enum. (B3 is deliberately NOT
                # evaluated on a B4-failed row: an unknown token's ReadWrite-semantics are
                # unknowable; case-only variants normalize above and DO reach B3.)
                # An unknown value used
                # to pass every check, land in the mapping, and die at the platform
                # mid-apply; normalizing through PERMISSIONS also canonicalizes case, so
                # rule B3 below compares the canonical token instead of a spelling.
                errors.append(
                    f"{rid}: permission '{raw_perm}' is not a supported value — use Read "
                    f"or ReadWrite (the DAR Action enum, rule B4)"
                )
                perm = raw_perm  # carried for display only — the error above blocks the config
            rls = row.get("rls_condition") or None
            if rls:
                # B2 — RLS needs a table: an rls_condition with no include_tables is rejected
                if not eff_tables:
                    errors.append(
                        f"{rid}: rls_condition applies to tables only — this row grants no tables (rule B2)"
                    )
                # C7 — RLS length: an rls_condition over MAX_RLS_CONDITION_CHARS chars is rejected
                if len(rls) > MAX_RLS_CONDITION_CHARS:
                    errors.append(
                        f"{rid}: rls_condition is {len(rls)} chars, over the "
                        f"{MAX_RLS_CONDITION_CHARS}-char platform limit — shorten the predicate (rule C7)"
                    )
                # C9 — unsupported RLS operator: rejects operators/keywords outside the OneLake-supported subset
                unsupported = RLS.unsupported_tokens(rls)
                if unsupported:
                    errors.append(
                        f"{rid}: rls_condition uses unsupported operator(s)/keyword(s) {unsupported} — "
                        f"OneLake RLS supports only = <> > >= < <= IN NOT AND OR IS BLANK NULL TRUE FALSE "
                        f"(plus columns, literals, parentheses); unsupported syntax fails at query time "
                        f"(rule C9)"
                    )
                # C14 — bare TRUE/FALSE as a value: OneLake parses an unquoted value as a column
                # name, so the predicate is refused at apply. Quoting it is the whole fix.
                bare_bools = RLS.bare_boolean_values(rls)
                if bare_bools:
                    errors.append(
                        f"{rid}: rls_condition compares against bare {bare_bools} — OneLake reads an "
                        f"unquoted value as a COLUMN NAME, so it refuses the predicate "
                        f"(InvalidRLSPredicate) instead of matching the boolean. Quote it, e.g. "
                        f"= 'true' — `IS TRUE` is refused too. Numbers are the exception and "
                        f"need no quotes (rule C14)"
                    )
                # C10 — RLS complexity: warns when AND/OR connectives exceed the heuristic
                connectives = RLS.connective_count(rls)
                if connectives > RLS_COMPLEXITY_WARN_CONNECTIVES:
                    warnings.append(
                        f"{rid}: rls_condition has {connectives} AND/OR connectives, over the complexity "
                        f"heuristic of {RLS_COMPLEXITY_WARN_CONNECTIVES} — highly complex roles can fail "
                        f"the security sync; consider splitting the role (rule C10)"
                    )

            inc_cols, exc_cols = (
                Parse.list(row.get("include_columns")),
                Parse.list(row.get("exclude_columns")),
            )
            if inc_cols and exc_cols:
                errors.append(
                    f"{rid}: include_columns and exclude_columns are both set — "
                    f"pick one CLS mode per row (whitelist or blacklist)"
                )
            cls_mode = "whitelist" if inc_cols else ("blacklist" if exc_cols else None)
            if cls_mode and not eff_tables:
                errors.append(
                    f"{rid}: column security applies to tables only — this row grants no tables"
                )

            if perm == "ReadWrite" and (rls or cls_mode):
                errors.append(
                    f"{rid}: a ReadWrite row cannot carry RLS/CLS — the platform forbids it (rule B3)"
                )

            # C13 — OLAF requires an RLS predicate to reference at least one column. This is an
            # authoring guard for constant conditions (`1=0`, `1 = 1`), not a statement about
            # platform rejection or access outcome. The neighbouring checks cover columns and case.
            if rls and not RLS.names_any_identifier(rls):
                errors.append(
                    f"{rid}: RLS condition references no column; OLAF blocks this constant "
                    f"predicate before request submission: '{rls}' (rule C13)"
                )

            # column-existence: every column named by rls or the CLS pair must exist (case-insensitively)
            # in every effective table. Uses the first-seen, case-insensitive reference set.
            referenced = list(dict.fromkeys(RLS.referenced_columns(rls) + inc_cols + exc_cols))
            if referenced and eff_tables:
                missing = {}
                for col in referenced:
                    for t in sorted(eff_tables):
                        tcols = [c.lower() for c in canon.get("columns", {}).get(t.lower(), [])]
                        if tcols and col.lower() not in tcols:
                            missing.setdefault(col, []).append(t)
                for col, tabs in sorted(missing.items()):
                    errors.append(
                        f"{rid}: column '{col}' referenced by RLS/CLS is missing in: {', '.join(tabs)}"
                    )
                # C11 — column case must match Delta schema: referenced column case must equal the Delta schema spelling
                # Column case-exactness: scan EVERY raw occurrence by its original spelling (all RLS refs
                # WITH duplicates + the CLS lists), not the case-folded `referenced` set — so a later
                # wrong-case repeat of an already-referenced column is still caught. An RLS mismatch
                # is an OLAF authoring error; this validation does not infer service enforcement or
                # reader outcome. RLS wins if a spelling is in both.
                rls_spellings = RLS.referenced_columns_all(rls)
                rls_seen = set(rls_spellings)
                case_mismatch = {}
                for col in dict.fromkeys(rls_spellings + inc_cols + exc_cols):
                    src = "rls" if col in rls_seen else "cls"
                    for t in sorted(eff_tables):
                        by_lower = {
                            c.lower(): c for c in canon.get("columns", {}).get(t.lower(), [])
                        }
                        actual = by_lower.get(col.lower())
                        if actual is not None and actual != col:
                            case_mismatch.setdefault((src, col), {})[t] = actual
                for (src, col), per_table in sorted(case_mismatch.items()):
                    spelled = ", ".join(f"{t} → '{per_table[t]}'" for t in sorted(per_table))
                    if src == "rls":
                        errors.append(
                            f"{rid}: RLS column '{col}' does not match the Delta schema's exact case "
                            f"(expected per table: {spelled}) — OLAF blocks this authoring mismatch "
                            f"before request submission; do not infer service enforcement from this "
                            f"validation. Write it exactly as the schema spells it (rule C11)"
                        )
                    else:
                        errors.append(
                            f"{rid}: CLS column '{col}' does not match the Delta schema's exact case "
                            f"(expected per table: {spelled}) — write it exactly as the schema spells it "
                            f"so the CLS column list names the real column (rule C11)"
                        )

            members_by_col, member_sig = Generate._members(row, rid, errors)

            # C1 — same members per role: all rows of a role must resolve to the same member set
            # Every row of a role must declare identical member columns (normalized)
            prev = role_members.get(name)
            if prev is None:
                role_order.append(name)
                role_members[name] = (member_sig, members_by_col)
            elif prev[0] != member_sig:
                errors.append(
                    f"role '{name}': member columns must be identical on every row of a role — "
                    f"unify the lists or split the roles (rule C1)"
                )

            tables = []
            for t in sorted(eff_tables):
                _vis = CLS.visible_for_table(t, cls_mode, inc_cols, exc_cols, canon)
                if _vis is not None and len(_vis) == 0:
                    errors.append(
                        f"{rid}: CLS on '{t}' would leave 0 visible columns — "
                        f"deny the table (omit it) instead of hiding every column"
                    )
                tables.append((t, rls, _vis))
            processed.append(
                {
                    "role": name,
                    "perm": perm,
                    "tables": tables,
                    "folders": sorted(eff_folders),
                    "excluded_paths": {ScopePath.table(t) for t in exc_tables} | set(exc_folders),
                    "subtracted": sub_t + sub_f,
                    "row_index": i,  # original config-row index (skips excluded)
                }
            )

        cross_errors, cross_warnings = Generate._validate_across_rows(processed, role_members)
        errors += cross_errors
        warnings += cross_warnings

        grants = Generate._build_grants(processed, role_order, role_members)
        summary = Generate._summary(processed, role_order, warnings)
        return grants, errors, warnings, summary

    @staticmethod
    def _validate_across_rows(processed, role_members):
        """Role-wide checks: C3 (one policy per role x table), carve-out warning, C4 over-exposure,
        C5 (no cross-role RLS x CLS mix per member), and the platform ceilings (paths/members/roles)
        that fail at generate."""
        errors, warnings = [], []

        # A role_name over the 124-char ceiling hard-fails the SQL analytics-endpoint security sync
        # (no workaround) — a per-role check distinct from B1's per-row name-format rule.
        for role in role_members:
            if len(role) > MAX_ROLE_NAME_CHARS:
                errors.append(
                    f"role '{role}': role_name is {len(role)} chars, over the {MAX_ROLE_NAME_CHARS}-char "
                    f"limit — the SQL analytics-endpoint security sync hard-fails for a longer name, with "
                    f"no workaround (rule C6)"
                )
            # C12 — role_name charset/start (Fabric's "Create a role" naming contract): letter-first,
            # alphanumeric only — no underscore, space, or other special char. Distinct nuance from
            # C6 above: Fabric's role-name field itself allows up to 128 chars, looser than the
            # 124-char ceiling C6 enforces (the tighter SQL-endpoint-sync limit) — so C12 checks
            # charset/start only, C6 stays the sole length check. Per-role coalesced counterpart to
            # B1's per-row charset+length check, the same way C6 coalesces B1's length half.
            if not ROLE_NAME_RE.match(role):
                errors.append(
                    f"role '{role}': role_name must start with a letter and contain only letters/digits "
                    f"(no underscore, space, or special char) — the SQL analytics-endpoint security sync "
                    f"mirrors the role as schema object OLS_<role_name>, which fails to sync when the "
                    f"name breaks Fabric's 'Create a role' naming contract (rule C12)"
                )

        # C3 — one policy per table per role: a table may carry only one RLS/CLS policy per role
        # Same scope must not get two different policies (permission + rls + hidden) from one role
        policy_by_scope = {}
        for prow in processed:
            role = prow["role"]
            scopes = [
                (ScopePath.table(t), (prow["perm"], rls, hidden))
                for t, rls, hidden in prow["tables"]
            ]
            scopes += [(f, (prow["perm"], None, ())) for f in prow["folders"]]
            for path, policy in scopes:
                key = (role, path)
                if key in policy_by_scope and policy_by_scope[key] != policy:
                    errors.append(
                        f"role '{role}' scope '{path}': two different policies from one role — "
                        f"same table cannot carry two rls_conditions or CLS sets (rule C3)"
                    )
                policy_by_scope[key] = policy

        # carve-out / accidental-re-grant warning: one row excludes what another row of the role includes.
        # Row numbers are the ORIGINAL config-row numbers (row_index), not processed-list positions, so
        # they stay correct after a duplicate row is skipped.
        by_role = {}
        for prow in processed:
            by_role.setdefault(prow["role"], []).append(prow)
        for role, rows in by_role.items():
            for arow in rows:
                included = Generate._scope_paths(arow)
                for brow in rows:
                    if arow is brow:
                        continue
                    overlap = sorted(brow["excluded_paths"] & included)
                    if overlap:
                        warnings.append(
                            f"role '{role}': {', '.join(overlap)} excluded in row "
                            f"{brow['row_index'] + 1} but included in row "
                            f"{arow['row_index'] + 1} — carve-out or accident?"
                        )

        # C4/C5 group members on the lowercased value, but their messages must echo the AUTHOR'S
        # original spelling (rule C5/C4 message fidelity) — map each lowercased key to its
        # first-seen original config casing (deterministic across the role/row iteration order).
        member_orig = {}
        for _role, (_sig, _by_col) in role_members.items():
            for _vals in _by_col.values():
                for _v in _vals:
                    member_orig.setdefault(_v.lower(), _v)

        # A member reaching one table through several roles (union over-exposure)
        member_scopes = {}
        for role, (sig, by_col) in role_members.items():
            member_vals = {v.lower() for vals in by_col.values() for v in vals}
            role_scopes = set()
            for prow in processed:
                if prow["role"] == role:
                    role_scopes |= Generate._scope_paths(prow)
            for m in member_vals:
                for s in role_scopes:
                    member_scopes.setdefault((m, s), set()).add(role)
        # sorted(): `member_scopes` is built by iterating SETS, so its insertion order follows
        # set iteration order, and CPython randomizes string hashing per process — the same code
        # against the same config emitted these warnings in a different ORDER on every run.
        # Never a correctness bug -- the multiset and every message were right -- but it made any
        # diff-based review of OLAF output noisy, which is the false signal that trains a reviewer
        # to skim a differential. Reproducible with the repo's own fixtures: run the C4 block over
        # a config of M members x S shared tables and the emitted order is the PRODUCT of two set
        # iterations, so it takes M!*S! values -- measured 238 distinct orders over 300 hash seeds
        # at 5x3. The C8 loop below already sorted for the same reason; both are pinned by
        # test_c{4,8}_warning_order_is_deterministic.
        for (m, s), roles in sorted(member_scopes.items()):
            if len(roles) > 1:
                warnings.append(
                    f"member '{member_orig[m]}' reaches '{s}' via several roles {sorted(roles)} — "
                    f"roles union, review over-exposure (rule C4)"
                )

        # C8 — restricted + unrestricted: an unrestricted role nullifies another role's RLS/CLS on the same table
        # Narrower than a plain multi-role reach: a member reaching the SAME table via a role that restricts it (RLS or
        # CLS on that table) AND another role granting the same table with NO RLS and NO CLS — the union
        # nullifies the restriction ("most permissive role wins"). C4 warns on ANY multi-role table
        # reach; C8 fires only when at least one reaching role is genuinely unrestricted on that table
        # and at least one other restricts it. Reuses member_scopes + member_orig from C4 (the
        # per-member/per-scope structure); adds only the per-(role, table) restriction status.
        role_table_restricted = {}  # (role, table_path) -> True when the role puts RLS/CLS on the table
        table_scope_paths = set()
        for prow in processed:
            for t, rls, hidden in prow["tables"]:
                path = ScopePath.table(t)
                table_scope_paths.add(path)
                if rls or hidden:
                    role_table_restricted[(prow["role"], path)] = True
        for (m, s), roles in sorted(member_scopes.items()):
            if s not in table_scope_paths or len(roles) < 2:
                continue
            restricted_roles = sorted(r for r in roles if role_table_restricted.get((r, s)))
            open_roles = sorted(r for r in roles if not role_table_restricted.get((r, s)))
            if restricted_roles and open_roles:
                warnings.append(
                    f"member '{member_orig[m]}' reaches table '{s}' via restricted role(s) "
                    f"{restricted_roles} (RLS/CLS) and unrestricted role(s) {open_roles} (no RLS, no "
                    f"CLS) — the union nullifies the restriction, most permissive role wins (rule C8)"
                )

        # OneLake breaks queries for a member who is in an RLS-bearing role AND a DIFFERENT
        # CLS-bearing role — the two policies must live in ONE role. Fail-closed (error), not a
        # review: any cross-role RLS x CLS mix for one member is unsupported (no same-table qualifier).
        role_has_rls, role_has_cls = {}, {}
        for prow in processed:
            role = prow["role"]
            role_has_rls[role] = role_has_rls.get(role, False) or any(
                rls for _t, rls, _hidden in prow["tables"]
            )
            role_has_cls[role] = role_has_cls.get(role, False) or any(
                hidden for _t, _rls, hidden in prow["tables"]
            )
        member_roles = {}
        for role, (sig, by_col) in role_members.items():
            for m in {v.lower() for vals in by_col.values() for v in vals}:
                member_roles.setdefault(m, set()).add(role)
        for m, roles in sorted(member_roles.items()):
            rls_roles = sorted(r for r in roles if role_has_rls.get(r))
            cls_roles = sorted(r for r in roles if role_has_cls.get(r))
            if any(r_rls != r_cls for r_rls in rls_roles for r_cls in cls_roles):
                errors.append(
                    f"member '{member_orig[m]}' is in role(s) {rls_roles} carrying RLS and role(s) "
                    f"{cls_roles} carrying CLS — OneLake does not support mixing RLS and CLS "
                    f"across roles for one user (queries fail); combine both policies into a "
                    f"single role or remove the member from one side (rule C5)"
                )

        # platform ceilings per role
        role_paths, role_member_counts = {}, {}
        for prow in processed:
            role_paths.setdefault(prow["role"], set())
            role_paths[prow["role"]] |= Generate._scope_paths(prow)
        for role, (sig, by_col) in role_members.items():
            role_member_counts[role] = sum(len(vals) for vals in by_col.values())
        for role, paths in role_paths.items():
            if len(paths) > MAX_PATHS_PER_ROLE:
                errors.append(
                    f"role '{role}': {len(paths)} paths exceed the {MAX_PATHS_PER_ROLE}-per-role "
                    f"platform limit — fails at generate, not at apply (shard the role, see RUNBOOK)"
                )
            elif len(paths) >= MAX_PATHS_PER_ROLE * WARN_FRACTION:
                warnings.append(
                    f"role '{role}': {len(paths)} paths nearing the {MAX_PATHS_PER_ROLE}-per-role limit"
                )
        for role, n in role_member_counts.items():
            if n > MAX_MEMBERS_PER_ROLE:
                errors.append(
                    f"role '{role}': {n} members exceed the {MAX_MEMBERS_PER_ROLE}-per-role platform limit"
                )
            elif n >= MAX_MEMBERS_PER_ROLE * WARN_FRACTION:
                warnings.append(
                    f"role '{role}': {n} members nearing the {MAX_MEMBERS_PER_ROLE}-per-role limit"
                )
        n_roles = len(role_members)
        if n_roles > MAX_ROLES_PER_ITEM:
            # The ceiling's own comment says "fail at generate, not at apply" — and the two
            # sibling ceilings above both do; this one only warned, so an over-limit config
            # sailed into a plan and died at the Fabric API mid-apply.
            errors.append(
                f"{n_roles} roles exceed the {MAX_ROLES_PER_ITEM}-per-item platform limit — "
                f"fails at generate, not at apply (raisable to 1,000 via Azure Support; if "
                f"your tenant's limit was raised, adjust MAX_ROLES_PER_ITEM)"
            )
        elif n_roles >= MAX_ROLES_PER_ITEM * WARN_FRACTION:
            warnings.append(
                f"{n_roles} roles nearing the {MAX_ROLES_PER_ITEM}-per-item platform limit "
                f"(raisable to 1,000 via Azure Support)"
            )
        return errors, warnings

    @staticmethod
    def _build_grants(processed, role_order, role_members):
        """Flatten processed rows into role x scope grants, deduped by (role, scope_path).
        Order = row order, tables/folders sorted within each row; roles keep first-appearance order.
        Each grant carries its four typed member-name columns (';' lists, None when the type is unused —
        the matching member_*_ids columns are populated later by generate's resolve step); generate
        stamps the attached workspace/lakehouse labels onto every grant afterwards."""
        grants, seen = [], set()
        ordered = sorted(processed, key=lambda p: role_order.index(p["role"]))
        for prow in ordered:
            role = prow["role"]
            _sig, by_col = role_members[role]
            member_cols = {
                col: (LIST_SEP.join(vals) if vals else None) for col, vals in by_col.items()
            }
            for t, rls, visible in prow["tables"]:
                path = ScopePath.table(t)
                if (role, path) in seen:
                    continue
                seen.add((role, path))
                grants.append(
                    {
                        "role_name": role,
                        "scope_path": path,
                        "scope_type": "Table",
                        "permission": prow["perm"],
                        "rls_condition": rls,
                        "visible_columns": LIST_SEP.join(visible) if visible else None,
                        **member_cols,
                    }
                )
            for f in prow["folders"]:
                if (role, f) in seen:
                    continue
                seen.add((role, f))
                grants.append(
                    {
                        "role_name": role,
                        "scope_path": f,
                        "scope_type": "Folder",
                        "permission": prow["perm"],
                        "rls_condition": None,
                        "visible_columns": None,
                        **member_cols,
                    }
                )
        return grants

    @staticmethod
    def _summary(processed, role_order, warnings):
        """Per-role tally the runtime surfaces: scopes granted (after subtraction), scopes subtracted,
        warnings referencing the role."""
        summary = {}
        for role in role_order:
            included = sum(
                len(p["tables"]) + len(p["folders"]) for p in processed if p["role"] == role
            )
            excluded = sum(p["subtracted"] for p in processed if p["role"] == role)
            warns = sum(1 for w in warnings if f"'{role}'" in w or f"({role})" in w)
            summary[role] = {"included": included, "excluded": excluded, "warnings": warns}
        return summary

    @staticmethod
    def to_log_grants(grants):
        """Mapping grants -> single-valued (role x scope x member) grains for the audit log (single values only).
        Member type comes from which typed mapping column the value sits in; each grain carries the
        member's display name (member_name) and its resolved objectId (member_id, positionally aligned)."""
        out = []
        for a in grants:
            members = []
            for name_col, id_col, mtype in MAPPING_MEMBER_COLUMNS:
                # Split WITHOUT deduping so names<->ids stay 1:1 aligned: two distinct display names
                # can resolve to the SAME objectId (a user's UPN + mail alias), and Parse.list's
                # case-insensitive dedup would collapse the id list and silently drop a name from the log.
                names = [x.strip() for x in str(a.get(name_col) or "").split(LIST_SEP) if x.strip()]
                ids = [x.strip() for x in str(a.get(id_col) or "").split(LIST_SEP) if x.strip()]
                for name, member_id in zip(names, ids):
                    members.append((name, member_id, mtype))
            for name, member_id, mtype in members or [(None, None, None)]:
                out.append(
                    {
                        "role_name": a["role_name"],
                        "scope_path": a["scope_path"],
                        "scope_type": a["scope_type"],
                        "member_name": name,
                        "member_id": member_id,
                        "member_type": mtype,
                    }
                )
        return out

## Classes

The four runtime classes in dependency order: `FabricClient` (Fabric REST for DAR) → `Log`
(writes `onelake_security_log`) → `Deployment` (one public method per mode; shared steps are
private) → `Audit` (read-only trace / audit queries). Definitions are pure Python — no Spark or
`notebookutils` at import time — so the CI extract-and-exec suite loads them directly.

### `FabricClient` — Fabric REST for OneLake data-access roles (DAR).

Wraps the DAR endpoints for **one** workspace/lakehouse target. Acquires the auth token once at construction from the **ambient identity** (the interactive user, or the pipeline connection's Workspace Identity / SP) and holds the auth header + roles URL so callers never thread tokens or ids through every call. Exposes `list_roles` (paged GET) and `put_roles` (bulk full-set PUT — the only write surface OLAF uses, and Microsoft labels that endpoint **Preview**, as it does the granular per-role endpoints OLAF does not call; see `put_roles`).

In [ ]:
# ══ FabricClient ═════════════════════════════════════════════════════════════
# Fabric REST for OneLake data-access roles (DAR) on one workspace/lakehouse target; token acquired once.
class FabricClient:
    """Fabric REST calls for OneLake data access roles (DAR) on ONE workspace/lakehouse target.
    Holds the auth header once so callers never thread tokens/ids through every call. Uses
    notebookutils.credentials.getToken("pbi"), the Fabric/Power BI audience selector for
    these REST calls. Runs as the
    AMBIENT identity — the interactive user (dev), or the pipeline connection's Workspace Identity / SP
    (prod). That identity must hold workspace Member for apply, Contributor for the read-only modes."""

    BASE = "https://api.fabric.microsoft.com/v1"
    # (connect, read) seconds on EVERY Fabric REST call. Without one, requests waits on the OS TCP
    # timeout -- minutes, and per page on a paginated call. The worst place that lands is the
    # failure path: the post-push re-read runs precisely when the network is what broke.
    TIMEOUT = (10, 60)
    # The re-read _record_push_failure does is best-effort forensics decorating an error the
    # operator is already waiting on, and a failed re-read is handled (live_roles_after: null).
    # It gets a tighter bound than a call whose result the run depends on.
    FAILURE_READ_TIMEOUT = (5, 15)
    # A per-request timeout bounds ONE page, not a long chain of them, so the whole paginated loop
    # carries its own ceiling. It bounds retry WAITS too — see _send_with_retry.
    #
    # It is a SLEEP bound, not a wall-clock one: the check runs between pages and before a retry
    # wait, never mid-request, so a page still costs up to RETRY_ATTEMPTS x TIMEOUT of round-trip
    # on top of it. Worst case per page is therefore ~3 x the read timeout plus the backoff, not
    # one. Bounding the round-trips as well would mean shrinking the per-request timeout for a
    # retried call, which is the opposite of what a retry is for.
    PAGED_BUDGET_SECONDS = 120

    # A Fabric REST call can answer with a status the SAME request survives: a throttle (429) or a
    # gateway/availability blip (502/503/504). Retried a bounded number of times those cost
    # seconds; unretried they fail a deploy that was never actually wrong. NOTHING else is
    # retried — every other 4xx is this caller's own request being malformed, unauthorized or
    # pointed at the wrong target, and repeating it changes nothing but the clock.
    # 412 (the If-Match precondition) is deliberately absent as well: re-sending the same
    # stale ETag cannot succeed, so a retry only burns attempts — see put_roles.
    RETRY_STATUSES = frozenset({429, 502, 503, 504})
    # ATTEMPTS, not retries: 3 attempts is the original call plus 2 more.
    RETRY_ATTEMPTS = 3
    # Exponential backoff between attempts, in seconds — one entry per GAP, so it is
    # RETRY_ATTEMPTS - 1 long. A numeric Retry-After from the service WINS over it: the service
    # knows its own throttle window, and guessing shorter is how one 429 becomes three.
    RETRY_BACKOFF = (1.0, 2.0)
    # ...but only up to here. Retry-After is a number the SERVICE chooses, and a throttled tenant
    # can legitimately answer with minutes or hours — which the PUT path would sleep through
    # verbatim, since a write has no paginated budget to trip. Capped, the worst case a caller can
    # be made to wait is bounded by attempts x this, and the request still fails honestly
    # afterwards instead of hanging a pipeline nobody is watching.
    RETRY_WAIT_CAP = 30.0

    def __init__(self, workspace_id, item_id):
        self.workspace_id, self.item_id = workspace_id, item_id
        import notebookutils

        token = notebookutils.credentials.getToken("pbi")
        self._headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
        self._roles_url = f"{self.BASE}/workspaces/{workspace_id}/items/{item_id}/dataAccessRoles"
        # The collection ETag captured off the LAST list_roles read (None before any read,
        # or when the service sent none) — the If-Match token for the next real PUT; see
        # put_roles.
        self.roles_etag = None

    def list_roles(
        self, timeout: tuple | None = None, attempts: int = RETRY_ATTEMPTS
    ) -> list[dict]:
        """Every DAR role on this target, following continuationUri. `timeout` overrides TIMEOUT for
        a caller that needs a tighter bound, and `attempts` its share of the retry budget; see
        list_roles_quick, which tightens both. Also captures the collection ETag off the
        FIRST page's response headers into self.roles_etag (reset to None first, so a
        header-less response never leaves a stale token) — the If-Match value put_roles
        sends on the next real PUT, closing the read→PUT race the drift gate cannot see."""
        self.roles_etag = None
        return self._get_paged(
            self._roles_url, timeout=timeout, attempts=attempts, capture_etag=True
        )

    def list_roles_quick(self) -> list[dict]:
        """list_roles under FAILURE_READ_TIMEOUT and a SINGLE attempt -- the best-effort re-read on
        the push-failure path, which must never outlive the error report it is decorating.

        Both halves of that bound are load-bearing, and the retry layer would quietly undo the
        second: three attempts with backoff turns a 15s read into ~48s, spent inside the exact
        incident window the tighter timeout exists to protect. Nothing here is worth waiting for
        either -- the result decorates an error the operator is already holding, and a failed
        re-read is already handled (live_roles_after: null). So this caller opts OUT of the retry
        that every other call wants: it is the one read whose VALUE is lower than its LATENCY."""
        return self.list_roles(timeout=self.FAILURE_READ_TIMEOUT, attempts=1)

    @classmethod
    def _retry_wait(cls, resp, attempt, attempts):
        """Seconds to wait before the attempt after `attempt`, or None when there must not be one.

        None means "this response is final": either the status is not one of the transient
        RETRY_STATUSES, or the caller's `attempts` budget is spent. Anything else returns a wait.
        `attempts` is the caller's, not the class's — list_roles_quick passes 1 to opt out of
        retrying entirely. It is CLAMPED to [1, RETRY_ATTEMPTS] here rather than trusted:
        the RETRY_BACKOFF index below is only in range for attempts <= RETRY_ATTEMPTS, and a
        caller passing more would IndexError on the last wait — a crash inside the retry
        layer, replacing the response it existed to deliver. Zero (or less) means the same
        thing 1 means — the send has already happened when this is consulted, so "no retry
        budget" and "one attempt" are the same contract, now explicit.

        `Retry-After` accepts the RFC 9110 delta-seconds and HTTP-date forms. Dates are converted
        to a UTC delay and floored at zero; malformed values fall back to the fixed backoff.

        Every wait, from either source, is then capped at RETRY_WAIT_CAP. The header is a number
        the service picks, and honouring a large one verbatim is how a bounded retry becomes an
        unbounded wait — with nothing but the attempt count left holding it on the PUT path, which
        has no paginated budget to trip.
        """
        attempts = max(1, min(attempts, cls.RETRY_ATTEMPTS))
        if attempt >= attempts - 1 or resp.status_code not in cls.RETRY_STATUSES:
            return None
        value = resp.headers.get("Retry-After")
        try:
            wait = max(0.0, float(value))
        except (TypeError, ValueError):
            try:
                from email.utils import parsedate_to_datetime

                retry_at = parsedate_to_datetime(str(value))
                if retry_at.tzinfo is None:
                    retry_at = retry_at.replace(tzinfo=datetime.timezone.utc)
                wait = max(
                    0.0,
                    (
                        retry_at.astimezone(datetime.timezone.utc)
                        - datetime.datetime.now(datetime.timezone.utc)
                    ).total_seconds(),
                )
            except (TypeError, ValueError, IndexError, OverflowError):
                wait = float(cls.RETRY_BACKOFF[attempt])
        return min(wait, cls.RETRY_WAIT_CAP)

    def _send_with_retry(self, send, deadline=None, attempts=RETRY_ATTEMPTS):
        """Perform one HTTP call with a bounded retry, and RETURN the final response.

        `send` is a zero-argument callable performing exactly ONE attempt; it is invoked
        synchronously, here, so it may close over the caller's locals. The retry layer adds
        attempts and NOTHING else — it never inspects a body and never raises — so each caller
        keeps deciding what its own >= 400 means (the GET path raise_for_status()es, the PUT path
        raises DARHTTPError carrying the API's message). That is what keeps the existing
        push-failure forensics intact for a failure that survives every attempt.

        Retrying the PUT is safe because it is a FULL-SET state PUT — so a second attempt
        re-asserts the same desired set rather than compounding a partial one; the operation is idempotent, which is the precondition for retrying a write
        at all.

        `deadline` is the paginated loop's time.monotonic() budget stamp. A wait that would cross
        it is NOT taken: the transient response is handed back unretried, and the caller's ordinary
        handling of it takes over (the paginated GET raise_for_status()es it) instead of the loop
        sleeping past its own ceiling. PAGED_BUDGET_SECONDS therefore still bounds the whole
        paginated call INCLUDING every retry wait — a retry allowed to outlive the budget would
        make that bound a lie. `None` (the PUT) has no loop budget to answer to; the per-request
        timeout still applies.

        `attempts` is this call's share of the retry budget. `1` means "send once, whatever comes
        back" — the failure-read path takes it, because there the latency of retrying costs more
        than the answer is worth (see list_roles_quick).
        """
        attempt = 0
        while True:
            resp = send()
            wait = self._retry_wait(resp, attempt, attempts)
            if wait is None:
                return resp
            if deadline is not None and time.monotonic() + wait >= deadline:
                return resp
            time.sleep(wait)
            attempt += 1

    @classmethod
    def _check_url(cls, url):
        """Every URL this client fetches — the caller-built first page and every
        server-provided continuationUri after it — must be a page of the SAME Fabric API
        this client was pointed at: BASE's scheme and host, the default port, no
        userinfo, a path under BASE's prefix. Anything else is refused rather than
        fetched, because the fetch carries the bearer token — following a poisoned
        continuation would hand a workspace-capable credential to whatever host it
        names. Derived from BASE rather than restated, so a repointed BASE (a sovereign
        cloud) moves the fence with it."""
        from urllib.parse import urlsplit

        try:
            base, parts = urlsplit(cls.BASE), urlsplit(str(url))
            ok = (
                parts.scheme == base.scheme
                and parts.hostname == base.hostname
                and parts.port in (None, 443)
                and not parts.username
                and not parts.password
                and parts.path.startswith(base.path + "/")
            )
        except ValueError:  # malformed authority (bad port, bad bracket) — refuse, not crash
            ok = False
        if not ok:
            raise DARHTTPError(f"refusing URL outside the Fabric API ({cls.BASE}): {url!r}")

    def _get_paged(self, url, timeout=None, attempts=RETRY_ATTEMPTS, capture_etag=False):
        """GET a paginated Fabric list endpoint -> flattened `value` items (follows continuationUri).
        `capture_etag` stores the FIRST page's ETag response header on self.roles_etag (the
        collection fingerprint — later pages are continuations of the same read). Every URL
        the loop is about to fetch — the first page and each continuationUri the service
        hands back — passes _check_url first, so a poisoned continuation can never carry
        the bearer token off the Fabric API host.
        Bounded twice: per request by `timeout` (default TIMEOUT), and across the whole loop by
        PAGED_BUDGET_SECONDS, since N slow pages otherwise multiply into an unbounded wait. The
        budget covers the retry WAITS as well as the pages, but it is a SLEEP bound rather than a
        wall-clock one, so a retried page still costs up to `attempts` round-trips of `timeout` on
        top of it (see PAGED_BUDGET_SECONDS)."""
        import requests

        out, pages = [], 0
        deadline = time.monotonic() + self.PAGED_BUDGET_SECONDS
        while url:
            self._check_url(url)
            if time.monotonic() >= deadline:
                raise DARHTTPError(
                    f"paginated GET exceeded the {self.PAGED_BUDGET_SECONDS}s budget after "
                    f"{pages} page(s) -- next page was {url}"
                )
            # `url` is read by the lambda during this call, before the loop reassigns it below.
            r = self._send_with_retry(
                lambda: requests.get(url, headers=self._headers, timeout=timeout or self.TIMEOUT),
                deadline=deadline,
                attempts=attempts,
            )
            r.raise_for_status()
            if capture_etag and pages == 0:
                self.roles_etag = r.headers.get("ETag")
            d = r.json()
            out += d.get("value", [])
            url = d.get("continuationUri")
            pages += 1
        return out

    def resolve_lakehouse(self, name: str) -> tuple[str, str]:
        """Resolve a lakehouse displayName to its canonical (displayName, id) in this client's
        workspace — the generate target-guard. Case-insensitive; >1 case-variant = ambiguous; 0 =
        not-found. Lets generate verify config.lakehouse_name names the ATTACHED lakehouse."""
        items = self._get_paged(f"{self.BASE}/workspaces/{self.workspace_id}/lakehouses")
        return Target._single_named(items, name, "lakehouse", self.workspace_id)

    def put_roles(
        self,
        roles: list[dict],
        dry_run: bool = False,
        etag: str | None = None,
        *,
        allow_unconditional: bool = False,
    ) -> int:
        """Bulk full-set PUT — the only write surface OLAF uses, and Microsoft labels
        that endpoint Preview (evaluation/development, not recommended for production).
        Retried on the transient statuses (see _send_with_retry); a failure that survives
        every attempt still raises the same DARHTTPError, so the push-failure forensics
        around it are unchanged.

        `etag` (the collection ETag list_roles captured) is sent as If-Match on the REAL
        PUT only — the zero-write dryRun validates a payload, not a concurrency window —
        so the SERVICE refuses a write against roles that changed after our read, closing
        the seconds-wide race the apply-time drift gate leaves between its re-read and
        this call. A 412 raises DARConflictError and is NOT in RETRY_STATUSES —
        re-sending the same stale ETag cannot succeed. On the FIRST attempt the 412
        means nothing landed (remedy: a fresh plan); on a RETRIED attempt it is
        AMBIGUOUS, and the error says so (`ambiguous=True`): the earlier attempt may
        itself have COMMITTED — a committed write rotates the collection ETag, drawing
        this very 412 — before a gateway/throttle status swallowed its response,
        indistinguishable from here from a concurrent edit between attempts. Only the
        mid-push forensics can tell (see Deployment._push_or_record_failure), which is
        why the attempt count rides on the exception at all.
        (A PUT that times out raises, which the retry loop never re-sends; if it actually
        LANDED, the next run's fresh read and drift gate see the new state — the 412 path
        does not arise there.) A real PUT without `etag` is refused unless the caller
        explicitly passes `allow_unconditional=True`; dry-run remains zero-write and exempt.
        The Preview bulk request is treated as a collection request: its published contract
        does not establish atomic replacement or deletion by omission. Differential outcomes
        therefore require a fresh post-state read; see error-handling.md's roadmap trade-off."""
        if not dry_run and not etag and allow_unconditional is not True:
            raise UsageError(
                "real DAR PUT requires a collection ETag; pass "
                "allow_unconditional=True only for an explicit concurrency opt-out"
            )
        import requests

        url = self._roles_url + ("?dryRun=true" if dry_run else "")
        headers = self._headers
        if etag and not dry_run:
            headers = {**self._headers, "If-Match": etag}
        attempts_made = 0

        def _send_once():
            # counted so a 412 can say whether a retry preceded it — the one fact that
            # separates "the service refused the write" from "an earlier attempt may have
            # committed and rotated the ETag on us" (see the docstring above)
            nonlocal attempts_made
            attempts_made += 1
            return requests.put(url, headers=headers, json={"value": roles}, timeout=self.TIMEOUT)

        resp = self._send_with_retry(_send_once)
        if resp.status_code == 412:
            if attempts_made > 1:
                raise DARConflictError(
                    f"PUT {url} -> 412 on a RETRIED attempt: an earlier attempt of this "
                    f"same PUT was answered with a transient status after possibly "
                    f"committing (a committed write rotates the collection ETag), or a "
                    f"concurrent edit landed between attempts — what this run wrote "
                    f"cannot be asserted without the live re-read in the push record",
                    ambiguous=True,
                )
            raise DARConflictError(
                f"PUT {url} -> 412: live OneLake security changed since read (the "
                f"collection ETag no longer matches) — re-run mode=plan and review "
                f"the new diff"
            )
        # Surface the API response body on 4xx/5xx so a policy-validation error is actionable
        # (raise_for_status alone drops it).
        if resp.status_code >= 400:
            raise DARHTTPError(f"PUT {url} -> {resp.status_code}: {resp.text[:2000]}")
        return resp.status_code

### `Log` — writer/reader for `onelake_security_log`.

Remembers the run context once (batch / run / env / target labels + resolved `run_by`) so every row carries it without re-passing arguments. Builds the start + per-grant validate + per-role action + complete rows for plan/apply; stamps `run_duration` on the complete row and `error_category` on failure rows; `find_plan_record` recovers the saved plan that gates apply. `COLUMNS` is the single source of truth `setup()` builds the log table from.

In [ ]:
# ══ Log ══════════════════════════════════════════════════════════════════════
# Writer/reader for onelake_security_log; remembers the run context once so every row carries it.
class Log:
    """Writes/reads onelake_security_log. Remembers the run context (batch/run/env/target labels + run_by
    + config_hash/config_version) once, so every row carries it without re-passing arguments per call.
    Stamps run_duration on the complete row and error_category on failure rows."""

    COLUMNS = (
        LOG_COLUMNS  # single source of truth — mode=setup creates the table from the same list
    )

    def __init__(
        self,
        spark,
        table,
        batch_id,
        run_id,
        env,
        mode,
        workspace_name,
        lakehouse_name,
        run_by=None,
        workspace_id=None,
        lakehouse_id=None,
        tenant_id=None,
        member_table=None,
    ):
        self.spark, self.table = spark, table
        self._start = time.monotonic()
        self._ctx = {
            "batch_id": batch_id,
            "run_id": run_id,
            "env": env,
            "mode": mode,
            "workspace_name": workspace_name,
            "lakehouse_name": lakehouse_name,
            "workspace_id": workspace_id,
            "lakehouse_id": lakehouse_id,
            "tenant_id": tenant_id,
            "config_hash": None,
            "config_version": None,
            "mapping_hash": None,
            "mapping_version": None,
            "framework_version": __version__,
            "run_by": Log.resolve_principal(
                spark, member_table, run_by if run_by is not None else Target.run_by(spark)
            ),
        }
        self._prewrite = None

    @staticmethod
    def resolve_principal(spark, member_table: str | None, value: str | None) -> str | None:
        """objectId -> `"<member_name> (<objectId>)"` from onelake_security_member; else unchanged.

        `run_by` is read by humans, and on a pipeline run it is a bare object id — a service
        principal or workspace identity has no user to name. The member table is already the
        No-Graph name<->objectId source, so a deploy identity listed there reads as its display
        name on every row it writes, and one that is not listed keeps its bare id, which stays
        correct and resolvable out of band (`az ad sp show --id <object-id>`). Same table, same
        read-seam trim as Audit._member_names — the id->name lookup who_can_access() uses.

        The name is APPENDED, never substituted: the object id is the only part of `run_by` the
        runtime attests, and `member_table` is a pipeline-overridable parameter over a schema
        anyone with write access can add rows to. One row `(ServicePrincipal, "alice@contoso.com",
        <deploy object id>)` is unique for its id AND unique for its (type, name), so no guard
        fires — and the duplicate-id guard runs only in `generate`, while this runs at
        `Log.__init__` on EVERY mode. Substituting would have made every subsequent row read
        `run_by = alice@contoso.com`, byte-identical to a genuinely authenticated interactive UPN.
        Appending keeps the attested value on the row, so a member row can add a LABEL, it can no
        longer silently REPLACE an identity. Listing the deploy identity is therefore optional
        enrichment, not a correctness requirement.

        Only a GUID-shaped value is looked up: a UPN is already a name. Read-only and
        exception-safe — a member table that does not exist yet (`setup` runs before it does), one
        that cannot be read, or an id with no row all return the value unchanged, because a display
        detail on a log row must never be able to fail the run that writes it."""
        if not value or not member_table or not GUID_RE.match(str(value)):
            return value
        try:
            wanted = str(value).strip().lower()
            names = {
                str(d.get("member_name")).strip()
                for d in (Parse.trim_row(r.asDict()) for r in spark.table(member_table).collect())
                if str(d.get("member_id") or "").strip().lower() == wanted and d.get("member_name")
            }
            # Matched on the objectId ALONE — deliberately, and member_type is neither passed nor
            # needed. An objectId is unique across principal types in Entra, so the id identifies
            # the row by itself; and the runtime could not supply a type anyway (the context has
            # none, and the token's `idtyp` only distinguishes user from app — a managed identity
            # and a service principal are both `app`, both servicePrincipal objects). Requiring a
            # type would mean guessing one, and a wrong guess misses a row that exists.
            #
            # One id spelled with two different names is data `generate` blocks (the member-table
            # duplicate-id guard), but this path can still meet it: apply/plan/show never reload
            # the member cache, so a table edited after the last generate reaches Log unchecked.
            # Picking either label would make run_by depend on row order, so keep the bare id: an
            # unlabelled value is honest, an arbitrary one is not.
            if len(names) == 1:
                return f"{names.pop()} ({value})"
        except Exception:
            pass
        return value

    def set_config_provenance(
        self, config_hash_value: str | None, config_version_value: int | str | None
    ) -> "Log":
        """Stamp the config generation identity onto every subsequent row. config_hash and
        config_version are first-class LOG_COLUMNS (not message JSON), so row() carries them on every
        row (start/validate/action/complete). Call once per run before any row is written."""
        self._ctx["config_hash"] = config_hash_value
        self._ctx["config_version"] = (
            None if config_version_value is None else str(config_version_value)
        )
        return self

    def set_mapping_provenance(
        self, mapping_hash_value: str | None, mapping_version_value: int | str | None
    ) -> "Log":
        """Stamp the mapping-generation identity onto every subsequent row (mirrors
        set_config_provenance) — mapping_hash/mapping_version are first-class LOG_COLUMNS, the
        config -> mapping -> run provenance chain. Call once per run, after the mapping is written
        (generate) or read (plan/apply)."""
        self._ctx["mapping_hash"] = mapping_hash_value
        self._ctx["mapping_version"] = (
            None if mapping_version_value is None else str(mapping_version_value)
        )
        return self

    @property
    def batch_token(self) -> str:
        """This run's batch_id reduced to a filename-safe token — the per-invocation component of
        apply's role-backup filename (see Deployment._backup_live_roles). run_mode mints a fresh
        uuid4 batch per call unless a pipeline passes its own run id, so two applies that land in
        the same clock second still name different files. Truncated for legibility, NOT for
        uniqueness: uniqueness is the exclusive-create claim's job, never this token's."""
        return re.sub(r"[^0-9A-Za-z]", "", str(self._ctx["batch_id"] or ""))[:12] or "nobatch"

    def row(self, action: str, status: str, **fields) -> dict:
        return {
            **{c: None for c in self.COLUMNS},
            **self._ctx,
            "run_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "action": action,
            "status": status,
            **fields,
        }

    def fail_row(self, action: str, exc: BaseException, **fields) -> dict:
        """Failure row with error_category classified from the exception (http/validation/guard/unexpected)."""
        return self.row(
            action, "failed", error_category=OLAFError.classify(exc), message=str(exc), **fields
        )

    def run_header_rows(self, grants: list[dict]) -> list[dict]:
        """'start' row + one 'validate' row per single-valued grain. config_hash/config_version ride as
        first-class columns (set via set_config_provenance), so the start row carries no message."""
        rows = [self.row("start", "success")]
        rows += [
            self.row(
                "validate",
                "success",
                role_name=a["role_name"],
                scope_path=a["scope_path"],
                scope_type=a["scope_type"],
                member_name=a["member_name"],
                member_id=a["member_id"],
                member_type=a["member_type"],
            )
            for a in grants
        ]
        return rows

    def action_rows(
        self, plan: dict, executed_message: str, omit_message: str, omit_status: str
    ) -> list[dict]:
        """One row per planned role action. `omit` is a request-construction candidate,
        not a claim about a platform deletion outcome."""
        return [
            self.row(
                "omission_candidate" if action == "omit" else action,
                omit_status if action == "omit" else "success",
                role_name=name,
                message=omit_message if action == "omit" else executed_message,
            )
            for name, action in sorted(plan.items())
        ]

    def complete_row(self, plan: dict, **extra) -> dict:
        """Versioned completion with the reviewed plan and its exact operation stage."""
        operation = extra.pop("operation", self._ctx["mode"])
        return self.row(
            "complete",
            "success",
            run_duration=str(round(time.monotonic() - self._start, 3)),
            message=json.dumps(
                {"schema": 1, "operation": operation, "plan": dict(sorted(plan.items())), **extra},
                sort_keys=True,
                separators=(",", ":"),
            ),
        )

    def run_complete(self, message: str, **extra) -> dict:
        """Versioned run-level completion for setup/generate, with a human summary."""
        operation = extra.pop("operation", self._ctx["mode"])
        return self.row(
            "complete",
            "success",
            run_duration=str(round(time.monotonic() - self._start, 3)),
            message=json.dumps(
                {"schema": 1, "operation": operation, "summary": message, **extra},
                sort_keys=True,
                separators=(",", ":"),
            ),
            **extra,
        )

    def write(self, rows: list[dict]) -> None:
        """Append with the explicit schema TableSchema derives from the DDL — run_at is a real
        TIMESTAMP and config_version a real BIGINT, and an appended frame whose types disagreed
        with the table's would be refused by Delta."""
        if not rows:
            return
        if self._prewrite is not None:
            self._prewrite()
        schema, to_row = TableSchema.frame_schema(self.COLUMNS)
        data = [to_row(r) for r in rows]
        self.spark.createDataFrame(data, schema).write.mode("append").saveAsTable(self.table)

    @property
    def _env_scope(self):
        """The WHERE clause that scopes a log read to THIS run's environment.

        `env` is optional, and a run that sets no label writes NULL — TableSchema coerces the
        blank like every other string column — while SQL's `env = ''` never matches NULL. So an
        unlabelled run asking the equality question would read none of its OWN rows. That is not
        a cosmetic miss: find_plan_record is the saved-plan gate, so an unlabelled apply would
        refuse the plan it had just written, and grant_provenance would report every grant it had
        just deployed as out-of-band. Blank asks for NULL; a label asks for equality. Written
        ONCE because three reads share it, and drifting them apart would fail one mode at a
        time."""
        return f"env = '{self._ctx['env']}'" if self._ctx["env"] else "env IS NULL"

    def find_plan_record(self, config_hash_value: str, mapping_hash_value: str) -> dict | None:
        """Latest successful 'complete' plan row for this config_hash AND mapping_hash — the
        saved-plan gate. Accepts
        both a standalone mode='plan' row AND the inline plan a mode='rollback' run writes (rollback
        stamps its whole chain mode='rollback', so its plan record would otherwise be invisible to this
        gate and apply() could never satisfy it). Matches on the config_hash and mapping_hash
        COLUMNS (first-class COLUMNS, not message fields), not by parsing message JSON. The
        mapping_hash match binds the plan to the exact mapping generation it reviewed:
        config_hash fingerprints the config ROWS only, so a mapping regenerated from the same
        rows with different member/scope resolution (an edited objectId in
        onelake_security_member) carries the same config_hash but a NEW mapping_hash, and the
        old plan must not unlock an apply of content nobody reviewed. A row with no stamped
        mapping_hash (an externally written or hand-edited log row — every framework release
        stamps it) never matches — SQL NULL compares unknown — so an
        unbindable plan fails closed to re-plan. Returns the
        parsed message dict
        (carries the plan for the drift check). None = no such plan (including: log table absent)."""
        try:
            matches = (
                self.spark.table(self.table)
                .where(
                    "mode IN ('plan', 'rollback') AND action = 'complete' AND status = 'success'"
                )
                .where(self._env_scope)
                .where(f"config_hash = '{config_hash_value}'")
                .where(f"mapping_hash = '{mapping_hash_value}'")
                .orderBy("run_at", ascending=False)
                .collect()
            )
        except Exception:
            # Also swallows a read against a log table MISSING these columns (hand-built or
            # ancient): the caller then reports "no plan" — a slightly confusing remedy for a
            # schema problem — but the gate stays fail-closed, the property that matters.
            return None
        for r in matches:
            try:
                return json.loads(r.message or "{}")
            except ValueError:
                continue
        return None

    def has_run_complete(self, mapping_hash_value: str) -> bool:
        """True when a successful generate-side 'complete' row for this mapping generation
        exists — the completion record generate's self-healing skip verifies (a prior run
        may have committed the mapping, then died before its audit rows landed). Matches
        the first-class mapping_hash column; mode IN ('generate','rollback') because a
        rollback chain's generate stamps its rows mode=rollback. False on any read failure:
        the repair row is then written, the safe direction for an audit trail — a duplicate
        completion record is noise, a missing one is a hole."""
        try:
            rows = (
                self.spark.table(self.table)
                .where(
                    "mode IN ('generate', 'rollback') AND action = 'complete' "
                    "AND status = 'success'"
                )
                .where(self._env_scope)
                .where(f"mapping_hash = '{mapping_hash_value}'")
                .collect()
            )
        except Exception:
            return False
        return bool(rows)

    def grant_provenance(self) -> dict:
        """Establishing-grant provenance per (role_name, scope_path, member_id), read from the log —
        powers show audit enrichment. The grant-grain 'validate' rows are written on every deploy;
        restricting to the DEPLOYING modes drops read-only plan runs, so both reported ends are
        instants at which the grant was actually pushed — a rollback chain's apply counts (its rows are stamped
        mode=rollback), matching the plan-gate loader's mode IN ('plan','rollback') precedent;
        a FAILED rollback apply re-stamps its rows failed, so it still establishes nothing.

        Per key: first_applied/first_granted_by (earliest run_at and who pushed it) and
        last_applied/last_granted_by (latest, and who), plus config_version FROM THE LATEST push
        -- the config actually in effect now. The ACTOR is split across both ends because they
        answer different questions -- who ORIGINALLY authorized this access, and who most recently
        re-asserted it -- and they are routinely different principals (a service principal
        deploying, a human re-running). One `granted_by` beside two timestamps would leave the
        reader guessing which end it belonged to, which is the same ambiguity `since` had.

        Both ends, and deliberately no single `since`. The log records what OLAF did; it cannot
        record a role deleted straight from the Fabric UI, so an apply, an out-of-band deletion
        and a re-apply are indistinguishable here from one unbroken grant. `since` asserted a
        continuity nothing observed -- the same overreach as calling an omitted role deleted.
        Taking the LATEST instead would be no better: it resets on every routine re-deploy of
        an unchanged config, which is exactly the question an access review is asking. Each end
        is exactly knowable; a wide gap between them is the operator's cue to look, and OLAF
        does not guess on their behalf.
        Per key it also carries the member display name (member_name) for human-facing surfacing.
        Returns {} when the log is unavailable/empty (-> everything reads as out-of-band, an honest signal)."""
        try:
            rows = (
                self.spark.table(self.table)
                .where("action = 'validate' AND status = 'success'")
                .where(self._env_scope)
                .collect()
            )
        except Exception:
            return {}
        prov = {}
        for r in rows:
            if (
                # 'replace' is a RETIRED pre-release mode: its full-truth semantics became
                # apply(keep_unmanaged=False), so no path here can emit it. It stays in the
                # predicate because this table is append-only history — a workspace deployed
                # before the fold still holds rows stamped with it, and dropping the token
                # would silently re-read those grants as out-of-band, with no provenance.
                getattr(r, "mode", None) not in ("apply", "replace", "rollback")
                or getattr(r, "member_id", None) is None
            ):
                continue
            key = (str(r.role_name).lower(), str(r.scope_path).lower(), str(r.member_id).lower())
            run_at = getattr(r, "run_at", None)
            cur = prov.get(key)
            if cur is None:
                prov[key] = {
                    "first_applied": run_at,
                    "first_granted_by": getattr(r, "run_by", None),
                    "last_applied": run_at,
                    "last_granted_by": getattr(r, "run_by", None),
                    "config_version": getattr(r, "config_version", None),
                    "member_name": getattr(r, "member_name", None),
                }
                continue
            if run_at is None:
                continue
            if cur["first_applied"] is None or str(run_at) < str(cur["first_applied"]):
                cur["first_applied"] = run_at
                cur["first_granted_by"] = getattr(r, "run_by", None)
            if cur["last_applied"] is None or str(run_at) > str(cur["last_applied"]):
                # config_version and member_name follow the LATEST push, not the first: they
                # describe the state in effect now, which is what a reader acts on.
                cur["last_applied"] = run_at
                cur["last_granted_by"] = getattr(r, "run_by", None)
                cur["config_version"] = getattr(r, "config_version", None)
                cur["member_name"] = getattr(r, "member_name", None)
        return prov

### `Deployment` — the pipeline façade.

One public method per runtime mode (`setup` / `generate` / `plan` / `apply` / `show`); shared steps (`_desired_state`, `_require_plan_and_no_drift`, `_push`) stay private. Holds the Spark session, the `FabricClient` and `Log`, the tenant id, the three table names and the target labels, and lazily caches the active short-config rows. Reads **only** the mapping lock-file as its input (TOCTOU closed) and returns the result dict the runtime hands back to the pipeline. Defined last because it depends on `FabricClient` and `Log`.

In [ ]:
# ══ Deployment ═══════════════════════════════════════════════════════════════
# The pipeline facade — one public method per runtime mode (setup/generate/plan/apply/show).
class Deployment:
    """The deployment pipeline on one lakehouse target — ONE PUBLIC METHOD PER RUNTIME MODE:
    generate(rebuild) · plan() · apply(keep_unmanaged) · rollback() · show(by, subject) · trace via Audit.
    Shared steps live in private methods (_desired_state, _require_plan_and_no_drift, ...).
    Every public method returns the result dict the runtime hands to the pipeline."""

    def __init__(
        self,
        spark,
        client,
        audit,
        tenant_id,
        config_table,
        mapping_table,
        mapping_history_dir,
        workspace_name="",
        lakehouse_name="",
        member_table="olaf.onelake_security_member",
        role_backup_dir=PARAM_DEFAULTS["role_backup_dir"],
        if_match=PARAM_DEFAULTS["if_match"],
        control_data_isolation_attestation=PARAM_DEFAULTS["control_data_isolation_attestation"],
    ):
        self.spark, self.client, self.audit, self.tenant_id = spark, client, audit, tenant_id
        self.config_table, self.mapping_table = config_table, mapping_table
        # Both lakehouse folder parameters pass the path guard ONCE, here — generate's CSV
        # export, apply's role backup and cleanup()'s delete loop all read the vetted value.
        self.mapping_history_dir = Deployment._require_safe_dir(
            "mapping_history_dir", mapping_history_dir
        )
        self.role_backup_dir = Deployment._require_safe_dir("role_backup_dir", role_backup_dir)
        self.workspace_name, self.lakehouse_name = workspace_name, lakehouse_name
        self.member_table = (
            member_table  # onelake_security_member: the name->objectId resolution cache
        )
        # The FOURTH control table. It is not a constructor argument because it is not a free
        # choice: it is the table this Deployment's own audit Log writes to, and setup() already
        # derives it exactly this way (self.audit.table). Taking it from anywhere else would let a
        # Deployment create, migrate and DROP a different log table than the one carrying its own
        # trail. Set HERE because cleanup() enumerates the four tables off the instance, and until
        # now this attribute existed only because the interactive facade patched it on after
        # construction — so a Deployment built any other way (explain(), a direct call, a test)
        # had a cleanup() that raised AttributeError on the third table, after dropping two.
        # explain() builds an audit-less, client-less Deployment for a pure read; it falls back to
        # the default name rather than to None, which would have made the drop loop issue
        # `DROP TABLE IF EXISTS None`.
        self.log_table = getattr(audit, "table", None) or DEFAULT_CONTROL_TABLES["log_table"]
        self._short_rows_cache = None  # lazy: mode=setup runs before the config table exists
        self._mapping_hash = None  # set by _desired_state; the plan gate refuses a falsy value
        self._live_etag = None  # set by _desired_state; the real PUT's If-Match token
        # The concurrency escape hatch (see PARAM_DEFAULTS): False sends the real PUT
        # unconditionally. The EFFECTIVE state is recorded per apply — see apply().
        self.if_match = bool(if_match)
        self.control_data_isolation_attestation = str(control_data_isolation_attestation or "")
        self._control_boundary = None
        self._control_depth = 0
        self._active_leases = []

    @staticmethod
    def _require_safe_dir(name, value):
        """Path guard for the two operator-editable lakehouse folder parameters
        (mapping_history_dir / role_backup_dir). cleanup() DELETES every file under these
        folders and generate/apply write into them, so the value must name a folder inside
        Files/ — the one lakehouse file area this framework owns. Accepted spellings follow
        ScopePath.folder (the config-side folder rule): an optional leading '/' and any
        letter case of the Files segment are canonicalized, so every spelling the config
        columns accept for a Files/ path works here too. An empty value, a backslash or
        NUL, anything that normalizes outside Files/ (a '..' escape, an absolute path such
        as /tmp/x), and bare '.'/'Files' (the WHOLE user file area) are refused, not
        coerced — the house rule bool_param/env/verbosity already follow: a silently
        repaired path would point the delete loop at a folder the operator never named.
        Returns the canonical 'Files/...' relative path."""
        import posixpath

        text = str(value or "").strip()
        normalized = posixpath.normpath(text.lstrip("/"))
        if (
            not text
            or "\\" in text
            or "\x00" in text
            or not normalized.lower().startswith("files/security/")
        ):
            raise SystemExit(
                f"{name} must name a folder below Files/security — not empty, no backslash, no "
                f"'..' escape, not the Files/security root — got {value!r}; cleanup() deletes "
                f"every file under this folder, so a path outside the framework-owned "
                f"Files/security area is refused, not coerced"
            )
        return "Files/" + normalized.split("/", 1)[1]

    @staticmethod
    def _require_contained(operation, folder):
        """Belt-and-braces realpath containment re-check of __init__'s path guard, run at
        the moment a folder is actually USED for writing or deleting (generate's CSV
        export, apply's role backup, cleanup's delete loop): the concrete lakehouse path
        must still resolve inside the framework-owned Files/ area even if the attribute
        was mutated after construction. A real if/SystemExit rather than an assert —
        `python -O` strips asserts, and a containment check that vanishes under an
        interpreter flag is no check at all (SystemExit also keeps the refusal in the
        house error class: 'guard', not 'unexpected')."""
        import os

        files_root = os.path.realpath("/lakehouse/default/Files/security")
        resolved = os.path.realpath(f"/lakehouse/default/{folder}")
        if not resolved.startswith(files_root + "/"):
            raise SystemExit(
                f"{operation} refused: {folder!r} resolves outside /lakehouse/default/Files/security "
                f"— the folder parameters are validated at construction, so this value "
                f"changed after; refused, not coerced"
            )

    @property
    def _isolation_state(self):
        """What this run may honestly claim about control-data isolation.

        Every record used to hardcode "attested". Once the evidence reference stopped being
        mandatory that became a lie for any run without one, so it is computed instead —
        the same test the health probe already applied."""
        return (
            "attested"
            if CONTROL_EVIDENCE_RE.fullmatch(str(self.control_data_isolation_attestation or ""))
            else "unknown"
        )

    def _boundary(self):
        if self._control_boundary is None:
            self._control_boundary = ControlBoundary(
                self.client,
                {
                    "config_table": self.config_table,
                    "mapping_table": self.mapping_table,
                    "log_table": self.log_table,
                    "member_table": self.member_table,
                },
                self.mapping_history_dir,
                self.role_backup_dir,
                self.control_data_isolation_attestation,
            )
        return self._control_boundary

    def _begin_sensitive(self, operation, desired_grants=None, snapshot=None):
        boundary = self._boundary()
        if desired_grants is not None:
            boundary.require_desired_safe(desired_grants)
        lease = boundary.begin(
            operation,
            snapshot=snapshot,
            sentinel_already_owned=self._control_depth > 0,
        )
        self._control_depth += 1
        self._active_leases.append(lease)
        if self.audit is not None:
            self.audit._prewrite = lease.prewrite_audit
        return lease

    def _current_lease(self):
        if not self._active_leases:
            raise ControlDataGuardError(
                "sensitive write requires an active control-data sentinel lease"
            )
        return self._active_leases[-1]

    def release_unwritten_leases(self):
        """Give back every lease this operation still holds that never authorized a write.

        Called once, where run_mode builds a `blocked` envelope. A refusal that got as far as
        a write keeps its marker -- that state is genuinely unknown. A validation refusal did
        not, and stranding the next operation behind an incident nobody had was the single
        most common way an operator met the sentinel at all."""
        while self._active_leases and not self._active_leases[-1].authorized_write:
            lease = self._active_leases.pop()
            self._control_depth -= 1
            lease.boundary.release_unwritten(lease)
        if self.audit is not None:
            self.audit._prewrite = (
                self._active_leases[-1].prewrite_audit if self._active_leases else None
            )

    def _finish_sensitive(self, lease, allow_dar_change=False):
        fresh = lease.postcheck(allow_dar_change=allow_dar_change)
        self._control_depth -= 1
        if not self._active_leases or self._active_leases[-1] is not lease:
            raise ControlDataGuardError("sensitive-operation lease ordering is invalid")
        self._active_leases.pop()
        if self.audit is not None:
            self.audit._prewrite = (
                self._active_leases[-1].prewrite_audit if self._active_leases else None
            )
        if lease.owns_sentinel:
            lease.clear()
        return fresh

    @staticmethod
    def _payload_hash(payload):
        canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"), default=str)
        return hashlib.sha256(canonical.encode()).hexdigest()

    def _prepared_intent(
        self, operation, payload, omission_candidates, backup_path, keep_unmanaged, snapshot
    ):
        intent = {
            "schema": 1,
            "operation": operation,
            "phase": "prepared",
            "payload_hash": self._payload_hash(payload),
            "intended_roles": sorted(str(r.get("name") or "") for r in payload),
            "omitted_role_candidates": sorted(omission_candidates),
            "post_state_review_required": True,
            "keep_unmanaged": keep_unmanaged,
            "backup_path": backup_path,
            "conditional": bool(self.if_match),
            "etag": snapshot.etag,
            "isolation_attestation": self.control_data_isolation_attestation,
            "reserved_digest": snapshot.reserved_digest,
            "dar_roles_digest": snapshot.roles_digest,
            "dar_observed_at": snapshot.observed_at,
        }
        self.audit.write(
            [
                self.audit.row(
                    "push",
                    "prepared",
                    message=json.dumps(intent, sort_keys=True, separators=(",", ":")),
                )
            ]
        )
        return intent

    def _mark_unknown_write(self, exc, operation, backup_path):
        # Preserve the original exception class/message for direct callers while carrying
        # the tri-state facts run_mode needs for its structured envelope.
        for name, value in (
            ("changed", None),
            ("possible_exposure", True),
            ("operation", operation),
            ("backup_path", backup_path),
        ):
            try:
                setattr(exc, name, value)
            except Exception:
                pass
        return exc

    def _write_confirmed_completion(self, operation, push_status, backup_path, rows):
        try:
            self.audit.write(rows)
        except BaseException as exc:
            ctx = getattr(self.audit, "_ctx", {})
            raise PostWriteAuditError(
                operation,
                push_status,
                backup_path,
                ctx.get("batch_id"),
                ctx.get("run_id"),
                exc,
            ) from exc

    def _postcheck_after_write(self, operation, lease, backup_path, allow_dar_change=False):
        try:
            return self._finish_sensitive(lease, allow_dar_change=allow_dar_change)
        except BaseException as exc:
            raise PostWriteBoundaryError(operation, backup_path, exc, changed=True) from exc

    @property
    def short_rows(self) -> list[dict]:
        # Catalog.active_config_rows: the CONFIG_AUTHOR_COLUMNS projection — foreign
        # columns on the physical table never reach config_hash (see that reader).
        if self._short_rows_cache is None:
            self._short_rows_cache = Catalog.active_config_rows(self.spark, self.config_table)
        return self._short_rows_cache

    @property
    def config_hash(self) -> str:
        return Hash.config(self.short_rows)

    # ---------- mode: setup ----------

    def _type_drift(self, table, cols):
        """[(column, live type, declared type)] for every column whose live Delta type disagrees
        with TableSchema's. Empty when they agree, when the column is absent (that is the
        additive migration's job), or when the table cannot report its types at all — the scan
        is diagnostic, so an engine that will not answer must not fail setup.

        Drift matters more than it looks: `ALTER` cannot retype a Delta column, so a table that
        drifts stays broken, and the failure it eventually causes is a Delta
        DELTA_FAILED_TO_MERGE_FIELDS naming a column pair and nothing else — far from here, and
        unreadable when it lands."""
        try:
            live_types = {n.lower(): str(t) for n, t in self.spark.table(table).dtypes}
        except Exception:  # noqa: BLE001 — no type info is not a reason to fail setup
            return []
        drifted = []
        for col in cols:
            want = TableSchema.ddl_type(col)
            have = live_types.get(col.lower())
            if have and have.lower() != want.lower():
                drifted.append((col, have, want))
        return drifted

    def setup(self, rebuild: bool = False) -> dict:
        """Idempotent create + additive schema-drift migration for the four control tables, then an
        audit trail — runnable anytime. Per table: create it if absent (outcome 'create'), else
        ADD COLUMNS for any expected column the live table is missing (outcome 'migrate', e.g. a table
        authored by an older framework version), else leave it untouched (outcome 'no_change').

        `rebuild=False` (the default) NEVER drops or retypes — unexpected live columns and type
        mismatches are warned about, never altered; only additive migration. That is the safe
        default, and it is also a dead end: `ALTER` cannot change a column's type, so a table
        whose type disagrees with the framework's stays broken and every write to it is refused
        by Delta.

        `rebuild=True` is the escape hatch: any table with TYPE drift is DROPPED and recreated
        from the DDL (outcome 'rebuilt'). It drops nothing else — a table that merely misses a
        column is still migrated additively, and a table that already agrees is untouched, so
        rebuild is not a blunt 'recreate everything'. DATA IS LOST: the config's authored rows,
        the log's audit history. Every drop prints the table and the row count going with it,
        and the run's audit trail records them — in the NEW log table, since the old one may be
        what was just dropped. Reload an author-owned table afterwards with OLAF.load_config().

        The log table is (created or migrated) in the SAME loop, so
        it is guaranteed to exist by the end of the loop; only THEN are audit rows written — on a
        first run the log table does not exist until this loop creates it, which is why setup logs
        AFTER the loop, not before."""
        boundary_lease = self._begin_sensitive("setup")
        expected = {
            self.config_table: CONFIG_AUTHOR_COLUMNS,
            self.mapping_table: MAPPING_COLUMNS + MAPPING_PROVENANCE_COLUMNS,
            self.audit.table: LOG_COLUMNS,
            self.member_table: MEMBER_CACHE_COLUMNS,
        }
        ddls = TableSchema.definitions(
            self.config_table, self.mapping_table, self.audit.table, self.member_table
        )
        created, migrated, unchanged, rebuilt = [], {}, [], {}
        for table, cols in expected.items():
            # ControlBoundary validates every name as schema.table before setup can enter.
            boundary_lease.prewrite()
            self.spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{table.split('.', 1)[0]}`")
            if not self.spark.catalog.tableExists(table):
                boundary_lease.prewrite()
                self.spark.sql(ddls[table])
                created.append(table)
                Say.out("detail", "   🆕 created   " + table)
                continue
            drifted = self._type_drift(table, cols)
            if drifted and rebuild:
                # DROP + recreate is the only way to retype a Delta column. Say what is being
                # destroyed BEFORE destroying it, with the row count, because for the config and
                # the log that is authored data and audit history rather than a derived file.
                try:
                    lost = self.spark.table(table).count()
                except Exception:  # noqa: BLE001 — an unreadable table is still droppable
                    lost = None
                Say.out(
                    "detail",
                    f"   💥 rebuilding {table} — DROP + recreate, "
                    f"{lost if lost is not None else 'an unknown number of'} row(s) lost "
                    f"({', '.join(f'{c}: {was} → {want}' for c, was, want in drifted)})",
                )
                boundary_lease.prewrite()
                self.spark.sql(f"DROP TABLE IF EXISTS {table}")
                boundary_lease.prewrite()
                self.spark.sql(ddls[table])
                rebuilt[table] = [c for c, _was, _want in drifted]
                continue
            # DataFrame.columns works on an empty Delta table and dodges qualified-name ambiguity.
            live = [str(c) for c in self.spark.table(table).columns]
            live_lower = {c.lower() for c in live}
            added = [c for c in cols if c.lower() not in live_lower]
            for col in added:
                boundary_lease.prewrite()
                self.spark.sql(
                    f"ALTER TABLE {table} ADD COLUMNS (`{col}` {TableSchema.ddl_type(col)})"
                )
            expected_lower = {c.lower() for c in cols}
            for extra in live:
                if extra.lower() not in expected_lower:
                    # ONE policy for a column the framework never declared (issue #2),
                    # split by who owns the table. Author-owned tables are shared ground:
                    # another framework's column there is SUPPORTED coexistence —
                    # config_hash never reads it and load_config carries it — so it gets
                    # a note, not a warning. Framework-owned tables are not shared:
                    # generate rewrites the mapping in full (overwriteSchema), so an
                    # extra column there is living on borrowed time, and the log is
                    # append-only history nothing else should be shaping.
                    if table in (self.config_table, self.member_table):
                        Say.out(
                            "detail",
                            f"   ℹ️   {table} has coexisting column '{extra}' outside the "
                            f"framework schema — left as-is (ignored by config_hash, "
                            f"preserved by load_config)",
                        )
                    elif table == self.mapping_table:
                        Say.out(
                            "detail",
                            f"WARN: {table} has unmanaged column '{extra}' — left as-is by "
                            f"setup, but the mapping is framework-derived: the next "
                            f"generate rewrites it in full (overwriteSchema) and will "
                            f"DROP this column",
                        )
                    else:
                        Say.out(
                            "detail",
                            f"WARN: {table} has unmanaged column '{extra}' not in the "
                            f"framework schema — left as-is (setup only adds, never drops "
                            f"or retypes), but the log is append-only audit history "
                            f"nothing else should be shaping",
                        )
            for col, have, want in drifted:
                Say.out(
                    "detail",
                    f"   ⚠️   {table}.`{col}` is {have}, the framework declares {want} "
                    f"— pass rebuild=True to DROP and recreate the table (data is lost); "
                    f"until then every write to it is refused",
                )
            if added:
                migrated[table] = added
                Say.out("detail", f"   🔧 migrated  {table} (added {', '.join(added)})")
            else:
                unchanged.append(table)
                Say.out("detail", "   ✔️  unchanged " + table)
        # Log table is guaranteed to exist now -> write the audit trail. config_hash/config_version
        # stay NULL: setup reads no config, so there is no provenance to stamp (no set_config_provenance).
        rows = [self.audit.row("create", "success", message=f"created {t}") for t in created]
        rows += [
            self.audit.row("migrate", "success", message=f"migrated {t}: added {', '.join(c)}")
            for t, c in migrated.items()
        ]
        # A rebuild row is a DESTRUCTIVE event, and it names the columns that forced it. Written
        # to the NEW log table — the old one may be exactly what was just dropped.
        rows += [
            self.audit.row(
                "rebuild", "success", message=f"rebuilt {t}: dropped + recreated for {', '.join(c)}"
            )
            for t, c in rebuilt.items()
        ]
        if rows:
            summary = (
                f"setup: created {len(created)}, rebuilt {len(rebuilt)}, "
                f"migrated {len(migrated)}, unchanged {len(unchanged)}"
            )
        else:
            summary = "setup rerun — no schema changes"
        rows.append(self.audit.run_complete(summary))
        self.audit.write(rows)
        affected = json.dumps({"tables": sorted(expected)}, sort_keys=True, separators=(",", ":"))
        boundary_snapshot = self._postcheck_after_write("setup", boundary_lease, affected)
        return {
            "changed": bool(created or migrated or rebuilt),
            "message": summary,
            "data": {
                "created": created,
                "rebuilt": rebuilt,
                "migrated": migrated,
                "unchanged": unchanged,
                "dar_snapshot_safe": True,
                "workspace_isolation": self._isolation_state,
                "dar_etag": boundary_snapshot.etag,
            },
        }

    # ---------- mode: generate ----------

    def _member_ids_drifted(self, stamped, cache):
        """True when the member objectIds stamped in the mapping rows `stamped` no longer match
        what onelake_security_member resolves those SAME rows' member names to now.

        The idempotency skip is keyed on config_hash, which fingerprints the CONFIG rows — so an
        edit to the member table is invisible to it. Correcting one member row's objectId (an
        Entra rotation, a fixed copy-paste) therefore left generate reporting `no_change` while
        every later plan/apply still pushed the OLD id: access granted to a principal that no
        longer exists, or to a different one, under an audit trail saying nothing had changed.
        The wildcard exception above covers a member table that gains or loses ROWS; this covers
        the ids ON the rows a wildcard-free config already names, which is the case no hash sees.

        Deliberately NOT a new stored fingerprint: the mapping already carries both halves — the
        effective member NAMES and the ids they resolved to — so this is a re-derivation over
        columns that already exist, and the control-table schemas (a documented contract) do not
        move. Resolution is not re-implemented either; Member.ids_for is the very helper that
        stamped the values being compared.

        Compared as strings with None folded to "": an empty id column round-trips through Delta
        as SQL NULL, and reading that back as a literal "None" would report drift on every run of
        every config with an unused member type.

        A PURE comparison: the caller (generate's skip branch) loads the member cache ONCE
        and passes it in, so the same load also hands the cache's ERROR list to the skip
        condition — a member table in an error state must refuse the skip outright, not
        merely when the errors happen to read as drift. (The load used to happen in here,
        and the error list was discarded by naming convention.)"""
        for row in stamped:
            expected = Member.ids_for(row, cache)
            for _name_col, id_col, _mtype in MAPPING_MEMBER_COLUMNS:
                if str(row.get(id_col) or "") != str(expected[id_col] or ""):
                    return True
        return False

    def _target_drifted(self, stamped):
        """True when any stamped workspace_id/lakehouse_id no longer names the ATTACHED
        target. The idempotency skip is keyed on config_hash, which cannot see WHERE the
        mapping was generated — so control tables copied to another lakehouse skipped
        forever: the plan/apply TARGET MISMATCH guard says 're-run mode=generate', and an
        unchanged config made that very generate a no-op, leaving the operator blocked
        unless they discovered rebuild=True. Target drift defeats the skip exactly like
        member-id drift, so the advertised remedy actually re-stamps. Ids compared as
        strings, stripped and lowercased (GUIDs are case-insensitive), with None folded
        to '' — the same folding _member_ids_drifted uses."""
        attached = (
            str(self.client.workspace_id or "").strip().lower(),
            str(self.client.item_id or "").strip().lower(),
        )
        return any(
            (
                str(row.get("workspace_id") or "").strip().lower(),
                str(row.get("lakehouse_id") or "").strip().lower(),
            )
            != attached
            for row in stamped
        )

    def _export_history_csv(self, df, mapping_hash, now_dt, lease):
        """Write (or reuse) the versioned mapping-history CSV for the generation whose
        content fingerprint is `mapping_hash`, returning its Files/ relpath. Deduped on
        mapping_hash: a history file already carrying this generation's hash is reused
        verbatim, so retries and the skip path's self-healing never pile up identical
        copies. Shared by generate's main path (which exports BEFORE the mapping commit)
        and its skip path (which re-exports a missing artifact). The filename's v-tag
        embeds the mapping table's Delta version AT EXPORT TIME — on the main path that is
        the moment just before this generation's commit, so the tag SYSTEMATICALLY trails
        the LOGGED mapping_version by one commit (and reads v0 on the first generate after
        setup created the table; vNA only when the table itself is absent). The tag is a
        human label, and the dedupe here plus the provenance chain key on mapping_hash
        alone.

        The dedupe matches the tool's own CANONICAL filename pattern, never a bare hash
        suffix: this path is load-bearing (it lands in the complete row), so a foreign
        file named to embed the hash must neither suppress the real export nor have its
        own path recorded. The write is atomic (temp + os.replace, exactly like the role
        backup), so a torn export can never sit at the canonical name and win the dedupe
        on a later run."""
        import os

        mapping_version = Catalog.config_version(self.spark, self.mapping_table)
        ts = now_dt.strftime("%Y%m%d-%H%M%S")
        mapping_basename = self.mapping_table.rsplit(".", 1)[-1]
        version_tag = f"v{mapping_version}" if mapping_version is not None else "vNA"
        name = f"{mapping_basename}_{ts}_{version_tag}_{mapping_hash}.csv"
        canonical = re.compile(
            rf"^{re.escape(mapping_basename)}_\d{{8}}-\d{{6}}_v(?:\d+|NA)_"
            rf"{re.escape(mapping_hash)}\.csv$"
        )
        Deployment._require_contained("mapping-history export", self.mapping_history_dir)
        history_fullpath = f"/lakehouse/default/{self.mapping_history_dir}"
        existing = next(
            (f for f in Catalog._export_lister(history_fullpath) if canonical.match(f)),
            None,
        )
        if existing is not None:
            # This generation is already exported (same mapping_hash) -> reuse, write nothing.
            return f"{self.mapping_history_dir}/{existing}"
        lease.prewrite()
        os.makedirs(history_fullpath, exist_ok=True)
        versioned_relpath = f"{self.mapping_history_dir}/{name}"
        final_path = f"/lakehouse/default/{versioned_relpath}"
        lease.prewrite()
        df.toPandas().to_csv(final_path + ".tmp", index=False)
        lease.prewrite()
        os.replace(final_path + ".tmp", final_path)
        return versioned_relpath

    def generate(self, rebuild: bool = False) -> dict:
        """Short config -> validated role x scope mapping table + review CSV. Blocks on any
        validation error; refuses an empty config (removing everything must be a deliberate apply).
        `rebuild=False` (default) is IDEMPOTENT: if the config is unchanged since the last
        generate (its content-fingerprint config_hash already equals the one stamped in the mapping),
        it rebuilds nothing, logs a single 'no_change' row, and returns `changed=False` — so a pipeline
        can run generate every time and only re-plan/apply when the config actually changed.
        `rebuild=True` regenerates regardless.

        FIVE EXCEPTIONS. The skip also requires the mapping's stamped framework_version to
        equal the RUNNING version — an older (or missing) stamp means the content has not
        been validated under this version's rules, so it revalidates and re-stamps rather
        than attesting success to rules it never met.
        Of the content-driven four: the first two share one root — config_hash fingerprints the config ROWS,
        so it cannot see onelake_security_member — which is a live input to every member list.
          1. a config carrying a member WILDCARD is never eligible for the skip at all. Adding a
             principal to the member table leaves config_hash identical, and the skip would then
             report no_change while the newly matching principal is never granted, silently, on
             the path a pipeline runs every deploy. Such a config returns `changed=True` on every
             run; that is the cost of the member table being a live input to the member list.
          2. any config whose STAMPED member ids no longer match what the member table resolves
             its own member names to today regenerates rather than skipping. Editing a member
             row's objectId is invisible to config_hash in exactly the same way, and skipping
             then keeps deploying the stale id. See _member_ids_drifted.
          3. a mapping stamped for a DIFFERENT workspace/lakehouse than the ATTACHED one
             regenerates rather than skipping (see _target_drifted). Different root:
             config_hash cannot see WHERE the mapping was generated, and the plan/apply
             TARGET MISMATCH guard's remedy is 're-run mode=generate' — a skip blind to
             the target would make that remedy a permanent no-op.
          4. a member table carrying resolution errors (conflicting objectIds, one id under
             two names, an invalid type) never skips. The fast path must not certify as
             clean a state the full gate defines as a hard error — falling through lands
             in _run_validation, which blocks with the complete error list and the
             forensic 'rejected' row."""
        boundary_lease = self._begin_sensitive("generate")
        if not self.short_rows:
            self._reject(
                "0 active rows in short config — refusing to generate an empty security config"
            )
        # A member wildcard makes onelake_security_member a live input to the member list, and
        # config_hash cannot see it — so such a config is never eligible for the skip. Checked on
        # the RAW rows and scoped to the member columns: a TABLE glob must not disable it.
        # Wildcard-free configs go on to the second member-table guard inside the branch.
        if not rebuild and not Member.has_wildcard(self.short_rows):
            try:
                # The lock-file is read WHOLE, not projected: beyond the config_hash /
                # target / member-id checks below, the skip re-derives the generation's
                # mapping_hash to verify its review CSV and completion audit row exist (the
                # self-healing below), and that fingerprint spans every MAPPING_COLUMNS
                # value. The row count the skip envelope returns comes off the same collect
                # rather than a second count() job. .get() below, never ["..."]: a mapping
                # predating a column must read as "do not skip", not raise a KeyError.
                stamped = [r.asDict() for r in self.spark.table(self.mapping_table).collect()]
            except Exception:
                stamped = []
            # A uniformly old schema has no generation identity to certify, but generate is
            # its repair path: force a full rebuild. Mixed or partly-stamped rows may be
            # tampering, so they still fail closed rather than inheriting row-zero trust.
            stamps = {
                tuple(row.get(field) for field in MappingProvenance.FIELDS) for row in stamped
            }
            uniform_legacy = len(stamps) == 1 and any(
                value in (None, "") for value in next(iter(stamps), ())
            )
            if stamped and not uniform_legacy:
                MappingProvenance.require(stamped)
            if uniform_legacy:
                stamped = []
            # Cheap, in-memory gates first: config_hash, the stamping framework version, and
            # the target ids. The VERSION gate makes every new validation rule retroactive:
            # a mapping stamped by another framework version has not been validated under
            # THIS version's rules, so it falls through to the full revalidation-and-restamp
            # path instead of collecting fresh success attestations for content the current
            # rules may reject.
            cheap_ok = (
                bool(stamped)
                and str(stamped[0].get("config_hash")) == str(self.config_hash)
                and str(stamped[0].get("framework_version")) == __version__
                and not self._target_drifted(stamped)
            )
            # COST, stated rather than buried: the member cache is a FULL read of a
            # directory-sized table on the no-op fast path, so it loads ONLY once the cheap
            # checks above pass — and ONCE, here rather than inside _member_ids_drifted, so
            # the skip condition also sees the cache's ERROR list: a member table in an
            # error state (conflicting objectIds, one id under two names, an invalid type)
            # must never be certified as clean, unchanged state. Falling through lands in
            # _run_validation, which blocks with the full collect-all list and writes the
            # forensic 'rejected' row — no second error path to maintain.
            cache, cache_errors = {}, []
            if cheap_ok:
                cache, _spellings, cache_errors = self._load_member_cache()
            if cheap_ok and not cache_errors and not self._member_ids_drifted(stamped, cache):
                self.audit.set_config_provenance(
                    self.config_hash, Catalog.config_version(self.spark, self.config_table)
                )
                # SELF-HEALING: a prior run may have committed this very mapping and then
                # died on the CSV export or the completion audit rows — that failure was
                # loud, but a plain retry lands HERE. Before declaring no_change, make sure
                # the generation's review artifact and completion record exist, and repair
                # the one that does not: the CSV re-exports from the committed table (the
                # mapping_hash dedupe makes this a no-op whenever the file exists), and the
                # audit gains a run_complete row explicitly marked as a repair.
                self._boundary().require_desired_safe(stamped)
                mapping_hash = Hash.mapping_content(stamped)
                self.audit.set_mapping_provenance(
                    mapping_hash, Catalog.config_version(self.spark, self.mapping_table)
                )
                csv_relpath = self._export_history_csv(
                    self.spark.table(self.mapping_table),
                    mapping_hash,
                    datetime.datetime.now(datetime.timezone.utc),
                    boundary_lease,
                )
                if not self.audit.has_run_complete(mapping_hash):
                    self.audit.write(
                        [
                            self.audit.row("start", "success"),
                            self.audit.run_complete(
                                f"generate: completion record added for this generation in "
                                f"this environment (none was found: an interrupted earlier "
                                f"run, or a mapping generated under another env), "
                                f"csv={csv_relpath}",
                                operation="generate",
                            ),
                        ]
                    )
                # One binding, two consumers: the audit row and the returned envelope must
                # say the same thing, and as separate literals they could drift apart unseen.
                _skip_msg = "config unchanged since last generate — mapping not rebuilt (skip)"
                self.audit.write([self.audit.row("generate", "no_change", message=_skip_msg)])
                self._postcheck_after_write("generate", boundary_lease, csv_relpath)
                Say.out(
                    "detail",
                    "   ⏭️  config unchanged — nothing rebuilt (pass rebuild=True to force)",
                )
                return {
                    "changed": False,
                    "message": _skip_msg,
                    "data": {"grants": len(stamped)},
                }
        grants, all_errors, warnings, summary, canonical_lakehouse, _cfg = self._run_validation()
        for w in warnings:
            Say.out("info", "   ⚠️  ", w)
        if all_errors:
            for e in all_errors:
                Say.out("detail", "   ❌ ", e)
            # One 'rejected' audit row NAMES every error (validation + member + target) — the whole
            # fix-list, so a blocked generate leaves a complete forensic trace.
            self._reject(f"generate blocked: {len(all_errors)} error(s): " + " | ".join(all_errors))
        # Target guard passed. Stamp the attached workspace/lakehouse ids + the CANONICAL lakehouse
        # name (the API's actual spelling of config.lakehouse_name) onto every grant, so the mapping
        # stays a self-describing, auditable lock-file.
        for a in grants:
            a["workspace_id"] = self.client.workspace_id
            a["lakehouse_id"] = self.client.item_id
            a["workspace_name"] = self.workspace_name
            a["lakehouse_name"] = canonical_lakehouse
            a["tenant_id"] = self.tenant_id  # captured here -> the mapping is self-contained
        now_dt = datetime.datetime.now(datetime.timezone.utc)
        write_cols = MAPPING_COLUMNS + MAPPING_PROVENANCE_COLUMNS
        provenance = {
            "generated_at": now_dt.isoformat(),
            "config_hash": self.config_hash,
            "config_version": Catalog.config_version(self.spark, self.config_table),
            "framework_version": __version__,
        }
        records = [{**{c: grant.get(c) for c in MAPPING_COLUMNS}, **provenance} for grant in grants]
        # Explicit schema from TableSchema, so the frame matches the DDL that created the table:
        # generated_at is a real TIMESTAMP and config_version a real BIGINT. Explicit either way —
        # the optional columns (rls_condition, visible_columns, member_*) are None for many grants,
        # and Spark cannot infer a type for a column that is None in every row.
        schema, to_row = TableSchema.frame_schema(write_cols)
        data = [to_row(r) for r in records]
        df = self.spark.createDataFrame(data, schema)
        # overwriteSchema: the mapping is a DERIVED lock-file that generate rewrites in FULL every
        # run, and its schema is the framework's (MAPPING_COLUMNS + MAPPING_PROVENANCE_COLUMNS),
        # not the author's. Without this, an existing table whose schema disagrees — an older
        # generation's column set, or a table someone created by hand or loaded with inferred
        # types — makes Delta refuse the write (DELTA_FAILED_TO_MERGE_FIELDS) and BLOCKS generate
        # permanently, with an error naming neither the table nor a remedy. Correct here and
        # nowhere else: the log is append-only history and the config is author-owned, so neither
        # write may reshape its table.
        # Versioned-only CSV export: every generate writes an immutable, browsable backup
        # directly under mapping_history_dir, named with the generation identity and deduped
        # on mapping_hash (see _export_history_csv). Exported BEFORE the mapping commit: of
        # the two artifacts, the mapping table is the one plan/apply act on, so a failure
        # between the two steps must leave the mapping UNCOMMITTED (the loud error says
        # re-run generate) rather than committed with no review artifact behind it. An
        # orphaned CSV in the immutable history dir is harmless — the dedupe reuses it on
        # the retry. There is no stable fixed-name CSV; the returned path names the exact
        # generation and is recorded in the log.
        mapping_hash = Hash.mapping_content(records)
        self._boundary().require_desired_safe(records)
        versioned_relpath = self._export_history_csv(df, mapping_hash, now_dt, boundary_lease)
        try:
            boundary_lease.prewrite()
            df.write.option("overwriteSchema", "true").mode("overwrite").saveAsTable(
                self.mapping_table
            )
        except Exception as exc:  # noqa: BLE001 — re-raised below, class preserved
            # Name the artifact and the remedy. The raw Delta/Spark message names a column pair and
            # nothing else, so an operator cannot tell WHICH table is wrong or what to do about it.
            raise type(exc)(
                f"mapping write failed for {self.mapping_table} "
                f"({len(records)} row(s), {len(write_cols)} columns) — if this is a schema "
                f"conflict, the table predates this framework version; it is a derived lock-file, "
                f"so DROP TABLE {self.mapping_table} and re-run generate. Original: {exc}"
            ) from exc
        mapping_version = Catalog.config_version(self.spark, self.mapping_table)
        Say.out(
            "detail",
            f"   ✅ {len(records)} grant(s) → {self.mapping_table}\n   📄 {versioned_relpath}",
        )
        # Audit the run: config provenance (the SAME config_hash/config_version stamped into the
        # mapping lock-file above) + a run-level start/complete pair. NO per-grant 'validate' rows —
        # those are exclusive to apply (they feed grant_provenance's since-when enrichment;
        # generate emitting them would corrupt it). mode is already 'generate' from the run context.
        # The complete message carries the export path so the log row points at the generation.
        self.audit.set_config_provenance(provenance["config_hash"], provenance["config_version"])
        self.audit.set_mapping_provenance(mapping_hash, mapping_version)
        self.audit.write(
            [
                self.audit.row("start", "success"),
                self.audit.run_complete(
                    f"generate: {len(records)} grants, {len(warnings)} warnings, csv={versioned_relpath}",
                    operation="generate",
                ),
            ]
        )
        boundary_snapshot = self._postcheck_after_write(
            "generate", boundary_lease, versioned_relpath
        )
        return {
            "changed": True,
            "message": f"generate: {len(records)} grants across {len(summary)} role(s), {len(warnings)} warning(s)",
            "data": {
                "grants": len(records),
                "roles": len(summary),
                "warnings": len(warnings),
                "csv": versioned_relpath,
                "summary": summary,
                "lakehouse": {"name": canonical_lakehouse, "id": self.client.item_id},
                "workspace": {"name": self.workspace_name, "id": self.client.workspace_id},
                "dar_snapshot_safe": True,
                "workspace_isolation": self._isolation_state,
                "dar_etag": boundary_snapshot.etag,
            },
        }

    def _run_validation(self, canon=None):
        """The pure, WRITE-FREE validation pipeline shared by generate (which then writes) and
        validate (a zero-write dry-run). Runs the IDENTICAL checks — config validation
        (Generate.rows: every rule in the docs/architecture.md rule catalog, per-row AND cross-row —
        NOT enumerated here, the range goes stale on every new rule), the No-Graph member gate
        (cache load then name->objectId
        resolution), and the lakehouse target guard (config.lakehouse_name must name the ATTACHED
        lakehouse) — on the best-effort grants, AGGREGATING every error into ONE collect-all list.
        Returns (grants, all_errors, warnings, summary, canonical_lakehouse): all_errors non-empty =
        the config is blocked; on the clean path grants/canonical_lakehouse are what generate stamps
        into the mapping. WRITES to no control table — it READS config and member — so the
        caller owns the reject/write decision."""
        # explain() reaches this with NO client -- it makes no DAR call by design -- but it HAS
        # resolved the same ids, so it passes them in. Defaulting to self.client keeps every
        # existing caller unchanged; the alternative was explain() re-implementing the pipeline,
        # which is exactly the drift this method exists to prevent.
        # explain() builds its own catalog, because it injects a folder lister for the
        # unresolvable-target case, and it has no client to take the ids from. Letting it hand the
        # catalog in is what makes the rest of this pipeline reusable by a caller without a client.
        if canon is None:
            canon = Catalog.canonical(
                self.spark, workspace_id=self.client.workspace_id, item_id=self.client.item_id
            )
        # The member cache loads FIRST because member wildcards expand FROM it, and that expansion
        # has to happen before Generate.rows ever sees a member column. Generate.rows' own
        # signature is untouched — it simply receives rows whose member columns are already
        # literal — so none of its call sites move. self.short_rows stays RAW, so config_hash keeps
        # meaning "the config as authored": expand_wildcards returns NEW dicts and mutating them
        # here would stamp a hash plan/apply re-derive differently, deadlocking every wildcard
        # config at "STALE: short config changed after generate".
        cache, spellings, cache_errors = self._load_member_cache()
        rows, expand_errors = Member.expand_wildcards(self.short_rows, spellings)
        grants, errors, warnings, summary = Generate.rows(rows, canon)
        # A dead member pattern (C15) is something the CONFIG AUTHOR fixes, so it joins `errors`
        # rather than only `all_errors` — that subset is what explain() shows a preview on, and a
        # preview built after silently dropping the pattern would show a role the author never
        # wrote. Expansion runs first, so its errors read first.
        errors = expand_errors + errors
        # COLLECT-ALL: run every check on the best-effort grants and AGGREGATE all errors so the whole
        # fix-list surfaces in one pass. (1) config validation (Generate.rows) · (2) member gate —
        # cache load (member_type + case-collision) then name->objectId resolution (GUID-in-name,
        # missing member, config-side case-collision); No-Graph: the member table is the only source
        # · (3) lakehouse target guard — config.lakehouse_name must name the ATTACHED lakehouse.
        resolution_errors = Member.resolve_ids(grants, cache, rows)
        # The ONLY step here that needs a live client: it resolves the declared lakehouse against
        # the workspace's items through the Fabric API. explain() runs without a client by design
        # (no DAR call at all), so it gets the other two layers and skips this one — the single
        # documented difference between what explain() and validate() check.
        target_errors, canonical_lakehouse = (
            self._resolve_lakehouse_target()
            if self.client is not None
            else ([], self.lakehouse_name)
        )
        all_errors = errors + cache_errors + resolution_errors + target_errors
        # `errors` is handed back separately as well: it is the subset the CONFIG AUTHOR can fix,
        # and explain() shows a preview or not on exactly that distinction. A member the cache does
        # not carry or an unattached lakehouse says nothing about what the config would produce.
        return grants, all_errors, warnings, summary, canonical_lakehouse, errors

    # ---------- mode: validate ----------

    def validate(self) -> dict:
        """Dry-run the IDENTICAL validation pipeline generate runs (every rule in the
        docs/architecture.md rule catalog, the No-Graph member gate, the lakehouse target guard)
        with ZERO writes — no mapping, no CSV, and no log row (not even a
        'rejected' one). Any error blocks natively with the full collect-all list, surfacing the SAME
        error set generate would; a clean config returns a success envelope with the grant / role
        counts and EVERY warning in `data`, so an author can preview a generate before committing it.
        Because it never WRITES to the control tables, validate is safe to run against a live
        deployment (unlike generate, which overwrites the mapping)."""
        grants, all_errors, warnings, summary, canonical_lakehouse, _cfg = self._run_validation()
        for w in warnings:
            Say.out("info", "   ⚠️  ", w)
        if all_errors:
            for e in all_errors:
                Say.out("detail", "   ❌ ", e)
            # Native failure — RAISE WITHOUT WRITING. Unlike generate's _reject, validate leaves NO
            # forensic 'rejected' row: a read-only dry-run must not mutate the log. Same message shape
            # as generate so the two blocked envelopes surface the identical error set.
            raise SystemExit(
                f"validate blocked: {len(all_errors)} error(s): " + " | ".join(all_errors)
            )
        message = (
            f"validate: {len(grants)} grant(s) across {len(summary)} role(s), "
            f"{len(warnings)} warning(s) — dry-run, no writes"
        )
        Say.out("detail", message)
        return {
            "changed": False,
            "message": message,
            "data": {
                "grants": len(grants),
                "roles": len(summary),
                "warnings": warnings,
                "summary": summary,
                "lakehouse": {"name": canonical_lakehouse, "id": self.client.item_id},
                "workspace": {"name": self.workspace_name, "id": self.client.workspace_id},
            },
        }

    def _resolve_lakehouse_target(self):
        """Lakehouse target guard: config.lakehouse_name must name the ATTACHED lakehouse. Resolve it
        (case-insensitive) against the attached workspace's items -> canonical spelling; block if the
        config rows name more than one lakehouse, or the name is ambiguous / not found / resolves to a
        DIFFERENT lakehouse than the attached one (no silent fallback — an apply to the wrong lakehouse
        is data exposure). Returns (errors, canonical_name): errors non-empty = block; canonical_name is
        the API's spelling to stamp into the mapping (falls back to the attached name on any error so
        the caller can still build grants for the collect-all reject). A fully-missing lakehouse_name is
        already a validation error in Generate.rows."""
        declared = sorted(
            {
                str(r.get("lakehouse_name") or "").strip()
                for r in self.short_rows
                if str(r.get("lakehouse_name") or "").strip()
            }
        )
        if not declared:
            return [], self.lakehouse_name
        if len({d.lower() for d in declared}) > 1:
            return (
                [
                    f"config rows name more than one lakehouse: {declared} — one lakehouse per config"
                ],
                self.lakehouse_name,
            )
        try:
            canonical, resolved_id = self.client.resolve_lakehouse(declared[0])
        except TargetResolutionError as e:
            return [str(e)], self.lakehouse_name
        if str(resolved_id) != str(self.client.item_id):
            return (
                [
                    f"config names lakehouse '{declared[0]}' (id {resolved_id}) but the notebook is "
                    f"attached to lakehouse id {self.client.item_id} — attach to '{declared[0]}' or "
                    f"fix onelake_security_config.lakehouse_name"
                ],
                canonical,
            )
        return [], canonical

    def _load_member_cache(self):
        """Read onelake_security_member into ({(member_type, member_name.lower()): member_id},
        {(member_type, member_name.lower()): member_name AS WRITTEN}, errors)
        — the ONLY member-resolution source (No-Graph gate), preloaded ENTIRELY from
        the member sheet of onelake_security.xlsx. `errors` collects FOUR kinds, all from usable rows:
          1. invalid member_type — not one of Group/User/ServicePrincipal/ManagedIdentity;
          2. case collision — two rows of one type whose names differ only by case (different
             principals — a silent-wrong-resolve hazard);
          3. conflicting objectIds — one (type, name) carrying more than one distinct objectId
             (a stale GUID after a rotation, a copy-paste error), which resolves non-deterministically
             to the WRONG principal;
          4. duplicate objectId — one objectId listed as more than one principal, which reads as two
             principals in config and makes every id->name read (run_by, who_can_access) resolve by
             row order. Cannot be enforced on the table (Fabric takes PRIMARY KEY / UNIQUE only as
             NOT ENFORCED), which is why it lives here.
        Tolerates an ABSENT/empty table (returns ({}, {}, [])), so generate then blocks on the first
        missing member. A row missing any key/value column, or whose id is not a GUID, is skipped.

        objectIds are case-INSENSITIVE hex (GUID_RE accepts [0-9a-fA-F]), so one principal spelled in
        two letter cases is ONE identity: every id KEY and every id COMPARISON below is lower-cased.
        The invariant, stated precisely: ids are compared case-insensitively at every consumer EXCEPT
        DAR._canon_members, which is byte-exact BY DESIGN — it compares the whole desired/live members
        blob with json.dumps(..., sort_keys=True), a payload-EQUALITY check against what the service
        echoes back rather than an identity comparison, so it has no id-aware layer in which to fold
        case (see the note there). The STORED and EMITTED value is preserved byte-for-byte, as
        written. That is deliberate and load-bearing: the stored id feeds MAPPING_COLUMNS ->
        Hash.mapping_content -> mapping_hash, which names the mapping-history CSV and drives its reuse
        check, so normalizing it would write a duplicate history file on the first post-fix generate
        of an unchanged config."""
        if not self.spark.catalog.tableExists(self.member_table):
            return {}, {}, []
        cache, errors = {}, []
        spelling_sets = {}  # (member_type, name.lower()) -> {member_name spellings} (case-collision guard)
        id_sets = {}  # (member_type, name.lower()) -> {distinct member_ids, LOWERED} (conflicting-id guard)
        identities_by_id = {}  # member_id (LOWERED) -> {(member_type, member_name)} (duplicate-id guard)
        id_spellings = {}  # member_id (LOWERED) -> first-seen written spelling (errors echo as written)
        for r in self.spark.table(self.member_table).collect():
            d = Parse.trim_row(r.asDict())
            mtype, name = d.get("member_type"), d.get("member_name")
            raw = d.get("member_id")
            member_id = str(raw).strip() if raw is not None else ""
            # Trust a row ONLY if the id is a non-blank, GUID-shaped objectId. A blank or garbage id
            # (a common preload/export artifact — blank cells become "" not NULL) is skipped, so
            # generate blocks with the No-Graph member-gate error rather than deploying a wrong member.
            # `str(name or "")`, NOT `str(name)`: member_name is a nullable STRING column and
            # str(None) is the TRUTHY literal "None", so a SQL NULL name passed all three
            # operands, cached as (mtype, "none") spelled "None", and — unlike a blank name,
            # which Parse.list drops downstream — reached the mapping and the DAR payload with
            # no error at all. The display paths already used this idiom (Log.resolve_principal,
            # _member_display_names); the security gate did not, until a review caught it.
            if not mtype or not str(name or "").strip() or not GUID_RE.match(member_id):
                continue
            if str(mtype) not in MEMBER_TYPES:
                errors.append(
                    f"onelake_security_member: invalid member_type {mtype!r} for '{name}' — "
                    f"must be one of {sorted(MEMBER_TYPES)}"
                )
                continue
            # A member_name is UNADDRESSABLE if it carries a glob metacharacter or the list
            # separator, so the row is refused rather than stored. Entra permits all three
            # in a displayName; this framework cannot represent them:
            #   * / ?  -- a config value containing one is read as a PATTERN, so such a name could
            #             never be written literally; worse, it would expand against its NEIGHBOURS
            #             and grant a different principal with no error at all.
            #   ;      -- expansion re-joins matches with LIST_SEP and every downstream reader
            #             re-splits, so one such row becomes several members, injecting principals
            #             the pattern never matched and dropping the one it did.
            # Refusing at the table is the only seam where the ambiguity can be removed rather than
            # merely detected -- the same seam that already refuses a non-GUID member_id.
            _unaddressable = [ch for ch in ("*", "?", LIST_SEP) if ch in str(name)]
            if _unaddressable:
                errors.append(
                    f"onelake_security_member: member_name '{name}' contains "
                    f"{_unaddressable} — a member name cannot hold a wildcard metacharacter or "
                    f"'{LIST_SEP}'; rename the principal or grant it via a group"
                )
                continue
            key = (str(mtype), str(name).lower())
            id_lower = member_id.lower()  # the comparison/grouping key — never what is stored
            id_spellings.setdefault(id_lower, member_id)
            spelling_sets.setdefault(key, set()).add(str(name))
            id_sets.setdefault(key, set()).add(id_lower)
            # Per-ROW identity, for the duplicate-id guard below. Collected here rather than
            # rebuilt from the two aggregates above: those are independent unions over a key, so
            # pairing them would cross-multiply — a key holding two case-variant names AND two ids
            # would report every id under every name, accusing each of a duplication it has no
            # part in (the other two guards already reject that input; the accusation would just
            # be a false line in a collect-all list an author is meant to act on).
            identities_by_id.setdefault(id_lower, set()).add((str(mtype), str(name)))
            cache[key] = member_id  # STORED as written — see the invariant in the docstring
        # Case-collision guard (mirrors the config side + the table guard): two member rows of the
        # same type whose names differ only by case are different principals — a hard error naming both.
        for (mtype, lower), names in sorted(spelling_sets.items()):
            if len(names) > 1:
                errors.append(
                    f"onelake_security_member: ambiguous {mtype} member names differing only by "
                    f"case: {sorted(names)} — different principals, rename to disambiguate"
                )
        # Conflicting-id guard: the same (type, name) with >1 distinct objectId (a stale GUID after a
        # rotation, a copy-paste error) would resolve non-deterministically to the WRONG principal —
        # hard error naming both ids (one principal, one id). Distinctness is judged on the LOWERED
        # id, so one id spelled in two letter cases is not accused of conflicting with itself (an
        # availability false positive that blocked a valid config); the message echoes the written
        # spellings.
        for (mtype, lower), ids in sorted(id_sets.items()):
            if len(ids) > 1:
                errors.append(
                    f"onelake_security_member: {mtype} member "
                    f"'{sorted(spelling_sets[(mtype, lower)])[0]}' has conflicting objectIds "
                    f"{sorted(id_spellings[i] for i in ids)} — one principal maps to one id"
                )
        # Duplicate-id guard — the mirror of the one above, and the reason it has to live here:
        # uniqueness cannot be enforced on the table itself (Fabric accepts PRIMARY KEY / UNIQUE
        # only as NOT ENFORCED, so a declared constraint would document the rule without applying
        # it). One objectId listed as more than one principal is one principal wearing two
        # identities, which reads as two principals in config and makes every id->name read
        # (run_by, who_can_access) resolve by row order. The key is the whole (member_type,
        # member_name) identity, so a second TYPE counts as much as a second name: an objectId is
        # unique across principal types in Entra, so the same id under ServicePrincipal and
        # ManagedIdentity is one principal described twice — and a config row could otherwise pick
        # either and hand the DAR API an objectType the principal does not have.
        # Grouped on the LOWERED id: an objectId is case-insensitive hex, so the same id written in
        # two letter cases is ONE principal — left case-sensitive this guard stayed SILENT on it and
        # the cache handed out two keys for one principal, which then collected BOTH roles' access.
        for id_lower, identities in sorted(identities_by_id.items()):
            if len(identities) > 1:
                shown = ", ".join(f"{t} '{n}'" for t, n in sorted(identities))
                errors.append(
                    f"onelake_security_member: objectId {id_spellings[id_lower]} is listed as more "
                    f"than one principal: {shown} — one id is one principal, remove the duplicates"
                )
        # The cache keys on the LOWERED name and stores only the objectId, so nothing in it
        # can say how a principal is actually SPELLED. Member wildcards expand from
        # this table, and an expansion emitting the lowered key would land in member_*_names
        # -> MAPPING_COLUMNS -> Hash.mapping_content -> mapping_hash, disagreeing with a
        # literal row about case and moving the hash (and the history CSV name with it).
        # sorted()[0] rather than insertion order: a case collision is a COLLECTED error,
        # not a raise, so the cache is still built for that key — and Delta row order, which
        # is what insertion order would follow, is not stable run to run.
        spellings = {k: sorted(v)[0] for k, v in spelling_sets.items()}
        return cache, spellings, errors

    # ---------- mode: plan ----------

    def plan(self) -> dict:
        """Diff desired vs live. No drift -> write ONE 'no_change' log row and return changes=False
        (a pipeline stops here). Drift -> log ONLY the roles that differ, plus the plan 'complete' row
        that unlocks apply's drift gate, and return changes=True with the drift so the pipeline knows to
        apply. Changes nothing live either way."""
        boundary_lease = self._begin_sensitive("plan")
        plan = self._desired_state(boundary_lease.snapshot)
        affected = json.dumps({"table": self.audit.table}, sort_keys=True, separators=(",", ":"))
        changed = {name: act for name, act in plan.items() if act != "no_change"}
        if not changed:
            # One binding, two consumers — see the same shape in generate's skip path.
            _no_drift_msg = f"no drift — {len(plan)} role(s) already match live"
            self.audit.write([self.audit.row("plan", "no_change", message=_no_drift_msg)])
            boundary_snapshot = self._postcheck_after_write("plan", boundary_lease, affected)
            Say.out("detail", "   ✔️  no drift — live already matches config")
            return {
                "changed": False,
                "message": _no_drift_msg,
                "data": {
                    "counts": self._counts(plan),
                    "dar_snapshot_safe": True,
                    "workspace_isolation": self._isolation_state,
                    "dar_etag": boundary_snapshot.etag,
                },
            }
        self.audit.write(
            self.audit.action_rows(changed, "planned", "planned", "drift")
            + [self.audit.complete_row(plan, operation="plan")]
        )
        boundary_snapshot = self._postcheck_after_write("plan", boundary_lease, affected)
        Say.out("detail", f"   ⚠️   {len(changed)} role(s) drift — apply needed")
        omission_candidates = sorted(n for n, a in plan.items() if a == "omit")
        # `info` keeps the request-construction warning visible at the default level. The Preview
        # endpoint does not document deletion by omission, so these are prior-live candidates
        # requiring post-state review, never confirmed platform deletions.
        Say.out(
            "info",
            f"   ⚠️  apply payload will omit {len(omission_candidates)} prior-live role candidate(s): {', '.join(omission_candidates)} · review post-state"
            if omission_candidates
            else "   ⚠️  apply payload has no prior-live omission candidates",
        )
        return {
            "changed": True,
            "message": f"{len(changed)} role(s) drift — apply needed",
            "data": {
                "counts": self._counts(plan),
                "drift": dict(sorted(changed.items())),
                "plan": dict(sorted(plan.items())),
                "dar_snapshot_safe": True,
                "workspace_isolation": self._isolation_state,
                "dar_etag": boundary_snapshot.etag,
            },
        }

    # ---------- mode: apply ----------

    def apply(self, keep_unmanaged: bool = False) -> dict:
        """Submit the config-derived bulk DAR request. `keep_unmanaged=False` builds the
        full config payload; prior-live roles absent from that payload are omission candidates,
        not confirmed platform deletions. `keep_unmanaged=True` carries unmanaged live roles into
        the payload. Both branches require post-state review and a matching `plan` with no
        since-plan drift."""
        boundary_lease = self._begin_sensitive("apply")
        plan = self._desired_state(boundary_lease.snapshot)
        self._require_plan_and_no_drift(plan)
        # The EFFECTIVE conditional state, recorded on the complete row and in the result so
        # an unconditional write is always visible in the audit trail — including the silent
        # degradation where the service sent no ETag on the roles listing.
        if not self.if_match:
            if_match_effective = "unconditional (if_match=false)"
        else:
            if_match_effective = "conditional"
        # The restore point comes FIRST — on BOTH branches, before anything reaches the client.
        backup_path = self._backup_live_roles("incremental" if keep_unmanaged else "replace")
        if not keep_unmanaged:
            payload = DAR.merge_replace(self._desired)
            omission_candidates = sorted(
                {r["name"] for r in self._live} - {r["name"] for r in payload}
            )
            # This is a pre-request warning. Keep it at the default level, but do not turn a
            # candidate omitted from a Preview payload into a platform-deletion assertion.
            Say.out(
                "info",
                f"   ⚠️  submitting payload with {len(omission_candidates)} prior-live omission candidate(s): {', '.join(omission_candidates)} · review post-state"
                if omission_candidates
                else "   ⚠️  submitting payload with no prior-live omission candidates",
            )
            status = self._push_or_record_failure(
                payload, plan, backup_path, omission_candidates, keep_unmanaged
            )
            self._write_confirmed_completion(
                "apply",
                status,
                backup_path,
                self._header_rows
                + self.audit.action_rows(
                    plan,
                    "submitted",
                    "submitted as omission candidate; post-state review required",
                    "submitted",
                )
                + [
                    self.audit.complete_row(
                        plan,
                        operation="apply",
                        payload_hash=self._payload_hash(payload),
                        request="config_payload",
                        omitted_role_candidates=omission_candidates,
                        post_state_review_required=True,
                        backup_path=backup_path,
                        if_match=if_match_effective,
                    )
                ],
            )
        else:
            omission_candidates = []
            payload = DAR.merge_upsert(self._live, self._desired)
            if len(payload) > MAX_ROLES_PER_ITEM:
                # generate's ceiling bounds the CONFIG's roles, but this payload is live ∪
                # desired — unmanaged live roles ride along, and the platform enforces the
                # ceiling on the PUT as a whole. Refused here, before the push, rather than
                # dying at the API mid-apply.
                self._reject(
                    f"{len(payload)} roles in the merged incremental payload exceed the "
                    f"{MAX_ROLES_PER_ITEM}-per-item platform limit — keep_unmanaged=true "
                    f"keeps every unmanaged live role, so they count too; adopt or delete "
                    f"unmanaged roles, or run the default REPLACE (limit raisable to 1,000 "
                    f"via Azure Support)"
                )
            status = self._push_or_record_failure(
                payload, plan, backup_path, omission_candidates, keep_unmanaged
            )
            self._write_confirmed_completion(
                "apply",
                status,
                backup_path,
                self._header_rows
                + self.audit.action_rows(
                    plan,
                    "submitted",
                    "not submitted as an omission candidate (incremental payload)",
                    "drift",
                )
                + [
                    self.audit.complete_row(
                        plan,
                        operation="apply",
                        payload_hash=self._payload_hash(payload),
                        request="incremental_payload",
                        omitted_role_candidates=omission_candidates,
                        post_state_review_required=True,
                        backup_path=backup_path,
                        if_match=if_match_effective,
                    )
                ],
            )
        boundary_snapshot = self._postcheck_after_write(
            "apply", boundary_lease, backup_path, allow_dar_change=True
        )
        return {
            "changed": True,
            "message": f"apply ({'incremental' if keep_unmanaged else 'replace'})",
            "data": {
                # push_status confirms the request response, not a role count or a post-state
                # classification. roles_written is the body count OLAF submitted.
                "push_status": status,
                "roles_written": len(payload),
                "keep_unmanaged": keep_unmanaged,
                "counts": self._counts(plan),
                "request": "incremental_payload" if keep_unmanaged else "config_payload",
                "backup_path": backup_path,
                "omitted_role_candidates": omission_candidates,
                "drift_omission_candidates": sorted(n for n, a in plan.items() if a == "omit"),
                "post_state_review_required": True,
                "if_match": if_match_effective,
                "dar_snapshot_safe": True,
                "workspace_isolation": self._isolation_state,
                "dar_etag": boundary_snapshot.etag,
            },
        }

    def _push_or_record_failure(
        self, payload, plan, backup_path, omission_candidates, keep_unmanaged
    ):
        """_push, plus the forensic record a mid-push failure needs, then RE-RAISE.

        The bulk endpoint accepts an OLAF-constructed role payload. OLAF records any prior-live
        roles omitted from that payload as candidates, not deletion facts; a failure leaves the
        resulting platform state unknown. Before this,
        the exception propagated out of apply() before `audit.write` ran, so the whole trail was
        run_mode's generic handler: one row, mode=apply action=run status=failed, no role_name,
        data={}. Nothing said which roles were meant to land, what was actually live afterwards, or
        where the restore point was, so the state could not be reconstructed from the log at all.

        NOT a rollback, deliberately. A PUT that timed out may well have SUCCEEDED (the response
        was lost, not the write), so replaying the backup could silently UNDO an intended
        deployment — a second unlogged full-set write on top of an unknown state. The backup path
        is NAMED here so the operator can restore deliberately; it is never replayed automatically.

        The exception that comes out is always the ORIGINAL one. Everything this method does on the
        failure path is best-effort and cannot displace it: the re-read is recorded as "could not
        be determined" if it fails too (the outage that broke the PUT is usually still in force a
        second later), and the write itself is swallowed exactly as _reject's denial row is.

        The dryRun validation call runs FIRST and OUTSIDE the guarded region, so the record covers
        the real PUT and nothing else. A dryRun rejection (a 400 policy-validation error — the
        likeliest of the two failures) writes NOTHING live: its blast radius is exactly zero, and
        a record that presents an omission candidate as a confirmed deletion would be read as a
        half-applied deletion, which the
        operator answers by restoring the backup — a needless full-set PUT that rotates every role
        id. Chosen over carrying a `real_put_attempted` flag in the forensic JSON because the
        record's EXISTENCE then carries that fact structurally: there is no state in which the
        flag could disagree with reality (an interrupt arriving between the two calls would set a
        hand-maintained flag to True having attempted nothing).

        BaseException, not Exception: an operator cancelling a Fabric cell during a hanging PUT
        raises KeyboardInterrupt, which is precisely the "the PUT may have landed" scenario the
        record exists for, and it bypassed `except Exception` entirely. Widening is safe because
        the guarded body is a single call, and it changes nothing for the load-bearing SystemExit
        pipeline-fail signal: as with every other exception the record is written and the ORIGINAL
        is re-raised unchanged (SystemExit keeps its own 'guard' error_category via
        OLAFError.classify)."""
        self._dry_run(payload)
        self._prepared_intent(
            "apply",
            payload,
            omission_candidates,
            backup_path,
            keep_unmanaged,
            self._boundary_snapshot,
        )
        lease = self._current_lease()
        lease.prewrite()
        try:
            return self._push(payload, lease)
        except DARConflictError as exc:
            if exc.ambiguous:
                # A 412 on a RETRIED attempt is NOT the clean refusal below: the earlier
                # attempt may have COMMITTED before its response was lost to the transient
                # status that triggered the retry (a committed write rotates the collection
                # ETag and draws this very 412), or a concurrent edit landed between
                # attempts — undecidable from here. That is exactly the "write in an
                # unknown state" the mid-push record exists for: live re-read, per-role
                # PRESENT/ABSENT, the restore-point pointer, and per-grant wording that
                # says "NOT confirmed pushed" rather than the conflict record's certainty
                # that nothing was written.
                self._record_push_failure(
                    exc, payload, plan, backup_path, omission_candidates, keep_unmanaged
                )
                self._mark_unknown_write(exc, "apply", backup_path)
                raise
            # A FIRST-attempt 412 is the SERVICE refusing the write outright — zero blast
            # radius, exactly the dryRun rationale above: nothing landed, so the mid-push
            # record (live re-read, per-role PRESENT/ABSENT, the restore-point pointer)
            # would document an incident that did not happen — and point the operator at a
            # restore that would clobber the very concurrent edit the 412 just protected.
            # If the immutable DAR boundary still matches, the trail records the refusal:
            # header rows re-stamped failed + one 'push'/'rejected' row naming the remedy.
            # A changed boundary instead refuses that follow-up audit append and retains the
            # sentinel; the already-safe prepared intent remains the durable record.
            self._record_push_conflict(exc)
            exc.changed = False
            exc.possible_exposure = False
            exc.operation = "apply"
            exc.backup_path = backup_path
            raise
        except BaseException as exc:
            self._record_push_failure(
                exc, payload, plan, backup_path, omission_candidates, keep_unmanaged
            )
            self._mark_unknown_write(exc, "apply", backup_path)
            raise

    def _record_push_conflict(self, exc):
        """The FIRST-attempt 412 outcome: the service refused the conditional PUT because
        live roles changed after this run's read — NOTHING landed. (A RETRIED-attempt 412
        — `exc.ambiguous` — never reaches here: _push_or_record_failure routes it to the
        mid-push record, because there the write is in an unknown state.) The per-grant validate rows are
        re-stamped failed (they must not read as pushed), and one 'push'/'rejected' row
        names the conflict and remedy only while the immutable DAR snapshot still holds.
        Otherwise Log.prewrite refuses the append and the incident sentinel persists.
        Deliberately NOT _record_push_failure: that
        record's framing (live re-read, PRESENT/ABSENT per role, backup pointer) describes
        a write in an unknown state, and here the state is known exactly — unchanged.
        Best-effort like every failure write: a logging failure never masks the conflict."""
        rows = [
            row
            if row["action"] != "validate"
            else {
                **row,
                "status": "failed",
                "error_category": "http",
                "message": "conflict: the service refused the conditional PUT — "
                "this grant was NOT pushed",
            }
            for row in self._header_rows
        ]
        rows.append(
            self.audit.row(
                "push",
                "rejected",
                error_category="http",
                message=(
                    f"conflict — nothing was written: {exc} — live OneLake security "
                    f"changed after this run's read; re-run mode=plan and review the "
                    f"new diff (do NOT restore the pre-push backup: there is nothing to "
                    f"restore, and an unconditional restore would overwrite the "
                    f"concurrent change the conflict protected)"
                ),
            )
        )
        try:
            self.audit.write(rows)
        except Exception:  # noqa: BLE001 — the conflict itself must surface, not the log write
            pass

    def _record_push_failure(
        self, exc, payload, plan, backup_path, omission_candidates, keep_unmanaged
    ):
        """The rows _push_or_record_failure writes: the pre-push header rows (start + per-grant
        validate — facts that were already true), ONE row per planned role stating the intended
        action against what the re-read actually found, and one 'push' summary row whose message is
        the JSON forensic record (intended payload vs live state vs backup path).

        The 'validate' header rows are re-stamped `failed` before they go in, and that is not
        cosmetic: action='validate' + status='success' + mode in (apply, replace, rollback)
        is EXACTLY the predicate
        Log.grant_provenance / Audit.grants / Audit.trace read as "this grant was actually
        pushed". Emitted as-is on the failure path, a push that provably wrote nothing asserts it
        wrote everything. That corrupts the EARLIEST end permanently: first_applied/first_granted_by are
        fixed by the oldest establishing row and no later successful apply can displace them, so a
        false one becomes the origin story of a grant it never made. It also ADOPTS a
        grant made out-of-band in the Fabric UI that happens to name the same triple: the
        framework never wrote it, yet out_of_band() would stop reporting it. The rows are kept
        rather than dropped — the grant-grain validation genuinely happened, and it is the only
        record of WHICH member × scope the failed push was carrying — but they now carry the push
        failure's own status and error_category, so no provenance query counts them as a deploy.

        The per-role rows preserve the request distinction: keep_unmanaged=True carries prior-live
        roles, while the config payload records them only as omission candidates. That candidate
        label is retained in forensic JSON without asserting a platform outcome.

        `live_roles_after` stays None — never [] — when the re-read fails: an empty list is a
        claim that nothing is live, which is the single most dangerous thing to assert wrongly
        here. Both the summary row and every per-role row say "could not be determined" instead.

        error_category classifies the PUSH failure on every row, including when it was the re-read
        that also failed: the run failed because the PUT failed, and the re-read's own class is
        forensic detail carried in `live_read_error`, not the run's category."""
        live_after, live_read_error = None, None
        try:
            live_after = sorted(r["name"] for r in self.client.list_roles_quick())
        except Exception as reread_exc:
            live_read_error = f"{type(reread_exc).__name__}: {reread_exc}"
        category = OLAFError.classify(exc)

        def _state(name):
            if live_after is None:
                return "live state after the failure: could not be determined"
            return f"role is {'PRESENT' if name in live_after else 'ABSENT'} in live OneLake security afterwards"

        def _intent(action):
            if action == "omit" and keep_unmanaged:
                return "not submitted as omission candidate (incremental payload)"
            if action == "omit":
                return "submitted payload omission candidate; post-state review required"
            return f"intended {action}"

        rows = [
            row
            if row["action"] != "validate"
            else {
                **row,
                "status": "failed",
                "error_category": category,
                "message": "push failed before confirmation — this grant was NOT confirmed pushed",
            }
            for row in self._header_rows
        ]
        rows += [
            self.audit.row(
                "omission_candidate" if action == "omit" else action,
                "failed",
                role_name=name,
                error_category=category,
                message=f"push failed before confirmation — {_intent(action)}; {_state(name)}",
            )
            for name, action in sorted(plan.items())
        ]
        rows += [
            self.audit.row(
                "push",
                "unknown",
                error_category=category,
                message=json.dumps(
                    {
                        "schema": 1,
                        "operation": "apply",
                        "phase": "unknown",
                        "error": str(exc),
                        "intended_roles": sorted(r["name"] for r in payload),
                        "omitted_role_candidates": omission_candidates,
                        "post_state_review_required": True,
                        "keep_unmanaged": keep_unmanaged,
                        "live_roles_after": live_after,
                        "live_read_error": live_read_error,
                        "backup_path": backup_path,
                        "rolled_back": False,
                    }
                ),
            )
        ]
        try:
            self.audit.write(rows)
        except Exception:
            pass  # a logging failure must never replace the push failure — same rule as _reject

    def _backup_live_roles(self, mode_word):
        """Capture the pre-request live role representations under role_backup_dir and return
        the Files/ relpath. Both payload-construction branches capture this recovery input before
        submitting a DAR request. The artifact does not prove an omission outcome, make the request
        atomic, or guarantee exact platform-state restoration.

        Costs no extra API call — self._live was already fetched for the drift gate. Same IO as
        generate's mapping-history CSV: build the /lakehouse/default path, makedirs, write. Stored
        as the BARE LIST FabricClient.put_roles() accepts (it wraps the {"value": ...} body itself)
        and WITHOUT the server-assigned id/etag, so a notebook restore is json.load ->
        client.put_roles(roles) with nothing to unwrap and nothing to strip.

        Every apply gets its own file — NO dedupe on identical content the way generate reuses a
        matching export: two applies from the same prior state are still two events, each retaining
        its own recovery input. The timestamp leads so a name sort is a time sort, and mode_word
        records which request-construction branch produced the artifact.

        NEVER overwrites an existing file. `ts` is second-granularity and mode_word/config_hash
        repeat, so a retry or two pipelines on one lakehouse CAN reach the same name — and the file
        that would be overwritten is the first request's pre-request capture. Sub-second precision
        only narrows that window; clocks are not a
        uniqueness source. Two mechanisms close it instead: the audit batch_token (a fresh uuid4 per
        run_mode call) separates the ordinary concurrent writers, and the name is then CLAIMED with
        an exclusive create ("x") that the OS resolves as one step — on FileExistsError a `_2`, `_3`, …
        sequence suffix is appended (after the extension-free stem, so a name sort stays a time
        sort) and the claim is retried, so a name is only ever written by the run that won
        the create. The loop terminates because every iteration tries a name it has not tried.

        ALL-OR-NOTHING content: the claim leaves a zero-byte placeholder, the JSON is written to a temp file
        beside it, and os.replace() swaps it in. A crash or short write therefore surfaces HERE, at
        write time, and aborts the apply — instead of leaving a truncated file at the name
        data.backup_path and the log row both advertise as a valid restore point (that failure would
        only surface at restore time, mid-incident). An ordinary failure removes both the temp
        file and placeholder. If cleanup's fresh boundary check is unsafe, it retains the
        recovery artifacts still present and the sentinel as incident evidence, and attaches that fact to
        the original write error instead of masking it.

        NOT best-effort: a failed write propagates and aborts the apply (run_mode turns it into a
        failed envelope). A backup that fails open is worse than none — the envelope would advertise
        a restore point that does not exist. The failure is RE-RAISED naming the artifact and its
        path, because the bare OSError says only e.g. "No space left on device" — indistinguishable
        from any other crash for whoever is on call. It also makes the poisoned-directory case
        legible: makedirs(exist_ok=True) still raises FileExistsError when a FILE sits where the
        directory belongs, which anyone who can write Files/security/ (a lower privilege than the
        workspace Member apply itself needs) can arrange in order to block every apply."""
        import contextlib
        import os

        if self._control_depth <= 0:
            raise ControlDataGuardError(
                "role backup requires an active sensitive-operation sentinel"
            )
        self._boundary()._read_sentinel()
        lease = self._current_lease()

        ts = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
        stem = (
            f"onelake_security_roles_{ts}_{mode_word}_{self.config_hash}_{self.audit.batch_token}"
        )
        Deployment._require_contained("role backup", self.role_backup_dir)
        dirpath = f"/lakehouse/default/{self.role_backup_dir}"
        relpath = f"{self.role_backup_dir}/{stem}.json"
        # Serialise without server-assigned id/etag fields so the artifact holds the role
        # representations OLAF captured, rather than stale response metadata. It is request input
        # for a reviewed recovery decision, not a promise that any restore is accepted or exact.
        # self._live is not mutated: the incremental branch still uses it to build its payload.
        restorable = [{k: v for k, v in r.items() if k not in ("id", "etag")} for r in self._live]
        cleanup_fact = None
        try:
            lease.prewrite()
            os.makedirs(dirpath, exist_ok=True)
            seq = 1
            while True:
                name = f"{stem}.json" if seq == 1 else f"{stem}_{seq}.json"
                relpath = f"{self.role_backup_dir}/{name}"
                try:
                    lease.prewrite()
                    open(f"{dirpath}/{name}", "x", encoding="utf-8").close()
                except FileExistsError:
                    seq += 1
                    continue
                break
            tmp = f"{dirpath}/.{name}.tmp"
            try:
                lease.prewrite()
                with open(tmp, "w", encoding="utf-8") as fh:
                    json.dump(restorable, fh)
                lease.prewrite()
                os.replace(tmp, f"{dirpath}/{name}")
            except BaseException as exc:
                cleanup_boundary = None
                for stray in (tmp, f"{dirpath}/{name}"):
                    try:
                        lease.prewrite()
                    except ControlDataGuardError as cleanup_exc:
                        cleanup_boundary = cleanup_exc
                        break
                    with contextlib.suppress(OSError):
                        os.remove(stray)
                if cleanup_boundary is not None:
                    cleanup_fact = {
                        "operation": "apply",
                        "artifacts_retained": True,
                        "reason": f"{type(cleanup_boundary).__name__}: {cleanup_boundary}",
                    }
                    try:
                        exc.cleanup_boundary = cleanup_fact
                    except Exception:
                        pass
                raise
        except OSError as exc:
            # The CLASS is re-raised, not a flattened OSError: FileExistsError is the tell that
            # distinguishes a poisoned directory from a full disk, and run_mode's envelope message
            # is f"{type(e).__name__}: {e}" — so the class IS the first word the operator reads.
            cleanup_note = (
                f"; cleanup boundary: {cleanup_fact['reason']}; recovery artifacts retained"
                if cleanup_fact is not None
                else ""
            )
            wrapped = type(exc)(f"role backup write failed ({relpath}): {exc}{cleanup_note}")
            if cleanup_fact is not None:
                try:
                    wrapped.cleanup_boundary = cleanup_fact
                except Exception:
                    pass
            raise wrapped from exc
        Say.out("detail", f"role backup ({len(self._live)} live role(s)) → {relpath}")
        return relpath

    # ---------- modes: show-* (read-only, no log) ----------

    def _config_rows_at_version(self, version):
        result = self.spark.sql(f"SELECT * FROM {self.config_table} VERSION AS OF {int(version)}")
        rows = []
        for row in result.collect():
            raw = row.asDict()
            by_lower = {str(k).lower(): v for k, v in raw.items()}
            active = by_lower.get("active")
            if active is not True and str(active).strip().lower() not in ("true", "1", "yes"):
                continue
            missing = [c for c in CONFIG_AUTHOR_COLUMNS if c.lower() not in by_lower]
            if missing:
                raise UsageError(
                    f"historical config v{version} is missing required columns {missing}"
                )
            rows.append(Parse.trim_row({c: by_lower[c.lower()] for c in CONFIG_AUTHOR_COLUMNS}))
        return rows

    def _historical_rollback_payload(self, rows, version):
        if not rows:
            raise SystemExit(
                f"rollback historical config v{version} has 0 active rows — refusing an empty target"
            )
        saved = self._short_rows_cache
        self._short_rows_cache = [dict(row) for row in rows]
        try:
            grants, errors, _warnings, _summary, canonical_lakehouse, _cfg = self._run_validation()
        finally:
            self._short_rows_cache = saved
        if errors:
            raise SystemExit(
                f"rollback historical config v{version} is invalid: " + " | ".join(errors)
            )
        for grant in grants:
            grant["workspace_id"] = self.client.workspace_id
            grant["lakehouse_id"] = self.client.item_id
            grant["workspace_name"] = self.workspace_name
            grant["lakehouse_name"] = canonical_lakehouse
            grant["tenant_id"] = self.tenant_id
        self._boundary().require_desired_safe(grants)
        return DAR.build_desired(grants, self.tenant_id)

    def _probe_rollback_artifacts(self, lease=None):
        import os

        lease = self._current_lease() if lease is None else lease
        if lease is not self._current_lease():
            raise ControlDataGuardError(
                "rollback artifact probe requires the active control-data sentinel lease"
            )

        for folder in (self.mapping_history_dir, self.role_backup_dir):
            Deployment._require_contained("rollback artifact probe", folder)
            base = f"/lakehouse/default/{folder}"
            probe = f"{base}/.olaf-rollback-probe-{self.audit.batch_token}"

            def _cleanup_probe():
                try:
                    lease.prewrite()
                    os.remove(probe)
                except OSError as exc:
                    raise OSError(
                        f"rollback artifact probe cleanup failed for {folder}: {exc}"
                    ) from exc

            try:
                lease.prewrite()
                os.makedirs(base, exist_ok=True)
                lease.prewrite()
                with open(probe, "x", encoding="utf-8"):
                    pass
                try:
                    with open(probe, encoding="utf-8") as handle:
                        if handle.read() != "":
                            raise OSError("rollback artifact probe read-back was not empty")
                except BaseException:
                    _cleanup_probe()
                    raise
            except BaseException as exc:
                raise type(exc)(f"rollback artifact preflight failed for {folder}: {exc}") from exc
            else:
                _cleanup_probe()

    def _require_rollback_schemas(self):
        for table, expected in (
            (self.mapping_table, MAPPING_COLUMNS + MAPPING_PROVENANCE_COLUMNS),
            (self.log_table, LOG_COLUMNS),
        ):
            if not self.spark.catalog.tableExists(table):
                raise UsageError(f"rollback preflight requires table {table}")
            live = {str(c).lower() for c in self.spark.table(table).columns}
            missing = [c for c in expected if c.lower() not in live]
            if missing:
                raise UsageError(
                    f"rollback preflight: {table} is missing required columns {missing}"
                )

    def _mark_rollback_progress(self, exc, changed, phase, from_v, to_v, target_hash):
        values = {
            "changed": changed,
            "possible_exposure": changed is not False,
            "operation": "rollback",
            "backup_path": f"{self.config_table} version {from_v}",
            "rollback_progress": {
                "phase": phase,
                "from_version": from_v,
                "to_version": to_v,
                "target_config_hash": target_hash,
            },
        }
        for name, value in values.items():
            try:
                setattr(exc, name, value)
            except Exception:
                pass
        return exc

    def _require_pinned_rollback_hash(self, target_hash, phase, from_v, to_v):
        actual = Hash.config(Catalog.active_config_rows(self.spark, self.config_table))
        if actual != target_hash:
            exc = SystemExit(
                f"rollback target config hash changed before {phase}: expected "
                f"{target_hash}, observed {actual}; config was restored but the remaining "
                f"mapping/DAR stages are blocked"
            )
            self._mark_rollback_progress(
                exc, True, f"blocked-before-{phase}", from_v, to_v, target_hash
            )
            raise exc
        return actual

    def rollback(self, to_version: int | str = "", reason: str = "") -> dict:
        """Preflight a historical config, durably prepare, restore it, then regenerate and submit
        the resulting config-derived DAR request. OLAF does not certify an exact platform state."""
        reason = str(reason).strip()
        if not reason:
            raise SystemExit("rollback requires a reason (set rollback_reason)")
        versions = sorted(
            int(r["version"])
            for r in self.spark.sql(f"DESCRIBE HISTORY {self.config_table}").collect()
        )
        if not versions:
            raise SystemExit("rollback: config table has no Delta history")
        from_v = versions[-1]
        if str(to_version).strip():
            to_v = int(to_version)
            if to_v not in set(versions):
                raise SystemExit(
                    f"rollback: config version {to_v} not found — history has {versions}"
                )
        else:
            if len(versions) < 2:
                raise SystemExit("rollback: no previous config version exists to roll back to")
            to_v = versions[-2]
        source_rows = Catalog.active_config_rows(self.spark, self.config_table)
        source_hash = Hash.config(source_rows)
        try:
            pre_rows = [r.asDict() for r in self.spark.table(self.mapping_table).collect()]
        except Exception:
            pre_rows = []
        if pre_rows:
            MappingProvenance.require(pre_rows)
            self._require_target_identity(pre_rows, audit_rejection=False)
        target_rows = self._config_rows_at_version(to_v)
        target_hash = Hash.config(target_rows)
        target_payload = self._historical_rollback_payload(target_rows, to_v)
        self._dry_run(target_payload)
        self._require_rollback_schemas()
        boundary = self._boundary()
        approved = boundary.snapshot()
        boundary_lease = self._begin_sensitive("rollback", snapshot=approved)
        self._probe_rollback_artifacts(boundary_lease)
        current = max(
            int(r["version"])
            for r in self.spark.sql(f"DESCRIBE HISTORY {self.config_table}").collect()
        )
        if current != from_v:
            exc = SystemExit(
                f"rollback config version changed during preflight: expected {from_v}, "
                f"observed {current}; nothing was restored"
            )
            self._mark_rollback_progress(
                exc, False, "pre-restore-version-race", from_v, to_v, target_hash
            )
            self._finish_sensitive(boundary_lease)
            raise exc
        prepared = {
            "schema": 1,
            "operation": "rollback",
            "phase": "prepared",
            "from_version": from_v,
            "to_version": to_v,
            "source_config_hash": source_hash,
            "target_config_hash": target_hash,
            "reason": reason,
            "workspace_id": self.client.workspace_id,
            "lakehouse_id": self.client.item_id,
            "etag": approved.etag,
            "isolation_attestation": self.control_data_isolation_attestation,
            "reserved_digest": approved.reserved_digest,
        }
        self.audit.write(
            [
                self.audit.row(
                    "rollback",
                    "prepared",
                    message=json.dumps(prepared, sort_keys=True, separators=(",", ":")),
                )
            ]
        )
        try:
            boundary_lease.prewrite()
            self.spark.sql(f"RESTORE TABLE {self.config_table} TO VERSION AS OF {to_v}")
        except BaseException as exc:
            self._mark_rollback_progress(
                exc, None, "restore-outcome-unknown", from_v, to_v, target_hash
            )
            raise
        self._short_rows_cache = None
        try:
            self._require_pinned_rollback_hash(target_hash, "restored-audit", from_v, to_v)
            restored = {**prepared, "phase": "restored"}
            self._write_confirmed_completion(
                "rollback",
                "delta-restore",
                f"{self.config_table} version {from_v}",
                [
                    self.audit.row(
                        "rollback",
                        "restored",
                        message=json.dumps(restored, sort_keys=True, separators=(",", ":")),
                    )
                ],
            )
            self._require_pinned_rollback_hash(target_hash, "generate", from_v, to_v)
            gen = self.generate(rebuild=True)
            self._require_pinned_rollback_hash(target_hash, "plan", from_v, to_v)
            self.plan()
            self._require_pinned_rollback_hash(target_hash, "apply", from_v, to_v)
            app = self.apply(keep_unmanaged=False)
        except BaseException as exc:
            if not isinstance(exc, PostWriteAuditError):
                self._mark_rollback_progress(
                    exc, True, "post-restore-chain-blocked", from_v, to_v, target_hash
                )
            raise
        boundary_lease.accept_dar_write()
        post_snapshot = self._postcheck_after_write(
            "rollback",
            boundary_lease,
            app.get("data", {}).get("backup_path"),
            allow_dar_change=True,
        )
        return {
            "changed": True,
            "message": f"rollback to config v{to_v}: {reason}",
            "data": {
                "rollback": {
                    "from_version": from_v,
                    "to_version": to_v,
                    "reason": reason,
                    "source_config_hash": source_hash,
                    "target_config_hash": target_hash,
                },
                "generate": gen.get("data", {}),
                "apply": app.get("data", {}),
                "dar_snapshot_safe": True,
                "workspace_isolation": self._isolation_state,
                "dar_etag": post_snapshot.etag,
            },
        }

    def _record_reset_conflict(self, exc, backup_path):
        try:
            self.audit.write(
                [
                    self.audit.row(
                        "push",
                        "rejected",
                        error_category="http",
                        message=json.dumps(
                            {
                                "schema": 1,
                                "operation": "reset",
                                "phase": "rejected",
                                "error": str(exc),
                                "backup_path": backup_path,
                                "changed": False,
                            },
                            sort_keys=True,
                        ),
                    )
                ]
            )
        except Exception:
            pass

    def _record_reset_unknown(self, exc, live_names, backup_path):
        live_after = live_error = None
        try:
            live_after = sorted(r["name"] for r in self.client.list_roles_quick())
        except Exception as reread_exc:
            live_error = f"{type(reread_exc).__name__}: {reread_exc}"
        message = {
            "schema": 1,
            "operation": "reset",
            "phase": "unknown",
            "error": str(exc),
            "intended_roles": [],
            "omitted_role_candidates": live_names,
            "post_state_review_required": True,
            "live_roles_after": live_after,
            "live_read_error": live_error,
            "backup_path": backup_path,
            "rolled_back": False,
        }
        try:
            self.audit.write(
                [
                    self.audit.row(
                        "push",
                        "unknown",
                        error_category=OLAFError.classify(exc),
                        message=json.dumps(message, sort_keys=True),
                    )
                ]
            )
        except Exception:
            pass

    def reset(self) -> dict:
        """DESTRUCTIVE, interactive-only containment request. OLAF submits an empty bulk
        DAR payload and records the roles observed before submission as omission candidates. The
        Preview contract does not establish deletion-by-omission, no-OneLake-security, or
        universal access-denial behavior; inspect the post-state in the intended engine/access mode.

        OLAF does not recreate platform-managed/default roles or infer their membership behavior.
        Its pre-request backup is a recovery input, not an exact platform-state restoration
        guarantee. If that capture fails, OLAF aborts before submitting the empty payload.

        Leaves the control tables alone: the config, the mapping and the audit log all survive, so
        `generate` -> `plan` -> `apply` redeploys everything config declares. Use `cleanup()` for
        the tables.
        """
        boundary_snapshot = self._boundary().snapshot()
        boundary_lease = self._begin_sensitive("reset", snapshot=boundary_snapshot)
        self._live = boundary_snapshot.roles
        # Snapshot the token BESIDE the read (as _desired_state does) — client.roles_etag
        # is mutable and a later read would silently swap the token under this run.
        reset_etag = boundary_snapshot.etag
        live_names = sorted(r["name"] for r in self._live)
        backup_path = self._backup_live_roles("reset")
        self.audit.write(
            [self.audit.row("start", "success", message=f"reset: {len(live_names)} live role(s)")]
        )
        self._prepared_intent("reset", [], live_names, backup_path, None, boundary_snapshot)
        # reset's own list_roles above captured the collection ETag — carried here so even
        # this full wipe refuses to race a concurrent edit (see FabricClient.put_roles).
        boundary_lease.prewrite()
        try:
            status = self.client.put_roles(
                [],
                etag=reset_etag if self.if_match else None,
                allow_unconditional=not self.if_match,
            )
            boundary_lease.accept_dar_write()
        except DARConflictError as exc:
            if exc.ambiguous:
                self._record_reset_unknown(exc, live_names, backup_path)
                self._mark_unknown_write(exc, "reset", backup_path)
            else:
                self._record_reset_conflict(exc, backup_path)
                exc.changed = False
                exc.possible_exposure = False
                exc.operation = "reset"
                exc.backup_path = backup_path
            raise
        except BaseException as exc:
            self._record_reset_unknown(exc, live_names, backup_path)
            self._mark_unknown_write(exc, "reset", backup_path)
            raise
        self._write_confirmed_completion(
            "reset",
            status,
            backup_path,
            [
                self.audit.row(
                    "omission_candidate",
                    "submitted",
                    role_name=n,
                    message="prior-live role omitted from submitted empty payload; post-state review required",
                )
                for n in live_names
            ]
            + [
                self.audit.row(
                    "complete",
                    "success",
                    message=(
                        f"reset: submitted empty payload; {len(live_names)} prior-live role candidate(s); "
                        f"post-state review required; backup artifact {backup_path}"
                    ),
                )
            ],
        )
        post_snapshot = self._postcheck_after_write(
            "reset", boundary_lease, backup_path, allow_dar_change=True
        )
        Say.out(
            "detail",
            f"   💥 reset — submitted empty payload; {len(live_names)} prior-live role candidate(s): {', '.join(live_names) or '(none)'} · post-state review required",
        )
        return {
            "push_status": status,
            "request": "empty_payload",
            "prior_live_role_candidates": live_names,
            "backup_path": backup_path,
            "post_state_review_required": True,
            "dar_snapshot_safe": True,
            "workspace_isolation": self._isolation_state,
            "dar_etag": post_snapshot.etag,
        }

    def clear_incident(self, access_review) -> dict:
        """Record reviewed clearance under the existing marker, then remove it only
        while the same safe immutable DAR snapshot still holds."""
        if self.audit is None:
            raise UsageError("incident clearance requires the durable audit log")
        boundary = self._boundary()
        boundary._require_evidence(access_review, "access review")
        approved = boundary.snapshot()
        boundary._read_sentinel()
        boundary.require_same(approved, boundary.snapshot())
        evidence = {
            "schema": 1,
            "access_review": access_review,
            "dar_etag": approved.etag,
            "dar_roles_digest": approved.roles_digest,
            "reserved_digest": approved.reserved_digest,
            "dar_observed_at": approved.observed_at,
            "exposure_remediated": False,
        }
        clearance_lease = ControlBoundaryLease(
            boundary, "sentinel_clearance", approved, owns_sentinel=False
        )
        prior_prewrite = self.audit._prewrite
        self.audit._prewrite = clearance_lease.prewrite
        try:
            self.audit.write(
                [
                    self.audit.row(
                        "sentinel_clearance",
                        "reviewed",
                        message=json.dumps(evidence, sort_keys=True, separators=(",", ":")),
                    )
                ]
            )
        finally:
            self.audit._prewrite = prior_prewrite
        result = boundary.clear_incident(access_review, approved=approved)
        result.update({"access_review": access_review, "dar_etag": approved.etag})
        return result

    def cleanup(self) -> dict:
        """DESTRUCTIVE, interactive-only, and the one operation with NO way back.

        Drops all four control tables and deletes every file under the mapping-history and
        role-backup folders. That is the authored config, the generated mapping, the member table,
        the entire audit history, every mapping-history CSV, and every pre-apply role backup —
        including the backups that would otherwise be the recovery for a bad apply or a `reset()`.
        After this there is nothing left to restore FROM.

        For standing up a NEW environment, where the lakehouse carries leftovers from a trial run
        and you want `setup()` to start from nothing. Not a maintenance operation.

        Does NOT touch the live data access roles — a role deployed before this stays deployed,
        now with no audit trail explaining it. The returned frame says so when it finds any, and
        `reset()` is the one that clears them; run it FIRST if that is what you want, because
        afterwards its backup is gone too.

        It cannot log what it did: the log table is one of the things it drops. The returned frame
        and the printed lines are the only record this run ever produces — the sole operation in
        this framework whose trail is not durable, which is itself a reason to keep it rare.
        """
        import os

        # Belt-and-braces re-check of __init__'s path guard, BEFORE anything is destroyed:
        # nothing is dropped past this point unless both delete roots are contained.
        for folder in (self.mapping_history_dir, self.role_backup_dir):
            Deployment._require_contained("cleanup", folder)
        dropped, files, not_removed, live_roles = [], [], [], []
        try:
            live_roles = sorted(r["name"] for r in self.client.list_roles()) if self.client else []
        except Exception as exc:  # noqa: BLE001 — a DAR read must not stop a lakehouse cleanup
            live_roles = [f"(could not read live roles: {type(exc).__name__})"]
        for table in (self.config_table, self.mapping_table, self.log_table, self.member_table):
            try:
                self.spark.sql(f"DROP TABLE IF EXISTS {table}")
                dropped.append(table)
                Say.out("detail", f"   💥 dropped {table}")
            except Exception as exc:  # best-effort containment; name every survivor
                not_removed.append(f"{table} ({type(exc).__name__})")
        for folder in (self.mapping_history_dir, self.role_backup_dir):
            base = f"/lakehouse/default/{folder}"
            try:
                names = sorted(os.listdir(base))
            except Exception:  # noqa: BLE001 — an absent folder is a clean state, not a failure
                continue
            for name in names:
                full = f"{base}/{name}"
                if os.path.realpath(full) == os.path.realpath(ControlBoundary.SENTINEL_FULL_PATH):
                    not_removed.append(f"{folder}/{name} (incident sentinel preserved)")
                    continue
                try:
                    os.remove(full)
                    files.append(f"{folder}/{name}")
                except Exception:  # noqa: BLE001 — a subfolder or locked file is reported, not fatal
                    not_removed.append(f"{folder}/{name}")
            Say.out("detail", f"   💥 cleared {folder} ({len(names)} entr(y/ies))")
        return {
            "dropped": dropped,
            "files_deleted": files,
            "not_removed": not_removed,
            "live_roles_left": live_roles,
            "incident_sentinel_preserved": True,
            "exposure_remediated": False,
            "containment_warning": (
                "exposure_remediated=false; cleanup cannot retract prior reads, and Delta "
                "history, caches, copies, and prior readers remain outside this cleanup"
            ),
        }

    def show(self, by: str, subject: str) -> dict:
        """One read mode, three axes: by = table | role | member + subject.
        Pivots the live DAR state and enriches every grant from onelake_security_log
        (first_applied/first_granted_by, last_applied/last_granted_by, config_version), flagging
        out-of-band grants that have no log row.
        Invalid/missing `by` aborts listing the valid axes. Result carries `by` for pipeline branching."""
        axis = str(by or "").strip().lower()
        if axis not in SHOW_AXES:
            raise SystemExit(
                f"show requires by={'|'.join(SHOW_AXES)} — the pivot axis (got {by!r})"
            )
        path_hit = member_hit = None
        if axis == "role":

            def predicate(r, paths, members):
                return Parse.subject_match(subject, r["name"])
        elif axis == "member":
            names = self._member_display_names()

            def member_hit(member):
                """Accept either spelling of a member subject: the objectId the live DAR
                exposes, or the display name from onelake_security_member. Matching the id
                ALONE meant the documented `by=member: objectId/name` was half true — a name
                came back 0 matches, and the enriched rows this very mode returns carry that
                same name in `member_name`, so it was the obvious thing to paste back in."""
                return Parse.subject_match(
                    subject, member, names.get(str(member).lower()) or member
                )

            def predicate(r, paths, members):
                return any(member_hit(m) for m in members)
        else:  # table

            def path_hit(path):
                """Accept either spelling of a table subject: the raw /Tables/<schema>/<table>
                path or its dotted schema.table form. ScopePath.to_table does the conversion
                leaves a non-table scope (a /Files/ folder) alone; spelling it inline here
                turned `/Files/raw` into `.Files.raw`, a candidate nobody meant to offer."""
                return Parse.subject_match(subject, path, ScopePath.to_table(path))

            def predicate(r, paths, members):
                return any(path_hit(p) for p in paths)

        return self._show(subject, predicate, axis, path_hit, member_hit)

    # ---------- private: shared steps ----------

    def _require_target_identity(self, exp_rows, audit_rejection=True):
        """TARGET-IDENTITY GUARD over the mapping lock-file: EVERY row's stamped
        workspace_id/lakehouse_id must name the ATTACHED target. config_hash is
        content-based, so control tables copied to another lakehouse (a dev→prod
        promotion, a re-created item) pass the STALE guard — only the stamped ids can
        tell the environments apart, and a mismatch means principals resolved for one
        environment are about to be deployed to another. Checked over ALL rows, not just
        the first (a poisoned tail row must not hide behind a clean lead row); ids
        compared stripped + lowercased (GUIDs are case-insensitive); a row with
        missing/blank ids cannot prove its target and is refused the same way. Shared by
        _desired_state (plan/apply) and rollback (against the PRE-restore mapping)."""
        stamped = {
            (
                str(r.get("workspace_id") or "").strip().lower(),
                str(r.get("lakehouse_id") or "").strip().lower(),
            )
            for r in exp_rows
        }
        attached = (
            str(self.client.workspace_id or "").strip().lower(),
            str(self.client.item_id or "").strip().lower(),
        )
        if any(not ws or not lh for ws, lh in stamped) or stamped != {attached}:
            shown = " · ".join(
                f"workspace {ws or '(unstamped)'} / lakehouse {lh or '(unstamped)'}"
                for ws, lh in sorted(stamped)
            )
            reason = (
                f"TARGET MISMATCH: mapping rows were generated for {shown} but this run "
                f"targets workspace {self.client.workspace_id} / lakehouse "
                f"{self.client.item_id} — re-run mode=generate against this lakehouse"
            )
            if audit_rejection:
                self._reject(reason, config_version=exp_rows[0].get("config_version"))
            raise SystemExit(reason)

    def _desired_state(self, boundary_snapshot=None):
        """Read the mapping lock-file (the ONLY build input), enforce the staleness and
        target-identity guards, build desired roles, fetch live roles, return the diff."""
        exp_rows = [r.asDict() for r in self.spark.table(self.mapping_table).collect()]
        if not exp_rows:
            raise SystemExit("mapping table empty — run mode=generate first")
        provenance = MappingProvenance.require(exp_rows)
        # Stamp mapping provenance BEFORE the staleness/target guards so a guard-denial row carries
        # mapping_hash/mapping_version too (auditors want denials — mirrors config_hash on _reject).
        self._mapping_hash = Hash.mapping_content(exp_rows)
        self.audit.set_mapping_provenance(
            self._mapping_hash, Catalog.config_version(self.spark, self.mapping_table)
        )
        if provenance["config_hash"] != self.config_hash:
            # STALE: version ambiguous for a superseded generation -> config_version stays None
            self._reject("STALE: short config changed after generate — re-run generate")
        self._require_target_identity(exp_rows)
        # config_version is read back from the lock-file (captured at generate time), not re-queried:
        # it must name the config generation this mapping was built from, even if the live config
        # table has since advanced to a newer Delta commit version.
        self.audit.set_config_provenance(self.config_hash, provenance["config_version"])
        tenants = {str(row.get("tenant_id") or "").strip() for row in exp_rows}
        if len(tenants) != 1:
            raise SystemExit(
                "TENANT MISMATCH: mapping rows carry more than one tenant identity — re-run mode=generate"
            )
        tenant = tenants.pop() or self.tenant_id  # mapping is the source of truth
        self._desired = DAR.build_desired(exp_rows, tenant)
        boundary = self._boundary()
        boundary.require_desired_safe(exp_rows)
        self._boundary_snapshot = boundary_snapshot or boundary.snapshot()
        self._live = self._boundary_snapshot.roles
        # The SAME read that feeds the drift gate also supplies the If-Match token: the
        # collection ETag captured here rides to put_roles, so the service itself refuses
        # a write against roles that changed in the seconds after this read.
        self._live_etag = self._boundary_snapshot.etag
        self._header_rows = self.audit.run_header_rows(Generate.to_log_grants(exp_rows))
        plan = DAR.diff(self._desired, self._live)
        Say.out("detail", json.dumps(dict(sorted(plan.items())), indent=2))
        return plan

    def _require_plan_and_no_drift(self, plan):
        """Three write-guards: (1) a successful plan for this exact config AND mapping
        generation must exist in the log — the plan is bound to the mapping content it
        reviewed, so a mapping regenerated after plan (same config rows, different
        member/scope resolution) voids it,
        (2) the live diff must still equal that recorded plan (drift check — interim stand-in for
        If-Match), (3) every member objectId must be a GUID — a safety net over the member_*_ids the
        mapping already carries (generate resolved the display names to objectIds from onelake_security_member)."""
        if not self._mapping_hash:
            # _desired_state sets this before any caller reaches the gate; falsy here means
            # the gate is being driven out of order — never unlock apply on an
            # unfingerprinted mapping.
            self._reject("no mapping fingerprint in hand — run mode=plan first")
        record = self.audit.find_plan_record(self.config_hash, self._mapping_hash)
        if record is None:
            self._reject(
                "no successful plan found for this config + mapping state — a plan unlocks "
                "apply only for the exact mapping generation it reviewed, so a mapping "
                "regenerated since plan (changed member/scope resolution) voids it — re-run "
                "mode=plan and review the diff"
            )
        if record.get("plan") != dict(sorted(plan.items())):
            self._reject(
                "live OneLake security changed since plan (drift) — re-run mode=plan and review the new diff"
            )
        unresolved = sorted(
            {
                m["objectId"]
                for r in self._desired
                for m in r["members"]["microsoftEntraMembers"]
                if not GUID_RE.match(str(m["objectId"]))
            }
        )
        if unresolved:
            self._reject(
                f"member objectIds must be GUIDs — the mapping holds an unresolved value, "
                f"re-run mode=generate: {unresolved[:5]}"
            )

    def _dry_run(self, payload):
        """The zero-write validation PUT. Deliberately its own method: _push_or_record_failure
        calls it OUTSIDE the guarded region, so a dryRun rejection — which changes nothing live —
        can never produce the mid-push forensic record."""
        Say.out("detail", "dryRun:", self.client.put_roles(payload, dry_run=True))

    def _push(self, payload, lease):
        """The real bulk PUT — the only write, and the only call the forensic record covers.
        Carries the If-Match ETag captured by _desired_state's live read (the same read the
        drift gate compares against), so the service refuses a write against roles that
        changed in the seconds since — see FabricClient.put_roles. if_match=False (the
        escape hatch) sends it unconditionally."""
        status = self.client.put_roles(
            payload,
            etag=self._live_etag if self.if_match else None,
            allow_unconditional=not self.if_match,
        )
        lease.accept_dar_write()
        Say.out(
            "detail",
            f"push status: {status} | roles written: {len(payload)} "
            f"| live roles before: {len(self._live)} | OLS_ sync ~5 min",
        )
        return status

    def _reject(self, reason, config_version=None):
        """Audit a denied plan/apply attempt BEFORE the guard aborts, so blocked attempts
        leave a forensic 'rejected' trace in onelake_security_log — who tried, when, why, which config
        (the audit trail is for denials too, not only successes). config_version is stamped when
        the guard knows it (target-label; TARGET MISMATCH — the mapping generation is current,
        so its captured version is in hand); left None when ambiguous (STALE — the mapping is a stale
        generation). Best-effort: a logging failure must never mask the guard, so the write is
        swallowed and the SystemExit is always raised."""
        try:
            self.audit.set_config_provenance(self.config_hash, config_version)
            self.audit.write([self.audit.row("guard", "rejected", message=reason)])
        except Exception:
            pass
        raise SystemExit(reason)

    def _member_display_names(self):
        """objectId (lowered) -> member_name, from onelake_security_member.

        Read only on the member axis, and only to widen what `subject` accepts. Deliberately
        NOT the validating read _load_member_cache performs: this is a display lookup, so a
        row with a blank name or a non-GUID id is simply absent from the map rather than an
        error — a caller searching by objectId must not be blocked by someone else's bad row.
        A missing table is an empty map, and matching falls back to the id alone."""
        if not self.spark.catalog.tableExists(self.member_table):
            return {}
        rows = (Parse.trim_row(r.asDict()) for r in self.spark.table(self.member_table).collect())
        return {
            str(d["member_id"]).lower(): str(d["member_name"])
            for d in rows
            if str(d.get("member_id") or "").strip() and str(d.get("member_name") or "").strip()
        }

    def _show(self, subject, hit, by, path_hit=None, member_hit=None):
        """Pivot live DAR roles by a predicate, enriched with audit provenance per grant.
        Join key = (role_name, scope_path, member objectId) — the live DAR exposes objectIds, so the
        log join is on member_id; the display name (member_name) rides along as a human-facing extra.
        Each matching (role x path x member) grant is annotated first_applied / first_granted_by /
        last_applied / last_granted_by / config_version / member_name from onelake_security_log; a live grant with NO
        log row is flagged 'out-of-band — no framework provenance' (a Fabric-UI edit the framework
        did not make) rather than hidden — the audit trail covers grants this framework never wrote."""
        if not subject:
            raise SystemExit(
                "show requires subject — by=role: role name · by=table: schema.table "
                "or /Tables/ path · by=member: objectId/name (globs ok)"
            )
        # show is strictly read-only: it reads live DAR state + the log for enrichment and writes
        # NO audit row — a read must never mutate state or provenance (plan/apply write rows,
        # show deliberately does not).
        provenance = self.audit.grant_provenance()
        lead = GRANT_LEAD_COLUMNS[by]  # show() refused anything outside SHOW_AXES
        columns = lead + [c for c in GRANT_COLUMNS if c not in lead]
        hits, grants = [], []
        for r in self.client.list_roles():
            paths, members = DAR.paths_and_members(r)
            if not hit(r, paths, members):
                continue
            hits.append(r["name"])
            perms = DAR.path_permissions(r)
            # Narrow to what was ASKED FOR, not everything the matching role happens to
            # reach. Selecting whole roles and then emitting every path × every member
            # answered "what else can whoever reaches this reach" — a real question, but not
            # the one `subject=` is asking, and the extra rows read as grants on tables the
            # caller never named, or as access held by people they never asked about.
            # by=role narrows neither: asking about a role means asking for all of it.
            paths = [p for p in paths if path_hit(p)] if path_hit else paths
            members = [m for m in members if member_hit(m)] if member_hit else members
            Say.out("detail", f"— {r['name']}: paths={paths} members={members}")
            for path in paths:
                for member in members:
                    prov = provenance.get(
                        (str(r["name"]).lower(), str(path).lower(), str(member).lower())
                    )
                    grant = {
                        "role_name": r["name"],
                        "scope_path": path,
                        "member": member,
                        "member_name": None,
                        "permission": perms.get(path),
                        "first_applied": None,
                        "first_granted_by": None,
                        "last_applied": None,
                        "last_granted_by": None,
                        "config_version": None,
                        "provenance": "out-of-band — no framework provenance",
                    }
                    if prov is not None:
                        grant.update(
                            provenance="framework",
                            first_applied=prov["first_applied"],
                            first_granted_by=prov["first_granted_by"],
                            last_applied=prov["last_applied"],
                            last_granted_by=prov["last_granted_by"],
                            config_version=prov["config_version"],
                            member_name=prov["member_name"],
                        )
                    grants.append({c: grant[c] for c in columns})
        return {
            "changed": False,
            # A 0-match result is where the exact-match rule is most likely to surprise, and
            # the one place the caller is already looking — so the hint rides the message
            # rather than the frame, whose shape pipelines depend on.
            "message": f"show by={by} subject={subject!r}: {len(hits)} match(es)"
            + ("" if hits else " — matched exactly; add * for a partial match"),
            "data": {
                "by": by,
                "subject": subject,
                "matches": len(hits),
                "roles": hits,
                "grants": grants,
                "out_of_band": sum(1 for g in grants if g["provenance"] != "framework"),
            },
        }

    @staticmethod
    def _counts(plan):
        from collections import Counter

        return dict(sorted(Counter(plan.values()).items()))

### `Audit` — read-only generation-tracing / audit queries over `onelake_security_log`.

Pairs with `Log`: `Log` WRITES the trail, `Audit` READS it. No SQL required — filters the in-memory log rows via `spark.table(...).collect()` the same way `Log.find_plan_record`/`grant_provenance` already do. `runs()`/`log_history()`/`batch()`/`failures()`/`last_run()` are the log query methods; `current_generation()`/`is_stale()`/`verify_chain()` (plus the `ChainStatus` result dataclass) are freshness/integrity checks over the mapping/config/log tables -- `is_stale()` mirrors the exact projection `Deployment.config_hash` uses, so it agrees with the real STALE guard.

In [ ]:
# ══ Audit ════════════════════════════════════════════════════════════════════
# Read-only generation-tracing / audit queries over onelake_security_log (pairs with Log's writes).
@dataclass(frozen=True)
class ChainStatus:
    """Result of Audit.verify_chain(): ok is True when the current generation's
    config_hash/mapping_hash both appear somewhere in the log (the chain is intact);
    details carries the per-hash booleans (or the no-mapping reason) for diagnostics."""

    ok: bool
    details: dict


# config_diff's per-row scope key: the raw onelake_security_config table has no literal
# scope_path column (that only exists post-generate, on onelake_security_mapping) -- a config row's
# own scope-DEFINING columns are include_tables/exclude_tables/include_folders/exclude_folders (the
# same columns Generate._DUP_KEY_COLS treats as identifying a role's authored scope, e.g. the
# config-examples.md carve-out recipe's two SalesTH rows differ only here). Joined deterministically
# for a stable (role_name, scope) diff key. Every OTHER authored column is compared field-by-field.
_CONFIG_DIFF_SCOPE_COLUMNS = (
    "include_tables",
    "exclude_tables",
    "include_folders",
    "exclude_folders",
)
_CONFIG_DIFF_FIELD_COLUMNS = [
    c for c in CONFIG_AUTHOR_COLUMNS if c not in {"role_name", *_CONFIG_DIFF_SCOPE_COLUMNS}
]

# A plausible Delta time-travel timestamp/date literal: YYYY-MM-DD, optionally with a space- or
# T-separated HH:MM[:SS[.fraction]] clock. config_at/at interpolate `date` RAW into
# TIMESTAMP AS OF '<date>', so this shape guard rejects an injection payload the way int() already
# rejects a non-numeric `version` (see _timetravel_sql).
_DELTA_TIMESTAMP_RE = re.compile(r"^\d{4}-\d{2}-\d{2}([ T]\d{2}:\d{2}(:\d{2}(\.\d+)?)?)?$")

# Rendered by _member_names in place of a display name when the member cache table carries MORE
# THAN ONE distinct name for one objectId. Log.resolve_principal already refuses to
# pick on exactly this data ("an unresolved value is honest, an arbitrary one is not"), but it can
# afford to fall back to the bare id -- Audit cannot: out_of_band/who_can_access/drift ALREADY
# render the bare id for an id ABSENT from the table, and drift's docstring documents comparing
# member_id with member_name as how a caller DETECTS that fallback. So ambiguity gets its own
# rendering, distinct from a real name AND from the absent-id fallback: a bracketed sentence no
# Entra display name or UPN can collide with, self-explanatory without docs, and BOUNDED -- it
# reports HOW MANY names the table holds, never lists them (a row must not grow with the data).
_AMBIGUOUS_MEMBER_NAME = "<ambiguous: {n} names in member table>"


class MappingProvenance:
    """Validate the generation identity carried by every mapping row.

    A mapping table is one generated lock-file, not a bag of independently trustworthy
    rows. Any missing or divergent provenance stamp makes the whole generation unusable.
    """

    FIELDS = ("config_hash", "config_version", "framework_version", "generated_at")

    @classmethod
    def require(cls, rows):
        if not rows:
            return None
        result = {}
        for field in cls.FIELDS:
            values = {row.get(field) for row in rows}
            if None in values or "" in values or len(values) != 1:
                raise UsageError(
                    f"mapping provenance is mixed or unstamped for {field} across all rows; "
                    "re-run generate before treating this mapping as a generation"
                )
            result[field] = values.pop()
        return result


class Audit:
    """Read-only generation-tracing / audit queries over the control tables (no SQL required).
    Pairs with Log: Log WRITES the trail, Audit READS it."""

    def __init__(
        self,
        spark,
        config_table,
        mapping_table,
        log_table,
        client=None,
        member_table="olaf.onelake_security_member",
    ):
        self.spark = spark
        self.config_table = config_table
        self.mapping_table = mapping_table
        self.log_table = log_table
        self.client = client
        self.member_table = member_table  # onelake_security_member: id->name cache (who_can_access)

    def _log_rows(self, **eq):
        """onelake_security_log rows as dicts, equality-filtered on the given columns (a filter whose
        value is None is skipped, not matched against — the caller's "not specified" case).
        The filters are pushed into Spark (.where, the pattern Log.find_plan_record set) so
        the driver never collects rows the caller is about to drop — the log grows one row
        per grant per apply, and collecting it whole made every Audit query scale with the
        table's lifetime. One exception, for identical results: a value that cannot ride
        inside a SQL string literal safely falls back to the original Python-side
        comparison — a quote, OR a backslash: Spark processes backslash escapes in string
        literals by default (escapedStringLiterals=false), so 'CONTOSO\\alice' would
        unescape into a comparison against the WRONG value and a trailing backslash would
        swallow the closing quote entirely (ParseException). AD-style names make both
        realistic inputs. The CI fakes cannot model that escape processing, so this guard
        must stay conservative — as must any future widening of it. Non-strings fall back
        too."""
        frame = self.spark.table(self.log_table)
        python_side = {}
        for k, v in eq.items():
            if v is None:
                continue
            if isinstance(v, str) and "'" not in v and "\\" not in v:
                frame = frame.where(f"{k} = '{v}'")
            else:
                python_side[k] = v
        rows = [r.asDict() for r in frame.collect()]
        for k, v in python_side.items():
            rows = [r for r in rows if r.get(k) == v]
        return rows

    def _df(self, rows, columns=None):
        """Explicit all-string schema, same rationale as Log.write: a log row's optional
        columns (role_name/scope_path/message/...) are None on many rows, and Spark cannot infer a
        type for a column that is None across the whole result. LOG_COLUMNS is the fallback column
        list for an empty result, so it still comes back typed rather than schema-less."""
        from pyspark.sql.types import StructType, StructField, StringType

        cols = list(rows[0].keys()) if rows else (columns or LOG_COLUMNS)
        schema = StructType([StructField(c, StringType(), True) for c in cols])
        data = [[None if r.get(c) is None else str(r[c]) for c in cols] for r in rows]
        return self.spark.createDataFrame(data, schema)

    def runs(
        self,
        mode: str | None = None,
        status: str | None = None,
        env: str | None = None,
        since: str | None = None,
        batch_id: str | None = None,
    ) -> "DataFrame":
        """Run-header/complete rows (and every other logged row), newest first. `since` is an
        inclusive run_at floor (string compare — run_at is an ISO-8601 timestamp, so lexical order
        matches chronological order)."""
        rows = self._log_rows(mode=mode, status=status, env=env, batch_id=batch_id)
        if since is not None:
            rows = [r for r in rows if (r.get("run_at") or "") >= since]
        rows.sort(key=lambda r: str(r.get("run_at")), reverse=True)
        return self._df(rows)

    def log_history(
        self, role: str | None = None, member: str | None = None, scope: str | None = None
    ) -> "DataFrame":
        """Rows touching a given subject (role/member/scope), oldest first (a chronological story
        of what happened to that subject, as opposed to runs()' newest-first operational view)."""
        rows = self._log_rows(role_name=role, member_name=member, scope_path=scope)
        rows.sort(key=lambda r: str(r.get("run_at")))
        return self._df(rows)

    def batch(self, batch_id: str) -> "DataFrame":
        """Every row written by one apply/plan invocation (its batch_id) -- the full blast radius
        of a single run."""
        return self._df(self._log_rows(batch_id=batch_id))

    def failures(self, since: str | None = None) -> "DataFrame":
        """Rows that are not a clean success: either a non-success status or a populated
        error_category. Newest first, optionally floored by `since` (see runs())."""
        rows = [
            r for r in self._log_rows() if r.get("status") != "success" or r.get("error_category")
        ]
        if since is not None:
            rows = [r for r in rows if (r.get("run_at") or "") >= since]
        rows.sort(key=lambda r: str(r.get("run_at")), reverse=True)
        return self._df(rows)

    def last_run(self, mode: str | None = None) -> dict | None:
        """The single newest row for a mode (or overall, if mode is None), as a plain dict --
        None when there is nothing logged yet."""
        rows = self._log_rows(mode=mode)
        return max(rows, key=lambda r: str(r.get("run_at"))) if rows else None

    def _mapping_rows(self):
        """Read and validate the entire mapping lock-file before interpreting any row."""
        rows = [r.asDict() for r in self.spark.table(self.mapping_table).collect()]
        return rows, MappingProvenance.require(rows)

    def current_generation(self) -> dict | None:
        """The all-row-validated mapping generation, or None before the first generate."""
        rows, provenance = self._mapping_rows()
        if not rows:
            return None
        return {
            **provenance,
            "mapping_hash": Hash.mapping_content(rows),
            "mapping_version": Catalog.config_version(self.spark, self.mapping_table),
        }

    def is_stale(self) -> bool:
        """True when the deployed mapping no longer matches the live active config -- the
        SAME comparison _desired_state()'s STALE guard makes (config_hash property vs.
        the mapping's stamped config_hash), so this agrees with what a real run would do.
        The rows come from Catalog.active_config_rows — the ONE reader
        Deployment.short_rows also uses — so both sides project to CONFIG_AUTHOR_COLUMNS
        and trim the same way; a private copy of that read here would drift, and then a
        config carrying incidental whitespace, or a foreign column another framework
        added, would read stale forever. No mapping yet counts as stale (nothing has been
        generated against the live config)."""
        gen = self.current_generation()
        if gen is None:
            return True
        active = Catalog.active_config_rows(self.spark, self.config_table)
        return gen["config_hash"] != Hash.config(active)

    @staticmethod
    def _message_object(row):
        try:
            message = json.loads(row.get("message") or "")
        except (TypeError, ValueError):
            return None
        return message if isinstance(message, dict) else None

    @classmethod
    def _completion_message(cls, row):
        message = cls._message_object(row)
        return message if message and message.get("schema") == 1 else None

    def last_successful_deployment(self, mode: str | None = None) -> dict | None:
        """Newest completed apply leg, including rollback's apply leg, with durable proof."""
        candidates = []
        for row in self._log_rows():
            message = self._message_object(row)
            completion = self._completion_message(row)
            if (
                row.get("mode") in ((mode,) if mode else ("apply", "rollback"))
                and row.get("action") == "complete"
                and row.get("status") == "success"
                and row.get("config_hash")
                and row.get("mapping_hash")
                and message is not None
                and (
                    (
                        completion is not None
                        and completion.get("operation") == "apply"
                        and completion.get("backup_path")
                        and completion.get("payload_hash")
                    )
                    or (
                        "schema" not in message
                        and isinstance(message.get("plan"), dict)
                        and isinstance(message.get("backup_path"), str)
                        and bool(message["backup_path"].strip())
                    )
                )
            ):
                candidates.append(row)
        return max(candidates, key=lambda row: str(row.get("run_at"))) if candidates else None

    def verify_chain(self) -> "ChainStatus":
        """Require ordered, versioned generate -> plan -> apply completions for this mapping."""
        gen = self.current_generation()
        if gen is None:
            return ChainStatus(False, {"reason": "no mapping"})
        stage_rows = {"generated": [], "planned": [], "applied": []}
        expected = {"generated": "generate", "planned": "plan", "applied": "apply"}
        for row in self._log_rows():
            message = self._completion_message(row)
            if (
                row.get("action") != "complete"
                or row.get("status") != "success"
                or row.get("config_hash") != gen["config_hash"]
                or row.get("mapping_hash") != gen["mapping_hash"]
                or str(row.get("run_at") or "") < str(gen["generated_at"])
                or message is None
            ):
                continue
            for stage, operation in expected.items():
                modes = (
                    ("generate", "rollback")
                    if stage == "generated"
                    else (("plan", "rollback") if stage == "planned" else ("apply", "rollback"))
                )
                if row.get("mode") in modes and message.get("operation") == operation:
                    if stage != "applied" or (
                        message.get("backup_path") and message.get("payload_hash")
                    ):
                        stage_rows[stage].append(row)
                    break
        counts = {stage: len(rows) for stage, rows in stage_rows.items()}
        details = {
            "config_hash": gen["config_hash"],
            "mapping_hash": gen["mapping_hash"],
            "generated": bool(stage_rows["generated"]),
            "planned": bool(stage_rows["planned"]),
            "applied": bool(stage_rows["applied"]),
            "stage_rows": counts,
        }
        if not stage_rows["generated"]:
            details["state"] = "missing_mapping"
            return ChainStatus(False, details)
        details["state"] = "generated"
        for generated in stage_rows["generated"]:
            for planned in stage_rows["planned"]:
                if str(planned.get("run_at")) < str(generated.get("run_at")):
                    continue
                details["state"] = "planned"
                for applied in stage_rows["applied"]:
                    if str(applied.get("run_at")) >= str(planned.get("run_at")):
                        details["state"] = "applied"
                        return ChainStatus(True, details)
        details["state"] = "incomplete"
        return ChainStatus(False, details)

    def grants(
        self, role: str | None = None, scope: str | None = None, member: str | None = None
    ) -> "DataFrame":
        """Establishing DAR grants read from the log -- one row per (role_name, scope_path,
        member_id) with provenance (member_name / first_applied /
        first_granted_by / last_applied / last_granted_by / config_version). The
        DataFrame-of-rows analogue of Log.grant_provenance's keyed dict, mirroring its grain
        EXACTLY: a 'validate' + 'success' row whose mode actually pushed the grant (apply —
        including the apply leg of a rollback chain, whose rows are stamped mode=rollback,
        matching the plan-gate loader's precedent; a FAILED apply re-stamps its rows, so a
        broken push still establishes nothing)
        and that names a member_id; deduped on the LOWERED (role, scope, member_id) triple, keeping
        BOTH ends of the run_at range plus the principal who pushed each end, and the ORIGINAL-case
        display values from the LATEST row (the state in effect now). Deliberately no single `since`:
        the log records what OLAF did and cannot see a role deleted straight from the Fabric UI, so
        one apply, an out-of-band deletion and a re-apply are indistinguishable here from one
        unbroken grant. Reporting only the earliest claimed a continuity nothing observed; reporting
        only the latest would reset on every routine re-deploy of an unchanged config, which is
        exactly the question an access review asks. Each END is exactly knowable, so both ship, and
        a wide gap between them is the operator's cue to look rather than OLAF's cue to guess. Optional role/scope/member narrow the listing (member matches the display name, as
        log_history() does). Aggregates across ALL envs -- unlike grant_provenance, which scopes to one
        env; Audit is cross-env by design (a noted follow-up, not a bug)."""
        rows = [
            r
            for r in self._log_rows(role_name=role, scope_path=scope, member_name=member)
            if r.get("action") == "validate"
            and r.get("status") == "success"
            # 'replace' is a RETIRED pre-release mode: its full-truth semantics became
            # apply(keep_unmanaged=False), so no path here can emit it. It stays in the
            # predicate because this table is append-only history — a workspace deployed
            # before the fold still holds rows stamped with it, and dropping the token
            # would silently re-read those grants as out-of-band, with no provenance.
            and r.get("mode") in ("apply", "replace", "rollback")
            and r.get("member_id") is not None
        ]
        agg = {}
        # Each end is established by COMPARISON, never by position. An earlier version sorted on
        # str(run_at) and let the last row win; str(None) is 'None', which sorts after every ISO
        # date ('N' > '2'), so one row with no run_at silently became the latest end and blanked
        # last_applied. Log.grant_provenance computes the same grain and skips such a row, so the
        # two disagreed -- while this docstring claimed they mirror each other EXACTLY.
        for r in rows:
            key = self._triple_key(r["role_name"], r["scope_path"], r["member_id"])
            run_at = r.get("run_at")
            cur = agg.get(key)
            if cur is None:
                agg[key] = {
                    "role_name": r["role_name"],
                    "scope_path": r["scope_path"],
                    "member_id": r["member_id"],
                    "member_name": r.get("member_name"),
                    "first_applied": run_at,
                    "first_granted_by": r.get("run_by"),
                    "last_applied": run_at,
                    "last_granted_by": r.get("run_by"),
                    "config_version": r.get("config_version"),
                }
                continue
            if run_at is None:
                # cannot be placed at either end, so it displaces neither
                continue
            if cur["first_applied"] is None or str(run_at) < str(cur["first_applied"]):
                cur["first_applied"] = run_at
                cur["first_granted_by"] = r.get("run_by")
            if cur["last_applied"] is None or str(run_at) > str(cur["last_applied"]):
                # ONE rule for everything else: a field that describes the grant as it stands NOW
                # -- its config version, its display spellings -- comes from the latest push,
                # because that is the push whose payload the live DAR is currently holding.
                # first_applied / first_granted_by are the only backward-looking fields.
                cur["last_applied"] = run_at
                cur["last_granted_by"] = r.get("run_by")
                cur["config_version"] = r.get("config_version")
                cur["role_name"] = r["role_name"]
                cur["scope_path"] = r["scope_path"]
                cur["member_id"] = r["member_id"]
                cur["member_name"] = r.get("member_name")
        return self._df(list(agg.values()))

    def provenance(self, role: str, scope: str, member: str | None = None) -> dict | None:
        """One establishing grant's provenance for a (role, scope[, member]) as a plain dict (the
        first row of grants()), or None when no such grant exists. Named for what it RETURNS, not
        for one end of it: it carries BOTH first_applied/first_granted_by and last_applied/
        last_granted_by, so the old name `since` overstated -- see grants() for why neither end
        alone can be reported as the moment the access began."""
        got = [r.asDict() for r in self.grants(role=role, scope=scope, member=member).collect()]
        return got[0] if got else None

    # A live path described MORE THAN ONCE resolves to this sentinel instead of a policy triple.
    # It is a 1-tuple, so it can never compare equal to a (permission, rls, columns) triple: an
    # ambiguous live shape is always a mismatch, never accidentally "matching".
    _POLICY_AMBIGUOUS = ("__ambiguous__",)

    @staticmethod
    def _scope_key(role, scope):
        """The lowered (role, scope) PAIR the POLICY comparison keys on -- _triple_key's identity
        formula minus the member, because DAR policy (permission / RLS / CLS) is a property of the
        (role, path) GRANT and every member of that grant shares it. One lowering formula for both
        keys, so a policy key and an identity key can never disagree about what is the same scope."""
        return (str(role).lower(), str(scope).lower())

    @staticmethod
    def _triple_key(role, scope, member):
        """The lowered (role, scope, member) identity triple that matches a live DAR grant against
        the framework's established set -- the ONE formula grants()/_established_set()/
        _iter_live_grants()/drift() all key on (previously re-derived inline in each). Built ON
        _scope_key so the identity axis and the policy axis lower identically."""
        return Audit._scope_key(role, scope) + (str(member).lower(),)

    @staticmethod
    def _policy_key(permission, rls, visible):
        """The comparable POLICY triple for one (role, scope) grant: (permission, rls_condition,
        visible-column allow-list). None on rls / visible means OPEN -- no RLS constraint, no CLS
        constraint -- and stays DISTINCT from an empty condition or an empty allow-list, the same
        distinction DAR.to_role's `has_cls` keeps. permission and the column names are lowered and
        the allow-list is compared as a SORTED tuple: exactly the normalization to_role's own
        decisionRule grouping key applies, so two spellings the framework ITSELF already treats as
        one policy compare equal here.

        Deliberately WEAKER than DAR.diff, which is a byte-exact comparison of the whole role
        payload including rule ORDER and member spelling: this answers "is the deployed POLICY
        different", plan answers "is the payload different". report() seeing a SUBSET of what plan
        sees is the intended relationship -- the reverse would have trace cry drift on a workspace
        plan calls clean. `effect` is deliberately NOT compared: to_role always emits "Permit", the
        service's normalization of that field is unprobed, and a permanent false red on every grant
        would cost more than the one shape it would catch (plan does catch it)."""
        perm = None if permission is None else str(permission).lower()
        cond = None if rls is None else str(rls)
        cols = None if visible is None else tuple(sorted(str(c).lower() for c in visible))
        return (perm, cond, cols)

    @classmethod
    def _desired_policy(cls, row):
        """One mapping lock-file row -> its comparable policy triple, derived by the SAME
        expressions DAR.to_role uses to BUILD the DAR payload out of that row: the permission
        default is to_role's own `get("permission", "Read")` (line-for-line, NOT `or` -- a row with
        an explicit NULL permission must yield None on both sides), `rls_condition or None` (an
        empty condition is no RLS), `visible_columns is not None` IS the has-CLS test, and
        constraints ride ONLY on a Table scope. That last gate is load-bearing: to_role emits
        neither a rows nor a columns constraint for a folder scope, so an rls_condition sitting on
        a folder row never reaches the live DAR and must not read as policy drift here. Deriving
        the desired side through to_role's expressions is what makes this a comparison against what
        WOULD BE PUT rather than a second, drifting interpretation of the mapping."""
        rls, visible = None, None
        if row.get("scope_type") == "Table":
            rls = row.get("rls_condition") or None
            raw_vis = row.get("visible_columns")
            if raw_vis is not None:
                visible = Parse.list(raw_vis)
        return cls._policy_key(row.get("permission", "Read"), rls, visible)

    @classmethod
    def _live_policies(cls, role):
        """One live DAR role -> {lowered (role, path): policy triple}, in ONE pass over
        decisionRules -- the live counterpart of _desired_policy, and the only live policy reader
        report()/drift() use.

        Not DAR.row_predicate/column_allowlist/path_permissions, deliberately. Those three answer
        "what does this role say about this path" by taking the FIRST matching constraint and the
        LAST matching rule's Action, which is right for effective_access's per-path question and
        WRONG here: an out-of-band editor who ADDS an unconstrained rule for a path, or a second
        Action value, or a second rows/columns entry for one tablePath, leaves every one of those
        readers returning the framework's own value -- three of the five edits this comparison
        exists to catch would read as clean. to_role puts each path in EXACTLY ONE rule with
        exactly one Action and at most one constraint entry per tablePath, so any multiplicity is a
        shape the framework cannot have written. Such a path resolves to _POLICY_AMBIGUOUS, which
        never equals a desired policy and so always reports as a mismatch -- fail-closed on a shape
        nobody can interpret. A path repeated with the SAME policy (two mapping rows for one
        (role, scope) that agree) is NOT ambiguous: to_role emits that as one rule listing the path
        twice, and reporting drift on it would be a false red."""
        out = {}
        for dr in role.get("decisionRules", []):
            actions = [
                v
                for att in dr.get("permission", [])
                if att.get("attributeName") == "Action"
                for v in att.get("attributeValueIncludedIn", [])
            ]
            constraints = dr.get("constraints", {})
            rows = constraints.get("rows") or []
            cols = constraints.get("columns") or []
            for att in dr.get("permission", []):
                if att.get("attributeName") != "Path":
                    continue
                for path in att.get("attributeValueIncludedIn", []):
                    hit_rows = [r for r in rows if r.get("tablePath") == path]
                    hit_cols = [c for c in cols if c.get("tablePath") == path]
                    if len(actions) != 1 or len(hit_rows) > 1 or len(hit_cols) > 1:
                        policy = cls._POLICY_AMBIGUOUS
                    else:
                        rls = None
                        if hit_rows:
                            rls = DAR.strip_predicate(path, hit_rows[0].get("value"))
                        visible = None
                        if hit_cols:
                            visible = list(hit_cols[0].get("columnNames") or [])
                        policy = cls._policy_key(actions[0], rls, visible)
                    key = cls._scope_key(role.get("name"), path)
                    if key in out and out[key] != policy:
                        policy = cls._POLICY_AMBIGUOUS
                    out[key] = policy
        return out

    @staticmethod
    def _policy_text(policy):
        """One policy triple rendered for drift()'s detail column. _POLICY_AMBIGUOUS renders as a
        named marker rather than a tuple: it means the live role describes that path more than
        once, which is a SHAPE the framework cannot emit, not a value that can be compared."""
        if policy == Audit._POLICY_AMBIGUOUS:
            return "ambiguous (the live role describes this path more than once)"
        perm, rls, cols = policy
        return "permission={} rls={} columns={}".format(
            perm, "open" if rls is None else repr(rls), "all" if cols is None else ";".join(cols)
        )

    def _established_set(self):
        """The set of lowered (role, scope, member) triples WITH framework provenance -- grants()'
        establishing grants, keyed via _triple_key for the live-DAR comparison out_of_band() and
        drift() both make."""
        return {
            self._triple_key(g["role_name"], g["scope_path"], g["member_id"])
            for g in (row.asDict() for row in self.grants().collect())
        }

    def _iter_live_grants(self):
        """Yield every live DAR grant as (role_name, scope_path, member, key, role): each live
        role's DAR.paths_and_members flattened to one tuple per (path, member), key =
        _triple_key(...), and `role` the RAW live role object the grant was flattened out of. The
        shared live-DAR walk out_of_band(), report() and drift() all iterate, over ONE
        client.list_roles() call. The raw role rides along because it is the only thing carrying
        the grant's POLICY (Audit._live_policies reads it), and report()'s policy comparison would
        otherwise need a SECOND list_roles(). It is yielded RAW rather than pre-resolved so
        out_of_band(), which never looks at policy, pays nothing; and it is APPENDED at the end,
        out_of_band()'s own convention, so the four positions the earlier callers unpack never
        shift. Assumes self.client is set -- callers raise their own UsageError first."""
        for r in self.client.list_roles():
            paths, members = DAR.paths_and_members(r)
            for path in paths:
                for member in members:
                    yield r["name"], path, member, self._triple_key(r["name"], path, member), r

    def out_of_band(self) -> "DataFrame":
        """Live DAR grants that have NO framework provenance -- the Audit listing of what
        _show counts as out_of_band. Flattens every live (role_name, scope_path, member objectId)
        grant via DAR.paths_and_members (as _show does) and returns those whose LOWERED triple is
        absent from grants()' ESTABLISHED set -- NOT merely absent from raw log rows: a grant seen
        only in a plan/read row is still out-of-band, exactly as _show treats it. Aggregates across
        ALL envs (cross-env by design; see grants()). member_name is resolved from the member cache
        table (self.member_table, the SAME id->name lookup who_can_access uses); an id absent from
        the cache surfaces AS the id -- an out-of-band member is usually NOT in the table, and the
        DAR is a live fact, so this never errors. An id the table gives MORE THAN ONE name surfaces
        as _AMBIGUOUS_MEMBER_NAME instead, never a row-order pick (see _member_names). Needs a
        FabricClient to read the live DAR."""
        if self.client is None:
            raise UsageError("out_of_band() needs a FabricClient to list the live DAR")
        established = self._established_set()
        columns = ["role_name", "scope_path", "member_id", "member_name"]
        hits = [
            (role, scope, member)
            for role, scope, member, key, _role in self._iter_live_grants()
            if key not in established
        ]
        if not hits:
            return self._df([], columns=columns)
        names = self._member_names()
        oob = [
            {
                "role_name": role,
                "scope_path": scope,
                "member_id": member,
                "member_name": names.get(str(member).lower(), member),
            }
            for role, scope, member in hits
        ]
        return self._df(oob, columns=columns)

    def effective_access(
        self, member: str, table: str, member_type: str | None = None, *, engine: str | None = None
    ) -> "DataFrame":
        """Live DAR (P2): the NET effective access to `table` for `member` -- one detail row
        per reaching role (its own rls_condition/visible_columns for that table, read via
        DAR.row_predicate/column_allowlist) plus ONE synthesized 'effective' row giving the
        union. Most-permissive-wins, mirroring rule C8's 'unrestricted nullifies filter'
        semantics: a reaching role with NO RLS on the table makes the union's row-predicate
        unconditionally true (full rows -- an open role nullifies every other role's filter);
        a reaching role with NO CLS makes the union's column set unrestricted (all columns).
        When every reaching role restricts the table, the union is instead the OR of the
        distinct conditions / the union of the column allow-lists (access via role A OR role
        B is the union of what either grants). No reaching role -> an EMPTY frame, still
        carrying the 6 typed columns. `member` is a name/UPN or an objectId, resolved via
        `_resolve_member` (GUID pass-through, strict No-Graph error) before matching the
        live DAR; the OPTIONAL `member_type` is forwarded to that resolution to disambiguate a
        display name shared by two principals of different types (a Group and a User both named
        "finance-team") -- omitted, such a name is a hard error rather than a silent pick.
        Needs a FabricClient to read the live DAR.
        `engine` is required: Spark/Direct Lake report CLS as a union, while SQL endpoint
        reports CLS as an intersection. This is a DAR reporting model, not proof of endpoint
        user identity mode or enforcement for a particular request.

        The member match against the live DAR is case-INSENSITIVE, and that is where the fix belongs
        rather than in the resolver: `_resolve_member` returns the member table's value UN-lowered
        (stored and emitted ids are preserved byte-for-byte -- they reach the DAR payload and the
        mapping hash), while the live DAR returns its own spelling of the same case-insensitive hex
        objectId. Compared exactly, a principal that DOES hold access reads as reaching no role at
        all -- a diagnostic false negative on the safe-looking side."""
        engine = str(engine or "").strip().lower()
        if engine not in ("spark", "direct_lake", "sql_endpoint"):
            raise UsageError("effective_access() requires engine=spark|direct_lake|sql_endpoint")
        if self.client is None:
            raise UsageError("effective_access() needs a FabricClient to read the live DAR")
        member = self._resolve_member(member, member_type=member_type)
        member_lower = str(member).lower()
        columns = [
            "role_name",
            "rls_condition",
            "visible_columns",
            "granting_role",
            "effective",
            "engine",
        ]
        path = ScopePath.table(table)
        detail_rows, granting_roles, conditions, column_sets = [], [], [], []
        any_open_rls, any_open_cls = False, False
        for r in self.client.list_roles():
            paths, members = DAR.paths_and_members(r)
            if path not in paths or member_lower not in {str(m).lower() for m in members}:
                continue
            rls_condition = DAR.row_predicate(r, path)
            visible_columns = DAR.column_allowlist(r, path)
            granting_roles.append(r["name"])
            detail_rows.append(
                {
                    "role_name": r["name"],
                    "rls_condition": rls_condition,
                    "visible_columns": (
                        LIST_SEP.join(visible_columns) if visible_columns is not None else None
                    ),
                    "granting_role": None,
                    "effective": False,
                    "engine": engine,
                }
            )
            if rls_condition is None:
                any_open_rls = True
            else:
                conditions.append(rls_condition)
            if visible_columns is None:
                any_open_cls = True
            else:
                column_sets.append(visible_columns)
        if not detail_rows:
            return self._df([], columns=columns)
        net_rls = None if any_open_rls else " OR ".join(f"({c})" for c in sorted(set(conditions)))
        if engine == "sql_endpoint":
            restricted = [cols for cols in column_sets if cols is not None]
            if not restricted:
                net_columns = None
            else:
                common = {str(c).lower(): c for c in restricted[0]}
                for cols in restricted[1:]:
                    allowed = {str(c).lower() for c in cols}
                    common = {key: value for key, value in common.items() if key in allowed}
                net_columns = LIST_SEP.join(sorted(common.values()))
        elif any_open_cls:
            net_columns = None
        else:
            seen, widest = set(), []
            for cols in column_sets:
                for c in cols:
                    if c.lower() not in seen:
                        seen.add(c.lower())
                        widest.append(c)
            net_columns = LIST_SEP.join(sorted(widest))
        detail_rows.append(
            {
                "role_name": None,
                "rls_condition": net_rls,
                "visible_columns": net_columns,
                "granting_role": LIST_SEP.join(sorted(granting_roles)),
                "effective": True,
                "engine": engine,
            }
        )
        return self._df(detail_rows, columns=columns)

    def who_can_access(self, table: str) -> "DataFrame":
        """Live DAR (P2): the REVERSE of effective_access() -- every member who can reach
        `table`, via which role. One row per (member, role) pair that reaches the table
        (unlike effective_access's single-member union, a member reachable via two roles
        gets two rows here, one per via_role -- exactly what a "who can see this" audit
        needs). member_name is resolved from the member cache table (self.member_table,
        keyed by member_id -- the SAME table diagnose_member reads, in the opposite
        direction); an objectId absent from the cache surfaces AS the id, never crashes,
        and one the cache gives MORE THAN ONE name surfaces as _AMBIGUOUS_MEMBER_NAME --
        a DISTINCT rendering from both a real name and the absent-id fallback, so "who can
        see this table" never prints whichever duplicate row sorted last.
        permission is DAR.path_permissions(role)[path]; rls_cls_summary is a one-cell
        digest of that role's row/column restriction on this table, built from
        DAR.row_predicate / DAR.column_allowlist: "rows: <condition>" and/or
        "cols: <col;col>" (LIST_SEP-joined, matching effective_access's visible_columns
        convention), joined with "; ", or "unrestricted" when the role carries neither.
        No reaching role -> an EMPTY frame, still carrying the 5 typed columns. Needs a
        FabricClient to read the live DAR."""
        if self.client is None:
            raise UsageError("who_can_access() needs a FabricClient to read the live DAR")
        columns = ["member_name", "member_id", "via_role", "permission", "rls_cls_summary"]
        path = ScopePath.table(table)
        hits = []
        for r in self.client.list_roles():
            paths, members = DAR.paths_and_members(r)
            if path not in paths:
                continue
            permission = DAR.path_permissions(r).get(path)
            rls_condition = DAR.row_predicate(r, path)
            visible_columns = DAR.column_allowlist(r, path)
            parts = []
            if rls_condition is not None:
                parts.append(f"rows: {rls_condition}")
            if visible_columns is not None:
                parts.append(f"cols: {LIST_SEP.join(visible_columns)}")
            summary = "; ".join(parts) if parts else "unrestricted"
            for member in members:
                hits.append((member, r["name"], permission, summary))
        if not hits:
            return self._df([], columns=columns)
        names = self._member_names()
        rows = [
            {
                "member_name": names.get(str(member).lower(), member),
                "member_id": member,
                "via_role": role_name,
                "permission": permission,
                "rls_cls_summary": summary,
            }
            for member, role_name, permission, summary in hits
        ]
        return self._df(rows, columns=columns)

    def _member_rows(self):
        """Member cache table rows, autotrimmed at the read seam (Parse.trim_row) -- the ONE
        read _member_names (id->name) and _resolve_member (name->id) share."""
        return [Parse.trim_row(r.asDict()) for r in self.spark.table(self.member_table).collect()]

    def _member_names(self):
        """objectId (lower) -> member_name, read from the member cache table -- the id->name
        resolution who_can_access() needs (the reverse keying of diagnose_member's name->id
        lookup over the SAME table). An id with no matching row is simply absent from the
        map; who_can_access() then surfaces the raw id instead of a name.
        Rows go through Parse.trim_row (the same autotrim Deployment._load_member_cache applies to
        this table), so the display NAME is trimmed too -- not only the id key -- and a stray-space
        preload artifact never surfaces in who_can_access/out_of_band/drift output.

        An id carrying MORE THAN ONE name takes the AMBIGUITY branch and maps to
        _AMBIGUOUS_MEMBER_NAME -- on THAT branch, never a row-order pick. The predicate
        is Log.resolve_principal's, character for character: the distinct set of STRIPPED, TRUTHY
        member_name values for that id, compared CASE-SENSITIVELY -- "Alice" and "alice" are two
        names (two different principals is the likelier reading of that data, and it is the reading
        generate's own duplicate guard takes), while "Alice" and "  Alice  " are one (both surfaces
        trim at the read seam AND strip again here).

        SCOPE, stated plainly: that "never a row-order pick" guarantee covers the ambiguity branch
        ONLY. A row whose member_name is blank/NULL is excluded from the distinct set, exactly as
        resolve_principal excludes it, so one real name beside a NULL row is NOT ambiguous -- it
        falls to the NON-ambiguous branch, which still returns values[-1], the LAST row read. That
        makes the blank/NULL case ROW-ORDER DEPENDENT: for the same id, rows [None, "Alice"] yield
        "Alice" while ["Alice", None] yield None. This is a DIFFERENT, PRE-EXISTING condition from
        the ambiguity above -- deliberately left untouched here and pinned as-is by
        test_member_names_leaves_a_null_display_name_as_none and its two ordering cases; changing it
        needs its own decision. (A lone NULL name likewise maps to the stored None; that value, not
        the id, is what consumers render, because names.get(key, member) falls back only when the
        KEY is ABSENT.)"""
        by_id = {}
        for r in self._member_rows():
            by_id.setdefault(str(r.get("member_id") or "").strip().lower(), []).append(
                r.get("member_name")
            )
        names = {}
        for member_id, values in by_id.items():
            distinct = {str(v).strip() for v in values if v}
            names[member_id] = (
                _AMBIGUOUS_MEMBER_NAME.format(n=len(distinct)) if len(distinct) > 1 else values[-1]
            )
        return names

    def _resolve_member(self, member, member_type=None):
        """Name/UPN or objectId -> objectId, strict (No-Graph): a GUID-shaped `member` (GUID_RE)
        passes through UNCHANGED -- already an objectId. Otherwise resolved by case-insensitive
        member_name match against the member cache table (the name->id lookup, the reverse
        keying of _member_names' id->name map over the SAME table). Rows go through
        Parse.trim_row first -- the SAME autotrim Deployment._load_member_cache applies to this
        table -- so the returned objectId is trimmed and can actually match the live DAR's clean
        objectIds. A name absent from the table is a HARD error naming it --
        effective_access()/diagnose_member() resolve config-declared members (groups/users/SPs);
        they do not expand group membership (No-Graph).

        `member_type` is OPTIONAL and scopes the match to the member table's logical PK
        (member_type, lower(member_name)) -- a Group and a User may legitimately share a display
        name (_load_member_cache only rejects same-TYPE collisions). Omitted, an unambiguous name
        resolves exactly as before; a name matching MORE THAN ONE member_type is a HARD error
        naming the ambiguity and asking for member_type, never a silent first-row pick.
        A None/blank `member` is a UsageError like every other input guard on this class.
        `member` itself is stripped before both the GUID match and the name comparison, the
        same autotrim diagnose_member already applies to its own input.

        The returned objectId is the STORED value, byte-for-byte as written (a GUID-shaped `member`
        likewise passes through unchanged) -- never lower-cased, because that value reaches the DAR
        payload and the mapping hash. Ids are instead compared case-insensitively at each consumer;
        effective_access case-folds its live-DAR membership test for exactly this reason."""
        if not isinstance(member, str) or not member.strip():
            raise UsageError(
                f"member must be a non-blank name/UPN or objectId string (got {member!r})"
            )
        member = member.strip()
        if GUID_RE.match(member):
            return member
        rows = self._member_rows()
        lower = member.lower()
        matches = [
            r
            for r in rows
            if str(r.get("member_name") or "").strip().lower() == lower
            and member_type in (None, str(r.get("member_type") or ""))
        ]
        if not matches:
            qualifier = f" (member_type {member_type!r})" if member_type is not None else ""
            raise UsageError(
                f"member {member!r}{qualifier} not in the member table -- effective_access "
                f"resolves config-declared members (groups/users/SPs); it does not expand group "
                f"membership (No-Graph)"
            )
        types = sorted({str(r.get("member_type") or "") for r in matches})
        if len(types) > 1:
            raise UsageError(
                f"member {member!r} is ambiguous in the member table -- it names rows of "
                f"member_type {types}; pass member_type= to say which principal you mean"
            )
        return matches[0].get("member_id")

    def timeline(self) -> "DataFrame":
        """Every logged config generation as one row: group ALL log rows by
        (config_version, config_hash) and report first_seen (min run_at) / last_seen (max run_at) /
        runs (count), ordered by config_version -- the lifetime of each generation as the audit
        trail saw it."""
        agg = {}
        for r in self._log_rows():
            k = (r.get("config_version"), r.get("config_hash"))
            a = agg.setdefault(
                k,
                {
                    "config_version": k[0],
                    "config_hash": k[1],
                    "first_seen": r.get("run_at"),
                    "last_seen": r.get("run_at"),
                    "runs": 0,
                },
            )
            a["runs"] += 1
            a["first_seen"] = min(a["first_seen"], r.get("run_at"), key=lambda x: str(x))
            a["last_seen"] = max(a["last_seen"], r.get("run_at"), key=lambda x: str(x))
        # Ordered by the generation's own version, NOT by its spelling: config_version is a
        # BIGINT, and str() would order 10 before 9. None sorts first (a generation logged
        # before the config table could report a version).
        return self._df(
            sorted(
                agg.values(),
                key=lambda a: (
                    a["config_version"] is not None,
                    int(a["config_version"]) if a["config_version"] is not None else 0,
                ),
            )
        )

    def trace(
        self, member: str | None = None, role: str | None = None, scope: str | None = None
    ) -> "DataFrame":
        """The DEPLOYING generation(s) behind a grant: validate+success rows whose mode actually
        pushed the grant (apply — including a rollback chain's apply, stamped mode=rollback;
        a plan-mode dry run never deployed, so it is dropped — the
        SAME rule grants() applies), narrowed to the given subject, newest first, projected to the
        generation coordinates (config_version, config_hash, run_at, run_by)."""
        rows = [
            r
            for r in self._log_rows(member_name=member, role_name=role, scope_path=scope)
            if r.get("action") == "validate"
            and r.get("status") == "success"
            # 'replace' is a RETIRED pre-release mode: its full-truth semantics became
            # apply(keep_unmanaged=False), so no path here can emit it. It stays in the
            # predicate because this table is append-only history — a workspace deployed
            # before the fold still holds rows stamped with it, and dropping the token
            # would silently re-read those grants as out-of-band, with no provenance.
            and r.get("mode") in ("apply", "replace", "rollback")
        ]
        rows.sort(key=lambda r: str(r.get("run_at")), reverse=True)
        keep = ("config_version", "config_hash", "run_at", "run_by")
        return self._df([{k: r.get(k) for k in keep} for r in rows])

    def authored_by(self, version: int) -> dict | None:
        """Who committed a given config Delta version, read from DESCRIBE HISTORY:
        {version, timestamp, user} for the matching commit, or None when no history row carries that
        version. user is the commit's userName, falling back to operationParameters.userName."""
        hist = self.spark.sql(f"DESCRIBE HISTORY {self.config_table}").collect()
        for h in hist:
            d = h.asDict()
            if d.get("version") == version:
                return {
                    "version": version,
                    "timestamp": d.get("timestamp"),
                    "user": (d.get("userName") or d.get("operationParameters", {}).get("userName")),
                }
        return None

    @staticmethod
    def _timetravel_sql(target, version, date):
        """Build a Delta time-travel SELECT for `target` at the ONE of version/date the caller
        selected (the caller owns the exactly-one-of guard -- config_at()/at() word their own
        message). `version` is int()-coerced (a non-numeric payload raises ValueError); `date` is
        shape-checked against _DELTA_TIMESTAMP_RE BEFORE it is interpolated raw into TIMESTAMP AS
        OF -- closing the same injection gap int() closes on the version side, so both branches
        validate identically."""
        if date is not None:
            if not _DELTA_TIMESTAMP_RE.match(str(date)):
                raise ValueError(
                    f"date must be a Delta timestamp/date like 'YYYY-MM-DD' or "
                    f"'YYYY-MM-DD HH:MM:SS' (got {date!r})"
                )
            return f"SELECT * FROM {target} TIMESTAMP AS OF '{date}'"
        return f"SELECT * FROM {target} VERSION AS OF {int(version)}"

    def config_at(self, version: int | str | None = None, date: str | None = None) -> "DataFrame":
        """The exact config rows that produced a generation, via Delta time-travel. Give EXACTLY ONE of
        version (VERSION AS OF, int-coerced) or date (TIMESTAMP AS OF, shape-validated). FakeSpark
        cannot time-travel; the query is asserted in unit tests and the real read is covered by the
        gated live smoke."""
        if (version is None) == (date is None):
            raise ValueError("config_at() needs exactly one of version or date")
        return self.spark.sql(self._timetravel_sql(self.config_table, version, date))

    def report(self) -> dict:
        """A one-call operational snapshot for mode=trace, answering the question an operator
        actually has after an apply: WHAT IS DEPLOYED RIGHT NOW, AND DOES IT MATCH THE CONFIG?

        Every count here is about the CURRENT state — the live DAR and the current generation's
        mapping. `established_ever` is the one exception and says so in its name: it is the
        cumulative count of everything the log has ever established, across every environment and
        config version, which is a provenance fact rather than a description of today.

        That distinction used to be missing. `role_count` and `grant_count` were the cumulative
        figures under names that read as current, so a workspace whose live DAR held 30 roles and
        430 grants reported 84 and 588 — the union of every version it had ever deployed — and a
        reader had no way to tell from the field name.

        TWO INDEPENDENT AXES, and `in_sync` is their conjunction. IDENTITY -- missing, unexpected,
        out_of_band -- compares (role, scope_path, member_id) triples: is the right principal still
        granted the right table? POLICY -- policy_checked, policy_mismatch -- compares the
        (permission, rls_condition, visible_columns) actually deployed on the grants BOTH sides
        agree exist: and does the rule on it still say what the mapping says? Keeping them separate
        is what lets each count keep one meaning; an edit to a predicate moves the policy axis and
        leaves the identity axis at zero, which is the true description of what happened.

        The policy axis was absent, and its absence was the sharpest edge in this method: in_sync
        was `not (desired ^ live)` alone, so a live RLS predicate rewritten to 1=1, a CLS allow-list
        widened to expose a column, or a permission raised Read -> ReadWrite ALL left it true. The
        triple had not moved -- only the rule attached to it had -- and `plan` was the only mode
        that could see it.

        Deliberately WEAKER than plan's DAR.diff, which compares the whole role payload byte for
        byte including rule ORDER and member spelling. This answers "is the deployed policy
        different"; plan answers "is the payload different". trace seeing a SUBSET of what plan sees
        is the intended relationship -- the reverse would have a daily glance cry drift on a
        workspace plan calls clean.

        Counts, not listings, so trace stays one row: `drift()` returns the itemised comparison
        (category "policy" for this axis) and `out_of_band()` the offending grants. The established
        set and the live scan are each collected ONCE and shared across every figure below; the
        earlier version collected the whole log twice per trace. The policy read costs no extra API
        call: it reads the roles _iter_live_grants already fetched.
        """
        last_deployment = self.last_successful_deployment()
        rep = {
            "current_generation": self.current_generation(),
            "last_generate": self.last_run("generate"),
            "last_apply": self.last_successful_deployment("apply"),
            "last_deployment": last_deployment,
            "last_deployment_mode": last_deployment.get("mode") if last_deployment else None,
            "is_stale": self.is_stale(),
        }
        established = self._established_set()
        rep["established_ever"] = len(established)
        if self.client is None:
            return rep

        live_roles, live, live_policy, out_of_band = set(), set(), {}, 0
        for role, _path, _member, key, role_obj in self._iter_live_grants():
            rname = str(role).lower()
            if rname not in live_roles:
                # Once per ROLE, never once per grant: _live_policies walks the whole role, and
                # the live_roles set is already the "have I seen this role" answer.
                live_roles.add(rname)
                live_policy.update(self._live_policies(role_obj))
            live.add(key)
            if key not in established:
                out_of_band += 1

        # The mapping lock-file IS the current generation's desired state — flattened per member
        # exactly as drift() and diagnose_member do, so all three agree on what "desired" means.
        desired, desired_policy, policy_conflict = set(), {}, set()
        mapping_rows, _provenance = self._mapping_rows()
        for row in mapping_rows:
            role, path = row.get("role_name"), row.get("scope_path")
            scope_key = self._scope_key(role, path)
            policy = self._desired_policy(row)
            # Two mapping rows for one (role, scope) that DISAGREE about policy: rule C3 refuses
            # that at generate, but C3 validates the CONFIG on an exact-case key, so a hand-edited
            # or rollback-restored mapping can still hold it. There is no single desired policy to
            # compare against, and to_role emits BOTH as rules so plan reports no_change — scoring
            # it either way would be a permanent red on a workspace plan calls clean. So the scope
            # ABSTAINS: it is dropped from the comparison, deterministically, whatever row order
            # the table returns.
            if scope_key in desired_policy and desired_policy[scope_key] != policy:
                policy_conflict.add(scope_key)
            desired_policy[scope_key] = policy
            for _name_col, id_col, _mtype in MAPPING_MEMBER_COLUMNS:
                for member in Parse.list(row.get(id_col)):
                    desired.add(self._triple_key(role, path, member))

        rep["live_role_count"] = len(live_roles)
        rep["live_grant_count"] = len(live)
        rep["desired_grant_count"] = len(desired)
        # `missing` is desired-but-absent; `unexpected` is live-but-not-in-this-generation, which
        # out_of_band alone cannot see: a grant an OLDER generation established is in the
        # cumulative established set forever, so a retired role reappearing in the DAR reads as
        # provenanced. Comparing against THIS generation is what catches it.
        rep["missing"] = len(desired - live)
        rep["unexpected"] = len(live - desired)
        rep["out_of_band"] = out_of_band
        # Policy is compared ONLY on the grants BOTH sides agree exist. A `missing` or `unexpected`
        # grant has no counterpart to read a policy off, and scoring its absent side as "open"
        # would report one identity fault twice — once as the count it already is, and again as
        # policy drift. Disjoint signals are what let in_sync be their conjunction without
        # double-counting, and are why missing/unexpected keep exactly their previous meaning.
        shared = {(role, scope) for role, scope, _member in desired & live} - policy_conflict
        rep["policy_checked"] = len(shared)
        # Direct indexing, not .get: every key in `shared` was written into BOTH maps by the same
        # pass that produced the triple it came from, so a KeyError here is an invariant break to
        # surface, not a case to swallow into a None-compares-unequal false positive.
        rep["policy_mismatch"] = sum(1 for k in shared if desired_policy[k] != live_policy[k])
        # in_sync = the identity sets match AND no shared grant's policy differs. The identity half
        # is untouched. An RLS predicate edited live to 1=1, a widened CLS allow-list and a
        # permission raised Read -> ReadWrite each move policy_mismatch and NONE of them move
        # missing/unexpected — before this they left in_sync true, and plan was the only mode that
        # saw them.
        rep["in_sync"] = not (desired ^ live) and rep["policy_mismatch"] == 0
        return rep

    def coverage(self) -> "DataFrame":
        """P1: protected vs unprotected table surface -- the compliance gap finder over the
        mapping lock-file. Table universe = Catalog.canonical(self.spark)['tables'] -- the SAME
        real-lakehouse lister generate() itself calls to validate config (spark-only: SHOW SCHEMAS
        / SHOW TABLES; columns are listed lazily and this method never touches them -- no
        FabricClient, no notebookutils; confirmed callable in the
        no-client interactive context, exactly like every other P1 Audit method). Deliberately the
        REAL table universe, not merely the tables named in the mapping: a table with ZERO mapping
        rows still gets a row here (protected=False) -- surfacing tables nobody configured at all is
        the actual gap this method finds, not a recap of what IS configured. Per table:
        roles_count = distinct role_name across its mapping rows (the mapping's grain is one row
        per role x scope, so >1 role reaching a table is common); has_rls/has_cls = True when ANY
        reaching role's row carries rls_condition / visible_columns (the same per-role fields
        effective_access reads, aggregated here to a yes/no rather than the condition itself). Only
        mapping rows whose scope_path is a table path (starts with '/Tables/') count -- a role's
        folder-scope rows are skipped, since a folder path can coincidentally parse into a
        schema.table-shaped string via ScopePath.to_table and must not collide with a real table of
        that parsed name. Empty mapping -> every universe table comes back protected=False (not an
        empty frame -- the universe itself still has rows). No live client needed."""
        columns = ["table", "protected", "roles_count", "has_rls", "has_cls"]
        universe = sorted(set(Catalog.canonical(self.spark)["tables"].values()))
        by_table = {}
        mapping_rows, _provenance = self._mapping_rows()
        for row in mapping_rows:
            path = row.get("scope_path")
            if not path or not str(path).startswith("/Tables/"):
                continue
            agg = by_table.setdefault(
                ScopePath.to_table(path), {"roles": set(), "has_rls": False, "has_cls": False}
            )
            agg["roles"].add(row.get("role_name"))
            if row.get("rls_condition") is not None:
                agg["has_rls"] = True
            if row.get("visible_columns") is not None:
                agg["has_cls"] = True
        rows = []
        for table in universe:
            agg = by_table.get(table)
            rows.append(
                {
                    "table": table,
                    "protected": agg is not None,
                    "roles_count": len(agg["roles"]) if agg else 0,
                    "has_rls": bool(agg and agg["has_rls"]),
                    "has_cls": bool(agg and agg["has_cls"]),
                }
            )
        return self._df(rows, columns=columns)

    def drift(self) -> "DataFrame":
        """Live DAR (P2): the FULL desired-vs-live comparison, CATEGORIZED and READ-ONLY -- decision
        4: drift() is a pure comparison VIEW, never a plan (it neither records a plan row nor gates
        apply -- that stays plan()'s job as the ONE apply-gating recorder; drift() and plan() coexist).
        One row per grant, category:
          - framework    = a live DAR grant WITH framework provenance -- in grants()' ESTABLISHED set,
                            the EXACT established-set-vs-live test out_of_band() makes (reused here,
                            not reimplemented).
          - out_of_band  = a live DAR grant with NO framework provenance -- out_of_band()'s own test,
                            applied the same way over the same established set.
          - missing      = a DESIRED grant -- present in the mapping lock-file (self.mapping_table),
                            flattened per member exactly as diagnose_member's in_mapping step does
                            (MAPPING_MEMBER_COLUMNS + Parse.list) -- that is ABSENT from the live DAR
                            entirely (checked against every live (role, scope, member) triple, not
                            just the established set: a grant pushed out-of-band still counts as
                            live, so it is never ALSO reported missing).
          - policy       = a live DAR grant WITH framework provenance whose (role, scope) is ALSO
                            in the mapping, but whose deployed POLICY -- permission / RLS predicate
                            / CLS allow-list -- differs from what the mapping declares. The
                            itemisation behind report()'s policy_mismatch count. out_of_band takes
                            precedence: a grant with no provenance at all is reported as that,
                            never as a policy difference.
        member_id + member_name are APPENDED at the END of the frame (out_of_band()'s convention,
        so the pre-existing role_name/scope_path/category/detail positions never shift), and the
        PAIR is kept for the same reason out_of_band/who_can_access/grants keep it: member_name is
        resolved per row from the member cache table (self.member_table, via _member_names -- the
        SAME id->name lookup those siblings use) for EVERY category, and an id absent from the
        cache surfaces AS the id -- comparing the two columns is how a caller DETECTS that
        fallback. That detection is why an AMBIGUOUS id (one the cache gives more than one name)
        renders as _AMBIGUOUS_MEMBER_NAME rather than as the id: were it the id too, "not in the
        member table" and "ambiguous in the member table" would be byte-identical here and the
        two-column comparison would no longer distinguish them. So member_name reads
        one of three ways -- a name, the id (absent), or the marker (ambiguous).
        detail stays a short human-readable note naming the member (by id) for each row.
        Needs a FabricClient to read the live DAR -- the same guard
        out_of_band/effective_access/who_can_access raise."""
        if self.client is None:
            raise UsageError("drift() needs a FabricClient to read the live DAR")
        columns = ["role_name", "scope_path", "category", "detail", "member_id", "member_name"]
        established = self._established_set()
        names = self._member_names()
        # The DESIRED read comes first now: categorizing a provenanced live grant needs the policy
        # the mapping declares for its scope. Same single read, same flattening — only the order
        # moved, and rows are still appended live-first so the frame's shape never changed.
        desired, desired_policy, policy_conflict = {}, {}, set()
        mapping_rows, _provenance = self._mapping_rows()
        for row in mapping_rows:
            role, path = row.get("role_name"), row.get("scope_path")
            scope_key = self._scope_key(role, path)
            policy = self._desired_policy(row)
            if scope_key in desired_policy and desired_policy[scope_key] != policy:
                policy_conflict.add(scope_key)  # see report(): the scope abstains
            desired_policy[scope_key] = policy
            for _name_col, id_col, _mtype in MAPPING_MEMBER_COLUMNS:
                for member in Parse.list(row.get(id_col)):
                    key = self._triple_key(role, path, member)
                    desired.setdefault(key, (role, path, member))
        live, live_roles, live_policy, rows = set(), set(), {}, []
        for role, path, member, key, role_obj in self._iter_live_grants():
            rname = str(role).lower()
            if rname not in live_roles:
                live_roles.add(rname)
                live_policy.update(self._live_policies(role_obj))
            live.add(key)
            scope_key = self._scope_key(role, path)
            if key not in established:
                # out_of_band WINS over policy: category is one string, and "a stranger holds this
                # grant" is the more urgent fact than "the rule on it reads differently".
                category = "out_of_band"
                detail = f"live grant has no framework provenance (member {member})"
            elif (
                key in desired
                and scope_key not in policy_conflict
                and desired_policy[scope_key] != live_policy[scope_key]
            ):
                category = "policy"
                detail = (
                    "live grant has framework provenance but its policy differs from the mapping "
                    f"— desired: {self._policy_text(desired_policy[scope_key])}; live: "
                    f"{self._policy_text(live_policy[scope_key])} (member {member})"
                )
            else:
                category = "framework"
                detail = f"live grant matches framework provenance (member {member})"
            rows.append(
                {
                    "role_name": role,
                    "scope_path": path,
                    "category": category,
                    "detail": detail,
                    "member_id": member,
                    "member_name": names.get(str(member).lower(), member),
                }
            )
        for key, (role, path, member) in desired.items():
            if key not in live:
                rows.append(
                    {
                        "role_name": role,
                        "scope_path": path,
                        "category": "missing",
                        "detail": f"desired grant absent from live DAR (member {member})",
                        "member_id": member,
                        "member_name": names.get(str(member).lower(), member),
                    }
                )
        return self._df(rows, columns=columns)

    @staticmethod
    def _config_key(row):
        """(role_name, scope) key for config_diff -- scope is the row's own scope-defining
        columns (see _CONFIG_DIFF_SCOPE_COLUMNS above), joined with '|' so two rows that grant the
        same role over the same authored scope collide into ONE key regardless of every other
        field (permission/rls_condition/members/...), which is what makes those other fields
        diffable as 'changed' instead of an added+removed pair."""
        scope = "|".join(str(row.get(c) or "") for c in _CONFIG_DIFF_SCOPE_COLUMNS)
        return (row.get("role_name"), scope)

    def config_diff(self, v1: int | str, v2: int | str) -> "DataFrame":
        """P3: role/scope/member changes between two config Delta versions, via time-travel
        (config_at) then diffed in PYTHON -- no SQL diffing, so this works identically whether v1/
        v2 are ints (VERSION AS OF) or whatever config_at itself accepts. Reads both versions,
        keys each row by (role_name, scope) (_config_key -- see _CONFIG_DIFF_SCOPE_COLUMNS for why
        raw config has no literal scope_path), and reports:
          - added   = a key present in v2 but not v1 (a new role/scope pair)
          - removed = a key present in v1 but not v2 (a dropped role/scope pair)
          - changed = a key in BOTH versions where at least one of the remaining authored columns
                      differs (_CONFIG_DIFF_FIELD_COLUMNS -- permission, rls_condition, the column
                      allow/deny lists, the eight member_* include/exclude columns, lakehouse_name,
                      active, notes) -- ONE row per differing field, not per key, so a role with
                      three changed fields yields three 'changed' rows.
        added/removed rows carry field/old/new as None (the whole row moved, not one field); a
        scope-column change (include_tables/... itself differing) surfaces as added+removed for
        that key, never double-reported as 'changed' too. Needs no FabricClient -- config_at reads
        the config Delta table directly, no live DAR involved."""
        v1_rows = {
            self._config_key(r): r for r in (row.asDict() for row in self.config_at(v1).collect())
        }
        v2_rows = {
            self._config_key(r): r for r in (row.asDict() for row in self.config_at(v2).collect())
        }
        columns = ["change_type", "role_name", "scope_key", "field", "old", "new"]
        rows = []
        for role_name, scope in sorted(v2_rows.keys() - v1_rows.keys()):
            rows.append(
                {
                    "change_type": "added",
                    "role_name": role_name,
                    "scope_key": scope,
                    "field": None,
                    "old": None,
                    "new": None,
                }
            )
        for role_name, scope in sorted(v1_rows.keys() - v2_rows.keys()):
            rows.append(
                {
                    "change_type": "removed",
                    "role_name": role_name,
                    "scope_key": scope,
                    "field": None,
                    "old": None,
                    "new": None,
                }
            )
        for key in sorted(v1_rows.keys() & v2_rows.keys()):
            role_name, scope = key
            old_row, new_row = v1_rows[key], v2_rows[key]
            for field in _CONFIG_DIFF_FIELD_COLUMNS:
                old_value, new_value = old_row.get(field), new_row.get(field)
                if old_value != new_value:
                    rows.append(
                        {
                            "change_type": "changed",
                            "role_name": role_name,
                            "scope_key": scope,
                            "field": field,
                            "old": old_value,
                            "new": new_value,
                        }
                    )
        return self._df(rows, columns=columns)

    def _control_table(self, name):
        """Maps a logical control-table name -- "config"/"mapping"/"log" -- to the actual
        configured table name (self.config_table/mapping_table/log_table). Shared plumbing:
        table_history (below) is the first caller, Task 12's `at` is the next."""
        tables = {
            "config": self.config_table,
            "mapping": self.mapping_table,
            "log": self.log_table,
        }
        if name not in tables:
            raise ValueError(f"unknown control table {name!r} (expected config, mapping, or log)")
        return tables[name]

    def table_history(self, table: str) -> "DataFrame":
        """Delta DESCRIBE HISTORY of a control table, made readable: version, timestamp, user,
        operation, rows. `table` is one of config/mapping/log (_control_table maps it to the
        actual table name; config/mapping are the primary use, log is accepted too). `rows` comes
        out of the nested operationMetrics map (numOutputRows), not a flat DESCRIBE HISTORY column;
        `user` falls back to operationParameters.userName when userName itself is null, same
        convention as authored_by. FakeSpark cannot run a real DESCRIBE HISTORY; the emitted SQL
        and the row projection are both asserted in unit tests (same approach as
        authored_by/config_at)."""
        hist = self.spark.sql(f"DESCRIBE HISTORY {self._control_table(table)}").collect()
        columns = ["version", "timestamp", "user", "operation", "rows"]
        rows = []
        for h in hist:
            d = h.asDict()
            metrics = d.get("operationMetrics") or {}
            rows.append(
                {
                    "version": d.get("version"),
                    "timestamp": d.get("timestamp"),
                    "user": d.get("userName") or d.get("operationParameters", {}).get("userName"),
                    "operation": d.get("operation"),
                    "rows": metrics.get("numOutputRows"),
                }
            )
        return self._df(rows, columns=columns)

    def at(
        self, table: str, version: int | str | None = None, date: str | None = None
    ) -> "DataFrame":
        """The snapshot of ANY control table (config/mapping/log) at a Delta version or
        date -- config_at generalized to any control table (Task 12's generic sibling;
        config_at itself stays as its own dedicated method for callers that only ever want
        the config table). `table` is resolved via _control_table FIRST -- so at("bogus",
        version=1) raises _control_table's ValueError before the exactly-one guard below
        ever runs. Give EXACTLY ONE of version (VERSION AS OF, int-coerced) or date
        (TIMESTAMP AS OF, shape-validated) -- the SAME guard + shared _timetravel_sql builder
        config_at uses. FakeSpark cannot time-travel;
        the emitted SQL is asserted in unit tests (same approach as
        config_at/table_history)."""
        target = self._control_table(table)
        if (version is None) == (date is None):
            raise ValueError("at() needs exactly one of version or date")
        return self.spark.sql(self._timetravel_sql(target, version, date))

    def value_history(self, subject: str, last: int | None = None) -> "DataFrame":
        """P3: how ONE role/scope's config value evolved across EVERY config Delta version --
        one row per version WHERE THE SUBJECT IS PRESENT: config_version, every
        _CONFIG_DIFF_FIELD_COLUMNS value (permission/rls_condition/the column allow-lists/the
        eight member_* lists/lakehouse_name/active/notes -- config_diff's own field set, reused
        so both methods agree on what "the value" means), plus `changed` (bool). Versions come
        from table_history("config")'s version list, walked OLDEST FIRST (DESCRIBE HISTORY
        itself is newest-first) so `changed` can compare each version against what came before
        it; each version's rows are read via config_at(version). A row is "the subject's row"
        when Parse.subject_match(subject, role_name, scope) is true -- scope is config_diff's
        OWN synthesized key (_config_key: include/exclude tables/folders joined with '|'), so a
        subject matches EXACTLY the same (role_name, scope) identity config_diff diffs by. When
        more than one row matches in the same version (a wildcard subject spanning several
        roles, or one role authored with more than one scope row), the LOWEST (role_name, scope)
        key wins -- deterministic, but this narrows an ambiguous subject to one row per version
        rather than raising ("one role/scope" is this method's documented contract, not a
        validated invariant).

        `changed` is True when this version's tracked fields differ from the LAST version where
        the subject was ALSO present. The very FIRST appearance -- no prior present version,
        whether that is the walk's first version ever or a reappearance after a gap where the
        subject was absent -- counts as a change from "absent" and is always True; a version
        with no match contributes NO row and resets this state, so a later reappearance is
        judged as a fresh first-appearance, never diffed against a stale pre-gap row.

        COST, stated plainly: ONE sequential time-travel snapshot read per walked version —
        on a long-lived config table that is one read per Delta commit ever made. `last=N`
        (an int, or an int-VALUED float/string; bool and non-integral values are refused)
        bounds the walk to the N NEWEST versions so the method stays usable interactively.
        WINDOW SEMANTICS, stated just as plainly: the oldest row of a bounded walk is judged
        a first appearance exactly like a reappearance after a gap, so its changed=True
        means "first appearance IN THE WINDOW", not necessarily in history — every row
        carries `window_truncated` (True when last=N genuinely cut versions off the walk)
        so a bounded result is self-describing. The one thing no column can carry: an EMPTY
        frame under last=N means "not present in the last N versions", NOT "never
        existed" — widen or drop `last` to tell those apart."""
        columns = [
            "config_version",
            "role_name",
            "scope_key",
            *_CONFIG_DIFF_FIELD_COLUMNS,
            "changed",
            "window_truncated",
        ]
        versions = sorted(
            int(r.asDict()["version"]) for r in self.table_history("config").collect()
        )
        truncated = False
        if last is not None:
            # int and int-VALUED inputs only, evaluated once. bool is an int subclass, but
            # last=True is a caller bug, not a request for one version; 2.9 silently floored
            # would walk a different window than the one asked for.
            try:
                bounded = None if isinstance(last, bool) else int(last)
                exact = bounded is not None and float(last) == bounded
            except (TypeError, ValueError):
                exact, bounded = False, None
            if not exact or bounded < 1:
                raise ValueError(f"last must be a positive number of versions (got {last!r})")
            truncated = bounded < len(versions)
            versions = versions[-bounded:]
        rows, last_values = [], None
        for version in versions:
            candidates = sorted(
                (row.asDict() for row in self.config_at(version).collect()), key=self._config_key
            )
            match = next(
                (r for r in candidates if Parse.subject_match(subject, *self._config_key(r))), None
            )
            if match is None:
                last_values = None
                continue
            role_name, scope = self._config_key(match)
            values = {field: match.get(field) for field in _CONFIG_DIFF_FIELD_COLUMNS}
            changed = last_values is None or any(
                values[field] != last_values[field] for field in _CONFIG_DIFF_FIELD_COLUMNS
            )
            rows.append(
                {
                    "config_version": version,
                    "role_name": role_name,
                    "scope_key": scope,
                    **values,
                    "changed": changed,
                    "window_truncated": truncated,
                }
            )
            last_values = values
        return self._df(rows, columns=columns)

## Runtime entrypoint — `run_mode` / `run_and_exit`

The engine of the single self-contained notebook. `run_mode` is the core:
resolve target/tenant → build `FabricClient`/`Log`/`Deployment` → dispatch →
**return** the full envelope (blocked/error come back as envelope status — it never exits or
raises on outcome; CI asserts envelopes by calling it directly). `run_and_exit` is what the
▶️ Run dispatch cell calls: reject `mode ∉ allowed` (= `KNOWN_MODES` — a mode-validity check;
least-privilege lives at the Fabric identity/workspace-role layer), else `run_mode`,
then the outcome contract — success/skipped → `notebook.exit(envelope)`;
blocked/error → raise a compact lean payload (Fabric truncates long exceptions; the log is the
authoritative failure channel). It also leaves the full envelope in the notebook namespace as
`envelope` for post-run inspection.

In [ ]:
# ══ Dispatch ═════════════════════════════════════════════════════════════════
# Runtime entrypoint: the engine every mode runs through — run_mode returns the envelope; run_and_exit raises.

# How many column names a returned-frame hint spells out before it summarises the rest. The
# hint exists because `DataFrame[mode: string, status: string, ...]` — the repr a notebook
# echoes for an unassigned Spark frame — reads as noise until you know it is the schema of the
# thing you just got back. Naming the columns turns the same line into the answer. Capped
# because the log passthroughs hand back 27 of them, which would bury the verdict above it.
HINT_COLUMNS = 8

_STATUS_BADGE = {
    "success": "✅ success",
    "skipped": "⏭️  skipped",
    "blocked": "🚫 blocked",
    "error": "❌ error",
}


def _compact(value):
    """One nested level, rendered without Python's repr quoting: a per-role summary reads
    `included=2 excluded=1`, not `{'included': 2, 'excluded': 1}`."""
    if isinstance(value, dict):
        return " ".join(f"{k}={v}" for k, v in value.items())
    if isinstance(value, list):
        return ", ".join(str(v) for v in value) if value else "(none)"
    return value


def _print_result(envelope):
    """Print the run outcome twice, for two different readers.

    A human block first — status badge, the message, and the envelope's own `data` as one
    `key: value` per line — because the operator reading a notebook cell needs the verdict at a
    glance, and a single-line JSON dump of a 10-key envelope is not that.

    Then the machine line, after a blank one so it reads as a separate thing rather than one
    more bullet: the durable, parseable record a pipeline log or a support ticket is read back
    from, and prettier output is no reason to take that away. It is LABELLED `OLAF.last_result`
    — the variable the same envelope is sitting in — so a reader who wants one field does not
    re-run the mode or hand-parse a wrapped JSON dump out of the cell output. run_and_exit sets
    that variable too, so the label is true on the ▶️Run path as well as the facade one. Nested
    values are summarised in the human block (a big dict renders as its size) and stay complete
    here."""
    badge = _STATUS_BADGE.get(envelope["status"], envelope["status"])
    changed = " · changed" if envelope.get("changed") else ""
    # a failed verdict is the one thing `silent` still says — see VERBOSITY_LEVELS
    _failed = envelope["status"] in ("blocked", "error")
    Say.out("silent" if _failed else "quiet", f"\n{badge}{changed} — {envelope['mode']}")
    if envelope.get("message"):
        Say.out("silent" if _failed else "quiet", f"   {envelope['message']}")
    for key, value in (envelope.get("data") or {}).items():
        if isinstance(value, dict):
            if not value:
                rendered = "(none)"
            elif len(value) <= 4:
                rendered = ", ".join(f"{k}={_compact(v)}" for k, v in value.items())
            else:
                rendered = f"{len(value)} entries"
        elif isinstance(value, list):
            rendered = ", ".join(str(v) for v in value) if value else "(none)"
        else:
            rendered = value
        Say.out("info", f"   · {key}: {rendered}")
    if envelope.get("error"):
        Say.out("silent", f"   ↳ {envelope['error']}")
    Say.out("verbose", "\nOLAF.last_result:", json.dumps(envelope, default=str))


def run_mode(mode, params, spark):
    """Run one mode end-to-end and RETURN the full envelope (never exits/raises on outcome).
    ALL failures — unknown mode, no attached lakehouse, missing tenant, auth error, a first-run
    missing config table, or a mode guard — become a proper envelope, never a raw traceback.
    COLLECT-ALL lives inside generate; plan/apply are fail-fast. show/trace are read-only (no log
    row). CI tests call this directly to assert envelopes without catching SystemExit."""
    # NOT bool(): a pipeline passes these as STRINGS and bool("false") is True — see
    # Parse.bool_param. An unparseable value is refused by the guard inside the try below, but
    # only for the modes that actually consume it (see that guard).
    keep_unmanaged, keep_unmanaged_error = Parse.bool_param(
        "keep_unmanaged", params.get("keep_unmanaged", PARAM_DEFAULTS["keep_unmanaged"])
    )
    rebuild, rebuild_error = Parse.bool_param(
        "rebuild", params.get("rebuild", PARAM_DEFAULTS["rebuild"])
    )
    if_match, if_match_error = Parse.bool_param(
        "if_match", params.get("if_match", PARAM_DEFAULTS["if_match"])
    )
    tenant_id = params.get("tenant_id", PARAM_DEFAULTS["tenant_id"])
    config_table = params.get("config_table", PARAM_DEFAULTS["config_table"])
    mapping_table = params.get("mapping_table", PARAM_DEFAULTS["mapping_table"])
    log_table = params.get("log_table", PARAM_DEFAULTS["log_table"])
    member_table = params.get("member_table", PARAM_DEFAULTS["member_table"])
    mapping_history_dir = params.get("mapping_history_dir", PARAM_DEFAULTS["mapping_history_dir"])
    role_backup_dir = params.get("role_backup_dir", PARAM_DEFAULTS["role_backup_dir"])
    control_data_isolation_attestation = params.get(
        "control_data_isolation_attestation",
        PARAM_DEFAULTS["control_data_isolation_attestation"],
    )
    verbosity = str(params.get("verbosity", PARAM_DEFAULTS["verbosity"]) or "").strip().lower()
    # verbosity/env are PARSED here but REFUSED inside the try below, beside their sibling
    # parameter guards, so each refusal becomes the same structured 'blocked' envelope —
    # see the guard block there. params_echo carries the raw env text, rejected or not.
    env, env_error = Parse.env_param(params.get("env", PARAM_DEFAULTS["env"]))
    # setup's lakehouse ASSERTION. A DISTINCT local on purpose: `lakehouse_name` below is the
    # ATTACHED lakehouse read from the runtime context, and the two must never be confused — the
    # whole point of the guard is that they can differ.
    declared_lakehouse = str(
        params.get("lakehouse_name", PARAM_DEFAULTS["lakehouse_name"]) or ""
    ).strip()
    batch_id = params.get("batch_id", PARAM_DEFAULTS["batch_id"])
    by = params.get("by", PARAM_DEFAULTS["by"])
    subject = params.get("subject", PARAM_DEFAULTS["subject"])
    rollback_to_version = params.get("rollback_to_version", PARAM_DEFAULTS["rollback_to_version"])
    rollback_reason = params.get("rollback_reason", PARAM_DEFAULTS["rollback_reason"])

    # Inputs echoed back per mode — computed up front so `params` is available for the envelope
    # even if the build below fails early.
    PARAMS_BY_MODE = {
        "setup": {
            "env": env,
            "rebuild": rebuild,
            "lakehouse_name": declared_lakehouse,
            "config_table": config_table,
            "mapping_table": mapping_table,
            "log_table": log_table,
            "member_table": member_table,
            "control_data_isolation_attestation": control_data_isolation_attestation,
        },
        "generate": {
            "env": env,
            "rebuild": rebuild,
            "control_data_isolation_attestation": control_data_isolation_attestation,
        },
        "validate": {"env": env},
        "plan": {
            "env": env,
            "control_data_isolation_attestation": control_data_isolation_attestation,
        },
        "apply": {
            "env": env,
            "keep_unmanaged": keep_unmanaged,
            "if_match": if_match,
            "control_data_isolation_attestation": control_data_isolation_attestation,
        },
        "rollback": {
            "env": env,
            "rollback_to_version": rollback_to_version,
            "rollback_reason": rollback_reason,
            "if_match": if_match,
            "control_data_isolation_attestation": control_data_isolation_attestation,
        },
        "show": {"env": env, "by": by, "subject": subject},
        "trace": {"env": env},
    }
    params_echo = PARAMS_BY_MODE.get(mode, {"env": env})
    deployment = audit = client = None
    batch = run = None
    envelope = None  # stays None only if a BaseException skipped every handler below
    # The pin is set from a VALID level only; an invalid one falls back to the DEFAULT level
    # so the blocked envelope its in-try guard produces still prints, and the finally still
    # releases the pin either way. Set immediately before the try, so no refusal can slip
    # between pin and release.
    Say.override = verbosity if verbosity in VERBOSITY_LEVELS else PARAM_DEFAULTS["verbosity"]

    def _envelope(status, changed, message, data, error):
        try:
            ch = deployment.config_hash
        except Exception:
            ch = None
        return {
            "mode": mode,
            "status": status,
            "changed": changed,
            "message": message,
            "params": params_echo,
            "data": data,
            "error": error,
            "batch_id": batch,
            "run_id": run,
            "config_hash": ch,
        }

    def _mutation_error_data(exc):
        if hasattr(exc, "as_data"):
            return exc.as_data()
        return {
            name: getattr(exc, name)
            for name in (
                "operation",
                "backup_path",
                "possible_exposure",
                "cleanup_boundary",
                "rollback_progress",
            )
            if hasattr(exc, name)
        }

    def _dispatch():
        if mode == "setup":
            return deployment.setup(rebuild=rebuild)
        if mode == "generate":
            outcome = deployment.generate(rebuild=rebuild)
            Say.out("detail", "   per role — included / excluded / warnings:")
            for _role, _s in outcome.get("data", {}).get("summary", {}).items():
                Say.out(
                    "detail",
                    f"     {_role}: included={_s['included']} "
                    f"excluded={_s['excluded']} warnings={_s['warnings']}",
                )
            return outcome
        if mode == "validate":
            return deployment.validate()
        if mode == "plan":
            return deployment.plan()
        if mode == "apply":
            return deployment.apply(keep_unmanaged=keep_unmanaged)
        if mode == "show":
            return deployment.show(by, subject)
        if mode == "rollback":
            return deployment.rollback(rollback_to_version, rollback_reason)
        # mode == "trace" — the only KNOWN_MODE left (the KNOWN_MODES gate in run_mode rejected
        # everything else before dispatch), so it is the unconditional tail: any trailing guard
        # or fallback raise here would be structurally unreachable and uncoverable.
        report = Audit(spark, config_table, mapping_table, log_table, client=client).report()
        return {"changed": False, "message": "trace report", "data": report}

    try:
        # adding a mode here REQUIRES a matching branch in _dispatch below — trace is the unconditional
        # tail, so an unmatched mode would silently run trace instead of failing loudly
        KNOWN_MODES = {
            "setup",
            "generate",
            "validate",
            "plan",
            "apply",
            "show",
            "trace",
            "rollback",
        }
        if mode in INTERACTIVE_ONLY:
            # Named rather than lumped into "unknown mode": these EXIST, and the refusal is the
            # point. reset() submits an empty DAR payload and cleanup() deletes the audit history AND the
            # backups; neither is something a scheduled pipeline should be able to select by
            # setting a string parameter.
            raise SystemExit(interactive_only_refusal(mode))
        if mode not in KNOWN_MODES:
            raise SystemExit(f"unknown mode: {mode} — expected one of {sorted(KNOWN_MODES)}")
        # Parameter refusals, raised INSIDE the try so each becomes the same structured
        # 'blocked' envelope every sibling guard produces. (During pre-release development
        # these three raised BEFORE the envelope boundary — the one blocked path whose exit
        # payload automation could not parse.) No log context exists yet either
        # way: no pre-target guard writes an audit row.
        if verbosity not in VERBOSITY_LEVELS:
            # Refused, not coerced — the same rule the boolean parameters follow. A typo
            # silently read as "silent" would hide the very run the operator was watching.
            raise SystemExit(
                f"verbosity must be one of {list(VERBOSITY_LEVELS)}, "
                f"got {params.get('verbosity')!r}"
            )
        if env_error:
            # Refused, not coerced — env is written into every audit row AND read back as a
            # SQL WHERE literal by the log reads, so a repaired value would run the deploy
            # under a label the operator never typed.
            raise SystemExit(env_error)
        # The two lakehouse FOLDER parameters, validated (and canonicalized) beside their
        # sibling refusals. Deployment.__init__ re-validates as the backstop for any
        # construction that does not come through this entrypoint (explain(), the
        # destructive utilities, direct use).
        mapping_history_dir = Deployment._require_safe_dir(
            "mapping_history_dir", mapping_history_dir
        )
        role_backup_dir = Deployment._require_safe_dir("role_backup_dir", role_backup_dir)
        # Refuse an unparseable boolean ONLY where it is actually consumed — PARAMS_BY_MODE echoes
        # each parameter for exactly the modes that read it, so the echo IS the definition of
        # "this mode uses it" and cannot drift from a second hand-kept list. Every other mode
        # ignores the parameter, and blocking those would break a pipeline that hands the SAME
        # Base-parameter set to every mode on operations where the value is never read.
        for _name, _error in (
            ("rebuild", rebuild_error),
            ("keep_unmanaged", keep_unmanaged_error),
            ("if_match", if_match_error),
        ):
            if _error and _name in params_echo:
                # a value that could not be parsed never reaches apply's REPLACE fork
                raise SystemExit(_error)
        # Every public mode either writes, prepares a write, or reports facts for the attached
        # target. Resolve it once; setup is deliberately included so it cannot bootstrap schema
        # without the immutable DAR snapshot required by the control-data boundary.
        workspace_id, item_id = Target.resolve()
        import notebookutils as _nu

        _ctx = _nu.runtime.context
        workspace_name = _ctx.get("currentWorkspaceName", "") or ""
        lakehouse_name = _ctx.get("defaultLakehouseName", "") or ""
        # `lakehouse_name`
        # names the lakehouse the operator believes they are setting up; it can never SELECT one,
        # because setup's DDL goes through two-part `olaf.…` names that always resolve against the
        # ATTACHED lakehouse (addressing another would need abfss paths, i.e. multi-lakehouse —
        # deliberately rejected: one lakehouse per deployment). It exists so a wrong attachment is
        # REFUSED here, rather than leaving four
        # control tables plus an audit row in someone else's workspace with nothing to say so.
        if mode == "setup":
            if not declared_lakehouse:
                raise SystemExit(
                    "lakehouse_name is required for setup — name the lakehouse you intend to "
                    "create the control tables in; setup always writes to the ATTACHED lakehouse "
                    "and refuses to run when the name does not match it"
                )
            _attached = str(_ctx.get("defaultLakehouseName", "") or "").strip()
            if declared_lakehouse.lower() != _attached.lower():
                raise SystemExit(
                    f"setup declares lakehouse_name '{declared_lakehouse}' but this notebook "
                    f"is attached to '{_attached}' — setup creates the control tables in the "
                    "ATTACHED lakehouse; attach the one you named (or fix the lakehouse_name "
                    "parameter) and re-run"
                )
        tenant_id = Target.tenant(tenant_id)
        if not tenant_id and mode in (DESIRED_STATE_MODES | {"generate"}):
            raise SystemExit(
                "tenant_id is required (or auto-resolvable from the runtime context) for "
                "generate/plan/apply — it is stamped into the DAR member payloads"
            )
        batch = batch_id or str(uuid.uuid4())
        run = Target.run_id()
        client = FabricClient(workspace_id, item_id)
        audit = Log(
            spark,
            log_table,
            batch,
            run,
            env,
            mode,
            workspace_name,
            lakehouse_name,
            workspace_id=workspace_id,
            lakehouse_id=item_id,
            tenant_id=tenant_id,
            member_table=member_table,
        )
        deployment = Deployment(
            spark,
            client,
            audit,
            tenant_id,
            config_table,
            mapping_table,
            mapping_history_dir,
            workspace_name,
            lakehouse_name,
            member_table=member_table,
            role_backup_dir=role_backup_dir,
            if_match=if_match,
            control_data_isolation_attestation=control_data_isolation_attestation,
        )
        Say.out("detail", f"\n▶️  {mode}")
        if mode == "setup":
            Say.out("detail", f"   batch {batch} · run {run}")
        else:
            Say.out(
                "detail",
                f"   {len(deployment.short_rows)} config row(s) · config {deployment.config_hash}"
                f"\n   batch {batch} · run {run}",
            )
        _out = _dispatch()
        _changed = bool(_out.get("changed"))
        # skipped iff a generate/plan ran but had nothing to do (idempotent generate / no-drift plan)
        _status = "skipped" if (mode in ("generate", "plan") and not _changed) else "success"
        envelope = _envelope(_status, _changed, _out.get("message", ""), _out.get("data", {}), None)
    except PostWriteAuditError as _e:
        _msg = f"{type(_e).__name__}: {_e}"
        envelope = _envelope("error", True, _msg, _mutation_error_data(_e), _msg)
    except PostWriteBoundaryError as _e:
        _msg = f"{type(_e).__name__}: {_e}"
        envelope = _envelope("error", _e.changed, _msg, _mutation_error_data(_e), _msg)
    except (SystemExit, ControlDataGuardError) as _e:
        # A guard refused (validation/member/lakehouse-target/STALE/no-plan/drift). A guard
        # that reaches _reject writes its rejected row only while the active immutable DAR
        # snapshot remains current; otherwise Log.prewrite blocks the sensitive append and the
        # envelope is the failure channel. Pre-target guards fail before any log context exists.
        if getattr(_e, "changed", False) is None:
            _msg = f"{type(_e).__name__}: {_e}"
            envelope = _envelope("error", None, _msg, _mutation_error_data(_e), _msg)
        else:
            envelope = _envelope("blocked", False, str(_e), {}, str(_e))
    except DARConflictError as _e:
        if _e.ambiguous:
            # A 412 on a RETRIED attempt: an earlier attempt of the same PUT may have
            # COMMITTED before its response was lost to the transient status that
            # triggered the retry — _record_push_failure attempts the mid-push forensics
            # only if a fresh immutable DAR snapshot permits that sensitive audit write.
            # A pipeline must see a failure demanding
            # investigation, not a clean guard refusal.
            _msg = f"{type(_e).__name__}: {_e}"
            envelope = _envelope("error", None, _msg, _mutation_error_data(_e), _msg)
        else:
            # The service refused the conditional PUT (412) on the FIRST attempt: a
            # concurrency GUARD outcome, not an unexpected failure — nothing landed, the
            # remedy is a fresh plan. _record_push_conflict appends rows only if the changed
            # DAR can still satisfy the immutable boundary. Blocked, no traceback.
            envelope = _envelope("blocked", False, str(_e), _mutation_error_data(_e), str(_e))
    except Exception as _e:
        traceback.print_exc()
        _msg = f"{type(_e).__name__}: {_e}"
        # unexpected failure — best-effort durable log row if the audit context was built before
        # the crash (audit is None if the crash preceded it; the write then AttributeErrors and is
        # swallowed).
        _changed = getattr(_e, "changed", False)
        if _changed is not None:
            try:
                audit.write([audit.fail_row("run", _e)])
            except Exception:
                pass
        envelope = _envelope("error", _changed, _msg, _mutation_error_data(_e), _msg)
    finally:
        # `finally`, not "after the handlers": the two excepts above catch SystemExit and
        # Exception, so a KeyboardInterrupt — or any other BaseException — skipped the release
        # entirely and leaked this run's verbosity into every later interactive call in the same
        # notebook namespace. The pin is scoped to THIS run and must come off on every exit.
        #
        # The result is printed INSIDE the finally, before the release, because printing it is
        # part of the run and must obey the level the run asked for (`silent` prints only a
        # blocked verdict; `verbose` echoes the envelope). Guarded on `envelope`: a BaseException
        # reaches here having built none, and there is nothing to print.
        try:
            if envelope is not None:
                _print_result(envelope)
        finally:
            # its own finally: a raise inside _print_result must not leak the pin either
            Say.override = None
    # Last thing run_mode does: hand back any lease that never authorized a write. A refusal
    # that got as far as a write keeps its marker -- that state is genuinely unknown. A
    # validation refusal did not, and stranding the next operation behind an incident nobody
    # had was the most common way an operator ever met the sentinel.
    #
    # Here rather than where the envelope is built: the audit trail is still being written at
    # that point, and removing the sentinel underneath it fails the next revalidation.
    if envelope is not None and envelope.get("status") == "blocked" and deployment is not None:
        try:
            deployment.release_unwritten_leases()
        except Exception:
            pass  # never let cleanup mask the refusal the caller needs to read
    return envelope


def run_and_exit(mode, allowed, params, spark):
    """THE production entrypoint this notebook calls for every mode (interactive AND pipeline).
    `mode` outside the caller's `allowed` set is blocked WITHOUT building anything. The ▶️Run
    cell passes every known mode, so this guards against an unknown/typo'd mode, NOT a
    privilege boundary — least-privilege is enforced at the Fabric identity/workspace-role
    layer. Otherwise run the mode, then the native-failure outcome: success/skipped →
    `notebook.exit` (activity Succeeds, exitValue = the FULL envelope; Fabric never truncates an
    exit value); blocked/error → raise (activity FAILS; pipelines branch on the native Failure
    arrow). Fabric wraps + TRUNCATES a long raised exception mid-JSON (lab-confirmed), so the
    raise path emits a COMPACT payload (same envelope shape, data dropped + reason capped) that
    stays valid + parseable — a best-effort secondary channel on top of onelake_security_log, the
    authoritative failure channel. The full envelope is left in the notebook namespace as
    `envelope` for post-run inspection (same affordance as the pre-split runtime)."""
    if mode not in allowed:
        # The two destructive utilities are named rather than lumped in with a typo: they EXIST,
        # and being unreachable from here is the guard, not an oversight. run_mode says the same
        # thing for a caller that reaches it directly.
        _msg = (
            interactive_only_refusal(mode)
            if mode in INTERACTIVE_ONLY
            else f"mode '{mode}' is not permitted by this notebook — allowed: {sorted(allowed)}"
        )
        envelope = {
            "mode": mode,
            "status": "blocked",
            "changed": False,
            "message": _msg,
            "params": {"env": params.get("env", PARAM_DEFAULTS["env"])},
            "data": {},
            "error": _msg,
            "batch_id": None,
            "run_id": None,
            "config_hash": None,
        }
        _print_result(envelope)
    else:
        envelope = run_mode(mode, params, spark)
    globals()["envelope"] = envelope  # post-run inspection + the test blackbox's capture path
    OLAF.last_result = envelope  # ...and the name the printed line points at, on every path
    if envelope["status"] in ("success", "skipped"):
        try:
            import notebookutils

            notebookutils.notebook.exit(json.dumps(envelope, default=str))
        except ImportError:
            pass  # outside Fabric — the envelope was already printed above
    else:
        _CAP = 400  # keep the raised payload well under Fabric's exception-truncation limit

        def _short(_s):
            _s = str(_s)
            if len(_s) <= _CAP:
                return _s
            return _s[:_CAP] + " …[truncated; full reason in onelake_security_log]"

        _lean = {
            **envelope,
            "data": {},
            "message": (
                f"{envelope['status']} — full reason in onelake_security_log "
                f"(batch {envelope.get('batch_id')})"
            ),
            "error": _short(envelope.get("error")),
        }
        raise SystemExit(json.dumps(_lean, default=str))

## OLAF — singleton-static interactive facade.

The interactive/driver surface beside the pipeline dispatch: every method is a FIRST-LEVEL member of `OLAF` (`OLAF.generate()`, `OLAF.plan()`, `OLAF.apply()`, `OLAF.show()`, `OLAF.grants(...)`, `OLAF.setup()`, …), callable without instantiation, every method returns a DataFrame, and nothing raises on a blocked/error outcome (the raw envelope stays at `OLAF.last_result`). `%run`-safe — defining it runs nothing.

In [ ]:
# ══ OLAF ══════════════════════════════════════════════════════════════════════
# Beside the pipeline dispatch (run_and_exit, which RAISES on outcome so a Fabric
# pipeline can branch on the native Success/Failure arrow), OLAF is the interactive
# driver surface: call OLAF.<action>() directly — every method is first-level, no
# instantiation, the ambient `spark` is bound lazily — and get a DataFrame back for
# an easy display().
#
#   OLAF.setup() · .health() · .diagnose_member(member) · .status()
#   OLAF.generate(rebuild=False) · .validate() · .plan() · .apply(keep_unmanaged=False)
#             · .rollback(rollback_to_version="", rollback_reason="") · .explain()
#   OLAF.show(by, subject="") · .trace()
#             + Audit read passthroughs (runs / grants / provenance / timeline /
#               is_stale / verify_chain / report / out_of_band / log_history / batch /
#               failures / last_run / current_generation / authored_by / config_at /
#               effective_access / who_can_access / coverage / drift / config_diff /
#               table_history / at / value_history)
#
# Two invariants set it apart from the pipeline path:
#   • NEVER raises on outcome — deployment/maintenance ops go through run_mode (the
#     non-raising engine); a blocked/error outcome comes back as a DataFrame whose
#     `status` column says so (and OLAF.last_result carries the raw dict). The audit
#     passthroughs read Audit directly.
#   • UNIFORM RETURN — every method returns ONE DataFrame type (a Spark DataFrame,
#     Fabric-native for display()): a query/audit method returns its result table; an
#     ops method returns a compact DataFrame VIEW of the outcome envelope (plan → the
#     role→action diff rows; apply → request status + body/candidate labels; generate → grants/warnings;
#     setup / rollback → a 1-row status summary). All-string schema — the same shape
#     Audit._df uses, so it builds under the CI fake-spark harness AND on Fabric.
#
# The pipeline path (mode param → Run → run_and_exit → notebook.exit JSON) is
# unchanged; OLAF only layers the DataFrame surface on top of the same run_mode engine.

# The Audit read methods OLAF forwards. Membership only: the ONE delegating
# code path lives in the metaclass __getattr__ below, so extending this set costs no new
# branch. (Write-side methods and Audit.trace are intentionally absent — trace is a
# first-class SNAPSHOT method on OLAF, mapping to the run_mode `trace` mode.)
# out_of_band/effective_access/who_can_access ARE passthroughs too: OLAF._trail() lazily binds a
# live FabricClient (see _build_client), so the live-DAR reads work interactively; off-Fabric
# (no client) they raise their clear "needs a FabricClient" error, exactly as a
# client-less Audit does.
_OLAF_AUDIT_PASSTHROUGH = frozenset(
    {
        "runs",
        "log_history",
        "batch",
        "failures",
        "last_run",
        "current_generation",
        "is_stale",
        "verify_chain",
        "grants",
        "provenance",
        "timeline",
        "authored_by",
        "config_at",
        "report",
        "out_of_band",
        "effective_access",
        "who_can_access",
        "coverage",
        "drift",
        "config_diff",
        "table_history",
        "at",
        "value_history",
    }
)


# explain()'s output schema -- also the fallback columns OLAF._frame uses for an EMPTY
# result (an all-inactive/absent config projects to no grants).
EXPLAIN_COLUMNS = [
    "role_name",
    "scope_path",
    "permission",
    "rls_condition",
    "visible_columns",
    "members",
]

# explain()'s error-surface schema: a config with any BLOCKING validation error (one generate()
# would hard-reject, all-or-nothing) projects to this 1-column frame -- one row per error --
# instead of a confident grant preview, keeping the dry-run preview honest.
EXPLAIN_ERROR_COLUMNS = ["error"]


def _explain_unresolvable_target_lister(_base):
    """The `folders` seam explain() injects when the attached target is UNRESOLVABLE (off-Fabric,
    a unit env, no lakehouse pinned, or a lakehouse pinned from ANOTHER workspace -- Target.resolve
    raised there). Raising ValidationError is deliberate: Generate._scope_pair catches exactly that
    and records it in the collect-all aggregate, so the config's OTHER validation errors survive
    alongside it -- which a short-circuit before Catalog.canonical would have thrown away.
    The message names the TARGET as the cause; the pre-fix failure was Catalog.onelake_uri's
    raw 'needs the target workspace/item GUIDs', which reads as a defect in the operator's config.

    The row is appended by _scope_pair's ordinary collect-all path -- the same path every other
    resolver error takes -- so ONE config can produce MORE THAN ONE of them: _scope_pair resolves
    include_folders and exclude_folders in two SEPARATE loops and appends from each. How many is a
    property of the config, not of this seam, and no number is stated here. That repeat is left
    undeduped deliberately: _scope_pair is shared machinery, and deduping there could swallow a
    genuine repeat raised by another rule. The message is kept to two short sentences instead, so
    meeting it more than once in a fix-list costs the reader almost nothing.

    It names the target WITHOUT prescribing a single remedy, deliberately. explain() catches
    (Exception, SystemExit) around Target.resolve and keeps no discriminator, so at this point the
    distinct causes are indistinguishable -- and they do not share a fix: a cross-workspace attached
    lakehouse (Target.resolve's second SystemExit) already HAS a lakehouse pinned, so telling that
    operator to pin one would misattribute the cause exactly the way the pre-fix message did. Pointing at the
    ATTACHMENT names the one place all of those causes are visible without asserting which it was.

    The closing clause states what the caller is actually HOLDING. explain() returns the 1-column
    `error` frame whenever `errors` is non-empty, so while this row stands nothing previews at all --
    not even the table/column scopes. The wording this replaces promised that they 'preview
    normally', which described a frame the same call cannot return."""
    raise ValidationError(
        "cannot preview folder scopes without a resolvable target — check the notebook's lakehouse "
        "attachment. This row is not a defect in your config; while any error stands explain() "
        "returns only this list, no preview"
    )


class _OLAFMeta(type):
    """Metaclass for OLAF: forwards any Audit read method named in _OLAF_AUDIT_PASSTHROUGH as a
    no-instantiation class call, coercing whatever it returns to the uniform DataFrame. Every
    explicit member — the promoted ops/maintenance methods plus show()/trace() — is found normally
    and shadows this. An unknown name raises AttributeError as usual."""

    @property
    def params(cls):
        """Every parameter a run will USE — the sticky values from OLAF.configure() over
        PARAM_DEFAULTS, as a plain dict.

        A property on the metaclass so it reads like its sibling `OLAF.last_result`: an
        attribute, no parentheses, no run needed. `show_params()` returns the same content as a
        DataFrame and adds where each value came from; this is the form you branch on.

        A COPY, so `OLAF.params[...] = ...` cannot quietly become sticky state — configure() is
        the one way in, and it is where the per-call parameters are refused.

        This used to report only what had been SET, on the reasoning that listing the defaults
        would mean a second copy of them. It meant the operator had to hold every default in
        their head to know what a run would do, which is the more expensive mistake. The copy
        was avoidable all along: PARAM_DEFAULTS is the map run_mode itself resolves against, so
        filling the gaps from it cannot drift from what actually runs.

        `keep_unmanaged` and `rebuild` are here because a call that passes neither gets exactly
        these values — but they are per-call and configure() refuses them, so nothing you do
        makes them sticky. show_params() labels them for that reason."""
        return {**PARAM_DEFAULTS, **cls._base_params}

    def __getattr__(cls, name):
        if name not in _OLAF_AUDIT_PASSTHROUGH:
            raise AttributeError(name)

        def _passthrough(*args, **kwargs):
            frame = OLAF._as_frame(getattr(OLAF._trail(), name)(*args, **kwargs))
            return OLAF._announce(name, None, frame)

        return _passthrough


class OLAF(metaclass=_OLAFMeta):
    """Singleton-static interactive facade (see the section header). Call every method directly
    on OLAF — OLAF.plan(), OLAF.grants(role="X"), OLAF.setup() — never instantiate. The Audit read
    methods resolve through the _OLAFMeta metaclass; the ops/maintenance methods plus show()/trace()
    are explicit staticmethods that shadow it. OLAF.configure(**kw) sets base params shared by every
    method (it refuses the per-call keep_unmanaged / rebuild and the REMOVED force — see its
    docstring); OLAF.last_result holds the raw dict of the last run_mode call."""

    _base_params = {}
    last_result = None
    _HEALTH_STALE_APPLY_DAYS = 30  # health(): warn when the newest apply is older than this
    # PER-OPERATION parameters configure() REFUSES -> the per-call home named in the refusal.
    _PER_CALL_PARAMS = {
        "keep_unmanaged": "OLAF.apply(...)",
        "rebuild": "OLAF.generate(...)",
    }
    # REMOVED parameter names configure() REFUSES -> the migration message. Kept SEPARATE from
    # _PER_CALL_PARAMS, which is derived from the live generate/apply signatures: `force` is not a
    # signature parameter anywhere any more, which is the whole point of refusing it.
    _REMOVED_PARAMS = {
        "force": (
            "force was removed and split in two — pass rebuild to OLAF.generate(...) "
            "and keep_unmanaged to OLAF.apply(...), per call. Mind the polarity: "
            "force=True asked for a REPLACE, and keep_unmanaged=False (the default) performs one"
        ),
    }

    # ---- shared plumbing -----------------------------------------------------
    @staticmethod
    def _spark():
        """The ambient Fabric spark session, bound in the notebook namespace by %run / the
        pipeline. Read lazily so `%run olaf` need not have a session at load time."""
        return globals().get("spark")

    @classmethod
    def configure(cls, **kw) -> "DataFrame":
        """Set base params (env, tenant_id, config_table, …) shared by every method.

        Returns a DataFrame of everything currently sticky — the same contract as every other
        method on this facade, and the reason is the notebook: returning the class rendered the
        cell output as `__main__.OLAF`, which tells the operator nothing and looks like a
        mistake. The frame answers the question the caller actually has after configuring,
        which is what is set now.
        `keep_unmanaged` and `rebuild` are REFUSED. They are per-operation parameters on paths that
        mutate live data, and a sticky value could not be honoured even if it were stored: every
        deployment method carries a SIGNATURE DEFAULT, so _run(...) always lands the key in
        _params()'s overrides and the override wins over _base_params — an operator who configured
        keep_unmanaged=True would still get the destructive default REPLACE. A value set once and
        inherited silently by a later destructive call is precisely the hazard, so these two cannot
        be made sticky at all: pass them per call. Refused BEFORE any update, so a rejected call
        stores nothing.

        The REMOVED `force` is refused too, and for a sharper reason: accepted, it would be stored
        as a dead key in the sticky _base_params and ride _params() into run_mode — the ONE path by
        which a legacy `force` still reaches the engine, since the ▶️Run cell builds its params from
        a hardcoded key list that no longer reads it. It is also the single place a migrating
        operator would ever be told about the rename, so the refusal names BOTH successors, where
        each now lives, and the inverted polarity."""
        for _name, _where in cls._PER_CALL_PARAMS.items():
            if _name in kw:
                raise UsageError(
                    f"{_name} is a per-call parameter — pass it to {_where}, it cannot be configured"
                )
        for _name, _message in cls._REMOVED_PARAMS.items():
            if _name in kw:
                raise UsageError(_message)
        if "env" in kw:
            # The interactive twin of run_mode's env guard, and deliberately the SAME rule from
            # the same place: a sticky env set here rides _params() into every later run, and
            # into the two destructive utilities that build their Log outside run_mode entirely.
            # Refused before the update below, so a rejected call stores nothing.
            _, _env_error = Parse.env_param(kw["env"])
            if _env_error:
                raise UsageError(_env_error)
        if "verbosity" in kw:
            # The interactive twin of run_mode's verbosity guard — same rule, same "refused,
            # not coerced" reason: a sticky typo would ride _params() into every later run
            # and be refused there anyway; refusing here names the mistake as it is made,
            # before anything is stored.
            _v = str(kw["verbosity"] or "").strip().lower()
            if _v not in VERBOSITY_LEVELS:
                raise UsageError(
                    f"verbosity must be one of {list(VERBOSITY_LEVELS)}, got {kw['verbosity']!r}"
                )
        cls._base_params.update(kw)
        return cls._announce("configure", cls._params_summary(), cls._params_frame())

    @classmethod
    def _params_summary(cls):
        """ "3 set · 12 default" — the split is the point: it is the one line that tells an
        operator how much of the coming run they actually chose."""
        set_count = len(cls._base_params)
        return f"{set_count} set · {len(cls.params) - set_count} default"

    @classmethod
    def _params_frame(cls):
        """The params table, shared by configure() and show_params() so there is one shape.

        The `source` column is what makes the full listing safe to show: without it a default
        and a deliberate choice render identically, and an operator reading `env dev` cannot
        tell whether someone configured dev or nobody configured anything."""
        rows = []
        for key, value in sorted(cls.params.items()):
            if key in cls._base_params:
                source = "set"
            elif key in cls._PER_CALL_PARAMS:
                source = "per-call"
            else:
                source = "default"
            rows.append({"parameter": key, "value": value, "source": source})
        return cls._frame(rows, columns=PARAMS_COLUMNS)

    @classmethod
    def show_params(cls) -> "DataFrame":
        """Every parameter a run will use, as a DataFrame — the display form of OLAF.params.

        Sets nothing. Before this existed the only way to render the table was to call
        configure() with no arguments: a setter answering a getter's question, which then
        printed "N set" after setting nothing at all.

        It RETURNS the frame and does not draw it, like every other method here — rendering is
        the caller's move (`display(...)`, `.show()`), and a method that drew its own output
        would be the one you could not quietly use as an input."""
        return cls._announce("show_params", cls._params_summary(), cls._params_frame())

    @classmethod
    def _params(cls, **overrides):
        params = dict(cls._base_params)
        params.update(overrides)
        return params

    @classmethod
    def _run(cls, mode, **overrides):
        """Run one mode through the non-raising engine, stash the raw envelope, return it."""
        cls.last_result = run_mode(mode, cls._params(**overrides), cls._spark())
        return cls.last_result

    @classmethod
    def _build_client(cls):
        """Lazily bind a live FabricClient the SAME way the run_mode `trace` path does — resolve the
        attached workspace + lakehouse from the runtime context (Target.resolve), then construct the
        client. Returns None on ANY target-resolution failure (off-Fabric / unit env / no attached
        lakehouse — Target.resolve raises SystemExit or ImportError there): the log-reading
        passthroughs keep working, and the live-DAR utils raise their own clear 'needs a FabricClient'
        error. A seam so tests can drive client construction. `params` is accepted for parity with the
        run_mode target path (today the target is read from the runtime context, not from params)."""
        try:
            workspace_id, item_id = Target.resolve()
            return FabricClient(workspace_id, item_id)
        except (Exception, SystemExit):
            return None

    @classmethod
    def _trail(cls):
        """An Audit bound to the ambient spark + the configured control tables, plus a lazily-bound
        live FabricClient (via _build_client) so the live-DAR passthrough (out_of_band) reads the live
        DAR interactively; off-Fabric the client is None and those utils raise their clear 'needs a
        FabricClient' error, while the log-only reads keep working with no client."""
        params = cls._params()
        client = cls._build_client()
        return Audit(
            cls._spark(),
            params.get("config_table", DEFAULT_CONTROL_TABLES["config_table"]),
            params.get("mapping_table", DEFAULT_CONTROL_TABLES["mapping_table"]),
            params.get("log_table", DEFAULT_CONTROL_TABLES["log_table"]),
            client=client,
            member_table=params.get("member_table", DEFAULT_CONTROL_TABLES["member_table"]),
        )

    # ---- uniform DataFrame builder ------------------------------------------
    @staticmethod
    def _frame_hint(frame):
        """`DataFrame[parameter, value, source]` — what came back, by name.

        Reads `frame.columns`, which is schema metadata on a Spark frame: no query runs, so
        this is safe on the lazy ones (`config_at`, `at`, `table_history`) that a row count
        would have made expensive. Long schemas are truncated rather than wrapped — the log
        passthroughs return 27 columns, and a hint that buries the verdict above it is worse
        than one that stops early."""
        columns = list(frame.columns)
        rest = len(columns) - HINT_COLUMNS
        shown = ", ".join(columns[:HINT_COLUMNS]) + (f", +{rest} more" if rest > 0 else "")
        return f"DataFrame[{shown}]"

    @staticmethod
    def _returned(frame):
        """The same hint as _announce, for the modes, printed as a footer.

        ALWAYS the last thing a call prints, always on a line of its own after a blank one. A
        mode's verdict block ends in a wrapped JSON dump and _announce's ends in a sentence
        about what was found; tacked onto either, the one line that says what you can use next
        is the one you cannot find. Given its own line it is in the same place every time.

        It also lands directly above the `DataFrame[mode: string, ...]` a notebook echoes for
        an unassigned frame, which is the line it exists to explain.

        Never printed on the ▶️Run path: that one exits or raises out of the notebook with no
        caller left to hand a frame to, and this is the log a pipeline failure is read back
        from — it never reaches a view builder, so the footer simply never runs there."""
        Say.out("info", f"\n→  {OLAF._frame_hint(frame)}")
        return frame

    @staticmethod
    def _announce(action, detail, frame, badge="✅"):
        """One line saying an interactive call ran, what it found, and what came back.

        The modes print their own verdict through _print_result; everything else returned a
        DataFrame and printed nothing, so a cell that succeeded and a cell that came back empty
        looked identical until you rendered the frame. This says which it was.

        Deliberately NOT a row count: `config_at`/`at`/`table_history` hand back a lazy Spark
        frame, and counting it here would run a query the caller never asked for. Where a count
        IS the answer — health, diagnose_member — the caller passes it as `detail`, computed
        from the rows it already has in hand.

        The returned frame is NAMED on the line below, by _returned — same footer the modes
        get, same place every time. Naming it matters because the frame is easy to miss and,
        once noticed, opaque: an unassigned Spark frame echoes as `DataFrame[mode: string,
        ...]`, which reads as noise until you know it is the schema of what you just got back.
        Columns, never a binding — an earlier version said "→ display(result)", which guessed a
        variable the caller may never have written. How to render it stays theirs.

        RETURNS the frame it was given, so every caller announces the object it is about to
        hand over and the hint cannot describe a different one.

        `badge` is ⚠️ for the paths that RAN but have nothing good to report — explain() over a
        config that would not generate, say. Those return a real frame and never raise, which
        is deliberate, and announcing them with a ✅ would read as a clean preview."""
        Say.out("quiet", f"{badge} {action}" + (f" · {detail}" if detail else ""))
        return OLAF._returned(frame)

    @staticmethod
    def _frame(rows, columns=None):
        """Build the uniform return DataFrame from a list of dicts — one all-string Spark
        DataFrame, the same shape Audit._df uses (so it builds under the fake-spark CI harness
        and displays natively on Fabric). `columns` is the explicit fallback schema for an EMPTY
        `rows` (Audit._df's same rationale: Spark can't infer a schema from zero sample rows) —
        every existing caller passes a non-empty `rows` and keeps inferring columns from rows[0]."""
        from pyspark.sql.types import StringType, StructField, StructType

        cols = list(rows[0].keys()) if rows else list(columns or [])
        schema = StructType([StructField(c, StringType(), True) for c in cols])
        data = [[None if row.get(c) is None else str(row.get(c)) for c in cols] for row in rows]
        return OLAF._spark().createDataFrame(data, schema)

    @staticmethod
    def _is_frame(value):
        """A DataFrame (Spark or the fake) exposes both collect() and columns; a scalar / dict /
        dataclass does not — the discriminator for pass-through vs. wrap."""
        return hasattr(value, "collect") and hasattr(value, "columns")

    @staticmethod
    def _as_row(value):
        """One dict row for a non-DataFrame Audit return: a dataclass (verify_chain →
        ChainStatus) is flattened, a dict (report / provenance / last_run / current_generation) is
        copied, None or a scalar (is_stale) rides in a single `value` column."""
        from dataclasses import asdict, is_dataclass

        if value is None:
            return {"value": None}
        if is_dataclass(value):
            return asdict(value)
        if isinstance(value, dict):
            return dict(value)
        return {"value": value}

    @classmethod
    def _as_frame(cls, value):
        """Coerce ANY Audit return to the uniform DataFrame: a method that already returns a
        DataFrame passes through; a scalar / dict / dataclass / None is wrapped into a 1-row frame."""
        if cls._is_frame(value):
            return value
        return cls._frame([cls._as_row(value)])

    # ---- ops → DataFrame view ------------------------------------------------
    @classmethod
    def _view(cls, env):
        """A compact DataFrame view of an ops outcome envelope. Every view carries the status
        columns (mode/status/changed/message); plan adds one role→action row per change, apply /
        generate flatten their salient counts, and setup / rollback (plus any blocked/error
        outcome, whose data is empty) get the 1-row status summary. The raw dict stays reachable
        at OLAF.last_result."""
        data = env.get("data", {})
        base = {
            "mode": env.get("mode"),
            "status": env.get("status"),
            "changed": env.get("changed"),
            "message": env.get("message"),
        }
        mode = env.get("mode")
        if mode == "plan":
            plan = data.get("plan", {})
            if plan:
                rows = [
                    {**base, "role": role, "action": action}
                    for role, action in sorted(plan.items())
                ]
            else:
                rows = [{**base, "role": None, "action": None}]
            return cls._frame(rows)
        if mode == "apply":
            return cls._frame(
                [
                    {
                        **base,
                        "push_status": data.get("push_status"),
                        "roles_written": data.get("roles_written"),
                        "keep_unmanaged": data.get("keep_unmanaged"),
                        "request": data.get("request"),
                        "backup_path": data.get("backup_path"),
                        "omitted_role_candidates": data.get("omitted_role_candidates"),
                        "drift_omission_candidates": data.get("drift_omission_candidates"),
                        "post_state_review_required": data.get("post_state_review_required"),
                    }
                ]
            )
        if mode == "generate":
            return cls._frame(
                [
                    {
                        **base,
                        "grants": data.get("grants"),
                        "roles": data.get("roles"),
                        "warnings": data.get("warnings"),
                        "csv": data.get("csv"),
                    }
                ]
            )
        return cls._frame([base])

    @classmethod
    def _show_view(cls, env):
        """audit.show → the enriched grant table (one row per role × scope × member, with
        provenance); when nothing matched, a 1-row summary of the query instead (by/subject/
        matches)."""
        data = env.get("data", {})
        grants = data.get("grants", [])
        if grants:
            return cls._frame(grants)
        return cls._frame(
            [
                {
                    "mode": env.get("mode"),
                    "status": env.get("status"),
                    "by": data.get("by"),
                    "subject": data.get("subject"),
                    "matches": data.get("matches"),
                }
            ]
        )

    @classmethod
    def _trace_view(cls, env):
        """audit.trace → the run_mode trace SNAPSHOT (Audit.report) as a 1-row frame.

        The columns describe WHAT IS DEPLOYED NOW — the live DAR against the current
        generation's mapping — because that is the question after an apply. `established_ever`
        is the log's cumulative total and is named so nobody reads it as today's figure; the
        columns it replaced (`role_count` / `grant_count`) carried that cumulative number under
        names that read as current.

        Two axes: identity (missing / unexpected / out_of_band) asks whether the right principal
        still holds the right table; policy (policy_checked / policy_mismatch) asks whether the
        rule deployed on it still matches the mapping. `in_sync` is true only when both agree.

        This list is HARDCODED and is the only thing that puts a report() key into the frame — a
        key left out of it is dropped from every trace silently, with the suite still green at
        100%. tests/test_olaf_queries.py pins the columns for that reason.

        Every live-state column is None when no FabricClient resolved, since each one needs to
        read the DAR."""
        data = env.get("data", {})
        return cls._frame(
            [
                {
                    "mode": env.get("mode"),
                    "status": env.get("status"),
                    "live_role_count": data.get("live_role_count"),
                    "live_grant_count": data.get("live_grant_count"),
                    "desired_grant_count": data.get("desired_grant_count"),
                    "missing": data.get("missing"),
                    "unexpected": data.get("unexpected"),
                    "out_of_band": data.get("out_of_band"),
                    "policy_checked": data.get("policy_checked"),
                    "policy_mismatch": data.get("policy_mismatch"),
                    "in_sync": data.get("in_sync"),
                    "is_stale": data.get("is_stale"),
                    "established_ever": data.get("established_ever"),
                }
            ]
        )

    # ---- ops · audit · maintenance methods (static — callable, no instantiation) ----
    @staticmethod
    def setup(rebuild: bool = False) -> "DataFrame":
        """Create/migrate the four control tables. → status-summary DataFrame.

        `rebuild=True` DROPS and recreates any control table whose live column types disagree
        with the framework's — the only way to retype a Delta column, and destructive: the
        config's authored rows and the log's audit history go with it. Tables that agree, or
        that merely miss a column, are left to the normal additive path. Reload an author-owned
        table afterwards with OLAF.load_config().

        Needs `lakehouse_name` set FIRST — `OLAF.configure(lakehouse_name="LH_Gold")`, then
        `OLAF.setup()`. It is REQUIRED for setup and is an ASSERTION, not a target:
        setup always writes to the ATTACHED lakehouse (two-part `olaf.…` names), and refuses when
        the name — or the workspace owning the attached lakehouse — does not match. Without it
        the run is refused with "lakehouse_name is required for setup"."""
        return OLAF._returned(OLAF._view(OLAF._run("setup", rebuild=rebuild)))

    _LOADABLE_TABLES = {
        # logical name -> (the _params() key holding the physical name, its expected columns)
        "config": ("config_table", CONFIG_AUTHOR_COLUMNS),
        "member": ("member_table", MEMBER_CACHE_COLUMNS),
    }
    # logical name -> the columns that IDENTIFY a row for load_config's foreign-column
    # carry. Matching is trimmed and case-folded, as everywhere else this framework
    # compares authored values.
    #
    # The two tables differ because their grains do. The member cache has a real logical
    # primary key (member_type + member_name — see data-model.md), so a row keeps its
    # identity across an edit to any other column: a refreshed member_id is the same
    # member, and coexisting provenance rides along with it. The config has NO unique
    # key — one role spans as many rows as it has policy statements, and data-model.md
    # says so outright ("not unique — dup detection is by full-row hash"). Keying the
    # carry on role_name there would collapse every row of a multi-row role onto
    # whichever one the engine happened to return last and hand its provenance to all
    # the others, so config's identity IS the full authored row — the same identity the
    # framework's own duplicate detection uses. An edited row is therefore a NEW row and
    # starts with NULL provenance, which is honest: what the other framework recorded
    # was recorded about the old content.
    _LOADABLE_KEYS = {
        "config": tuple(CONFIG_AUTHOR_COLUMNS),
        "member": ("member_type", "member_name"),
    }

    @staticmethod
    def _read_sheet(fullpath, sheet):
        """One worksheet as a list of dicts, NaN normalised to None.

        Its own method because it is the only place this framework touches a file format rather
        than a table, and because it is the seam a caller without Excel has to stand in for.
        `sheet=None` takes the workbook's first sheet, which is what a single-sheet export is."""
        import pandas

        frame = pandas.read_excel(fullpath) if sheet is None else pandas.read_excel(fullpath, sheet)
        return [
            {k: (None if v is None or v != v else v) for k, v in row.items()}
            for row in frame.to_dict("records")
        ]

    @classmethod
    def _destructive_deployment(cls, need_client, operation="reset"):
        """A Deployment wired from the sticky params for the two DESTRUCTIVE interactive utilities.

        Deliberately NOT reachable through _run/run_mode: reset and cleanup are not modes, so a
        pipeline cannot select them by passing a `mode` parameter. That is their primary guard —
        see run_mode's refusal, which names them rather than saying "unknown mode"."""
        params = cls._params()
        client = cls._build_client()
        if need_client and client is None:
            raise UsageError(
                "needs a live FabricClient — run this from a Fabric notebook with the target "
                "lakehouse attached. Off-Fabric there is no live DAR to change."
            )
        spark = cls._spark()
        tables = {k: params.get(k, DEFAULT_CONTROL_TABLES[k]) for k in DEFAULT_CONTROL_TABLES}
        audit = Log(
            spark,
            tables["log_table"],
            str(uuid.uuid4()),
            Target.run_id(),
            params.get("env", PARAM_DEFAULTS["env"]),
            operation,
            "",
            params.get("lakehouse_name", ""),
            run_by=Target.run_by(spark),
            member_table=tables["member_table"],
        )
        dep = Deployment(
            spark,
            client,
            audit,
            params.get("tenant_id", ""),
            tables["config_table"],
            tables["mapping_table"],
            params.get("mapping_history_dir", PARAM_DEFAULTS["mapping_history_dir"]),
            "",
            params.get("lakehouse_name", ""),
            member_table=tables["member_table"],
            role_backup_dir=params.get("role_backup_dir", PARAM_DEFAULTS["role_backup_dir"]),
            if_match=params.get("if_match", PARAM_DEFAULTS["if_match"]),
            control_data_isolation_attestation=params.get(
                "control_data_isolation_attestation",
                PARAM_DEFAULTS["control_data_isolation_attestation"],
            ),
        )
        # No `dep.log_table = ...` patch here any more. Deployment.__init__ takes it from the
        # audit Log built just above — the SAME tables["log_table"] this used to assign — so
        # cleanup() enumerates four real tables no matter how the Deployment was constructed.
        # Patching it on from outside meant only the object built HERE ever had the attribute.
        return dep

    @staticmethod
    def reset() -> "DataFrame":
        """🔥 DESTRUCTIVE containment request — submits an empty DAR payload.

        The returned role names are roles observed before submission and omitted from that request;
        they are candidates, not confirmed deletions. The Preview contract does not establish
        deletion-by-omission, no-OneLake-security, or universal reader outcomes. Review the
        post-state in the target engine/access mode before drawing an access conclusion.

        OLAF does not recreate platform-managed/default roles. The pre-request backup artifact is
        a recovery input only and does not guarantee exact platform-state restoration. If its
        capture fails, OLAF aborts before submitting the request.

        Control tables are untouched, so `generate` → `plan` → `apply` can submit the config's
        derived payload again. Interactive only — there is no `mode="reset"`.

        → DataFrame: prior-live role candidates, request label, backup artifact, and review flag.
        """
        dep = OLAF._destructive_deployment(need_client=True)
        out = dep.reset()
        rows = [
            {
                "prior_live_role_candidate": n,
                "request": out["request"],
                "backup_path": out["backup_path"],
                "post_state_review_required": out["post_state_review_required"],
            }
            for n in out["prior_live_role_candidates"]
        ] or [
            {
                "prior_live_role_candidate": "(none observed before submission)",
                "request": out["request"],
                "backup_path": out["backup_path"],
                "post_state_review_required": out["post_state_review_required"],
            }
        ]
        return OLAF._announce(
            "reset",
            f"submitted empty payload · {len(out['prior_live_role_candidates'])} prior-live role candidate(s) · post-state review required · backup artifact {out['backup_path']}",
            OLAF._frame(rows),
            badge="🔥",
        )

    @staticmethod
    def cleanup() -> "DataFrame":
        """🔥 DESTRUCTIVE, and the ONE operation with no way back. Empties the lakehouse of every
        trace of this framework so a new environment can start from nothing.

        Drops all four control tables — the authored config, the generated mapping, the member
        table, the **entire audit history** — and deletes every file under the mapping-history and
        role-backup folders. That includes **every pre-apply role backup**, i.e. the recovery for a
        bad `apply` and for `reset()`. Afterwards there is nothing left to restore from.

        Live roles are NOT touched. A role deployed before this stays deployed, now with no audit
        trail explaining where it came from; the returned frame says so when it finds any. If you
        want those gone too, run `reset()` **first** — after `cleanup()` its backup is gone.

        ⚠️ It cannot log what it did: the log table is one of the things it drops. The returned
        frame and the printed lines are the only record this run will ever produce.

        Interactive only — there is no `mode="cleanup"`.

        → DataFrame: one row per dropped table and deleted file, plus any live role left behind.
        """
        dep = OLAF._destructive_deployment(need_client=False)
        out = dep.cleanup()
        rows = (
            [{"kind": "dropped table", "name": t} for t in out["dropped"]]
            + [{"kind": "deleted file", "name": f} for f in out["files_deleted"]]
            + [{"kind": "NOT REMOVED", "name": item} for item in out["not_removed"]]
            + [{"kind": "LIVE ROLE LEFT BEHIND", "name": r} for r in out["live_roles_left"]]
            + [
                {
                    "kind": "INCIDENT SENTINEL PRESERVED",
                    "name": ControlBoundary.SENTINEL_REL_PATH,
                },
                {
                    "kind": "EXPOSURE NOT REMEDIATED",
                    "name": out["containment_warning"],
                },
            ]
        )
        left = len(out["live_roles_left"])
        return OLAF._announce(
            "cleanup",
            f"dropped {len(out['dropped'])} table(s), deleted {len(out['files_deleted'])} file(s)"
            + f", {len(out['not_removed'])} item(s) NOT removed"
            + (
                f" · ⚠️ {left} live role(s) still listed — reset() requires a reviewed empty-payload request and post-state review"
                if left
                else ""
            )
            + " · exposure_remediated=false",
            OLAF._frame(rows),
            badge="🔥",
        )

    @staticmethod
    def clear_incident(access_review: str) -> "DataFrame":
        """Remove the durable incident sentinel only after new isolation evidence and a
        recorded access-review reference. This is containment clearance, not disclosure
        remediation."""
        dep = OLAF._destructive_deployment(need_client=True, operation="sentinel_clearance")
        out = dep.clear_incident(access_review)
        return OLAF._announce(
            "clear_incident",
            "incident sentinel cleared after reviewed fresh boundary evidence; "
            "exposure_remediated=false",
            OLAF._frame([out]),
            badge="⚠️",
        )

    @staticmethod
    def load_config(table: str, path: str, sheet: str | None = None) -> "DataFrame":
        """Load an author-owned control table from a workbook on the lakehouse. → summary DataFrame.

            OLAF.load_config("config", "Files/security/onelake_security.xlsx", "config")
            OLAF.load_config("member", "Files/security/onelake_security.xlsx", sheet="member")

        `table` is the LOGICAL name — "config" or "member" — not the physical table, so the same
        call works whichever names OLAF.configure() points at, exactly like OLAF.at("mapping").

        Only those two are loadable, and the refusal for the other two is the point: the mapping is
        a lock-file `generate` derives (loading one would deploy grants nobody authored) and the log
        is append-only audit history (loading one would forge it).

        The workbook path must live inside the lakehouse `Files/` area — the same
        containment rule the folder parameters follow (an optional leading '/' and any
        letter case of the Files segment are canonicalized; a '..' escape, an absolute
        path outside Files/, a backslash or NUL are refused, not coerced). What this call
        reads becomes deployed access, so it reads only from the one file area the
        framework owns.

        The sheet's columns must match the table's EXACTLY — missing and unexpected are both
        refused, naming them. A sheet loaded with a column missing is a config that silently means
        something different from the one the author edited, and this is the last point before it
        becomes deployed access.

        The write is a full REPLACE of OLAF's OWN columns: the workbook is the source of
        truth, so a row deleted there is deleted here. Columns another framework added to
        the same table are NOT OLAF's to destroy: the schema is never rewritten (no
        overwriteSchema), and a row that comes back UNCHANGED keeps its foreign values —
        matched by the table's row identity (see _LOADABLE_KEYS: the member cache's
        primary key, but the whole authored row for the config, which has no unique key).
        A row the workbook added, and a row whose authored columns were edited, start
        with NULL there; a foreign value disappears when its row does.
        Types come from TableSchema, so `active` lands as a real BOOLEAN. No audit row
        is written — this is authoring, not deployment; the Delta commit is the record, readable via
        OLAF.table_history(...)."""
        if table not in OLAF._LOADABLE_TABLES:
            raise UsageError(
                f"{table!r} is not loadable — expected one of {sorted(OLAF._LOADABLE_TABLES)}. "
                "The mapping is derived by generate and the log is append-only audit history; "
                "loading either would fabricate a record nobody authored."
            )
        param_key, expected = OLAF._LOADABLE_TABLES[table]
        params = OLAF._params()
        target = params.get(param_key, DEFAULT_CONTROL_TABLES[param_key])
        import posixpath

        text = str(path or "").strip()
        normalized = posixpath.normpath(text.lstrip("/"))
        if (
            not text
            or "\\" in text
            or "\x00" in text
            or not normalized.lower().startswith("files/security/")
        ):
            raise UsageError(
                f"path must name a workbook below Files/security — not empty, no backslash, no "
                f"'..' escape — got {path!r}; what load_config reads becomes a control "
                f"table, so a path outside the lakehouse Files/ area is refused, not "
                f"coerced (the same rule the folder parameters follow)"
            )
        path = "Files/" + normalized.split("/", 1)[1]
        client = OLAF._build_client()
        boundary = ControlBoundary(
            client,
            {key: params.get(key, default) for key, default in DEFAULT_CONTROL_TABLES.items()},
            params.get("mapping_history_dir", PARAM_DEFAULTS["mapping_history_dir"]),
            params.get("role_backup_dir", PARAM_DEFAULTS["role_backup_dir"]),
            params.get(
                "control_data_isolation_attestation",
                PARAM_DEFAULTS["control_data_isolation_attestation"],
            ),
        )
        boundary_lease = boundary.begin("load_config")
        rows = OLAF._read_sheet(f"/lakehouse/default/{path}", sheet)
        if not rows:
            raise UsageError(f"{path} sheet {sheet or '(first)'} has no rows — nothing to load")

        found = list(rows[0].keys())
        missing = [c for c in expected if c not in found]
        unexpected = [c for c in found if c not in expected]
        if missing or unexpected:
            raise UsageError(
                f"{path} sheet {sheet or '(first)'} does not match {target}: "
                + ", ".join(
                    part
                    for part in (
                        f"missing {missing}" if missing else "",
                        f"unexpected {unexpected}" if unexpected else "",
                    )
                    if part
                )
            )

        schema, to_row = TableSchema.frame_schema(expected)
        spark = OLAF._spark()
        # The rows are REPLACED; the table's schema is not. A live control table may carry
        # columns another framework added, and the old overwriteSchema write silently
        # dropped them — schema and history both. Now the foreign columns ride along:
        # carried over by row key for rows that survive the replace, NULL on rows the
        # workbook just added, gone only when their row is. mergeSchema only ever ADDS a
        # column OLAF's own contract gained (setup's additive migration is the primary
        # path for that); this write drops nothing, ever.
        prior, live_schema = [], None
        if spark.catalog.tableExists(target):
            live = spark.table(target)
            prior = [r.asDict() for r in live.collect()]
            live_schema = live.schema
        # Case-insensitively, like the engine: a contract column the live table spells
        # `Role_Name` is OLAF's, not a foreign one, and writing it again under the
        # contract spelling would ask Delta for two columns that differ only by case.
        expected_lower = {c.lower() for c in expected}
        foreign = [c for c in (list(prior[0]) if prior else []) if c.lower() not in expected_lower]
        data = [to_row(r) for r in rows]
        if foreign:
            from pyspark.sql.types import StructType

            keys = OLAF._LOADABLE_KEYS[table]

            def _rowkey(d):
                low = {str(k).lower(): v for k, v in d.items()}
                return tuple(str(low.get(c.lower()) or "").strip().lower() for c in keys)

            # A key that matches prior rows disagreeing about the foreign values cannot
            # be resolved — the incoming row is equally "the same row" as either — so it
            # carries NOTHING rather than a coin-flip, and says so. For config that means
            # duplicate authored rows (which generate refuses anyway); for the member
            # cache it means the table has two rows under one primary key.
            carried, ambiguous = {}, {}
            for p in prior:
                k = _rowkey(p)
                seen = carried.get(k)
                if seen is not None and [seen.get(c) for c in foreign] != [
                    p.get(c) for c in foreign
                ]:
                    low = {str(n).lower(): v for n, v in p.items()}
                    ambiguous[k] = low.get(keys[0].lower())
                carried[k] = p
            for label in ambiguous.values():
                Say.out(
                    "detail",
                    f"WARN: {target} has more than one row with {keys[0]}={label!r} "
                    f"carrying different {foreign} values — a coexisting column is "
                    f"loaded as NULL for that row rather than guessed",
                )

            def _carry(typed):
                key = _rowkey(dict(zip(expected, typed)))
                prev = {} if key in ambiguous else (carried.get(key) or {})
                return typed + [prev.get(c) for c in foreign]

            data = [_carry(typed) for typed in data]
            live_fields = {f.name: f for f in live_schema.fields}
            schema = StructType(list(schema.fields) + [live_fields[c] for c in foreign])
        frame = spark.createDataFrame(data, schema)
        boundary_lease.prewrite()
        frame.write.option("mergeSchema", "true").mode("overwrite").saveAsTable(target)
        affected_version = None
        try:
            affected_version = Catalog.config_version(spark, target)
            boundary_lease.postcheck()
            boundary_lease.clear()
        except BaseException as exc:
            affected = json.dumps(
                {"table": target, "version": affected_version, "source": path},
                sort_keys=True,
                separators=(",", ":"),
            )
            raise PostWriteBoundaryError("load_config", affected, exc, changed=True) from exc
        return OLAF._announce(
            "load_config",
            f"{len(rows)} row(s) into {target} · from {path}" + (f" · {sheet}" if sheet else ""),
            OLAF._frame([{"table": target, "rows": len(rows), "source": path, "sheet": sheet}]),
        )

    @staticmethod
    def health() -> "DataFrame":
        """One-call doctor → a DataFrame with cols check · status(pass/warn/fail) · detail, ONE
        row per check. A P4 util: added explicitly here (not via the audit passthrough) because it
        assembles its own multi-row dict-list, exactly like setup()/generate() build their view.

        Nine independent checks, each wrapped so ONE failing check can never abort the others and
        health() ALWAYS returns a frame (never raises):
          • control_tables    — the four control tables exist with their expected schema (the same
                               table→columns map setup() migrates against; member_table needs no
                               client).
          • table_location    — the tables this session reads are the ones it is pointed at. Every
                               other check reads "the control tables" as though there were only one
                               set; they are whatever the ATTACHED lakehouse holds, so a changed
                               attachment silently swaps them (and a setup run before the
                               cross-workspace guard existed left a set orphaned elsewhere).
          • mapping_staleness — the deployed mapping still matches the live active config, via the
                               existing Audit.is_stale() (byte-for-byte the generator's STALE guard).
          • dar_reachable     — a live FabricClient resolved (OLAF._build_client succeeded); fail
                               off-Fabric / no lakehouse attached.
          • control_data_exposure — a bounded DAR snapshot and separate control-boundary facts
                               (ETag, reserved paths, and workspace isolation attestation).
          • identity_preflight— a Fabric API token was acquired for the AMBIENT identity (the exact
                               mechanism FabricClient uses — notebookutils.credentials
                               .getToken, exercised by _build_client) and that principal can be named
                               via Target.run_by; warn off-Fabric where it can't be exercised.
          • runtime_prerequisites — observed Spark baseline; Fabric Runtime labels are not inferred.
          • last_apply_age    — how long since the newest completed deployment apply leg.
                               warn when never applied or older than _HEALTH_STALE_APPLY_DAYS.
          • out_of_band       — count of live DAR grants with no framework provenance, via the
                               existing Audit.out_of_band() (needs the client).

        When no client resolves (off-Fabric / _build_client → None) the DAR-dependent rows
        (dar_reachable/identity_preflight/out_of_band) report fail/warn WITH a clear detail while
        the log/mapping-based rows (control_tables/mapping_staleness/last_apply_age) are STILL
        evaluated. Returned via OLAF._frame (the same all-string multi-row builder setup()/generate()
        reach through _view), so a partial result still builds under the fake-spark harness AND
        displays natively on Fabric."""
        trail = OLAF._trail()
        spark = OLAF._spark()
        client = trail.client
        params = OLAF._params()
        member_table = params.get("member_table", "olaf.onelake_security_member")
        boundary_probe = {"done": False}

        def _probe_control_boundary():
            if boundary_probe["done"]:
                return boundary_probe
            boundary_probe.update(
                {
                    "done": True,
                    "snapshot": None,
                    "snapshot_error": None,
                    "reserved_paths": ["/files/security"],
                }
            )
            attestation = params.get("control_data_isolation_attestation", "")
            boundary_probe["workspace_isolation"] = (
                "attested" if CONTROL_EVIDENCE_RE.fullmatch(str(attestation or "")) else "unknown"
            )
            if client is None:
                boundary_probe["snapshot_error"] = "FabricClient unavailable"
                return boundary_probe
            try:
                boundary = ControlBoundary(
                    client,
                    {
                        "config_table": trail.config_table,
                        "mapping_table": trail.mapping_table,
                        "log_table": trail.log_table,
                        "member_table": member_table,
                    },
                    params.get("mapping_history_dir", PARAM_DEFAULTS["mapping_history_dir"]),
                    params.get("role_backup_dir", PARAM_DEFAULTS["role_backup_dir"]),
                    attestation,
                )
                boundary_probe["reserved_paths"] = list(boundary.reserved)
                roles = client.list_roles_quick()
                boundary_probe["snapshot"] = boundary.snapshot_from(
                    roles, getattr(client, "roles_etag", None)
                )
            except Exception as exc:
                boundary_probe["snapshot_error"] = f"{type(exc).__name__}: {exc}"
            return boundary_probe

        def _control_tables():
            expected = {
                trail.config_table: CONFIG_AUTHOR_COLUMNS,
                trail.mapping_table: MAPPING_COLUMNS + MAPPING_PROVENANCE_COLUMNS,
                trail.log_table: LOG_COLUMNS,
                member_table: MEMBER_CACHE_COLUMNS,
            }
            # Extra columns follow setup()'s ownership policy (issue #2): on the
            # author-owned config/member tables they are supported coexistence (a pass
            # with a note — config_hash ignores them, load_config preserves them); on
            # the framework-owned mapping/log they warrant a WARN — generate rewrites
            # the mapping in full and will drop them, and the log is append-only
            # history nothing else should be shaping.
            problems, warned, coexisting = [], [], []
            for table, cols in expected.items():
                if not spark.catalog.tableExists(table):
                    problems.append(f"{table} (absent)")
                    continue
                live_cols = [str(c) for c in spark.table(table).columns]
                live = {c.lower() for c in live_cols}
                missing = [c for c in cols if c.lower() not in live]
                if missing:
                    problems.append(f"{table} (missing {', '.join(missing)})")
                declared = {c.lower() for c in cols}
                extras = [c for c in live_cols if c.lower() not in declared]
                if extras and table in (trail.mapping_table, trail.log_table):
                    warned.append(f"{table} (unmanaged {', '.join(extras)})")
                elif extras:
                    coexisting.append(f"{table} ({', '.join(extras)})")
            if problems:
                return "fail", "control-table issue(s) — " + "; ".join(problems)
            if warned:
                return "warn", (
                    "unmanaged column(s) on framework-owned table(s) — "
                    + "; ".join(warned)
                    + " (the mapping is rewritten in full by generate; the log is "
                    "append-only history)"
                )
            detail = "all 4 control tables present with the expected schema"
            if coexisting:
                detail += (
                    " · coexisting column(s) " + "; ".join(coexisting) + " — outside the "
                    "framework contract (ignored by config_hash, preserved by load_config)"
                )
            return "pass", detail

        def _table_location():
            try:
                import notebookutils
            except ImportError:
                return "warn", "no runtime context off Fabric — cannot tell where the tables live"
            ctx = notebookutils.runtime.context
            here_ws, here_lh = ctx.get("currentWorkspaceId"), ctx.get("defaultLakehouseId")
            there_ws = ctx.get("defaultLakehouseWorkspaceId")
            if there_ws and here_ws and there_ws != here_ws:
                return (
                    "fail",
                    "attached lakehouse lives in "
                    f"{Target.ws_label(ctx, 'defaultLakehouseWorkspaceId', 'defaultLakehouseWorkspaceName')}"
                    " but this notebook runs in "
                    f"{Target.ws_label(ctx, 'currentWorkspaceId', 'currentWorkspaceName')}"
                    " — every mode refuses this pairing, and a setup that ran before that guard "
                    "existed left four control tables plus an audit row over there",
                )
            rows = [r.asDict() for r in spark.table(trail.mapping_table).collect()]
            if not rows:
                return (
                    "pass",
                    "attached lakehouse is in this workspace; no mapping generated yet, so there "
                    "is no stamped target to cross-check against",
                )
            MappingProvenance.require(rows)
            was_ws, was_lh = rows[0].get("workspace_id"), rows[0].get("lakehouse_id")
            if (was_ws, was_lh) != (here_ws, here_lh):
                return (
                    "fail",
                    f"the mapping in this lakehouse was generated against workspace {was_ws} / "
                    f"lakehouse {was_lh}, but this notebook is attached to {here_ws} / {here_lh} — "
                    "the attachment changed under the tables; re-attach the original lakehouse, or "
                    "re-run generate to re-target these tables",
                )
            return (
                "pass",
                f"control tables are in the attached lakehouse "
                f"({rows[0].get('lakehouse_name')}) in this workspace",
            )

        def _mapping_staleness():
            gen = trail.current_generation()
            if trail.is_stale():
                if gen is None:
                    return (
                        "warn",
                        "no mapping generated yet — run generate to build the lock-file",
                    )
                return "warn", "mapping is stale vs the active config — regenerate to refresh"
            return (
                "pass",
                f"mapping matches the active config (version {gen.get('config_version')})",
            )

        def _dar_reachable():
            probe = _probe_control_boundary()
            snapshot = probe.get("snapshot")
            if snapshot is None:
                return (
                    "fail",
                    "bounded DAR read failed or was incomplete — "
                    + str(probe.get("snapshot_error") or "unknown response"),
                )
            return (
                "pass",
                f"bounded DAR read succeeded with collection ETag {snapshot.etag}",
            )

        def _control_data_exposure():
            probe = _probe_control_boundary()
            snapshot = probe.get("snapshot")
            facts = {
                "dar_snapshot_safe": snapshot is not None,
                "dar_etag": snapshot.etag if snapshot is not None else None,
                "reserved_paths": probe.get("reserved_paths", []),
                "snapshot_error": probe.get("snapshot_error"),
                "workspace_isolation": probe.get("workspace_isolation", "unknown"),
            }
            status = (
                "pass"
                if facts["dar_snapshot_safe"] and facts["workspace_isolation"] == "attested"
                else "fail"
            )
            return status, json.dumps(facts, sort_keys=True, separators=(",", ":"))

        def _identity_preflight():
            who = Target.run_by(spark) or "unknown"
            if client is None:
                return (
                    "warn",
                    f"identity preflight needs a live Fabric session — running as {who}, "
                    "no Fabric token acquired",
                )
            return "pass", f"Fabric token acquired for the ambient identity ({who})"

        def _runtime_prerequisites():
            version = str(getattr(spark, "version", "") or "")
            match = re.match(r"^(\d+)\.(\d+)", version)
            if not match or (int(match.group(1)), int(match.group(2))) < (3, 5):
                return (
                    "fail",
                    "Fabric Runtime 1.3 / Spark 3.5+ baseline not met; observed Spark "
                    + (version or "unknown")
                    + ". Verify the Fabric Runtime label in the environment; OLAF does not infer it.",
                )
            return (
                "pass",
                f"Spark {version} meets the Fabric Runtime 1.3 / Spark 3.5+ baseline; "
                "verify the Fabric Runtime label in the environment",
            )

        def _last_apply_age():
            last = trail.last_successful_deployment()
            if not last or not last.get("run_at"):
                return "warn", "no successful deployment recorded — run apply to deploy the mapping"
            when = datetime.datetime.fromisoformat(str(last["run_at"]))
            age_days = (datetime.datetime.now(when.tzinfo) - when).days
            if age_days > OLAF._HEALTH_STALE_APPLY_DAYS:
                return (
                    "warn",
                    f"last apply was {age_days} day(s) ago — the live DAR may be drifting",
                )
            return "pass", f"last apply was {age_days} day(s) ago"

        def _out_of_band():
            if client is None:
                return (
                    "warn",
                    "cannot count out-of-band grants without a live DAR client (off-Fabric)",
                )
            count = len(trail.out_of_band().collect())
            if count:
                return (
                    "warn",
                    f"{count} out-of-band grant(s) with no framework provenance — review",
                )
            return "pass", "no out-of-band grants — every live grant has framework provenance"

        checks = [
            ("control_tables", _control_tables),
            ("table_location", _table_location),
            ("mapping_staleness", _mapping_staleness),
            ("dar_reachable", _dar_reachable),
            ("control_data_exposure", _control_data_exposure),
            ("identity_preflight", _identity_preflight),
            ("runtime_prerequisites", _runtime_prerequisites),
            ("last_apply_age", _last_apply_age),
            ("out_of_band", _out_of_band),
        ]
        rows = []
        for check, fn in checks:
            try:
                status, detail = fn()
            except Exception as exc:  # a health check must NEVER abort the others
                status, detail = "fail", f"check could not run: {exc}"
            rows.append({"check": check, "status": status, "detail": detail})
        failed = sum(1 for r in rows if r["status"] != "pass")
        return OLAF._announce(
            "health",
            f"{len(rows)} check(s) · all pass"
            if not failed
            else f"{len(rows)} check(s) · {failed} not passing",
            OLAF._frame(rows),
        )

    @staticmethod
    def status() -> "DataFrame":
        """One-call at-a-glance deployment snapshot -> a 1-ROW DataFrame: n_roles ·
        n_members · last_generate · last_apply · last_deployment · last_deployment_mode ·
        live_config_version · pending_change(bool).
        Built purely from the log + mapping (no live client needed -- unlike health()'s
        DAR-dependent checks). A P4 util: explicit on OLAF, following health()'s
        shape -- OLAF._trail() for the log/mapping reads, OLAF._frame([row]) for the uniform
        return (the same multi-row builder health() uses; a 1-element list gives a 1-row frame).

          • n_roles / n_members  — distinct counts straight off the mapping lock-file:
                                   n_roles is the distinct role_name across every mapping row;
                                   n_members flattens every row's typed member_*_ids column via
                                   MAPPING_MEMBER_COLUMNS + Parse.list (the exact pair
                                   diagnose_member's in_mapping step and drift()'s desired-side
                                   reads use) into one case-insensitively deduped objectId set.
                                   An empty mapping -> 0/0 (no grants to count).
          • last_generate        — the newest logged generate-mode row's run_at
                                   (Audit.last_run("generate")). None when nothing has been
                                   generated yet.
          • last_apply           — the newest durably-proven successful apply's run_at
                                   (Audit.last_successful_deployment("apply")); health() and
                                   diagnose_member's apply_in_sync step read). None when nothing
                                   has been applied yet.
          • last_deployment / last_deployment_mode — newest successful completed deployment
                                   across apply and rollback, selected only from a durable
                                   completion record with its backup and payload proof.
          • live_config_version  — the config version the CURRENT mapping was generated from
                                   (Audit.current_generation()'s config_version -- the mapping's
                                   own stamped provenance, not a fresh catalog lookup). None when
                                   no mapping has been generated.
          • pending_change       — True when the deployed mapping is newer than the last apply:
                                   no mapping yet -> False (nothing pending); a mapping with no
                                   successful apply yet -> True (an unapplied generation IS a
                                   pending change); otherwise it requires the exact ordered
                                   versioned generate -> plan -> apply chain for this mapping.
        """
        trail = OLAF._trail()
        mapping_rows, _provenance = trail._mapping_rows()

        n_roles = len(
            {row.get("role_name") for row in mapping_rows if row.get("role_name") is not None}
        )
        member_ids = set()
        for row in mapping_rows:
            for _name_col, id_col, _mtype in MAPPING_MEMBER_COLUMNS:
                for member in Parse.list(row.get(id_col)):
                    member_ids.add(member.lower())
        n_members = len(member_ids)

        last_generate = trail.last_run("generate")
        last_deployment = trail.last_successful_deployment()
        last_apply = trail.last_successful_deployment("apply")
        gen = trail.current_generation()

        if gen is None:
            pending_change = False  # nothing generated yet -- no change can be pending
        else:
            pending_change = not trail.verify_chain().ok

        row = {
            "n_roles": n_roles,
            "n_members": n_members,
            "last_generate": last_generate.get("run_at") if last_generate else None,
            "last_apply": last_apply.get("run_at") if last_apply else None,
            "last_deployment": last_deployment.get("run_at") if last_deployment else None,
            "last_deployment_mode": last_deployment.get("mode") if last_deployment else None,
            "live_config_version": gen.get("config_version") if gen else None,
            "pending_change": pending_change,
        }
        return OLAF._announce(
            "status",
            f"{row['n_roles']} role(s) · {row['n_members']} member(s)",
            OLAF._frame([row]),
        )

    @staticmethod
    def diagnose_member(member: str) -> "DataFrame":
        """Why can't `member` see data? -> a DataFrame with cols step · ok(bool) · detail, ONE
        row per step, walking the same chain a human troubleshooter would, IN ORDER:

          1. member_in_table — is `member` present (any type, case-insensitive name match) in
                                the onelake_security_member cache? An objectId-shaped `member`
                                (GUID_RE — the same pass-through Audit._resolve_member gives
                                effective_access) needs no lookup: it IS the objectId, so this
                                step reports ok=True and steps 2-5 run against it unchanged.
          2. id_resolved     — does that cache row carry a non-blank, GUID-shaped objectId
                                (GUID_RE — the same guard _load_member_cache enforces)?
          3. in_mapping      — does that objectId appear in any onelake_security_mapping row's
                                member_*_ids columns, for any role (MAPPING_MEMBER_COLUMNS +
                                Parse.list — the pair Member.resolve_ids/DAR.to_role read)?
          4. live_in_dar     — does that objectId show up as a live DAR member of any role
                                (self.client.list_roles() + DAR.paths_and_members — the same
                                read out_of_band()/effective_access() use)? Needs a
                                FabricClient: with none resolved (OLAF._trail().client is None)
                                this reports ok=False, detail "no live client" — it never
                                raises the Audit methods' usual "needs a FabricClient" error.
          5. apply_in_sync   — is the deployed mapping NOT newer than the last successful
                                apply (Audit.current_generation()'s generated_at vs.
                                Audit.last_successful_deployment()'s run_at — the same timestamps
                                health()'s last_apply_age check reads)? A mapping regenerated
                                since the last apply means even a correctly-mapped,
                                correctly-resolved member may not be live yet.

        Steps 1-3 are a DEPENDENCY CHAIN, not health()'s six independent checks: once one of
        them fails the member's identity/role membership is unknown, so probing steps 4-5
        would report misleading detail (e.g. "not live" when the real reason is "never in the
        cache") — every step after the first broken link is SHORT-CIRCUITED instead, with
        detail "skipped — prerequisite failed". Steps 4 and 5 are independent of EACH OTHER:
        apply_in_sync is a deployment-wide signal, not member-specific, so a client-less step
        4 does NOT skip it.
        Every step is wrapped so one broken step reports ok=False with a detail instead of
        aborting the rest — diagnose_member ALWAYS returns a frame, mirroring health()'s
        never-raises invariant, via the same OLAF._frame(rows) multi-row builder."""
        trail = OLAF._trail()
        spark = OLAF._spark()
        member_table = OLAF._params().get("member_table", "olaf.onelake_security_member")
        name = str(member or "").strip()
        resolved_id = None  # set by _member_in_table; read by every step after it

        def _member_in_table():
            nonlocal resolved_id
            # GUID pass-through, mirroring Audit._resolve_member: an objectId copied out of
            # who_can_access()'s member_id column is already resolved — no name lookup can
            # match it, and reporting "not found" for it would be plain wrong.
            if GUID_RE.match(name):
                resolved_id = name
                return True, f"'{name}' is objectId-shaped — used as-is, no name lookup"
            match = next(
                (
                    row
                    for row in (r.asDict() for r in spark.table(member_table).collect())
                    if str(row.get("member_name") or "").strip().lower() == name.lower()
                ),
                None,
            )
            if match is None:
                return False, f"'{name}' not found in {member_table}"
            resolved_id = str(match.get("member_id") or "").strip()
            return True, f"found in {member_table} (type {match.get('member_type')})"

        def _id_resolved():
            if resolved_id and GUID_RE.match(resolved_id):
                return True, f"resolved to objectId {resolved_id}"
            return (
                False,
                f"'{name}' has no valid objectId in {member_table} (got {resolved_id!r})",
            )

        def _in_mapping():
            mid = resolved_id.lower()
            mapping_rows, _provenance = trail._mapping_rows()
            roles = sorted(
                {
                    row.get("role_name")
                    for row in mapping_rows
                    for _name_col, id_col, _mtype in MAPPING_MEMBER_COLUMNS
                    if mid in {v.lower() for v in Parse.list(row.get(id_col))}
                }
            )
            if roles:
                return True, f"in role(s): {', '.join(roles)}"
            return (
                False,
                f"objectId {resolved_id} not found in any role's member_*_ids in "
                f"{trail.mapping_table}",
            )

        def _live_in_dar():
            if trail.client is None:
                return False, "no live client"
            mid = resolved_id.lower()
            roles = sorted(
                r["name"]
                for r in trail.client.list_roles()
                if mid in {m.lower() for m in DAR.paths_and_members(r)[1]}
            )
            if roles:
                return True, f"live in role(s): {', '.join(roles)}"
            return (
                False,
                "not found live in any DAR role — apply may not have run, or the grant was "
                "removed out-of-band",
            )

        def _apply_in_sync():
            gen = trail.current_generation()
            last_apply = trail.last_successful_deployment()
            if not last_apply or not last_apply.get("run_at"):
                return False, "no successful apply recorded — run apply to deploy the mapping"
            if str(gen["generated_at"]) > str(last_apply["run_at"]):
                return (
                    False,
                    f"mapping generated at {gen['generated_at']} is newer than the last apply "
                    f"at {last_apply['run_at']} — re-run apply to sync",
                )
            return True, f"mapping is in sync with the last apply ({last_apply['run_at']})"

        prerequisite_steps = {"member_in_table", "id_resolved", "in_mapping"}
        steps = [
            ("member_in_table", _member_in_table),
            ("id_resolved", _id_resolved),
            ("in_mapping", _in_mapping),
            ("live_in_dar", _live_in_dar),
            ("apply_in_sync", _apply_in_sync),
        ]
        rows, broken = [], False
        for step, fn in steps:
            if broken:
                rows.append({"step": step, "ok": False, "detail": "skipped — prerequisite failed"})
                continue
            try:
                ok, detail = fn()
            except Exception as exc:  # a diagnose step must NEVER abort the rest of the chain
                ok, detail = False, f"check could not run: {exc}"
            rows.append({"step": step, "ok": ok, "detail": detail})
            if step in prerequisite_steps and not ok:
                broken = True
        stopped = [r for r in rows if r["ok"] is not True]
        return OLAF._announce(
            "diagnose_member",
            f"{len(rows)} step(s) · all ok"
            if not stopped
            else f"first failing step: {stopped[0]['step']}",
            OLAF._frame(rows),
        )

    @staticmethod
    def generate(rebuild: bool = False) -> "DataFrame":
        """Build the mapping lock-file from the short config. → grants/roles/warnings DataFrame."""
        return OLAF._returned(OLAF._view(OLAF._run("generate", rebuild=rebuild)))

    @staticmethod
    def validate() -> "DataFrame":
        """Dry-run generate's IDENTICAL validation pipeline with ZERO writes — no mapping, no
        CSV, no log row (not even a 'rejected' one), so it is safe to run against a live
        deployment. → the same error set generate would surface (blocked, collect-all), or a
        clean success envelope with the grant / role counts and every warning."""
        return OLAF._returned(OLAF._view(OLAF._run("validate")))

    @staticmethod
    def plan() -> "DataFrame":
        """Diff desired (mapping) vs. live DAR. → one role→action row per change."""
        return OLAF._returned(OLAF._view(OLAF._run("plan")))

    @staticmethod
    def apply(keep_unmanaged: bool = False) -> "DataFrame":
        """Submit the planned DAR payload. → request status, body count, and omission candidates."""
        return OLAF._returned(OLAF._view(OLAF._run("apply", keep_unmanaged=keep_unmanaged)))

    @staticmethod
    def rollback(rollback_to_version: int | str = "", rollback_reason: str = "") -> "DataFrame":
        """Restore a prior config version and re-run the chain. → status-summary DataFrame.

        `rollback_to_version` is forwarded ONLY when the call actually names one. Every
        facade method carries a signature default, so the key otherwise always lands in
        _run()'s overrides and always beats _base_params — which made a CONFIGURED
        rollback_to_version unreachable: it was silently dropped and you rolled back ONE
        version instead of to N. Blank now means "not stated here", so the configured value is
        used; a value passed to the call still wins over it. The cost is that a call cannot
        re-request "the previous version" over a configured pin — but that IS the default, so
        expressing it means simply not configuring one.

        `rollback_reason` is deliberately NOT given the same treatment, and this asymmetry is
        the whole reason a general _UNSET sentinel across the facade was declined: made
        omittable, a configured reason would become STICKY and stamp a stale justification
        into the audit log of a later, unrelated rollback. Passing it unconditionally means a
        reasonless call is REFUSED by run_mode's reason guard. A loud failure beats a quiet
        falsification of an audit trail."""
        overrides = {"rollback_reason": rollback_reason}
        if str(rollback_to_version).strip() != "":
            overrides["rollback_to_version"] = rollback_to_version
        return OLAF._returned(OLAF._view(OLAF._run("rollback", **overrides)))

    @staticmethod
    def explain() -> "DataFrame":
        """Preview the roles/scopes/predicates a config WOULD produce, BEFORE generate runs --
        a dry projection over generate's OWN resolution chain (Catalog.canonical ->
        Generate.rows -> Generate._build_grants), reading the SAME config
        Deployment.short_rows reads, but stopping BEFORE generate's write step: no mapping
        row, no log row, no live FabricClient (no DAR call at all) -- nothing here ever calls
        .write()/.saveAsTable(). It DOES read onelake_security_member: the member gate runs here
        (paragraph 3), and read-only is not the same as call-free.

        SAME CHECKS AS validate() AND generate(), different OUTPUT. All three call
        Deployment._run_validation(): config rules (Generate.rows -- A2 / C9 / C11 / C14 /
        column-existence), the No-Graph member gate, and the lakehouse target guard. explain()
        returns the projection or the error list and never raises; validate() returns an
        envelope; generate() writes when clean and rejects when not.

        ONE documented exception: the lakehouse TARGET guard. It resolves the declared lakehouse
        against the workspace's items through the Fabric API, so it needs a live client, and
        explain() has none by design. The member gate is a plain Delta read and IS run here --
        it was the layer that actually misled, since a config previewing clean on an unseeded
        member is the common case, while an unattached lakehouse announces itself.

        READ-ONLY is not the same as call-free. Exactly like generate, explain() resolves the
        attached workspace/lakehouse GUIDs (Target.resolve, inside the same swallow-everything
        try/except OLAF._build_client uses, so an unresolvable target yields (None, None)
        instead of raising) and hands them to Catalog.canonical, whose production folder lister
        issues REAL read-only notebookutils.fs.ls listings when this runs on Fabric. HOW MANY
        depends on the config -- how many folder entries it declares, how deep each path runs,
        and any glob, which multiplies it. Catalog.resolve_folders is the AUTHORITY on that
        walk; its arithmetic is not restated here or anywhere else. A folder entry of exactly
        '/Files' is where the walk already starts, so it lists nothing and reports nothing.
        That listing IS generate's folder resolution: previewing folder scopes without it was
        the bug this fixed -- every folder-scoped config came back as an error frame blaming
        the operator.
        Table/column-only configs need no GUIDs and touch no filesystem (Catalog.canonical's
        own docstring blesses omitting them). A P4 util: built directly (not via run_mode),
        exactly like health()/status()/diagnose_member() build their own multi-row dict-list
        and return it through OLAF._frame.

        FOR THE NEXT EDITOR, and the reason the wording above names no numbers: describe what
        KIND of thing happens and where the AUTHORITY for it lives -- never a count. A count
        here cannot be checked without tracing _scope_pair, Catalog.resolve_folders and
        ScopePath.folder at once; a shape plus a pointer can be checked by reading one.

        -> one row per role x scope grant: role_name / scope_path / permission / rls_condition /
        visible_columns / members -- the four typed member-name columns MAPPING_MEMBER_COLUMNS
        carries (member_group_names/member_user_names/member_sp_names/member_mi_names),
        flattened into ONE ';'-joined, human-scannable column. The id columns generate resolves
        from the member cache are an internal DAR-payload detail, irrelevant to a pre-generate
        preview, so they are left off. That is a projection choice, NOT evidence the table is
        untouched: explain() reads onelake_security_member and reports what the gate finds there
        (see paragraph 3) -- the ids are simply not carried into the frame.
        An empty/all-inactive config (0 active rows) -> an EMPTY but
        TYPED frame (EXPLAIN_COLUMNS) -- generate's own 'nothing to build' case, without
        generate's hard refusal.

        When the config carries a BLOCKING validation error (one generate() would hard-reject,
        all-or-nothing), explain() does NOT show a confident grant preview: it returns a
        1-column `error` frame (EXPLAIN_ERROR_COLUMNS), one row per blocking error, so the
        dry-run stays honest. A clean config's preview is unchanged. That SAME frame is also
        what a VALID config gets when its folder scopes cannot be resolved (below), so the
        frame's SHAPE alone does not mean 'this config would be rejected' -- the row text is
        the only discriminator between a config rejection and an unreachable target.

        The OneLake listing this preview now issues does not break the never-raise promise, and
        THAT much is kept by construction rather than by luck: the resolve step is swallowed, and the
        Catalog.canonical -> Generate.rows chain is guarded too, because Generate._scope_pair
        catches ValidationError ONLY and a live fs.ls fails with things that are not one (403,
        missing path, throttle) -- those become one more `error` row instead of a traceback out
        of a preview. When the target is unresolvable AND the config declares folder scopes,
        the `folders` seam is filled with _explain_unresolvable_target_lister, so the failure is
        reported as what it is -- no target to list against -- ALONGSIDE (not instead of) every
        other validation error the config carries.

        The promise is NOT total, and the one gap is PRE-EXISTING, not opened here: `dep.short_rows`
        is read ABOVE the try/except below (it decides the empty-config early return), so a
        missing or unreadable config table still raises straight out of explain(). That is
        unchanged from before the folder-listing change; the claim is narrowed
        here rather than the guard widened, because moving that read inside the try would change
        explain()'s behaviour, which this change deliberately does not do.

        Those two failure modes treat the collect-all aggregate DIFFERENTLY, and that asymmetry
        is a DECISION, not an accident of where each one happens to be caught:

          • UNRESOLVABLE TARGET -> the injected lister raises ValidationError, which
            Generate._scope_pair catches, so it joins the aggregate by the same collect-all
            path every other resolver error takes -- a config can therefore carry MORE THAN ONE
            such row -- and the config's other validation errors SURVIVE beside it.
          • ANYTHING ELSE ESCAPING Generate.rows (403 / missing path / throttle from the live
            listing, but also any non-ValidationError defect in the pure rule checks) -> the
            whole preview collapses to a SINGLE error row; the config's other validation errors
            are LOST. The guard is broad on purpose, which is exactly why its message does not
            assert a live-service cause it cannot know.

        The difference is one of CONTROL FLOW, not of how much of the catalog is readable.
        Catalog.canonical builds `tables` EAGERLY from spark.sql and merely STORES the
        folders callable and a lazy per-table column view (Catalog.LazyColumnMap) -- neither
        folders nor columns are listed until Generate.rows first asks --
        so a failing listing leaves the catalog exactly as readable as an unresolvable target
        does. What differs is the exception TYPE, and so whether the row loop survives:
        _scope_pair catches ValidationError and nothing else, so the injected lister's
        ValidationError is recorded and every remaining config row is still validated, while a
        403 unwinds Generate.rows and the rows after it are never validated at all. Those
        errors were never computed, so they cannot be shown. The same guard also wraps the pure
        rule checks, and a non-ValidationError escaping THOSE means the checks themselves
        misbehaved -- an aggregate that looks complete but is not is worse than one honest
        "could not resolve the config" row. Pinned by the two tests named for each mode."""
        spark = OLAF._spark()
        params = OLAF._params()
        dep = Deployment(
            spark,
            None,
            None,
            "",
            params.get("config_table", DEFAULT_CONTROL_TABLES["config_table"]),
            params.get("mapping_table", DEFAULT_CONTROL_TABLES["mapping_table"]),
            # explain() never writes, but __init__'s path guard vets every construction —
            # hand it the real (default) folder rather than an empty string it would refuse.
            params.get("mapping_history_dir", PARAM_DEFAULTS["mapping_history_dir"]),
            member_table=params.get("member_table", DEFAULT_CONTROL_TABLES["member_table"]),
        )
        if not dep.short_rows:
            return OLAF._announce(
                "explain",
                "0 grant(s) previewed · no active config rows",
                OLAF._frame([], columns=EXPLAIN_COLUMNS),
            )
        try:
            # SAME shape as OLAF._build_client (the in-repo precedent): Target.resolve raises
            # SystemExit (no/cross-workspace lakehouse) or ImportError (off-Fabric), and
            # swallowing both keeps explain()'s never-raise promise on the RESOLVE step.
            workspace_id, item_id = Target.resolve()
        except (Exception, SystemExit):
            workspace_id, item_id = None, None
        try:
            # ...but that try/except guards the resolve step ONLY. Everything the resolved ids
            # then REACH -- real notebookutils.fs.ls listings -- is guarded here:
            # _scope_pair catches ValidationError only, so a 403/missing-path/throttle would
            # otherwise escape a method documented never to raise. Mirrors the sibling seam
            # Catalog._export_lister, which likewise guards its own listing (except OSError).
            canon = Catalog.canonical(
                spark,
                list_folders=(
                    None if workspace_id and item_id else _explain_unresolvable_target_lister
                ),
                workspace_id=workspace_id,
                item_id=item_id,
            )
            # THE SAME pipeline validate() and generate() run — not a re-implementation of a
            # subset of it. Every check they make, explain makes: config rules, the No-Graph member
            # gate, and the lakehouse target guard. All three are READS (the config table, the
            # catalog, the member table, the runtime context); none writes, and none calls the DAR
            # API, so running the full set costs explain nothing it was protecting.
            grants, all_errors, _warnings, _summary, _lh, errors = dep._run_validation(canon)
        except (Exception, SystemExit) as exc:
            return OLAF._announce(
                "explain",
                "preview unavailable · the config could not be resolved",
                OLAF._frame(
                    [
                        {
                            "error": "preview could not resolve the config — this step both "
                            "reads the live catalog/OneLake and runs generate's own pure rule "
                            "checks, so the cause may be either a live-service failure or a "
                            f"defect in the framework itself; read the message: {exc}"
                        }
                    ],
                    columns=EXPLAIN_ERROR_COLUMNS,
                ),
                badge="⚠️ ",
            )

        if errors:
            # generate() is all-or-nothing: ANY config-validation error hard-rejects the whole
            # config, so a confident grant preview would be dishonest. Surface the blocking
            # errors (one row each) in a 1-column `error` frame instead -- honest that the
            # config would NOT generate, still a Spark DataFrame, still never raising.
            return OLAF._announce(
                "explain",
                f"{len(all_errors)} blocking error(s) · this config would NOT generate",
                OLAF._frame(
                    [{"error": message} for message in all_errors],
                    columns=EXPLAIN_ERROR_COLUMNS,
                ),
                badge="⚠️ ",
            )
        # Config rules are clean, so the projection below is honest. Anything left in all_errors is
        # about the ENVIRONMENT -- a member the cache does not carry, a lakehouse that is not the
        # attached one -- which generate would still refuse, and which the config author cannot fix
        # by editing the config. Report it loudly, and still show what the config produces.
        for message in all_errors:
            Say.out("quiet", "   ❌ ", message)
        rows = [
            {
                "role_name": a["role_name"],
                "scope_path": a["scope_path"],
                "permission": a["permission"],
                "rls_condition": a["rls_condition"],
                "visible_columns": a["visible_columns"],
                "members": LIST_SEP.join(
                    name
                    for name_col, _id_col, _mtype in MAPPING_MEMBER_COLUMNS
                    for name in Parse.list(a.get(name_col))
                )
                or None,
            }
            for a in grants
        ]
        return OLAF._announce(
            "explain",
            f"{len(rows)} grant(s) previewed · nothing written"
            + (f" · {len(all_errors)} problem(s) above" if all_errors else "")
            + " · lakehouse target unchecked (needs a live client) — validate() covers it",
            OLAF._frame(rows, columns=EXPLAIN_COLUMNS),
        )

    @staticmethod
    def show(by: str, subject: str = "") -> "DataFrame":
        """Pivot the live DAR by table|role|member, enriched with log provenance. → grant table."""
        return OLAF._returned(OLAF._show_view(OLAF._run("show", by=by, subject=subject)))

    @staticmethod
    def trace() -> "DataFrame":
        """The operational snapshot (deployed generation, counts, staleness). → 1-row frame."""
        return OLAF._returned(OLAF._trace_view(OLAF._run("trace")))

## ▶️ Run

One self-contained runtime. The dispatch cell below runs whatever `mode` is passed — the
notebook-level least-privilege gate is dropped (`allowed` is every known mode); configure and
independently review authorization for your Fabric environment. `run_mode`'s internal `KNOWN_MODES`
guard still rejects an unknown mode; success/skipped →
`notebook.exit` (the full envelope), blocked/error → raise (a compact envelope) per the
native-failure contract.

In [ ]:
# ══ Run ══════════════════════════════════════════════════════════════════════
# Dispatch every mode from this one notebook. `allowed` is the full KNOWN_MODES set
# (no notebook-level least-privilege); run_mode still rejects a mode outside KNOWN_MODES.
# GUARD: dispatch ONLY when `mode` is set. An empty `mode` (the parameters-cell
# default) makes this notebook a pure library — `%run olaf` loads the OLAF facade + the
# classes without running anything and, crucially, WITHOUT calling notebook.exit (which would stop a
# parent that %run-ed this as a library) — and without RAISING, which would stop that parent just
# the same. Keep the empty-`mode` branch free of both. A pipeline / notebook.run sets `mode` -> real dispatch.
if mode:
    run_and_exit(
        mode,
        allowed={"setup", "generate", "validate", "plan", "apply", "rollback", "show", "trace"},
        params={
            "rebuild": rebuild,
            "keep_unmanaged": keep_unmanaged,
            "if_match": if_match,
            "control_data_isolation_attestation": control_data_isolation_attestation,
            "tenant_id": tenant_id,
            "lakehouse_name": lakehouse_name,
            "config_table": config_table,
            "mapping_table": mapping_table,
            "log_table": log_table,
            "member_table": member_table,
            "mapping_history_dir": mapping_history_dir,
            "role_backup_dir": role_backup_dir,
            "verbosity": verbosity,
            "env": env,
            "batch_id": batch_id,
            "by": by,
            "subject": subject,
            "rollback_to_version": rollback_to_version,
            "rollback_reason": rollback_reason,
        },
        spark=spark,
    )

else:
    Say.out(
        "info",
        f"\n🦉 OLAF {__version__} loaded — no mode set, so nothing ran."
        "\n   Interactive: OLAF.<action>() — setup · generate · validate · plan · apply · rollback · show · trace"
        "\n   Pipeline:    pass the `mode` parameter (notebook.run / Base parameters)",
    )